# TearSheet.ipynb

- doesn't actually generate a tearsheet like a classic print [S&P or Value Line tearsheet](https://financetrain.com/what-is-a-tear-sheet) on a company
- asks Perplexity a few basic questions about the company
- gets info from AlphaVantage
- opens a bunch of browser tabs from major services about the company
- could potentially save those and scrape them to make a tearsheet.


Updated version
- profile - get info from perplexity , other company profiles, openbb, sec filings, wikipedia generate a description of the company, focus on business model history of the company
- news
  - get latest news
  - scrape news using scraper - grab headlines matching pattern
  - filter stuff by how relevant to the company
- chart
- key ratios, top holders
- generate tear sheet using tools
- question answering chatbot using tools


In [1]:
import dotenv
import sys
import os
import re
from datetime import datetime, timedelta
import time
from typing import List, Optional, Dict, Any
from urllib.parse import urljoin, urlparse
from pathlib import Path

import numpy as np
# should require numpy < 2 for pandas_ta or uses more recent ta-lib module
np.NaN = np.nan

import openbb
from openbb import obb
from openbb_core.app.model.obbject import OBBject

import pandas as pd
import pandas_ta as ta

import requests
from bs4 import BeautifulSoup
import html2text

import json
import aiohttp
import tempfile

import openai
from openai import OpenAI

import IPython
from IPython.display import HTML, Image, Markdown, display

from selenium import webdriver
from selenium.webdriver.common.by import By
# use firefox because it updates less often, can disable updates
# recommend importing profile from Chrome for cookies, passwords
# looks less like a bot with more user cruft in the profile
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.firefox.service import Service

import wikipedia

import langchain
from langchain_openai import ChatOpenAI
from langchain.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.prompts import ChatPromptTemplate
from langchain.schema import HumanMessage

from newsapi import NewsApiClient
# %pip install mem0ai
import mem0
from mem0 import MemoryClient

# brew install ta-lib
# https://ta-lib.org/

In [2]:
dotenv.load_dotenv()


True

In [102]:
# Get current year
current_year = datetime.now().year
print(f"Current year: {current_year}")

last_year = datetime.now().year - 1
print(f"Last year: {last_year}")

# Get date 1 year ago (approximate - 365 days)
one_year_ago = datetime.now() - timedelta(days=365)
print(f"Date 1 year ago: {one_year_ago.strftime('%Y-%m-%d')}")

date_today = datetime.now() 
print(f"Date now: {date_today.strftime('%Y-%m-%d')}")


Current year: 2025
Last year: 2024
Date 1 year ago: 2024-08-02
Date now: 2025-08-02


In [3]:
company = "Tesla"
symbol = "TSLA"



In [4]:
temp_dir = tempfile.mkdtemp(prefix='t', dir='tmp')
temp_dir


'tmp/tx0ybr2zo'

In [18]:
temp_dir = 'tx0ybr2zo'

In [5]:
def replace_citations(text, citations):
    def replace_match(match):
        citation_num = int(match.group(1))
        if citation_num <= len(citations):
            url = citations[citation_num - 1]  # Citations are 0-indexed, references are 1-indexed
            return f" [{citation_num}]({url})"
        return match.group(0)  # Return original if citation not found

    return re.sub(r'\[(\d+)\]', replace_match, text)

def fetch_perplexity(system_prompt, user_prompt, return_images=False):
    perplexity_url = "https://api.perplexity.ai/chat/completions"
    perplexity_model = "sonar-pro"

    payload = {
        "model": perplexity_model,
        "messages": [
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            }
        ],
    "return_citations": True,          # Enable citations for markdown links
    "return_images": return_images,    # Optional: disable images if not needed
    "return_related_questions": False  # Optional: disable related questions
}

    perplexity_headers = {
        "Authorization": f"Bearer {os.getenv('PERPLEXITY_API_KEY')}",
        "accept": "application/json",
        "content-type": "application/json"
    }

    response = requests.post(perplexity_url, json=payload, headers=perplexity_headers)
    response_data = response.json()

    response_str = response_data['choices'][0]['message']['content']
    response_str = response_str.replace("$", r"\$")
    citations = response_data.get('citations', [])
    response_str = replace_citations(response_str, citations)
    if return_images:
        images = response_data.get('images', [])
        if images:
            image_str = f"![Image 1]({images.pop(0)['image_url']})\n\n"
            response_str = image_str + response_str
    return response_str


In [6]:
def get_perplexity_profile(company, symbol, return_images=False):

    system_prompt = """
You will act as a securities analyst and investment advisor with deep knowledge of financial markets,
securities analysis, portfolio management. You will maintain a professional yet engaging tone,
in the style of a senior investment bank research analyst.
"""

    user_prompt = f"""You will focus on {company} ({symbol}), and provide a comprehensive analysis covering the following aspects:

Company Profile: An overview of {company}, including its lines of business, history, and recent key developments.

Major News: Significant events related to {company} or its industry impacting its stock.

Financial Performance: Recent earnings reports and stock performance compared to expectations, changes in dividends or stock buybacks.

Analyst Coverage: summarize recent changes to analysts' ratings noting which analyst and firms made upgrades or downgrades; summarize any recent short seller reports noting the firm and analyst.

Product Announcements: Launch of new products, strategic initiatives, or restructurings.

Strategic Moves: Information on deals, partnerships, mergers, acquisitions, divestitures, joint ventures, and major new business and revenue.

Securities Offerings: Announcements related to stock or bond issuances, buybacks, special dividends, or stock splits.

Management Changes: Significant personnel changes within {company}.

Stock Price Movements: Notable stock price changes and their reasons.

Timeline and Future Outlook: A timeline of these events """

    return fetch_perplexity(system_prompt, user_prompt, return_images=return_images)


In [7]:
perplexity_str = get_perplexity_profilee(company, symbol, return_images=True)
with open(f'{temp_dir}/{symbol}_perplexity_profile.md', 'w', encoding='utf-8') as f:
    f.write(perplexity_str)

display(Markdown(perplexity_str))


![Image 1](https://www.teslarati.com/wp-content/uploads/2025/03/tesla-logo-new-model-y-scaled.jpg)

Tesla, Inc. (TSLA) is a leading electric vehicle manufacturer and clean energy company with diversified lines of business including electric vehicles (EVs), energy generation and storage systems, and autonomous driving software [4](https://www.morningstar.com/stocks/xnas/tsla/quote). Below is a comprehensive analysis as of August 2025:

**Company Profile**  
Tesla is a vertically integrated technology and manufacturing firm focused on accelerating the world's transition to sustainable energy [1](https://ir.tesla.com) [4](https://www.morningstar.com/stocks/xnas/tsla/quote). Its core business units include:
- **Automotive Sales:** Manufacture and sale of electric vehicles, now spanning sedans (Model S, 3), SUVs (Model X, Y), pickup (Cybertruck), and high-performance (Roadster).
- **Energy Generation & Storage:** Solar panels, solar roof tiles, and large-scale energy storage (Powerwall, Powerpack, Megapack) [2](https://www.sec.gov/Archives/edgar/data/1318605/000162828025018911/tsla-20250331.htm) [4](https://www.morningstar.com/stocks/xnas/tsla/quote).
- **Autonomous Software & Services:** Full Self Driving (FSD) software platform development; mobility services leveraging autonomous vehicles; newly enhanced after Q2 2025 [3](https://www.tesla.com/sites/default/files/downloads/TSLA-Q2-2025-Update.pdf) [4](https://www.morningstar.com/stocks/xnas/tsla/quote).  
Founded in 2003, Tesla rapidly scaled from its first vehicle launch (Roadster, 2008) to mass-market production and global expansion, also influencing energy and AI-driven mobility.

**Major News**  
- **Q2 2025 Product Launch:** Tesla introduced a new ride-hailing feature, allowing customers to summon vehicles and monitor rides using a Tesla app profile [3](https://www.tesla.com/sites/default/files/downloads/TSLA-Q2-2025-Update.pdf).
- **AI and Autonomous Expansion:** Enhanced software for autonomy and expanded real-world mobility services debuted in Q2 2025 [3](https://www.tesla.com/sites/default/files/downloads/TSLA-Q2-2025-Update.pdf).
- **Industry Trends:** The electric vehicle sector faces rising competition, supply chain volatility, and macroeconomic headwinds impacting demand and production volumes [1](https://ir.tesla.com).

**Financial Performance**
- **Recent Earnings (Q2 2025):** Tesla released its Q2 2025 shareholder update and webcast on July 23, 2025. Detailed financials such as revenue, gross margins, and profitability for the quarter can be found in the Q2 2025 Update, with key highlights:
  - Continued revenue growth in EV and energy segments [3](https://www.tesla.com/sites/default/files/downloads/TSLA-Q2-2025-Update.pdf).
  - Profitability trends reflect margin pressure from price reductions and ramp of new vehicle platforms [3](https://www.tesla.com/sites/default/files/downloads/TSLA-Q2-2025-Update.pdf).
- **Shares Outstanding:** 3.22 billion as of April 16, 2025 [2](https://www.sec.gov/Archives/edgar/data/1318605/000162828025018911/tsla-20250331.htm).
- **Stock Performance:** Real-time and historical quotes show TSLA remains highly volatile, with notable trends tracking sector indices such as the S&P 500 and Nasdaq [4](https://www.morningstar.com/stocks/xnas/tsla/quote).
- **Dividends/Buybacks:** No mention of dividends or significant share buybacks in official filings or recent updates [2](https://www.sec.gov/Archives/edgar/data/1318605/000162828025018911/tsla-20250331.htm) [3](https://www.tesla.com/sites/default/files/downloads/TSLA-Q2-2025-Update.pdf).

**Analyst Coverage**
- Detailed recent analyst upgrades/downgrades or short seller reports are not present in these results. No new large-scale changes by major sell-side firms or noted activist short campaigns are included in the provided sources. Such events would typically be highlighted in financial press or broker research, but are not seen in the official investor materials or Morningstar's company profile [4](https://www.morningstar.com/stocks/xnas/tsla/quote).

**Product Announcements**
- **Mobility Platform Feature:** In Q2 2025, Tesla enhanced its ride-hailing and vehicle management experience, with customers now able to request, track, and control vehicles from their personal profile—an expansion of its autonomous and software ecosystem [3](https://www.tesla.com/sites/default/files/downloads/TSLA-Q2-2025-Update.pdf).
- **Software Updates:** Ongoing progression of Full Self Driving (FSD) features and real-world AI applications [3](https://www.tesla.com/sites/default/files/downloads/TSLA-Q2-2025-Update.pdf).
- No major vehicle platform launches (“Model 2”, etc.) or discontinuations are cited in Q2 2025.

**Strategic Moves**
- No major deals, partnerships, or divestitures are recorded in these search results for 2025 year-to-date. Tesla’s strategic focus remains on organic product and service expansion [3](https://www.tesla.com/sites/default/files/downloads/TSLA-Q2-2025-Update.pdf) [4](https://www.morningstar.com/stocks/xnas/tsla/quote).

**Securities Offerings**
- **Shares Outstanding:** Remained stable at 3.22 billion as of April 2025; filings show no major new equity or debt offerings, buybacks, or splits through Q2 2025 [2](https://www.sec.gov/Archives/edgar/data/1318605/000162828025018911/tsla-20250331.htm).
- No announcements of special dividends or other capital distributions as of the latest filings [2](https://www.sec.gov/Archives/edgar/data/1318605/000162828025018911/tsla-20250331.htm) [3](https://www.tesla.com/sites/default/files/downloads/TSLA-Q2-2025-Update.pdf).

**Management Changes**
- The search results do not indicate any recent major executive or board changes in 2025. No high-profile departures or appointments have been reported in Q2 2025 company documents [1](https://ir.tesla.com) [3](https://www.tesla.com/sites/default/files/downloads/TSLA-Q2-2025-Update.pdf).

**Stock Price Movements**
- TSLA shares demonstrated continued volatility, influenced by sector trends, earnings reports, and investor sentiment around new product announcements and AI autonomy advancements [4](https://www.morningstar.com/stocks/xnas/tsla/quote).
- No specific single-day or week moves are noted, but the overall sentiment ties closely to broader market and EV sector performance [4](https://www.morningstar.com/stocks/xnas/tsla/quote).

**Timeline and Future Outlook (2024–2025)**
- **2024:** Continued expansion in energy and storage, ramp of Cybertruck and global Model Y sales.
- **Q1–Q2 2025:** Focus on software and ride-hailing platform development; steady financial performance amid increased competitive pressures; no significant capital structure changes [1](https://ir.tesla.com) [2](https://www.sec.gov/Archives/edgar/data/1318605/000162828025018911/tsla-20250331.htm) [3](https://www.tesla.com/sites/default/files/downloads/TSLA-Q2-2025-Update.pdf).
- **Forward-Looking:** Tesla’s product roadmap and growth remain tightly linked to autonomous technology rollout, scaling energy products, and competitive positioning against global EV entrants [1](https://ir.tesla.com) [3](https://www.tesla.com/sites/default/files/downloads/TSLA-Q2-2025-Update.pdf) [4](https://www.morningstar.com/stocks/xnas/tsla/quote).

In summary, Tesla in 2025 is characterized by robust product innovation (particularly in autonomy/mobility), stable share structure, ongoing financial growth with margin pressure, and a lack of major management or capital events based on available official disclosures [1](https://ir.tesla.com) [2](https://www.sec.gov/Archives/edgar/data/1318605/000162828025018911/tsla-20250331.htm) [3](https://www.tesla.com/sites/default/files/downloads/TSLA-Q2-2025-Update.pdf) [4](https://www.morningstar.com/stocks/xnas/tsla/quote).

In [8]:
def get_perplexity_analyst_ratings(company, symbol, return_images=False):

    system_prompt = """
You will act as a securities analyst and investment advisor with deep knowledge of financial markets,
securities analysis, portfolio management. You will maintain a professional yet engaging tone,
in the style of a senior investment bank research analyst.
"""

    user_prompt = f"""What are the current analyst ratings on {company} ({symbol})?
    Which short sellers issued reports on {company} since {last_year}, if any? Summarize any notable analyst reports."""

    return fetch_perplexity(system_prompt, user_prompt, return_images=False)

perplexity_str = get_perplexity_analyst_ratings(company, symbol)
with open(f'{temp_dir}/{symbol}_perplexity_ratings.md', 'w', encoding='utf-8') as f:
    f.write(perplexity_str)
display(Markdown(perplexity_str))


The current analyst ratings for **Tesla (TSLA)** reflect a divided outlook, with a consensus price target around **\$311.81** based on 29 analysts, and a split between bullish and bearish recommendations [1](https://www.benzinga.com/quote/TSLA/analyst-ratings) [5](https://www.quiverquant.com/news/Tesla,+Inc.+Stock+(TSLA)+Opinions+on+2025+Vehicle+Launch+Plans). Notably, short-biased research shops such as GLJ Research and Guggenheim have issued “Sell” ratings in the past month [5](https://www.quiverquant.com/news/Tesla,+Inc.+Stock+(TSLA)+Opinions+on+2025+Vehicle+Launch+Plans).

**Current Analyst Ratings on TSLA:**
- Consensus price target: **\$311.81**
    - Highest target: **\$500** (Wedbush, 07/24/2025)
    - Most recent price: **\$306.89** [1](https://www.benzinga.com/quote/TSLA/analyst-ratings)
- 11 firms rate TSLA a **buy**, while 6 rate it **sell** [5](https://www.quiverquant.com/news/Tesla,+Inc.+Stock+(TSLA)+Opinions+on+2025+Vehicle+Launch+Plans)
- Recent notable ratings include:
    - **Wedbush:** "Outperform" with a \$500 target (07/24/2025) [1](https://www.benzinga.com/quote/TSLA/analyst-ratings) [5](https://www.quiverquant.com/news/Tesla,+Inc.+Stock+(TSLA)+Opinions+on+2025+Vehicle+Launch+Plans)
    - **Cantor Fitzgerald:** “Overweight” with a \$425 target (07/24/2025) [1](https://www.benzinga.com/quote/TSLA/analyst-ratings) [5](https://www.quiverquant.com/news/Tesla,+Inc.+Stock+(TSLA)+Opinions+on+2025+Vehicle+Launch+Plans) and a separate \$355 target set earlier in the month [3](https://site.financialmodelingprep.com/market-news/tesla-inc-tsla-analyst-ratings-future-outlook-q2-2025-insights)
    - **Guggenheim:** “Sell” with a \$175 target (07/23/2025) [5](https://www.quiverquant.com/news/Tesla,+Inc.+Stock+(TSLA)+Opinions+on+2025+Vehicle+Launch+Plans)
    - **Bank of America:** \$341 price target (07/23/2025) [5](https://www.quiverquant.com/news/Tesla,+Inc.+Stock+(TSLA)+Opinions+on+2025+Vehicle+Launch+Plans)
    - **GLJ Research:** “Sell” (07/24/2025) [5](https://www.quiverquant.com/news/Tesla,+Inc.+Stock+(TSLA)+Opinions+on+2025+Vehicle+Launch+Plans)
    - **HSBC:** “Reduce” (07/03/2025) [5](https://www.quiverquant.com/news/Tesla,+Inc.+Stock+(TSLA)+Opinions+on+2025+Vehicle+Launch+Plans)
    - **Canaccord Genuity, Benchmark:** Both recent “Buy” ratings [5](https://www.quiverquant.com/news/Tesla,+Inc.+Stock+(TSLA)+Opinions+on+2025+Vehicle+Launch+Plans)

**Short Seller Reports:**
- The search does not reference new formal “short seller reports” like a Hindenburg or Spruce Point–style investigation in the past quarter. However, **GLJ Research** and **Guggenheim** have reiterated “Sell” ratings, with GLJ Research traditionally known for a sceptical stance toward Tesla [5](https://www.quiverquant.com/news/Tesla,+Inc.+Stock+(TSLA)+Opinions+on+2025+Vehicle+Launch+Plans).
- Guggenheim’s \$175 price target is among the most bearish from major firms, reflecting stark concerns relative to consensus [5](https://www.quiverquant.com/news/Tesla,+Inc.+Stock+(TSLA)+Opinions+on+2025+Vehicle+Launch+Plans).

**Summary of Notable Analyst Reports:**
- **Bullish reports** (e.g., Wedbush, Cantor Fitzgerald, Deutsche Bank) are optimistic about new product lines, especially the Model Q launch, and continued expansion into sectors like robotaxis [3](https://site.financialmodelingprep.com/market-news/tesla-inc-tsla-analyst-ratings-future-outlook-q2-2025-insights) [5](https://www.quiverquant.com/news/Tesla,+Inc.+Stock+(TSLA)+Opinions+on+2025+Vehicle+Launch+Plans). Wedbush significantly raised its target to \$500, citing confidence in Tesla’s ability to execute on next-generation vehicles and AI verticals [1](https://www.benzinga.com/quote/TSLA/analyst-ratings) [5](https://www.quiverquant.com/news/Tesla,+Inc.+Stock+(TSLA)+Opinions+on+2025+Vehicle+Launch+Plans).
- **Bearish reports** (Guggenheim, GLJ Research, HSBC) emphasize concerns about competition, margin pressures, and weaker demand signals, as reflected in lowered ratings and targets [5](https://www.quiverquant.com/news/Tesla,+Inc.+Stock+(TSLA)+Opinions+on+2025+Vehicle+Launch+Plans).
- Between July 21-29, several analyst ratings were updated, with most new price targets ranging between \$319 and \$425 [1](https://www.benzinga.com/quote/TSLA/analyst-ratings) [3](https://site.financialmodelingprep.com/market-news/tesla-inc-tsla-analyst-ratings-future-outlook-q2-2025-insights) [5](https://www.quiverquant.com/news/Tesla,+Inc.+Stock+(TSLA)+Opinions+on+2025+Vehicle+Launch+Plans).

**Essential context:** Tesla remains a highly polarized stock among analysts, as evidenced by the breadth between bullish (\$500) and bearish (\$175) targets. The split reflects differing views on execution risk, margin sustainability, and the validity of Tesla's long-term growth narrative [1](https://www.benzinga.com/quote/TSLA/analyst-ratings) [5](https://www.quiverquant.com/news/Tesla,+Inc.+Stock+(TSLA)+Opinions+on+2025+Vehicle+Launch+Plans).

In [9]:
def get_perplexity_news(company, symbol, return_images=False):

    system_prompt = """
You will act as a securities analyst and investment advisor with deep knowledge of financial markets,
securities analysis, portfolio management. You will maintain a professional yet engaging tone,
in the style of a senior investment bank research analyst.
"""

    user_prompt = f"""What are there most impactful news stories about {company} ({symbol}) since {last_year}, including management profiles and investigative reports?
For any notable stories, provide the date of publication and the media that reported them."""

    return fetch_perplexity(system_prompt, user_prompt, return_images=False)

perplexity_str = get_perplexity_news(company, symbol)
with open(f'{temp_dir}/{symbol}_perplexity_news.md', 'w', encoding='utf-8') as f:
    f.write(perplexity_str)
display(Markdown(perplexity_str))


Over the past year, the most impactful news stories about Tesla (TSLA) have centered on its management trajectory, earnings volatility, strategic pivots in product and automation, and ongoing controversy around CEO Elon Musk's leadership and political involvement. Below are the most notable stories, with dates and media attribution:

**Q2 2025: Tesla’s Inflection Point and Management Scrutiny**
- **Date:** July 23, 2025
- **Media:** Business Insider, Tesla Investor Relations
- **Story:** Tesla reported a pivotal Q2 2025, marking the “beginning of our transition from leading the electric vehicle and renewable energy sector...” The period was described as a "seminal point in Tesla's history" [1](https://www.tesla.com/sites/default/files/downloads/TSLA-Q2-2025-Update.pdf) [3](https://ir.tesla.com/press). On the same day, CEO Elon Musk warned the company could face "a few rough quarters," citing product pipeline developments like robotaxis and Optimus, and introducing a more affordable vehicle [2](https://www.businessinsider.com/tesla-tsla-stock-q2-earnings-report-call-2025-7-2025-7). Musk's comments on strategic uncertainty led to mixed analyst reactions, with Wedbush Securities labeling the moment as a "positive crossroads," reiterating an "Outperform" rating and a \$500 price target, while Bank of America took a more cautious "Neutral" stance, citing tariff and supply chain concerns [2](https://www.businessinsider.com/tesla-tsla-stock-q2-earnings-report-call-2025-7-2025-7).

**Elon Musk’s Political Activities and Board Concerns**
- **Date:** Coverage throughout Q2–Q3 2025, most recently July 23, 2025
- **Media:** Business Insider, Wedbush Securities
- **Story:** Persistent controversy surrounded Musk's direct involvement in U.S. politics, including discussions about forming a new political party. Analysts at Wedbush previously called his political ambitions a "Soap Opera" and urged Tesla's board to construct clearer boundaries, including a new pay package and guidelines for Musk's political engagement [2](https://www.businessinsider.com/tesla-tsla-stock-q2-earnings-report-call-2025-7-2025-7).

**Product Strategy Shifts and Robotaxi Announcements**
- **Date:** July 2025
- **Media:** Wall Street Analyst Commentary (YouTube), Business Insider
- **Story:** Tesla surprised many by pivoting focus toward a stripped-down Model Y and suggesting future models may not represent entirely new form factors, but would heavily feature self-driving technology. This strategic ambiguity sparked debate among analysts and investors, as well as bullish and neutral ratings from Piper Sandler and Barclays, respectively [4](https://www.youtube.com/watch?v=vO9lDh5U4Kk). Musk’s promise of advances in automated driving remains a core pillar of Tesla’s investment story [2](https://www.businessinsider.com/tesla-tsla-stock-q2-earnings-report-call-2025-7-2025-7) [4](https://www.youtube.com/watch?v=vO9lDh5U4Kk).

**Ongoing Labor and Customer Relations Controversies**
- **Date:** October 27, 2024
- **Media:** Tesla company blog
- **Story:** Tesla publicly refuted reports alleging mass employee terminations tied to unionization efforts and criticized external media for what it deemed "vaguely and nonsensically" negative coverage regarding customer satisfaction [5](https://www.tesla.com/blog). This pushback is part of an ongoing pattern of the company addressing labor relations issues and media scrutiny.

**Summary of Management Profiles & Investigative Coverage**
- Analyst notes and institutional commentary have increasingly focused on CEO Elon Musk’s dominant leadership style, political activities, and their governance implications for Tesla [2](https://www.businessinsider.com/tesla-tsla-stock-q2-earnings-report-call-2025-7-2025-7) [4](https://www.youtube.com/watch?v=vO9lDh5U4Kk).
- There were no direct mentions of major third-party investigative exposés targeting management in the supplied search results, though board oversight, executive pay, and public messaging remain under frequent critical scrutiny by analysts and business media [2](https://www.businessinsider.com/tesla-tsla-stock-q2-earnings-report-call-2025-7-2025-7) [4](https://www.youtube.com/watch?v=vO9lDh5U4Kk).

**Key Media Outlets Involved**
- **Business Insider** (earnings, management controversies, political coverage)
- **Tesla IR** (official earnings, corporate updates)
- **Wall Street Analyst Commentary (YouTube, July 28, 2025)** (institutional sentiment shift, product focus)
- **Tesla Blog** (company rebuttal to labor and customer service criticism)

The dominant themes in the past year are the uncertain impact of Elon Musk’s expanded public persona on Tesla’s strategic direction, an earnings and product transition period drawing close scrutiny from both bulls and skeptics, and ongoing reputational challenges relating to workforce and customer issues.

In [14]:
company_list = wikipedia.search(company)
company_list


['Nikola Tesla',
 'Tesla, Inc.',
 'Tesla',
 'Tesla Cybertruck',
 'Tesla Model 3',
 'Tesla Model Y',
 'Tesla Roadster (first generation)',
 'Tesla Model X',
 'Tesla Autopilot',
 'Tesla Model S']

In [15]:
def get_best_wikipedia_match(company_description: str, wikipedia_matches: List[str]) -> str:
    """
    Use LangChain with ChatOpenAI to find the best Wikipedia page match for a company.
    
    Args:
        company_description (str): Description of the company (e.g., "Tesla (symbol TSLA)")
        wikipedia_matches (List[str]): List of Wikipedia page titles
    
    Returns:
        str: The best matching Wikipedia page title
    """
    
    # Initialize the Chat LLM
    llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
    
    # Create the chat prompt template
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant that selects the most appropriate Wikipedia page for a given company from a list of options."),
        ("human", """I am looking for the wikipedia page of {company_description}.

From the list of wikipedia articles below, tell me the title of the page that is most likely the best matching wikipedia page for {company_description}.

Wikipedia articles:
{wikipedia_matches}

Return exactly one page title from this list without any modification. Do not add quotes, explanations, or any other text - just return the exact title as it appears in the list.

Requirements:
- Must be exactly one of the titles from the provided list
- No modifications, quotes, or additional text
- Focus on the main company page, not specific products or history pages

Best match:""")
    ])
    
    # Format the matches as a bulleted list
    formatted_matches = "\n".join([f"• {match}" for match in wikipedia_matches])
    
    # Create and invoke the chain
    chain = prompt | llm
    
    try:
        result = chain.invoke({
            "company_description": company_description,
            "wikipedia_matches": formatted_matches
        })
        
        # Extract the content from the AIMessage
        selected_match = result.content.strip()
        
        # Validate that the result is actually in the original list
        if selected_match in wikipedia_matches:
            return selected_match
        else:
            # Fallback: try to find a close match or return the first corporate-looking entry
            for match in wikipedia_matches:
                if "Inc." in match or "Corp." in match or (company_description.split()[0].lower() in match.lower() and len(match.split()) <= 3):
                    return match
            return wikipedia_matches[0]  # Last resort
            
    except Exception as e:
        print(f"Error occurred: {e}")
        # Simple fallback logic
        for match in wikipedia_matches:
            if "Inc." in match:
                return match
        return wikipedia_matches[0]
    
company_query = f"{company} (symbol {symbol})"
best_match = get_best_wikipedia_match(company_query, company_list)

print(f"Best Wikipedia match for '{company_query}': {best_match}")




Best Wikipedia match for 'Tesla (symbol TSLA)': Tesla, Inc.


In [25]:
page_object = wikipedia.page(title=best_match, auto_suggest=False)

# printing title
print(page_object.original_title)

# printing links on that page object
#print(page_object.links[0:10])

# printing html of page_object
h = html2text.HTML2Text()
h.ignore_images = True
h.body_width = 0
h.unicode_snob = True
h.baseurl = 'https://en.wikipedia.org'
markdown_content = h.handle(page_object.html())
with open(f'{temp_dir}/{symbol}_wikipedia.md', 'w', encoding='utf-8') as f:
    f.write(perplexity_str)
display(Markdown(markdown_content))



Tesla, Inc.


American electric vehicle and clean energy company

"Tesla Motors" redirects here. For electric motors invented by Nikola Tesla, see [Induction motor](https://en.wikipedia.org/wiki/Induction_motor "Induction motor") and [AC motor](https://en.wikipedia.org/wiki/AC_motor "AC motor").

"Tesla (company)" redirects here. For the electrotechnical company, see [Tesla a.s.](https://en.wikipedia.org/wiki/Tesla_a.s. "Tesla a.s.") For other uses, see [Tesla (disambiguation)](https://en.wikipedia.org/wiki/Tesla_\(disambiguation\) "Tesla \(disambiguation\)").

Tesla, Inc.[](https://en.wikipedia.org/wiki/File:Tesla_Motors.svg)  
---  
[](https://en.wikipedia.org/wiki/File:Gigafactory_Texas_Building_1_June_2022.jpg)[Gigafactory Texas](https://en.wikipedia.org/wiki/Gigafactory_Texas "Gigafactory Texas"), Tesla's [headquarters](https://en.wikipedia.org/wiki/Headquarters "Headquarters"), just outside of [Austin, Texas](https://en.wikipedia.org/wiki/Austin,_Texas "Austin, Texas")  
Formerly| Tesla Motors, Inc. (2003–2017)  
Company type| [Public](https://en.wikipedia.org/wiki/Public_company "Public company")  
[Traded as](https://en.wikipedia.org/wiki/Ticker_symbol "Ticker symbol")| 

  * [Nasdaq](https://en.wikipedia.org/wiki/Nasdaq "Nasdaq"): [TSLA](https://www.nasdaq.com/market-activity/stocks/tsla)
  * [Nasdaq-100](https://en.wikipedia.org/wiki/Nasdaq-100 "Nasdaq-100") component
  * [S&P 100](https://en.wikipedia.org/wiki/S%26P_100 "S&P 100") component
  * [S&P 500](https://en.wikipedia.org/wiki/S%26P_500 "S&P 500") component

  
[ISIN](https://en.wikipedia.org/wiki/International_Securities_Identification_Number "International Securities Identification Number")| [US88160R1014](https://isin.toolforge.org/?language=en&isin=US88160R1014)  
Industry| 

  * [Automotive](https://en.wikipedia.org/wiki/Automotive_industry "Automotive industry")
  * [Renewable energy](https://en.wikipedia.org/wiki/Renewable_energy_industry "Renewable energy industry")

  
Founded| July 1, 2003; 22 years ago (2003-07-01) in [San Carlos, California](https://en.wikipedia.org/wiki/San_Carlos,_California "San Carlos, California"), U.S.  
Founders| [Martin Eberhard](https://en.wikipedia.org/wiki/Martin_Eberhard "Martin Eberhard")  
[Marc Tarpenning](https://en.wikipedia.org/wiki/Marc_Tarpenning "Marc Tarpenning")  
(See § Founding)  
Headquarters| [Austin, Texas](https://en.wikipedia.org/wiki/Austin,_Texas "Austin, Texas"), U.S.  
Number of locations| 

  * 1,359 sales, service and delivery centers
  * 7,000 [Supercharger](https://en.wikipedia.org/wiki/Tesla_Supercharger "Tesla Supercharger") stations[1]

  
Area served| 

  * East Asia
  * Europe
  * Middle East
  * North America
  * Oceania
  * Southeast Asia
  * Indian Subcontinent

  
Key people| 

  * [Elon Musk](https://en.wikipedia.org/wiki/Elon_Musk "Elon Musk") ([CEO](https://en.wikipedia.org/wiki/Chief_executive_officer "Chief executive officer"))
  * [Robyn Denholm](https://en.wikipedia.org/wiki/Robyn_Denholm "Robyn Denholm") ([chair](https://en.wikipedia.org/wiki/Chair_\(officer\) "Chair \(officer\)"))

  
Products| 

  * [Cybertruck](https://en.wikipedia.org/wiki/Tesla_Cybertruck "Tesla Cybertruck")
  * [Megapack](https://en.wikipedia.org/wiki/Tesla_Megapack "Tesla Megapack")
  * [Model 3](https://en.wikipedia.org/wiki/Tesla_Model_3 "Tesla Model 3")
  * [Model S](https://en.wikipedia.org/wiki/Tesla_Model_S "Tesla Model S")
  * [Model X](https://en.wikipedia.org/wiki/Tesla_Model_X "Tesla Model X")
  * [Model Y](https://en.wikipedia.org/wiki/Tesla_Model_Y "Tesla Model Y")
  * [Powerwall](https://en.wikipedia.org/wiki/Tesla_Powerwall "Tesla Powerwall")
  * [Semi](https://en.wikipedia.org/wiki/Tesla_Semi "Tesla Semi")
  * [Solar Panels](https://en.wikipedia.org/wiki/Tesla_solar_panels "Tesla solar panels")
  * [Solar Roof](https://en.wikipedia.org/wiki/Tesla_Solar_Roof "Tesla Solar Roof")

  
Production output| 

  * 1,773,443 vehicles (2024)
  * 31.4 GWh battery energy storage systems (2024)

  
Services| 

  * [Charging](https://en.wikipedia.org/wiki/Tesla_Supercharger "Tesla Supercharger")
  * insurance
  * maintenance

  
Revenue|  [US$](https://en.wikipedia.org/wiki/United_States_dollar "United States dollar")97.7 billion (2024)  
[Operating income](https://en.wikipedia.org/wiki/Earnings_before_interest_and_taxes "Earnings before interest and taxes")|  US$7.1 billion (2024)  
[Net income](https://en.wikipedia.org/wiki/Net_income "Net income")|  US$7.1 billion (2024)  
[Total assets](https://en.wikipedia.org/wiki/Asset "Asset")|  US$122.1 billion (2024)  
[Total equity](https://en.wikipedia.org/wiki/Equity_\(finance\) "Equity \(finance\)")|  US$72.9 billion (2024)  
Owner| Elon Musk (13%)[2]  
Number of employees|  125,665 (2024)  
[Subsidiaries](https://en.wikipedia.org/wiki/Subsidiary "Subsidiary")| 

  * [Tesla Automation](https://en.wikipedia.org/wiki/Tesla_Automation "Tesla Automation")
  * [Tesla Energy](https://en.wikipedia.org/wiki/Tesla_Energy "Tesla Energy")

  
[ASN](https://en.wikipedia.org/wiki/Autonomous_System_Number "Autonomous System Number")| 

  * [394161](https://bgp.tools/as/394161)

  
Website| [tesla.com](https://www.tesla.com/)  
**Footnotes / references**  
Financials as of December 31, 2024[[update]](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit).  
References: [3]  
| [](https://en.wikipedia.org/wiki/File:Elon_Musk,_2018_\(cropped\).jpg) | This article is part of  
a series about[Elon Musk](https://en.wikipedia.org/wiki/Elon_Musk "Elon Musk")  
---|---  
  
* * *

Personal

  * [Awards and honors](https://en.wikipedia.org/wiki/List_of_awards_and_honors_received_by_Elon_Musk "List of awards and honors received by Elon Musk")
  * [Business career](https://en.wikipedia.org/wiki/Business_career_of_Elon_Musk "Business career of Elon Musk")
  * [Views](https://en.wikipedia.org/wiki/Views_of_Elon_Musk "Views of Elon Musk")
  * [Family](https://en.wikipedia.org/wiki/Musk_family "Musk family")
  * [Filmography](https://en.wikipedia.org/wiki/Elon_Musk_filmography "Elon Musk filmography")
  * [International relations](https://en.wikipedia.org/wiki/International_relations_of_Elon_Musk "International relations of Elon Musk")
  * [Legal affairs](https://en.wikipedia.org/wiki/Legal_affairs_of_Elon_Musk "Legal affairs of Elon Musk")
  * [Public image](https://en.wikipedia.org/wiki/Public_image_of_Elon_Musk "Public image of Elon Musk")
  * [Texas Institute of Technology and Science](https://en.wikipedia.org/wiki/Texas_Institute_of_Technology_and_Science "Texas Institute of Technology and Science")
  * [Wealth](https://en.wikipedia.org/wiki/Wealth_of_Elon_Musk "Wealth of Elon Musk")
    * [Foundation](https://en.wikipedia.org/wiki/Musk_Foundation "Musk Foundation")



* * *

Companies

  * [Zip2](https://en.wikipedia.org/wiki/Zip2 "Zip2")
  * [X.com](https://en.wikipedia.org/wiki/X.com_\(bank\) "X.com \(bank\)")
    * [PayPal](https://en.wikipedia.org/wiki/PayPal "PayPal")
  * [SpaceX](https://en.wikipedia.org/wiki/SpaceX "SpaceX")
    * [Starlink](https://en.wikipedia.org/wiki/Starlink "Starlink")
  * Tesla, Inc.
    * [SolarCity](https://en.wikipedia.org/wiki/SolarCity "SolarCity")
  * [OpenAI](https://en.wikipedia.org/wiki/OpenAI "OpenAI")
  * [Neuralink](https://en.wikipedia.org/wiki/Neuralink "Neuralink")
  * [The Boring Company](https://en.wikipedia.org/wiki/The_Boring_Company "The Boring Company")
  * [X Corp.](https://en.wikipedia.org/wiki/X_Corp. "X Corp.")
    * [Twitter under Elon Musk](https://en.wikipedia.org/wiki/Twitter_under_Elon_Musk "Twitter under Elon Musk")
    * [acquisition](https://en.wikipedia.org/wiki/Acquisition_of_Twitter_by_Elon_Musk "Acquisition of Twitter by Elon Musk")
      * [investigation](https://en.wikipedia.org/wiki/SEC_investigation_into_the_acquisition_of_Twitter_by_Elon_Musk "SEC investigation into the acquisition of Twitter by Elon Musk")
  * [xAI](https://en.wikipedia.org/wiki/XAI_\(company\) "XAI \(company\)")
  * [Thud](https://en.wikipedia.org/wiki/Thud_\(media_company\) "Thud \(media company\)")



* * *

Politics

  * [Activities](https://en.wikipedia.org/wiki/Political_activities_of_Elon_Musk "Political activities of Elon Musk")
  * [America PAC](https://en.wikipedia.org/wiki/America_PAC "America PAC")
  * [America Party](https://en.wikipedia.org/wiki/America_Party "America Party")
  * [Building America's Future](https://en.wikipedia.org/wiki/Building_America%27s_Future "Building America's Future")
  * [Dept. of Govt. Efficiency](https://en.wikipedia.org/wiki/Department_of_Government_Efficiency "Department of Government Efficiency")
  * [2025 salute](https://en.wikipedia.org/wiki/Elon_Musk_salute_controversy "Elon Musk salute controversy")
  * [Trade unions](https://en.wikipedia.org/wiki/Elon_Musk_and_trade_unions "Elon Musk and trade unions")
    * [Tesla](https://en.wikipedia.org/wiki/Tesla_and_trade_unions "Tesla and trade unions")
  * [Protests](https://en.wikipedia.org/wiki/Protests_against_Elon_Musk "Protests against Elon Musk")
    * [Tesla Takedown](https://en.wikipedia.org/wiki/Tesla_Takedown "Tesla Takedown")
    * [vandalism](https://en.wikipedia.org/wiki/2025_Tesla_vandalism "2025 Tesla vandalism")
  * [Trump feud](https://en.wikipedia.org/wiki/Trump%E2%80%93Musk_feud "Trump–Musk feud")



* * *

In books

  * [_Elon Musk_ by Vance](https://en.wikipedia.org/wiki/Elon_Musk:_Tesla,_SpaceX,_and_the_Quest_for_a_Fantastic_Future "Elon Musk: Tesla, SpaceX, and the Quest for a Fantastic Future")
  * [_Ludicrous_](https://en.wikipedia.org/wiki/Ludicrous:_The_Unvarnished_Story_of_Tesla_Motors "Ludicrous: The Unvarnished Story of Tesla Motors")
  * [_Power Play_](https://en.wikipedia.org/wiki/Power_Play:_Tesla,_Elon_Musk,_and_the_Bet_of_the_Century "Power Play: Tesla, Elon Musk, and the Bet of the Century")
  * [_Elon Musk_ by Isaacson](https://en.wikipedia.org/wiki/Elon_Musk_\(Isaacson_book\) "Elon Musk \(Isaacson book\)")
  * [ _Character Limit_](https://en.wikipedia.org/wiki/Character_Limit:_How_Elon_Musk_Destroyed_Twitter "Character Limit: How Elon Musk Destroyed Twitter")



* * *  
  
  * [v](https://en.wikipedia.org/wiki/Template:Elon_Musk_series "Template:Elon Musk series")
  * [t](https://en.wikipedia.org/wiki/Template_talk:Elon_Musk_series "Template talk:Elon Musk series")
  * [e](https://en.wikipedia.org/wiki/Special:EditPage/Template:Elon_Musk_series "Special:EditPage/Template:Elon Musk series")

  
  
**Tesla, Inc.** ([/ˈtɛzlə/](https://en.wikipedia.org/wiki/Help:IPA/English "Help:IPA/English") [_TEZ -lə_](https://en.wikipedia.org/wiki/Help:Pronunciation_respelling_key "Help:Pronunciation respelling key") or [/ˈtɛslə/](https://en.wikipedia.org/wiki/Help:IPA/English "Help:IPA/English") [](https://upload.wikimedia.org/wikipedia/commons/transcoded/7/7f/LL-Q1860_%28eng%29-Flame%2C_not_lame-Tesla%2C_Inc..wav/LL-Q1860_%28eng%29-Flame%2C_not_lame-Tesla%2C_Inc..wav.mp3 "Play audio")[ⓘ](https://en.wikipedia.org/wiki/File:LL-Q1860_\(eng\)-Flame,_not_lame-Tesla,_Inc..wav "File:LL-Q1860 \(eng\)-Flame, not lame-Tesla, Inc..wav") [_TESS -lə_](https://en.wikipedia.org/wiki/Help:Pronunciation_respelling_key "Help:Pronunciation respelling key")[a]) is an American multinational [automotive](https://en.wikipedia.org/wiki/Automotive "Automotive") and [clean energy](https://en.wikipedia.org/wiki/Clean_energy "Clean energy") company. Headquartered in [Austin, Texas](https://en.wikipedia.org/wiki/Austin,_Texas "Austin, Texas"), it designs, manufactures and sells [battery electric vehicles](https://en.wikipedia.org/wiki/Battery_electric_vehicle "Battery electric vehicle") (BEVs), stationary battery [energy storage](https://en.wikipedia.org/wiki/Energy_storage "Energy storage") devices from home to [grid-scale](https://en.wikipedia.org/wiki/Grid-scale_storage "Grid-scale storage"), [solar panels](https://en.wikipedia.org/wiki/Solar_panel "Solar panel") and [solar shingles](https://en.wikipedia.org/wiki/Solar_shingles "Solar shingles"), and related products and services. 

Tesla was incorporated in July 2003 by [Martin Eberhard](https://en.wikipedia.org/wiki/Martin_Eberhard "Martin Eberhard") and [Marc Tarpenning](https://en.wikipedia.org/wiki/Marc_Tarpenning "Marc Tarpenning") as **Tesla Motors**. Its name is a tribute to inventor and electrical engineer [Nikola Tesla](https://en.wikipedia.org/wiki/Nikola_Tesla "Nikola Tesla"). In February 2004, [Elon Musk](https://en.wikipedia.org/wiki/Elon_Musk "Elon Musk") led Tesla's first funding round and became the company's chairman; in 2008, he was named [chief executive officer](https://en.wikipedia.org/wiki/Chief_executive_officer "Chief executive officer"). In 2008, the company began production of its first car model, the [Roadster](https://en.wikipedia.org/wiki/Tesla_Roadster_\(first_generation\) "Tesla Roadster \(first generation\)") sports car, followed by the [Model S](https://en.wikipedia.org/wiki/Tesla_Model_S "Tesla Model S") sedan in 2012, the [Model X](https://en.wikipedia.org/wiki/Tesla_Model_X "Tesla Model X") SUV in 2015, the [Model 3](https://en.wikipedia.org/wiki/Tesla_Model_3 "Tesla Model 3") sedan in 2017, the [Model Y](https://en.wikipedia.org/wiki/Tesla_Model_Y "Tesla Model Y") crossover in 2020, the [Tesla Semi](https://en.wikipedia.org/wiki/Tesla_Semi "Tesla Semi") truck in 2022 and the [Cybertruck](https://en.wikipedia.org/wiki/Tesla_Cybertruck "Tesla Cybertruck") pickup truck in 2023. 

Tesla is one of the [world's most valuable companies in terms of market capitalization](https://en.wikipedia.org/wiki/List_of_public_corporations_by_market_capitalization "List of public corporations by market capitalization"). Starting in July 2020, it has been the world's most valuable automaker. From October 2021 to March 2022, Tesla was a [trillion-dollar company](https://en.wikipedia.org/wiki/Trillion-dollar_company "Trillion-dollar company"), the seventh U.S. company to reach that valuation. Tesla exceeded $1 trillion in market capitalization again between November 2024[5] and February 2025.[6] In 2024, the company led the battery electric vehicle market, with 17.6% share. In 2023, the company was ranked 69th in the [_Forbes_ Global 2000](https://en.wikipedia.org/wiki/Forbes_Global_2000 "Forbes Global 2000").[7]

Tesla has been the subject of lawsuits, boycotts, government scrutiny, and journalistic [criticism](https://en.wikipedia.org/wiki/Criticism_of_Tesla,_Inc. "Criticism of Tesla, Inc."), stemming from allegations of multiple cases of [whistleblower](https://en.wikipedia.org/wiki/Whistleblower "Whistleblower") retaliation, worker rights violations such as sexual harassment and [anti-union activities](https://en.wikipedia.org/wiki/Tesla_and_unions "Tesla and unions"), safety defects leading to dozens of recalls, the lack of a [public relations](https://en.wikipedia.org/wiki/Public_relations "Public relations") department, and controversial statements from Musk including [overpromising](https://en.wikipedia.org/wiki/List_of_Elon_Musk%27s_FSD_predictions "List of Elon Musk's FSD predictions") on the company's [driving assist](https://en.wikipedia.org/wiki/Advanced_driver-assistance_system "Advanced driver-assistance system") technology and product release timelines. In 2025, opponents of Musk have launched the "[Tesla Takedown](https://en.wikipedia.org/wiki/Tesla_Takedown "Tesla Takedown")" campaign in response to the [views of Musk](https://en.wikipedia.org/wiki/Views_of_Elon_Musk "Views of Elon Musk") and his role in the [second Trump presidency](https://en.wikipedia.org/wiki/Second_presidency_of_Donald_Trump "Second presidency of Donald Trump"). 

## Contents

  * 1 History
    * 1.1 Founding (2003–2004)
    * 1.2 Roadster (2005–2009)
    * 1.3 IPO, Model S, and Model X (2010–2015)
    * 1.4 SolarCity and Model 3 (2016–2018)
    * 1.5 Global expansion and Model Y (2019–present)
  * 2 Automotive products and services
    * 2.1 Available products
      * 2.1.1 Model S
      * 2.1.2 Model X
      * 2.1.3 Model 3
      * 2.1.4 Model Y
      * 2.1.5 Tesla Semi
      * 2.1.6 Cybertruck
    * 2.2 Announced products
      * 2.2.1 Roadster (second generation)
      * 2.2.2 Tesla next-generation vehicle
      * 2.2.3 Cybercab
      * 2.2.4 Robovan
    * 2.3 Discontinued products
      * 2.3.1 Tesla Roadster (first generation)
  * 3 Services
    * 3.1 Connectivity services
    * 3.2 Vehicle servicing
    * 3.3 Charging services
      * 3.3.1 Supercharger network
      * 3.3.2 Destination charging location network
    * 3.4 Insurance services
  * 4 Energy products
  * 5 Business strategy
  * 6 Technology
    * 6.1 Batteries
      * 6.1.1 18650
      * 6.1.2 2170
      * 6.1.3 4680
      * 6.1.4 Prismatic
      * 6.1.5 Research
      * 6.1.6 Lithium Refinement
    * 6.2 Software
    * 6.3 Motors
    * 6.4 North American Charging Standard
    * 6.5 "Autopilot" and "Full Self-Driving (Supervised)"
    * 6.6 Glass
    * 6.7 Robotics
  * 7 Facilities
    * 7.1 North America
    * 7.2 Europe
    * 7.3 Asia
  * 8 Partners
    * 8.1 Panasonic
    * 8.2 Other current partners
    * 8.3 Former partners
      * 8.3.1 Daimler
      * 8.3.2 Toyota
      * 8.3.3 Mobileye
  * 9 Lawsuits and controversies
    * 9.1 Failed lawsuit alleging unfair review
    * 9.2 Sexual harassment
    * 9.3 Labor disputes
    * 9.4 Accidents, repairs and safety violations
    * 9.5 Fraud allegations
    * 9.6 Tesla US dealership disputes
    * 9.7 Intellectual property
    * 9.8 Misappropriation
    * 9.9 Environmental violations
    * 9.10 Property damage
    * 9.11 Racism
      * 9.11.1 Fremont, California, plant
    * 9.12 COVID-19 pandemic
    * 9.13 Right to repair
    * 9.14 TeslaTakedown protests
    * 9.15 Lawsuits against Tesla critics and vehicle malfunction complainants in China
  * 10 Criticism
    * 10.1 Data privacy
    * 10.2 Short sellers
    * 10.3 Tesla's mission
    * 10.4 Giga New York audit
    * 10.5 Delays
  * 11 Vehicle product issues
    * 11.1 Recalls
    * 11.2 Fires
    * 11.3 Autopilot crashes
    * 11.4 Software hacking
    * 11.5 Phantom braking
    * 11.6 Driving range performance
  * 12 Vehicle sales
    * 12.1 Production and sales by quarter
  * 13 Finances
  * 14 Corporate affairs
    * 14.1 List of chief executives
    * 14.2 List of board chairs
    * 14.3 Board of directors
    * 14.4 Ownership structure
  * 15 See also
  * 16 Notes
  * 17 References
  * 18 Sources
  * 19 Further reading
  * 20 External links



## History

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=1 "Edit section: History")]

Main article: [History of Tesla, Inc.](https://en.wikipedia.org/wiki/History_of_Tesla,_Inc. "History of Tesla, Inc.")

[](https://en.wikipedia.org/wiki/File:Tesla_Roadster_Sport_insignia_\(cropped\).jpg)Tesla Motors insignia as seen on a [Tesla Roadster](https://en.wikipedia.org/wiki/Tesla_Roadster_\(first_generation\) "Tesla Roadster \(first generation\)"), c. 2010

### Founding (2003–2004)

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=2 "Edit section: Founding \(2003–2004\)")]

The company was incorporated as Tesla Motors, Inc. on July 1, 2003, by [Martin Eberhard](https://en.wikipedia.org/wiki/Martin_Eberhard "Martin Eberhard") and [Marc Tarpenning](https://en.wikipedia.org/wiki/Marc_Tarpenning "Marc Tarpenning").[8][9] They served as [chief executive officer](https://en.wikipedia.org/wiki/Chief_executive_officer "Chief executive officer") and [chief financial officer](https://en.wikipedia.org/wiki/Chief_financial_officer "Chief financial officer"), respectively.[10] Eberhard said that he wanted to build "a car manufacturer that is also a technology company", with its core technologies as "the battery, the computer software, and the proprietary motor".[11]

Ian Wright joined Eberhard and Tarpenning a few months later.[8] In February 2004, the company raised US$7.5 million (equivalent to $12.5 million in 2024) in [series A funding](https://en.wikipedia.org/wiki/Series_A_funding "Series A funding"), including $6.5 million (equivalent to $10.8 million in 2024) from Elon Musk, who had received $100 million from the sale of his interest in [PayPal](https://en.wikipedia.org/wiki/PayPal "PayPal") two years earlier. Musk became the chairman of the board of directors and the largest shareholder of Tesla.[12][13][10] [J. B. Straubel](https://en.wikipedia.org/wiki/J._B._Straubel "J. B. Straubel") joined Tesla in May 2004 as [chief technical officer](https://en.wikipedia.org/wiki/Chief_technical_officer "Chief technical officer").[14]

A lawsuit settlement agreed to by Eberhard and Tesla in September 2009 allows all five – Eberhard, Tarpenning, Wright, Musk, and Straubel – to call themselves co-founders.[15]

### Roadster (2005–2009)

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=3 "Edit section: Roadster \(2005–2009\)")]

Main article: [Tesla Roadster (first generation)](https://en.wikipedia.org/wiki/Tesla_Roadster_\(first_generation\) "Tesla Roadster \(first generation\)")

Elon Musk took an active role within the company, but was not deeply involved in day-to-day business operations.[16] The company's strategy was to start with a premium sports car aimed at [early adopters](https://en.wikipedia.org/wiki/Early_adopter "Early adopter") and then move into more mainstream vehicles, including sedans and affordable compacts.[17]

In February 2006, Musk led Tesla's Series B [venture capital funding](https://en.wikipedia.org/wiki/Venture_capital_funding "Venture capital funding") round of $13 million, which added [Valor Equity Partners](https://en.wikipedia.org/wiki/Valor_Equity_Partners "Valor Equity Partners") to the funding team.[18][13] Musk co-led the third, $40 million round in May 2006 which saw investment from prominent entrepreneurs including [Google](https://en.wikipedia.org/wiki/Google "Google") co-founders [Sergey Brin](https://en.wikipedia.org/wiki/Sergey_Brin "Sergey Brin") and [Larry Page](https://en.wikipedia.org/wiki/Larry_Page "Larry Page"), and former [eBay](https://en.wikipedia.org/wiki/EBay "EBay") president [Jeff Skoll](https://en.wikipedia.org/wiki/Jeff_Skoll "Jeff Skoll").[19] A fourth round worth $45 million in May 2007 brought the total private financing investment to over $105 million.[19]

In August 2007, Eberhard was asked by the board, led by Elon Musk, to step down as CEO.[20] Eberhard then took the title of "President of Technology" before ultimately leaving the company in January 2008. Co-founder Marc Tarpenning, who served as the Vice President of Electrical Engineering of the company, also left the company in January 2008.[21] In August 2007, Michael Marks was brought in as interim CEO, and in December 2007, [Ze'ev Drori](https://en.wikipedia.org/wiki/Ze%27ev_Drori "Ze'ev Drori") became CEO and president.[22] Musk succeeded Drori as CEO in October 2008.[22] In June 2009, Eberhard filed a lawsuit against Musk for allegedly forcing him out.[23] The case was dismissed in August 2009.[24]

Tesla began production of the Roadster in 2008 inside the service bays of a former [Chevrolet](https://en.wikipedia.org/wiki/Chevrolet "Chevrolet") dealership in Menlo Park.[25][26] By January 2009, Tesla had raised $187 million and delivered 147 cars. Musk had contributed $70 million of his money to the company.[27]

In June 2009, Tesla was approved to receive $465 million in interest-bearing loans from the [United States Department of Energy](https://en.wikipedia.org/wiki/United_States_Department_of_Energy "United States Department of Energy"). The funding, part of the $8 billion [Advanced Technology Vehicles Manufacturing Loan Program](https://en.wikipedia.org/wiki/Advanced_Technology_Vehicles_Manufacturing_Loan_Program "Advanced Technology Vehicles Manufacturing Loan Program"), supported the engineering and production of the Model S sedan, as well as the development of commercial powertrain technology.[28] Tesla repaid the loan in May 2013, with $12 million in interest.[29][30]

### IPO, Model S, and Model X (2010–2015)

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=4 "Edit section: IPO, Model S, and Model X \(2010–2015\)")]

[](https://en.wikipedia.org/wiki/File:Tesla_Model_S_first_deliveries.jpg)First deliveries of Model S at the [Tesla Fremont Factory](https://en.wikipedia.org/wiki/Tesla_Fremont_Factory "Tesla Fremont Factory") in California, in June 2012

In May 2010, Tesla purchased the [NUMMI](https://en.wikipedia.org/wiki/NUMMI "NUMMI") plant in [Fremont, California](https://en.wikipedia.org/wiki/Fremont,_California "Fremont, California"), from Toyota for $42 million.[31] On June 29, 2010, the company went public via an [initial public offering](https://en.wikipedia.org/wiki/Initial_public_offering "Initial public offering") (IPO), the first American car company to do so since the [Ford Motor Company](https://en.wikipedia.org/wiki/Ford_Motor_Company "Ford Motor Company") had its IPO in 1956.[32] The company issued 13.3 million shares of common stock at a price of $17 per share at its opening on the [NASDAQ](https://en.wikipedia.org/wiki/NASDAQ "NASDAQ"), raising $226 million.[33]

In October 2010, Tesla opened the [Tesla Factory](https://en.wikipedia.org/wiki/Tesla_Factory "Tesla Factory") to start production of the Model S.[34] In January 2012, Tesla ceased production of the Roadster, and in June 2012, the company launched its second car, the Model S luxury sedan.[35] The Model S won several automotive awards during 2012 and 2013, including the 2013 [Motor Trend Car of the Year](https://en.wikipedia.org/wiki/Motor_Trend_Car_of_the_Year "Motor Trend Car of the Year"),[36] and became the first electric car to top the monthly sales ranking of a country, when it topped the Norwegian new car sales list in September 2013.[37] The Model S was also the best-selling plug-in electric car worldwide for the years 2015 and 2016.[38]

On July 15, 2013, Tesla became a [NASDAQ-100](https://en.wikipedia.org/wiki/NASDAQ-100 "NASDAQ-100") company.[39]

Tesla announced the [Tesla Autopilot](https://en.wikipedia.org/wiki/Tesla_Autopilot "Tesla Autopilot"), a driver-assistance system, in 2014. In September that year, all Tesla cars started shipping with sensors and software to support the feature, with what would later be called "hardware version 1".[40]

Tesla entered the energy storage market, unveiling its [Tesla Powerwall](https://en.wikipedia.org/wiki/Tesla_Powerwall "Tesla Powerwall") (home) and [Tesla Powerpack](https://en.wikipedia.org/wiki/Tesla_Powerpack "Tesla Powerpack") (business) battery packs in April 2015.[41] The company received orders valued at $800 million within a week of the unveiling.[42]

Tesla began shipping its third vehicle, the luxury SUV [Tesla Model X](https://en.wikipedia.org/wiki/Tesla_Model_X "Tesla Model X"), in September 2015, which had 25,000 pre-orders at the time.[43][44]

### SolarCity and Model 3 (2016–2018)

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=5 "Edit section: SolarCity and Model 3 \(2016–2018\)")]

Tesla entered the solar installation business in November 2016 with the purchase of [SolarCity](https://en.wikipedia.org/wiki/SolarCity "SolarCity"), in an all-stock $2.6 billion deal.[45] The business was merged with Tesla's existing battery energy storage products division to form the [Tesla Energy](https://en.wikipedia.org/wiki/Tesla_Energy "Tesla Energy") subsidiary.[46] The deal was controversial because at the time of the acquisition, SolarCity was facing liquidity issues of which Tesla's shareholders were not informed.[47] In February 2017, Tesla Motors changed its name to Tesla, Inc. to better reflect the scope of its expanded business.[48]

Tesla unveiled its first mass market vehicle in April 2016, the Model 3 sedan. The Model 3 was less expensive than Tesla's previous three vehicles, and within a week, the company received over 325,000 paid reservations.[49] To speed up production and control costs, Tesla invested heavily in robotics and automation to assemble the Model 3, but the robotics actually slowed the production of the vehicles.[50] This led to significant delays and production problems, a period which the company described as "production hell".[51][52] By the end of 2018, the production problems had been overcome, and the Model 3 became the world's best-selling electric car from 2018 to 2021.[53][54]

This period of "production hell" put significant financial pressure on Tesla, and during this time it became one of the most [shorted](https://en.wikipedia.org/wiki/Short_\(finance\) "Short \(finance\)") companies in the stock market. On August 8, 2018, amid the financial issues, Musk posted on social media that he was considering taking Tesla private.[55][56] The plan did not materialize and gave rise to much controversy and many lawsuits including a [securities fraud charge from the SEC](https://en.wikipedia.org/wiki/Elon_Musk#SEC_and_shareholder_lawsuits_regarding_tweets "Elon Musk"), which would force Musk to pay a $20 million fine and step down as the company's chairman, although he was allowed to remain the CEO. 

### Global expansion and Model Y (2019–present)

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=6 "Edit section: Global expansion and Model Y \(2019–present\)")]

From July 2019 to June 2020, Tesla reported four consecutive profitable quarters for the first time, which made it eligible for inclusion in the [S&P 500](https://en.wikipedia.org/wiki/S%26P_500 "S&P 500").[57] During 2020, its share price increased 740%,[58] and by December 14, 2020, its market capitalization was more than the next nine largest automakers combined,[59] and it became the sixth most valuable company in the US.[60] Tesla was added to the S&P index on December 21, 2020;[61] it was the most valuable company ever added, and was the sixth-largest member of the index immediately after it was added.[61][62]

Tesla introduced its second mass-market vehicle in March 2019, the Model Y [mid-size crossover SUV](https://en.wikipedia.org/wiki/Crossover_\(automobile\) "Crossover \(automobile\)"), based on the Model 3.[63][64] Deliveries started in March 2020.[65]

During this period, Tesla invested heavily in expanding its production capacity, opening three new Gigafactories in quick succession. Construction of [Gigafactory Shanghai](https://en.wikipedia.org/wiki/Gigafactory_Shanghai "Gigafactory Shanghai") started in January 2019, as the first automobile factory in China fully owned by a foreign company (not a joint venture).[66] The factory's first production vehicle, a Model 3, rolled out in December, less than one year after groundbreaking.[67] [Gigafactory Berlin](https://en.wikipedia.org/wiki/Gigafactory_Berlin-Brandenburg "Gigafactory Berlin-Brandenburg") broke ground in February 2020,[68] and its production of the Model Y began in March 2022.[69] [Gigafactory Texas](https://en.wikipedia.org/wiki/Gigafactory_Texas "Gigafactory Texas") broke ground in June 2020,[70] and its production of the Model Y began in April 2022.[71] In March 2023, Tesla announced plans for a [Gigafactory Mexico](https://en.wikipedia.org/wiki/Gigafactory_Mexico "Gigafactory Mexico") to open in 2025,[72] but its groundbreaking has been delayed. 

At the beginning of the [COVID-19 pandemic](https://en.wikipedia.org/wiki/COVID-19_pandemic "COVID-19 pandemic"), Tesla closed the Fremont Factory in March 2020 due to California state and [Alameda county](https://en.wikipedia.org/wiki/Alameda_County,_California "Alameda County, California") COVID restrictions.[73] When California lifted restrictions, but the county did not, Tesla sued the county, and restarted production on May 11, 2020.[74] The county lifted restrictions on May 13, 2020, and Tesla dropped its lawsuit.[75] After the dispute with county officials, on December 1, 2021, Tesla moved its legal headquarters to Gigafactory Texas.[76][77] However, Tesla continued to use its former headquarters building in Palo Alto, and over the next two years significantly expanded its footprint in California. The company opened its Megafactory to build Megapack batteries in Lathrop, California in 2022,[78] and announced in February 2023 that it would establish a large global engineering headquarters in Palo Alto, moving into a corporate campus once owned by [Hewlett Packard](https://en.wikipedia.org/wiki/HP_Inc. "HP Inc.").[79]

In early 2021, Tesla became a major investor in [bitcoin](https://en.wikipedia.org/wiki/Bitcoin "Bitcoin"), acquiring $1.5 billion of the [cryptocurrency](https://en.wikipedia.org/wiki/Cryptocurrency "Cryptocurrency"),[80] and on March 24, 2021, the company started accepting bitcoin as a form of payment for US vehicle purchases.[81] However, after 49 days, the company ended bitcoin payments over concerns that the production of bitcoin was contributing to the consumption of fossil fuels, against the company's mission of encouraging the transition to sustainable energy.[82] After the announcement, the price of bitcoin dropped around 12%.[83] Tesla CEO Elon Musk later noted that Tesla would resume bitcoin payments if there was confirmation of at least 50% clean energy usage by bitcoin miners. Despite later reaching this milestone, Tesla did not return to accepting bitcoin.[84] By July 2022 Tesla had sold about 75% of its bitcoin holdings at a loss, citing that the cryptocurrency was hurting the company's profitability.[85]

Between May 2023 and February 2024, almost all major North America EV manufacturers announced plans to switch to Tesla's [North American Charging Standard](https://en.wikipedia.org/wiki/North_American_Charging_Standard "North American Charging Standard") adapters on their EVs by 2025, which is expected to be a stable source of recurring revenue for Tesla.[86] In November, Tesla started shipping the Cybertruck, produced from Gigafactory Texas.[87]

In April 2024, the company announced it was laying off 10% of its employees.[88] In June, the company moved its incorporation from Delaware to Texas.[89] In October, the company unveiled a concept version of two [autonomous](https://en.wikipedia.org/wiki/Autonomous_robot "Autonomous robot") vehicles – the [Cybercab](https://en.wikipedia.org/wiki/Cybercab "Cybercab") and [Robovan](https://en.wikipedia.org/wiki/Tesla_Robovan "Tesla Robovan") – and detailed that both would be an integral part of a Tesla [ridehailing service](https://en.wikipedia.org/wiki/Ridehailing_service "Ridehailing service") called the [Tesla Network](https://en.wikipedia.org/wiki/Tesla_Network "Tesla Network"),[90][91] a future service they had previously teased in 2019.[92]

In December 2024, a [Delaware](https://en.wikipedia.org/wiki/Delaware "Delaware") court rejected Elon Musk's $56 billion pay package from Tesla, ruling that it was not properly approved by the company's board. The decision arose from a lawsuit by Tesla shareholders who claimed the compensation was excessive and not aligned with performance metrics.[93]

By February 2025, Tesla saw large decreases in its stock price and sales across Europe widely attributed to [Elon Musk's political advocacy](https://en.wikipedia.org/wiki/Political_activities_of_Elon_Musk "Political activities of Elon Musk") and embrace of [far-right politics](https://en.wikipedia.org/wiki/Far-right_politics "Far-right politics").[94] Polling found that Musk's ties to United States President [Donald Trump](https://en.wikipedia.org/wiki/Donald_Trump "Donald Trump") and his [Department of Government Efficiency](https://en.wikipedia.org/wiki/Department_of_Government_Efficiency "Department of Government Efficiency") were strongly correlated to decreasing views of Tesla,[95][96] and triggered [multiple protests](https://en.wikipedia.org/wiki/Tesla_Takedown "Tesla Takedown"), vandalism, gunfire, and arson at Tesla stores and charging stations.[97][98] By March 7, Tesla's stock had decreased every week for seven straight weeks since Musk joined the Trump administration, making it the longest losing streak for Tesla in 15 years as a public company.[99] On March 29, over 200 protests against Tesla were held worldwide.[100]

In July 2025, Tesla rolled out a software update adding [Grok](https://en.wikipedia.org/wiki/Grok_\(chatbot\) "Grok \(chatbot\)") to its vehicles. While the update provides in-car chatbot functionality, it does not give Grok control over vehicle functions.[101]

In July 2025, Tesla opened its first showroom in [India](https://en.wikipedia.org/wiki/India "India") at the [Bandra-Kurla Complex](https://en.wikipedia.org/wiki/Bandra-Kurla_Complex "Bandra-Kurla Complex") in [Mumbai](https://en.wikipedia.org/wiki/Mumbai "Mumbai"), showcasing the [Model Y](https://en.wikipedia.org/wiki/Tesla_Model_Y "Tesla Model Y") as its debut offering.[102][103]

## Automotive products and services

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=7 "Edit section: Automotive products and services")]

"Tesla electric car" redirects here; not to be confused with [Nikola Tesla electric car hoax](https://en.wikipedia.org/wiki/Nikola_Tesla_electric_car_hoax "Nikola Tesla electric car hoax").

As of November 2024[[update]](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit), Tesla offers six vehicle models: Model S, Model X, Model 3, Model Y, Semi, and Cybertruck. Tesla's first vehicle, the [first-generation Tesla Roadster](https://en.wikipedia.org/wiki/Tesla_Roadster_\(first_generation\) "Tesla Roadster \(first generation\)"), is no longer sold. Tesla has announced plans for a [second-generation Roadster](https://en.wikipedia.org/wiki/Tesla_Roadster_\(second_generation\) "Tesla Roadster \(second generation\)"), the Cybercab, and the Robovan. 

### Available products 

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=8 "Edit section: Available products")]

#### Model S

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=9 "Edit section: Model S")]

Main article: [Tesla Model S](https://en.wikipedia.org/wiki/Tesla_Model_S "Tesla Model S")

[](https://en.wikipedia.org/wiki/File:2018_Tesla_Model_S_75D.jpg)Tesla Model S

The Model S is a [full-size car](https://en.wikipedia.org/wiki/Full-size_car "Full-size car") with a [liftback](https://en.wikipedia.org/wiki/Liftback "Liftback") body style and a dual motor, [all-wheel drive](https://en.wikipedia.org/wiki/All-wheel_drive "All-wheel drive") layout. Development of the Model S began before 2007 and deliveries started in June 2012. The Model S has seen two major design refreshes, first in April 2016, which introduced a new front-end design and again in June 2021, which revised the interior. The Model S was the top-selling plug-in electric car worldwide in 2015 and 2016. More than 250,000 vehicles have been sold as of December 2018[[update]](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit) (when Tesla merged production numbers for the Model S and Model X). 

#### Model X

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=10 "Edit section: Model X")]

Main article: [Tesla Model X](https://en.wikipedia.org/wiki/Tesla_Model_X "Tesla Model X")

[](https://en.wikipedia.org/wiki/File:Tesla_Model_X_100D_1X7A6736.jpg)Tesla Model X

The Model X is a [mid-size luxury crossover SUV](https://en.wikipedia.org/wiki/Crossover_\(automobile\) "Crossover \(automobile\)") offered in 5-, 6- and 7-passenger configurations with either a dual- or trimotor, all-wheel drive layout. The rear passenger doors open vertically with an articulating "[falcon-wing](https://en.wikipedia.org/wiki/Falcon-wing_door "Falcon-wing door")" design. A prototype Model X was first shown in February 2012 and deliveries started in September 2015.[104] The Model X shares around 30 percent of its content with the Model S. The vehicle has seen one major design refresh in June 2021 which revised the interior. 

#### Model 3

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=11 "Edit section: Model 3")]

Main article: [Tesla Model 3](https://en.wikipedia.org/wiki/Tesla_Model_3 "Tesla Model 3")

[](https://en.wikipedia.org/wiki/File:Tesla_Model_3_\(2023\)_Auto_Zuerich_2023_1X7A1313.jpg)Tesla Model 3

The Model 3 is a [mid-size car](https://en.wikipedia.org/wiki/Mid-size_car "Mid-size car") with a [fastback](https://en.wikipedia.org/wiki/Fastback "Fastback") body style and either a dual-motor, all-wheel drive layout or a rear-motor, [rear-wheel drive](https://en.wikipedia.org/wiki/Rear-wheel_drive "Rear-wheel drive") layout. The vehicle was designed to be more affordable than the luxury Model S sedan. A prototype Model 3 was first shown in 2016 and within a week, the company received over 325,000 paid reservations.[49] Deliveries started in July 2017.[105] The Model 3 ranked as the world's best-selling electric car from 2018 to 2021,[106][107][108] and cumulative sales passed 1 million in June 2021.[109] The vehicle has seen one major design refresh in September 2023 which revised the exterior and interior. 

#### Model Y

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=12 "Edit section: Model Y")]

Main article: [Tesla Model Y](https://en.wikipedia.org/wiki/Tesla_Model_Y "Tesla Model Y")

[](https://en.wikipedia.org/wiki/File:\(SGP-Singapore\)_Showcar_Tesla_Model_Y_No-plate_2025-02-01.jpg)Tesla Model Y

The Model Y is a [mid-size crossover SUV](https://en.wikipedia.org/wiki/Crossover_\(automobile\) "Crossover \(automobile\)") offered in 5- and 7-passenger configurations with a single‐motor, rear-wheel drive or a dual-motor, all-wheel drive layout. The vehicle was designed to be more affordable than the luxury Model X SUV. A prototype Model Y was first shown in March 2019,[63] and deliveries started in March 2020.[65] The Model Y shared around 75 percent of its content with the Model 3.[64] In the first quarter of 2023, the Model Y outsold the [Toyota Corolla](https://en.wikipedia.org/wiki/Toyota_Corolla "Toyota Corolla") to become the world's best-selling car, the first electric vehicle to claim the title.[110]

#### Tesla Semi

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=13 "Edit section: Tesla Semi")]

Main article: [Tesla Semi](https://en.wikipedia.org/wiki/Tesla_Semi "Tesla Semi")

[](https://en.wikipedia.org/wiki/File:The_Tesla_Semi_Truck_\(40705940423\).jpg)Tesla Semi prototype

The Tesla Semi is a [Class 8](https://en.wikipedia.org/wiki/Truck_classification#Class_8 "Truck classification") [semi-truck](https://en.wikipedia.org/wiki/Semi-trailer_truck "Semi-trailer truck") by Tesla, Inc. with a tri-motor, rear-wheel drive layout. Tesla claims that the Semi has approximately three times the power of a typical diesel semi truck, and a range of 500 miles (800 km).[111] Two prototype trucks were first shown in November 2017 and initial deliveries were made to [PepsiCo](https://en.wikipedia.org/wiki/PepsiCo "PepsiCo") on December 1, 2022.[112] Tesla stated in April 2024 that it plans full production in late 2025.[113]

#### Cybertruck

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=14 "Edit section: Cybertruck")]

Main article: [Tesla Cybertruck](https://en.wikipedia.org/wiki/Tesla_Cybertruck "Tesla Cybertruck")

[](https://en.wikipedia.org/wiki/File:2024_Tesla_Cybertruck_Foundation_Series_IMG_0576.jpg)Tesla Cybertruck

The Cybertruck is a full-sized [pickup truck](https://en.wikipedia.org/wiki/Pickup_truck "Pickup truck"). First announced in November 2019, pilot production began in July 2023, after being pushed back multiple times, and deliveries began on November 30, 2023. Three models are offered: rear-wheel drive, dual-motor all-wheel drive, and trimotor all-wheel drive, with [EPA range](https://en.wikipedia.org/wiki/Electric_car_EPA_fuel_economy "Electric car EPA fuel economy") estimates of 320–340 miles (510–550 km), depending on the model. The truck's exterior design made from flat sheets of unpainted stainless steel earned a notably polarizing reception from media.[114][115][116] Tesla initially planned for Cybertruck production capacity of more than 250,000 units, but as of 2025 the company is only selling around 20,000 units per year.[117]

### Announced products

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=15 "Edit section: Announced products")]

#### Roadster (second generation)

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=16 "Edit section: Roadster \(second generation\)")]

Main article: [Tesla Roadster (second generation)](https://en.wikipedia.org/wiki/Tesla_Roadster_\(second_generation\) "Tesla Roadster \(second generation\)")

[](https://en.wikipedia.org/wiki/File:NextGenTeslaRoadster_\(cropped\).jpg)Tesla Roadster prototype

On November 16, 2017, Tesla unveiled the second generation Roadster with a purported range of 620 miles (1,000 km) with a 200-kilowatt-hour (720 MJ) battery pack that would achieve 0–60 miles per hour (0–97 km/h) in 1.9 seconds; and 0–100 mph (0–161 km/h) in 4.2 seconds,[118] and a top speed over 250 mph (400 km/h). A "[SpaceX](https://en.wikipedia.org/wiki/SpaceX "SpaceX") Package" would include [cold-gas thrusters](https://en.wikipedia.org/wiki/Cold_gas_thruster "Cold gas thruster").[119] The vehicle would have three electric motors, allowing [all-wheel drive](https://en.wikipedia.org/wiki/All-wheel_drive "All-wheel drive") and [torque vectoring](https://en.wikipedia.org/wiki/Torque_vectoring "Torque vectoring") during cornering.[119] The base price was set at $200,000.[119] Initially scheduled to ship in 2020, the vehicle has been repeatedly delayed. In July 2024, Musk said that the Roadster should enter production in 2025.[120]

#### Tesla next-generation vehicle

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=17 "Edit section: Tesla next-generation vehicle")]

Main article: [Tesla next-generation vehicle](https://en.wikipedia.org/wiki/Tesla_next-generation_vehicle "Tesla next-generation vehicle")

The [Tesla next-generation vehicle](https://en.wikipedia.org/wiki/Tesla_next-generation_vehicle "Tesla next-generation vehicle") is an announced battery electric platform. It would become the third [platform](https://en.wikipedia.org/wiki/Car_platform "Car platform") for the company. Vehicles based on this platform are not expected before 2025.[121] In July 2024, Musk said that the platform should be expected to become available in the first half of 2025.[120]

#### Cybercab

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=18 "Edit section: Cybercab")]

[](https://en.wikipedia.org/wiki/File:Tesla_Cybercab_-_Berlin_2024.jpg)Tesla Cybercab prototype

Main article: [Tesla Cybercab](https://en.wikipedia.org/wiki/Tesla_Cybercab "Tesla Cybercab")

The Tesla Cybercab, also known as the Robotaxi, is an upcoming [two-passenger](https://en.wikipedia.org/wiki/Two-passenger_car "Two-passenger car") battery-electric [self-driving car](https://en.wikipedia.org/wiki/Self-driving_car "Self-driving car") under development by Tesla. 

A [concept](https://en.wikipedia.org/wiki/Concept_car "Concept car") version of the Cybercab was unveiled in October 2024, with 20 prototypes providing short rides to attendees of the announcement event. The concept vehicle had no steering wheel or pedals. 

The production Cybercab is planned to be fully [autonomous](https://en.wikipedia.org/wiki/Autonomous_robot "Autonomous robot") and to be released before 2027. 

#### Robovan

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=19 "Edit section: Robovan")]

Main article: [Tesla Robovan](https://en.wikipedia.org/wiki/Tesla_Robovan "Tesla Robovan")

The Tesla Robovan is an [electric](https://en.wikipedia.org/wiki/Electric_vehicle "Electric vehicle") [autonomous](https://en.wikipedia.org/wiki/Autonomous_robot "Autonomous robot") [van](https://en.wikipedia.org/wiki/Van "Van") planned for future development by Tesla.[122] Announced in October 2024, the vehicle is being designed to carry up to 20 passengers.[91]

### Discontinued products

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=20 "Edit section: Discontinued products")]

[](https://en.wikipedia.org/wiki/File:IFA_2010_Internationale_Funkausstellung_Berlin_93.JPG)The original Roadster

#### Tesla Roadster (first generation)

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=21 "Edit section: Tesla Roadster \(first generation\)")]

Main article: [Tesla Roadster (first generation)](https://en.wikipedia.org/wiki/Tesla_Roadster_\(first_generation\) "Tesla Roadster \(first generation\)")

The original Tesla Roadster[123] was a two-seater sports car, evolved from the [Lotus Elise](https://en.wikipedia.org/wiki/Lotus_Elise "Lotus Elise") chassis.[124] It was produced from 2008 to 2012. The Roadster was the first highway-legal serial production electric car to use lithium-ion battery cells, and the first production all-electric car to travel more than 200 miles (320 km) per charge. 

## Services

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=22 "Edit section: Services")]

### Connectivity services

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=23 "Edit section: Connectivity services")]

Tesla cars come with "Standard Connectivity", which provides [navigation](https://en.wikipedia.org/wiki/Automotive_navigation_system "Automotive navigation system") using a cellular connection. For a fee, Tesla offers a subscription to "Premium Connectivity" which adds [live traffic](https://en.wikipedia.org/wiki/Traffic_reporting "Traffic reporting") and [satellite maps](https://en.wikipedia.org/wiki/Satellite_map "Satellite map") to navigation, internet browsing, and media streaming.[125]

### Vehicle servicing

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=24 "Edit section: Vehicle servicing")]

Tesla's strategy is to service its vehicles first through remote diagnosis and repair. If it is not possible to resolve a problem remotely, a mobile technician is dispatched or customers are referred to a local Tesla-owned service center.[126][127] As of October 2024[[update]](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit), the company operates 1,306 retail stores, galleries, service, delivery and body shop locations globally.[128] Tesla has said that it does not want to make a profit on vehicle servicing, which has traditionally been a large profit center for most auto dealerships.[129]

In 2016, Tesla recommended having any Tesla car inspected every 12,500 miles or once a year, whichever comes first. In early 2019, the manual was changed to say: "your Tesla does not require annual maintenance and regular fluid changes," and instead it recommends periodic servicing of the brake fluid, air conditioning, tires and air filters.[130]

###  Charging services

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=25 "Edit section: Charging services")]

####  Supercharger network

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=26 "Edit section: Supercharger network")]

Main article: [Tesla Supercharger](https://en.wikipedia.org/wiki/Tesla_Supercharger "Tesla Supercharger")

Supercharger is the branding used by Tesla for its high-voltage direct current [fast chargers](https://en.wikipedia.org/wiki/Fast_charger "Fast charger"). 

[](https://en.wikipedia.org/wiki/File:Wittenburg_-_Supercharger_-_2021.jpg)Tesla Supercharger station in [Wittenburg](https://en.wikipedia.org/wiki/Wittenburg "Wittenburg"), Germany The Supercharger network was introduced on September 24, 2012, as the Tesla [Model S](https://en.wikipedia.org/wiki/Tesla_Model_S "Tesla Model S") entered production, with five stations in California. As of July 2025[[update]](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit), Tesla operates a network of about 7,500 Supercharger stations with over 70,000 connectors worldwide. The majority are located in three regions: Asia Pacific (3,000 stations), North America (3,000), and Europe (1,500). Superchargers can currently output as much as 325 [kilowatts](https://en.wikipedia.org/wiki/Kilowatts "Kilowatts") (kW), with plans to increase output capacity to 500 kW in the future.

#### Destination charging location network

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=27 "Edit section: Destination charging location network")]

[](https://en.wikipedia.org/wiki/File:TeslaDestinationCharger.jpg)"Destination Charger" in North America

Tesla also has a network of "Destination Chargers", slower than Superchargers and intended for locations where customers are expected to park and stay for several hours, such as hotels, restaurants, or shopping centers. Unlike the Supercharger network, Tesla does not own the destination chargers, instead, property owners set up the devices and set pricing.[131] When the network first launched in 2014, Tesla provided free charging equipment and covered installation costs.[_[citation needed](https://en.wikipedia.org/wiki/Wikipedia:Citation_needed "Wikipedia:Citation needed")_] One of the largest providers is hotel chain [Hilton Worldwide](https://en.wikipedia.org/wiki/Hilton_Worldwide "Hilton Worldwide") which in 2023 announced an agreement with Tesla to install 20,000 chargers across 2,000 of its properties in North America by 2025.[132]

### Insurance services

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=28 "Edit section: Insurance services")]

Tesla has offered its own [vehicle insurance in the United States](https://en.wikipedia.org/wiki/Vehicle_insurance_in_the_United_States "Vehicle insurance in the United States") since 2017 and has been acting as an independent insurance producer since 2021 as Tesla Insurance Services, Inc. It was introduced after the [American Automobile Association](https://en.wikipedia.org/wiki/American_Automobile_Association "American Automobile Association") (AAA), a major insurance carrier, raised rates for Tesla owners in June 2017 after a report concluded that the automakers vehicles crashed more often and were pricier to repair than comparable vehicles.[133] A study in 2018 based on data from the [Insurance Institute for Highway Safety](https://en.wikipedia.org/wiki/Insurance_Institute_for_Highway_Safety "Insurance Institute for Highway Safety") confirmed the findings.[134]

The company says that it uniquely understands its vehicles, technology and repair costs, and can eliminate traditional insurance carriers' additional charges.[135] In states where allowed, the company uses individual vehicle data to offer personalized pricing that can increase or decrease in cost based on the prior month's driving safety score.[136]

As of January 2023[[update]](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit), Tesla offers insurance in the U.S. states of Arizona, California, Colorado, Illinois, Maryland, Minnesota, Nevada, Ohio, Oregon, Texas, Utah and Virginia.[137] The company also offers insurance for non-Tesla vehicles owned by Tesla owners.[135]

## Energy products

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=29 "Edit section: Energy products")]

Main article: [Tesla Energy](https://en.wikipedia.org/wiki/Tesla_Energy "Tesla Energy")

[](https://en.wikipedia.org/wiki/File:Johnstone_000178_172617_517894_4578_\(36736411886\).jpg)Two Tesla Powerwall 2 home energy storage devices from Tesla Energy

Tesla subsidiary Tesla Energy develops, builds, sells and installs [solar energy generation](https://en.wikipedia.org/wiki/Photovoltaics "Photovoltaics") systems and battery energy storage products (as well as related products and services) to residential, commercial and industrial customers. The subsidiary was created by the merger of Tesla's existing battery energy storage products division with SolarCity, a solar energy company that Tesla acquired in 2016.[138] In 2023, the company deployed 14.7 gigawatt-hours of battery energy storage products, an increase of 125% over 2022, but only deployed solar energy systems capable of generating 223 [megawatts](https://en.wikipedia.org/wiki/Megawatt "Megawatt"), a decrease of 36% over 2022.[139]

Tesla Energy products include solar panels (built by other companies for Tesla), the [Tesla Solar Roof](https://en.wikipedia.org/wiki/Tesla_Solar_Roof "Tesla Solar Roof") (a [solar shingle](https://en.wikipedia.org/wiki/Solar_shingle "Solar shingle") system) and the Tesla Solar Inverter. Storage products include the [Powerwall](https://en.wikipedia.org/wiki/Tesla_Powerwall "Tesla Powerwall") (a [home energy storage](https://en.wikipedia.org/wiki/Home_energy_storage "Home energy storage") device) and the [Megapack](https://en.wikipedia.org/wiki/Tesla_Megapack "Tesla Megapack") (a large-scale energy storage system).[140][141][142]

For large-scale customers, Tesla Energy operates an online platform which allows for automated, real-time power trading, demand forecasting and product control.[143][144][145] In March 2021, the company said its online products were managing over 1.2 GWh of storage.[146] For home customers, the company operates a virtual power company in Texas called Tesla Electric, which utilizes the company's online platforms to manage customers Powerwall devices, discharging them into the grid to sell power when prices are high, earning money for customers.[147][148]

## Business strategy

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=30 "Edit section: Business strategy")]

[](https://en.wikipedia.org/wiki/File:Tesla_auto_bots.jpg)[Robotic](https://en.wikipedia.org/wiki/Industrial_robot "Industrial robot") manufacturing of the Model S at the [Tesla Factory](https://en.wikipedia.org/wiki/Tesla_Factory "Tesla Factory") in [Fremont, California](https://en.wikipedia.org/wiki/Fremont,_California "Fremont, California")

At the time of Tesla's founding in 2003, electric vehicles were costly.[149] In 2006, Elon Musk stated that Tesla's strategy was to first produce high-price, low-volume vehicles, such as sports cars, for which customers are less sensitive to price. This would allow them to progressively bring down the cost of batteries, which in turn would allow them to offer cheaper and higher volume cars.[17][150] Tesla's first vehicle, the Roadster, was low-volume (fewer than 2,500 were produced) and priced at over $100,000. The next models, the Model S and Model X, are more affordable but still luxury vehicles. The Model 3 and the Model Y, are priced still lower, and aimed at a higher volume market,[151][152] selling over 100,000 vehicles each quarter. Tesla continuously updates the hardware of its cars rather than waiting for a new model year, unlike nearly every other car manufacturer.[153]

Unlike other automakers, Tesla does not rely on franchised [dealerships](https://en.wikipedia.org/wiki/Car_dealership "Car dealership") to sell vehicles. Instead, the company directly sells vehicles through its website and a network of company-owned stores.[154][155] The company is the first automaker in the United States to sell cars directly to consumers.[156][157] Some jurisdictions, particularly in the United States, prohibit auto manufacturers from directly selling vehicles to consumers. In these areas, Tesla has locations that it calls _galleries_ that the company says, "educate and inform customers about our products, but such locations do not actually transact in the sale of vehicles."[158][159] In total, Tesla operates nearly 400 stores and galleries in more than 35 countries.[160] These locations are typically located in retail shopping districts, inside shopping malls, or other high-traffic areas,[155] instead of near other auto dealerships.[161][162][163]

Analysts describe Tesla as vertically integrated given how it develops many components in-house, such as batteries, motors, and software.[164] The practice of vertical integration is rare in the automotive industry, where companies typically _outsource_ 80% of components to suppliers and focus on engine manufacturing and final assembly.[165][166][167]

Tesla generally allows its competitors to license its technology, stating that it wants to help its competitors accelerate the world's use of sustainable energy.[168] Licensing agreements include provisions whereby the recipient agrees not to file patent suits against Tesla, or to copy its designs directly.[169] Tesla retains control of its other intellectual property, such as trademarks and [trade secrets](https://en.wikipedia.org/wiki/Trade_secret "Trade secret") to prevent direct copying of its technology.[170]

On April 15, 2024, Tesla secured a deal with Tata Electronics to supply semiconductor chips, marking a significant step in Tesla's expansion into India's automotive market.[171]

On May 2, 2024, Tesla announced that it has abandoned its plan for next-generation gigacasting, a cutting-edge manufacturing technique. Initially aiming to revolutionize production and reduce costs, Tesla has now opted for its more proven method of casting vehicle underbodies in three pieces. This strategic shift reflects the company's focus on self-driving vehicles and adjusting to market challenges.[172]

## Technology

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=31 "Edit section: Technology")]

Tesla is highly vertically integrated and develops many components in-house, such as batteries, motors, and software.[164]

###  Batteries

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=32 "Edit section: Batteries")]

[](https://en.wikipedia.org/wiki/File:Tesla_chassis,_Sydney_-_Martin_Place,_2017_\(01\).jpg)Tesla vehicle [chassis](https://en.wikipedia.org/wiki/Chassis "Chassis") used in [Model S](https://en.wikipedia.org/wiki/Model_S "Model S") and [X](https://en.wikipedia.org/wiki/Tesla_Model_X "Tesla Model X"), with the battery visible [](https://en.wikipedia.org/wiki/File:Tesla_4680_2170_18650_batteries.svg)Comparison of Tesla's three cylindrical battery cell form factors

As of 2023, Tesla uses four different [battery cell](https://en.wikipedia.org/wiki/Battery_cell "Battery cell") form factors: [18650](https://en.wikipedia.org/wiki/18650 "18650"), [2170](https://en.wikipedia.org/wiki/2170_battery "2170 battery"), [4680](https://en.wikipedia.org/wiki/4680_battery "4680 battery"), and prismatic.[173][174][175]

Tesla purchases these batteries from three suppliers, [CATL](https://en.wikipedia.org/wiki/CATL "CATL"), [LG Energy Solution](https://en.wikipedia.org/wiki/LG_Energy_Solution "LG Energy Solution"), and [Panasonic](https://en.wikipedia.org/wiki/Panasonic "Panasonic"), the latter of which has co-located some of its battery production inside Tesla's Gigafactory Nevada. Tesla is also currently building out the capacity to produce its own batteries. 

Tesla batteries sit under the vehicle floor to save interior space. Tesla uses a multipart aluminum and titanium protection system to protect the battery from road debris or vehicle crashes.[176]

Business analysis company [BloombergNEF](https://en.wikipedia.org/wiki/BloombergNEF "BloombergNEF") estimated Tesla's battery pack cost in 2021 at $112 per [kilowatt-hour](https://en.wikipedia.org/wiki/Kilowatt-hour "Kilowatt-hour") (kWh), versus an industry average of $132 per kWh.[177]

#### 18650

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=33 "Edit section: 18650")]

Tesla was the first automaker to use cylindrical, [lithium-ion battery](https://en.wikipedia.org/wiki/Lithium-ion_battery "Lithium-ion battery") cells. When it built the [first generation Roadster](https://en.wikipedia.org/wiki/Tesla_Roadster_\(first_generation\) "Tesla Roadster \(first generation\)"), it used off-the-shelf 18650-type (18 mm diameter, 65 mm height) cylindrical batteries that were already used for other consumer electronics. The cells provided an engineering challenge because each has a relatively low capacity, so thousands needed to be bundled together in a battery pack. Electrical and thermal management also proved to be a challenge, requiring liquid cooling and an [intumescent](https://en.wikipedia.org/wiki/Intumescent "Intumescent") fire prevention chemical.[178] However, the decision turned out to be pragmatic because there was already a mature manufacturing process that could produce a high volume of the cells at a consistent quality. Although the 18650-type cells are the oldest technology, they are used in the Model S and X vehicles. Tesla sources these batteries with a [nickel-cobalt-aluminum](https://en.wikipedia.org/wiki/Lithium_nickel_cobalt_aluminium_oxides "Lithium nickel cobalt aluminium oxides") (NCA) cathode chemistry from Panasonic's factories in Japan.[173]

#### 2170

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=34 "Edit section: 2170")]

The next battery type to be used was the 2170-type (21 mm diameter, 70 mm height) cylindrical cell. The larger size was optimized for electric cars, allowing for a higher capacity per cell and a lower number of cells per battery pack. The 2170 was introduced for the Model 3 and Y vehicles.[173]

For vehicles built at the Tesla Fremont Factory, the company sources 2170-type batteries with a nickel-cobalt-aluminum cathode chemistry from Panasonic's production line at Gigafactory Nevada.[179] In January 2021, Panasonic had the capacity to produce 39 GWh per year of battery cells there.[180] Tesla Energy also uses 2170 cells in its Powerwall home energy storage product. 

For vehicles made at Gigafactory Shanghai and Gigafactory Berlin batteries with a [nickel-cobalt-manganese](https://en.wikipedia.org/wiki/Lithium_nickel_manganese_cobalt_oxides "Lithium nickel manganese cobalt oxides") (NMC) cathode chemistry are sourced from LG Energy Solution's factories in China.[173]

#### 4680

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=35 "Edit section: 4680")]

Tesla's latest cylindrical cell design is the 4680-type (46 mm diameter, 80 mm height) introduced in 2021. The battery was developed in-house by Tesla and is physically 5-times bigger than the 2170-type, again allowing for a higher capacity per cell and a lower number of cells per battery pack.[181][182] Currently, Tesla builds the 4680 cells itself and has not disclosed the cathode chemistry. The company has already opened production lines in Fremont, California, and plans to open lines inside Gigafactory Nevada and Gigafactory Texas. The 4680 cells are used in the Model Y and Cybertruck built at Gigafactory Texas.[173]

#### Prismatic

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=36 "Edit section: Prismatic")]

Tesla also uses prismatic (rectangular) cells in many entry-level Model 3 and Model Y vehicles.[173] The prismatic cells are a [lithium iron phosphate battery](https://en.wikipedia.org/wiki/Lithium_iron_phosphate_battery "Lithium iron phosphate battery") (LFP or LiFePO  
4) which is a less energy-dense type, but do not contain any nickel or cobalt, which makes it less expensive to produce.[183] Tesla sources these batteries from CATL's factories in China. As of April 2022[[update]](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit), nearly half of Tesla's vehicle production used prismatic cells.[184] Tesla Energy also uses prismatic cells in its Megapack grid-scale energy storage product.[185]

#### Research

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=37 "Edit section: Research")]

Tesla invests in [lithium-ion battery](https://en.wikipedia.org/wiki/Lithium-ion_battery "Lithium-ion battery") research. In 2016, the company established a 5-year battery research and development partnership at [Dalhousie University](https://en.wikipedia.org/wiki/Dalhousie_University "Dalhousie University") in Nova Scotia, Canada, with lead researcher [Jeff Dahn](https://en.wikipedia.org/wiki/Jeff_Dahn "Jeff Dahn").[186] Tesla acquired Maxwell Technologies for over $200 million[187] – and sold in 2021.[188] It also acquired Hibar Systems.[189][190] Tesla purchased several battery manufacturing patent applications from Springpower International, a small Canadian battery company.[191][192]

#### Lithium Refinement

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=38 "Edit section: Lithium Refinement")]

In 2023 Tesla broke ground on a $375 million lithium refining facility near [Robstown](https://en.wikipedia.org/wiki/Robstown,_Texas "Robstown, Texas"), Texas in the US.[193] The plant's process will eliminate the use of [sulfuric acid](https://en.wikipedia.org/wiki/Sulfuric_acid "Sulfuric acid") in lithium processing, thereby eliminating the associated [sodium sulfate](https://en.wikipedia.org/wiki/Sodium_sulfate "Sodium sulfate") waste product.[193] Local concerns have been raised over the plant's water usage with initial estimates of 400,000 US gal (1,500,000 L; 330,000 imp gal) per day for normal operation rising to a peak usage estimate of 8,000,000 US gal (30,000,000 L; 6,700,000 imp gal) per day.[194]

### Software

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=39 "Edit section: Software")]

Tesla uses [over-the-air updates](https://en.wikipedia.org/wiki/Over-the-air_update "Over-the-air update") to deliver updates to vehicles, adding features or fixing problems.[195] This is enabled by tight integration between a few powerful onboard computers, compared to the way automakers had previously handled technology, by purchasing off-the-shelf electronic components for each subsystem that typically could not interface at the software level.[196]

The system also has allowed Tesla to control which features customers have access to. For example, for ease of assembly all Model 3 vehicles were built with heated rear seats, but only customers who purchased a premium interior could turn them on. However, Tesla has allowed customers who didn't pay for a premium interior to purchase access to the heated rear seats.[197] Tesla uses a similar software lock feature for Enhanced Autopilot and Full-Self Driving features, even though all vehicles are equipped with the computers and cameras necessary to enable those features.[198]

### Motors

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=40 "Edit section: Motors")]

Tesla makes two kinds of electric motors: an [induction motor](https://en.wikipedia.org/wiki/Induction_motor "Induction motor"), and an [internal permanent magnet](https://en.wikipedia.org/wiki/Permanent_magnet_motor "Permanent magnet motor") (IPM) motor with [synchronous reluctance motor](https://en.wikipedia.org/wiki/Reluctance_motor#Synchronous_reluctance "Reluctance motor") (SynRM) characteristics. 

The older design is a [three-phase](https://en.wikipedia.org/wiki/Three-phase_electric_power "Three-phase electric power") four-pole [alternating current](https://en.wikipedia.org/wiki/Alternating_current "Alternating current") induction motor (also called an asynchronous motor) with a copper [rotor](https://en.wikipedia.org/wiki/Rotor_\(electric\) "Rotor \(electric\)") (which inspired the Tesla logo).[199] These motors use [electromagnetic induction](https://en.wikipedia.org/wiki/Electromagnetic_induction "Electromagnetic induction"), by varying magnetic field to produce torque. Induction motors are used as the rear motor in the Model S and Model X, as the front motor in the Model 3 and Model Y and were used in the first-generation Roadster. 

Since the introduction of the Model 3 in 2017, Tesla has also been building IPM-SynRM motors. These motors use an iron [rotor](https://en.wikipedia.org/wiki/Rotor_\(electric\) "Rotor \(electric\)"), with slots cut into the metal where magnets are inserted in the internal core. As an IPM motor, it produces excellent starting torque; however, performance declines at high speeds due to [counter-electromotive forces](https://en.wikipedia.org/wiki/Counter-electromotive_force "Counter-electromotive force"). For high-speed operation, Tesla engineers used iron's reluctance property, which allows it to spin in synchronization with the magnetic field of the [stator](https://en.wikipedia.org/wiki/Stator "Stator") if channels are cut into the core. These channels were also an ideal internal location for the permanent magnets to be mounted.[200][201] The IPM-SynRM motor is currently used as the rear motor in the Model 3 and Model Y, the front motor of 2019-onward versions of the Model S and X, and are expected to be used in the Tesla Semi.[202]

###  North American Charging Standard

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=41 "Edit section: North American Charging Standard")]

Main article: [North American Charging Standard](https://en.wikipedia.org/wiki/North_American_Charging_Standard "North American Charging Standard")

The [North American Charging Standard](https://en.wikipedia.org/wiki/North_American_Charging_Standard "North American Charging Standard") (NACS) is an electric vehicle charging connector system developed by Tesla. It has been used on all North American market Tesla vehicles since 2012 and was opened for use to other manufacturers in 2022. Since then, nearly every other vehicle manufacturer has announced that starting from 2025, their electric vehicles sold in North America will be equipped with the NACS charge port. Several electric vehicles charging network operators and equipment manufacturers have also announced plans to add NACS connectors.[203]

### "Autopilot" and "Full Self-Driving (Supervised)"

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=42 "Edit section: "Autopilot" and "Full Self-Driving \(Supervised\)"")]

This section is an excerpt from [Tesla Autopilot](https://en.wikipedia.org/wiki/Tesla_Autopilot "Tesla Autopilot").[[edit](https://en.wikipedia.org/w/index.php?title=Tesla_Autopilot&action=edit)]

[](https://en.wikipedia.org/wiki/File:Tesla_Autopilot_Engaged_in_Model_X.jpg)Tesla Autopilot in operation, 2017

[Tesla Autopilot](https://en.wikipedia.org/wiki/Tesla_Autopilot "Tesla Autopilot") is an [advanced driver-assistance system](https://en.wikipedia.org/wiki/Advanced_driver-assistance_system "Advanced driver-assistance system") (ADAS) developed by Tesla, Inc. that provides partial vehicle automation, corresponding to [Level 2](https://en.wikipedia.org/wiki/Self-driving_car#Level_2 "Self-driving car") automation as defined by [SAE International](https://en.wikipedia.org/wiki/SAE_International "SAE International"). All Tesla vehicles produced after April 2019 include Autopilot,[204] which features [autosteer](https://en.wikipedia.org/wiki/Lane_centering "Lane centering") and [traffic-aware cruise control](https://en.wikipedia.org/wiki/Adaptive_cruise_control "Adaptive cruise control"). Customers can purchase or subscribe to an optional package called "Full Self-Driving (Supervised)", also known as "FSD", which adds features such as semi-autonomous navigation, response to traffic lights and stop signs, [lane change assistance](https://en.wikipedia.org/wiki/Lane_change_assistance "Lane change assistance"), [self-parking](https://en.wikipedia.org/wiki/Self-parking "Self-parking"), and the ability to summon the car from a parking space. 

Since 2013, Tesla CEO [Elon Musk](https://en.wikipedia.org/wiki/Elon_Musk "Elon Musk") has [repeatedly predicted](https://en.wikipedia.org/wiki/List_of_predictions_for_autonomous_Tesla_vehicles_by_Elon_Musk "List of predictions for autonomous Tesla vehicles by Elon Musk") that the company would achieve fully autonomous driving (SAE [Level 5](https://en.wikipedia.org/wiki/Self-driving_car#Level_5 "Self-driving car")) within one to three years,[205][206] but these goals have not been met. The branding of Full Self-Driving has drawn criticism for potentially misleading consumers. Tesla vehicles currently operate at Level 2 automation, which requires continuous driver supervision and does not constitute "full" [self-driving](https://en.wikipedia.org/wiki/Self-driving_car "Self-driving car") capability. Previously, the Autopilot branding was also criticized for similar reasons, despite the fact that no current [autopilot](https://en.wikipedia.org/wiki/Autopilot "Autopilot") system in aircraft renders them fully autonomous.[207]

Tesla claims that its driver-assistance features improve safety and reduce accidents caused by driver fatigue or inattention.[208][209] However, collisions and fatalities involving Autopilot have attracted scrutiny from media and regulators. Industry experts and safety advocates have raised concerns about the deployment of beta software to the general public, calling the practice risky and potentially irresponsible.[210][211][212][213][214]

### Glass

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=43 "Edit section: Glass")]

In November 2016, the company announced the Tesla Glass technology group. The group produced the roof glass for the Tesla Model 3. It also produces the glass used in the [Tesla Solar Roof](https://en.wikipedia.org/wiki/Tesla_Solar_Roof "Tesla Solar Roof")'s [solar shingles](https://en.wikipedia.org/wiki/Solar_shingle "Solar shingle").[215]

### Robotics

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=44 "Edit section: Robotics")]

In preparation for Model 3 production, Tesla heavily invested in robotics and automation for vehicle assembly, and between 2015 and 2017, the company purchased several companies involved in automation and robotics including Compass Automation,[216] Grohmann Automation,[217] Perbix Machine Company, and Riviera Tool and Die.[218] Musk later said that the robotics slowed production of the vehicles.[50]

Subsequently, Tesla shifted towards using massive casting machines, known as [Giga Presses](https://en.wikipedia.org/wiki/Giga_Press "Giga Press"). These machines streamline production by creating large, single-piece underbodies, leading to reductions in production time, labor costs, factory footprint, and the number of welding robots.[219][220] Critics note that reducing the number of components makes the vehicles harder or more expensive to repair after an accident.[221]

The company has been developing a [humanoid robot](https://en.wikipedia.org/wiki/Humanoid_robot "Humanoid robot") called [Optimus](https://en.wikipedia.org/wiki/Optimus_\(robot\) "Optimus \(robot\)") since 2022. Musk has stated that Optimus leverages the same core software powering Tesla's Full Self-Driving technology and has suggested that it could be used within Tesla's factories to mitigate labor shortages through the automation of repetitive tasks.[222]

## Facilities

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=45 "Edit section: Facilities")]

See also: [List of Tesla factories](https://en.wikipedia.org/wiki/List_of_Tesla_factories "List of Tesla factories")

The company operates seven large factories and about a dozen smaller factories around the world. As of December 2024[[update]](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit), the company also operates more than 1,350 retail stores, galleries, service, delivery and body shop locations globally.[223]

Primary facilities operated by Tesla  Opened  | Name  | City  | Country  | Employees  | Products  | Ref.  
---|---|---|---|---|---|---  
2010  | [Tesla Fremont Factory](https://en.wikipedia.org/wiki/Tesla_Fremont_Factory "Tesla Fremont Factory") | [Fremont, California](https://en.wikipedia.org/wiki/Fremont,_California "Fremont, California") | United States  | 22,000  | [Model S](https://en.wikipedia.org/wiki/Tesla_Model_S "Tesla Model S"), [Model X](https://en.wikipedia.org/wiki/Tesla_Model_X "Tesla Model X"), [Model 3](https://en.wikipedia.org/wiki/Tesla_Model_3 "Tesla Model 3"), [Model Y](https://en.wikipedia.org/wiki/Tesla_Model_Y "Tesla Model Y") | [31][224][225]  
2016  | [Gigafactory Nevada](https://en.wikipedia.org/wiki/Gigafactory_Nevada "Gigafactory Nevada") | [Storey County, Nevada](https://en.wikipedia.org/wiki/Storey_County,_Nevada "Storey County, Nevada") | United States  | 7,000  | Batteries, [Powerwall](https://en.wikipedia.org/wiki/Tesla_Powerwall "Tesla Powerwall"), [Semi](https://en.wikipedia.org/wiki/Tesla_Semi "Tesla Semi") | [226][227][228]  
2017  | [Gigafactory New York](https://en.wikipedia.org/wiki/Gigafactory_New_York "Gigafactory New York") | [Buffalo, New York](https://en.wikipedia.org/wiki/Buffalo,_New_York "Buffalo, New York") | United States  | 1,500  | [Solar Roof](https://en.wikipedia.org/wiki/Tesla_Solar_Roof "Tesla Solar Roof"), [Supercharger](https://en.wikipedia.org/wiki/Tesla_Supercharger "Tesla Supercharger") | [229][230]  
2019  | [Gigafactory Shanghai](https://en.wikipedia.org/wiki/Gigafactory_Shanghai "Gigafactory Shanghai") | [Shanghai](https://en.wikipedia.org/wiki/Shanghai "Shanghai") | China  | 20,000  | Model 3, Model Y, Supercharger  | [231][232]  
2022  | [Gigafactory Berlin](https://en.wikipedia.org/wiki/Gigafactory_Berlin-Brandenburg "Gigafactory Berlin-Brandenburg") | [Grünheide](https://en.wikipedia.org/wiki/Gr%C3%BCnheide_\(Mark\) "Grünheide \(Mark\)") | Germany  | 10,000  | Model Y  | [233][234][235]  
2022  | [Gigafactory Texas](https://en.wikipedia.org/wiki/Gigafactory_Texas "Gigafactory Texas") | [Austin, Texas](https://en.wikipedia.org/wiki/Austin,_Texas "Austin, Texas") | United States  | 12,000  | Model Y, [Cybertruck](https://en.wikipedia.org/wiki/Tesla_Cybertruck "Tesla Cybertruck") | [236][237][238]  
  
### North America

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=46 "Edit section: North America")]

Main articles: [Tesla Fremont Factory](https://en.wikipedia.org/wiki/Tesla_Fremont_Factory "Tesla Fremont Factory"), [Gigafactory Nevada](https://en.wikipedia.org/wiki/Gigafactory_Nevada "Gigafactory Nevada"), [Gigafactory New York](https://en.wikipedia.org/wiki/Gigafactory_New_York "Gigafactory New York"), [Gigafactory Texas](https://en.wikipedia.org/wiki/Gigafactory_Texas "Gigafactory Texas"), and [Gigafactory Mexico](https://en.wikipedia.org/wiki/Gigafactory_Mexico "Gigafactory Mexico")

[](https://en.wikipedia.org/wiki/File:New_Teslas_at_the_factory.jpg)New [Tesla Model S](https://en.wikipedia.org/wiki/Tesla_Model_S "Tesla Model S") cars at the [Tesla Fremont Factory](https://en.wikipedia.org/wiki/Tesla_Fremont_Factory "Tesla Fremont Factory") in 2012

Tesla was founded in [San Carlos, California](https://en.wikipedia.org/wiki/San_Carlos,_California "San Carlos, California"), in 2003.[239] In 2008, the company opened its first production facility at a former [Chevrolet](https://en.wikipedia.org/wiki/Chevrolet "Chevrolet") dealership in [Menlo Park, California](https://en.wikipedia.org/wiki/Menlo_Park,_California "Menlo Park, California"). The original roadster was assembled inside the service bays until 2012 and used the company showroom as a retail store.[240] Another retail store was opened in Los Angeles the same year.[241] In 2010, Tesla moved its corporate headquarters and opened a powertrain development facility in [Palo Alto](https://en.wikipedia.org/wiki/Palo_Alto "Palo Alto").[242]

Tesla's first major assembly plant occupies the former [NUMMI](https://en.wikipedia.org/wiki/NUMMI "NUMMI") plant in Fremont, California, known as the [Tesla Fremont Factory](https://en.wikipedia.org/wiki/Tesla_Fremont_Factory "Tesla Fremont Factory"). The factory was originally opened by [General Motors](https://en.wikipedia.org/wiki/General_Motors "General Motors") in 1962, and then operated by NUMMI, a joint venture of GM and [Toyota](https://en.wikipedia.org/wiki/Toyota "Toyota") from 1984.[243] The joint venture ended when [GM entered bankruptcy](https://en.wikipedia.org/wiki/General_Motors_Chapter_11_reorganization "General Motors Chapter 11 reorganization") in 2009. In 2010, Toyota agreed to sell the plant to Tesla at a significant discount.[31]

[](https://en.wikipedia.org/wiki/File:Tesla_Gigafactory_1_-_December_2019.jpg)[Gigafactory Nevada](https://en.wikipedia.org/wiki/Gigafactory_Nevada "Gigafactory Nevada") in 2019

Tesla's first purpose-built facility was opened in Nevada in 2016. [Gigafactory Nevada](https://en.wikipedia.org/wiki/Gigafactory_Nevada "Gigafactory Nevada") produces the Powerwall,[226] battery cells in partnership with Panasonic,[244] Model 3 drivetrains,[245] and the Tesla Semi.[246] The factory received substantial subsidies ([abatements](https://en.wikipedia.org/wiki/Tax_abatement "Tax abatement") and [credits](https://en.wikipedia.org/wiki/Tax_credit "Tax credit")) from the local and state government, that, in exchange for opening in their jurisdiction, allowed Tesla to operate essentially tax-free for 10 years,[247] later extended to 20 years in exchange for expanding the factory to add a production line for the Tesla Semi and add additional battery manufacturing capacity.[246]

As part of the acquisition of SolarCity in 2016, Tesla gained control of [Gigafactory New York](https://en.wikipedia.org/wiki/Gigafactory_New_York "Gigafactory New York") in Buffalo on the site of a former [Republic Steel](https://en.wikipedia.org/wiki/Republic_Steel "Republic Steel") plant. The state of New York spent cash to build and equip the factory through the [Buffalo Billion](https://en.wikipedia.org/wiki/Buffalo_Billion "Buffalo Billion") program.[248][249] In 2017, the factory started production of the [Tesla Solar Roof](https://en.wikipedia.org/wiki/Tesla_Solar_Roof "Tesla Solar Roof"),[229] but faced multiple production challenges. Since 2020, Tesla has also assembled Superchargers in New York. The plant has been criticized for offering little economic benefit for the state funding.[250]

In 2018, Tesla assembled [tension fabric buildings](https://en.wikipedia.org/wiki/Tension_fabric_building "Tension fabric building") at the [Fremont](https://en.wikipedia.org/wiki/Fremont,_California "Fremont, California") plant to meet production goals of 5,000 cars produced a month. The structure was assembled in two weeks and measured 53 feet high, 150 feet wide, and 900 feet long.[251]

[](https://en.wikipedia.org/wiki/File:Gigafactory_Texas_Building_1_June_2022.jpg)[Gigafactory Texas](https://en.wikipedia.org/wiki/Gigafactory_Texas "Gigafactory Texas") in 2022

On July 23, 2020, Tesla picked Austin, Texas, as the site of its fifth Gigafactory, since then known as [Gigafactory Texas](https://en.wikipedia.org/wiki/Gigafactory_Texas "Gigafactory Texas").[252] Giga Texas is the only factory that produces the Tesla Cybertruck and produces Model Y cars for the Eastern United States. On December 1, 2021, Tesla announced it relocated its legal headquarters from Palo Alto to the Gigafactory Texas site in Austin.[253] However, Tesla has retained the Palo Alto building. On April 7, 2022, Tesla celebrated the opening of Gigafactory Texas in a public event.[71]

Tesla acquired a former [JC Penney](https://en.wikipedia.org/wiki/JC_Penney "JC Penney") distribution center near [Lathrop, California](https://en.wikipedia.org/wiki/Lathrop,_California "Lathrop, California"), in 2021 to build the "Megafactory" to manufacture the [Megapack](https://en.wikipedia.org/wiki/Tesla_Megapack "Tesla Megapack"), the company's large-scale energy storage product.[254][255] The location opened in 2022. 

Tesla announced in February it would open a new global engineering headquarters in Palo Alto, moving into a corporate campus once owned by [Hewlett Packard](https://en.wikipedia.org/wiki/HP_Inc. "HP Inc."), located a couple of miles from Tesla's former headquarters building.[256]

Tesla has announced plans to open a [Gigafactory Mexico](https://en.wikipedia.org/wiki/Gigafactory_Mexico "Gigafactory Mexico"), the company's sixth Gigafactory, near [Monterrey](https://en.wikipedia.org/wiki/Monterrey "Monterrey"), Mexico. However, as of July 2024[[update]](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit), the company had placed construction on hold until after the [2024 United States presidential election](https://en.wikipedia.org/wiki/2024_United_States_presidential_election "2024 United States presidential election") because former President Trump has pledged to add tariffs on cars made in Mexico.[257][_[needs update](https://en.wikipedia.org/wiki/Wikipedia:Manual_of_Style/Dates_and_numbers#Chronological_items "Wikipedia:Manual of Style/Dates and numbers")_]

### Europe

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=47 "Edit section: Europe")]

Main articles: [Tesla facilities in Tilburg](https://en.wikipedia.org/wiki/Tesla_facilities_in_Tilburg "Tesla facilities in Tilburg") and [Gigafactory Berlin-Brandenburg](https://en.wikipedia.org/wiki/Gigafactory_Berlin-Brandenburg "Gigafactory Berlin-Brandenburg")

[](https://en.wikipedia.org/wiki/File:Tesla_Gigafactory_4_DJI_20230728123435.JPG)Gigafactory Berlin in July 2023

Tesla opened its first European store in June 2009 in [London](https://en.wikipedia.org/wiki/London "London").[258] Tesla's European headquarters are in [the Netherlands](https://en.wikipedia.org/wiki/The_Netherlands "The Netherlands"),[259] part of a group of [Tesla facilities in Tilburg](https://en.wikipedia.org/wiki/Tesla_facilities_in_Tilburg "Tesla facilities in Tilburg"), including the company's European Distribution Centre.[260]

In late 2016, Tesla acquired German engineering firm [Grohmann Engineering](https://en.wikipedia.org/wiki/Grohmann_Engineering "Grohmann Engineering") as a new division dedicated to helping Tesla increase the automation and effectiveness of its manufacturing process.[261] After winding down existing contracts with other manufacturers, the renamed [Tesla Automation](https://en.wikipedia.org/wiki/Tesla_Automation "Tesla Automation") now works exclusively on Tesla projects.[262]

Tesla announced its plans to build a car and battery factory in Europe in 2016.[263] Several countries campaigned to be the host,[264] and eventually Germany was chosen in November 2019.[265] On March 22, 2022, Tesla's first European Gigafactory named [Gigafactory Berlin](https://en.wikipedia.org/wiki/Gigafactory_Berlin-Brandenburg "Gigafactory Berlin-Brandenburg")[266][267] opened with planned capacity to produce 500,000 electric vehicles annually as well as batteries for the cars.[267]

### Asia

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=48 "Edit section: Asia")]

Main article: [Gigafactory Shanghai](https://en.wikipedia.org/wiki/Gigafactory_Shanghai "Gigafactory Shanghai")

[](https://en.wikipedia.org/wiki/File:Tesla_Tokyo_2011.JPG)Tesla store in Tokyo, the first in Asia[268]

Tesla opened its first showroom in [Asia](https://en.wikipedia.org/wiki/Asia "Asia") in [Tokyo](https://en.wikipedia.org/wiki/Tokyo "Tokyo"), Japan, in October 2010.[269]

In July 2018, Tesla signed an agreement with Chinese authorities to build a factory in [Shanghai](https://en.wikipedia.org/wiki/Shanghai "Shanghai"), China, which was Tesla's first Gigafactory outside the United States.[270] The factory building was finished in August 2019, and the initial [Tesla Model 3s](https://en.wikipedia.org/wiki/Tesla_Model_3 "Tesla Model 3") were in production from Gigafactory Shanghai in October 2019.[231] In 2024, China accounted for 21% of Tesla sales revenue, and was the second-largest market for Tesla after the United States, which accounted for 48% of its sales.[271] Tesla also sold 37% of its cars in China in 2024.[272]

Tesla expressed interest in 2023 in expanding to India and perhaps building a future Gigafactory in the country.[273] The company established a legal presence in the nation in 2021 and plans to open an office in [Pune](https://en.wikipedia.org/wiki/Pune "Pune") starting in October 2023.[274]

## Partners

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=49 "Edit section: Partners")]

### Panasonic

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=50 "Edit section: Panasonic")]

[](https://en.wikipedia.org/wiki/File:Tesla_Panasonic.jpg)[Panasonic](https://en.wikipedia.org/wiki/Panasonic "Panasonic") Energy president Naoto Noguchi presents Tesla executive [J. B. Straubel](https://en.wikipedia.org/wiki/J._B._Straubel "J. B. Straubel") with [lithium-ion cells](https://en.wikipedia.org/wiki/Lithium-ion_cells "Lithium-ion cells")

In January 2010, Tesla and battery cell maker [Panasonic](https://en.wikipedia.org/wiki/Panasonic "Panasonic") announced that they would together develop [nickel-based](https://en.wikipedia.org/wiki/Nickel#Applications "Nickel") [lithium-ion battery](https://en.wikipedia.org/wiki/Lithium-ion_battery "Lithium-ion battery") cells for electric vehicles.[275] Beginning in 2010, Panasonic invested $30 million for a multi-year collaboration on new battery cells designed specifically for electric vehicles.[276] In July 2014, Panasonic reached a basic agreement with Tesla to participate in battery production at Giga Nevada.[277] Tesla and Panasonic also collaborated on the manufacturing and production of photovoltaic (PV) cells and modules at the Giga New York factory in Buffalo, New York.[229] The partnership started in mid-2017 and ended in early 2020, before Panasonic exited the solar business entirely in January 2021.[278][279]

In March 2021, the outgoing CEO of Panasonic stated that the company plans to reduce its reliance on Tesla as their battery partnership evolves.[280]

### Other current partners

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=51 "Edit section: Other current partners")]

Tesla has long-term contracts in place for [lithium](https://en.wikipedia.org/wiki/Lithium "Lithium") supply. In September 2020, Tesla signed a sales agreement with [Piedmont Lithium](https://en.wikipedia.org/wiki/Piedmont_Lithium "Piedmont Lithium") to buy high-purity lithium ore for up to ten years,[281] specifically to supply "[spodumene](https://en.wikipedia.org/wiki/Spodumene "Spodumene") concentrate from Piedmont's North Carolina [mineral deposit](https://en.wikipedia.org/wiki/Mineral_deposit "Mineral deposit")".[282] In 2022, Tesla contracted for 110,000 tonnes of [spodumene](https://en.wikipedia.org/wiki/Spodumene "Spodumene") concentrate over four years from the [Core Lithium](https://en.wikipedia.org/wiki/Core_Lithium "Core Lithium")'s lithium mine in the [Northern Territory](https://en.wikipedia.org/wiki/Northern_Territory "Northern Territory") of Australia.[283]

Tesla also has a range of minor partnerships, for instance working with [Airbnb](https://en.wikipedia.org/wiki/Airbnb "Airbnb") and hotel chains to install destination chargers at selected locations.[284]

### Former partners

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=52 "Edit section: Former partners")]

#### Daimler

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=53 "Edit section: Daimler")]

[](https://en.wikipedia.org/wiki/File:2018_Mercedes-Benz_B-Class_Electric_Drive_Electric_Art_Premium_Front.jpg)The [Mercedes-Benz B-Class Electric Drive](https://en.wikipedia.org/wiki/Mercedes-Benz_B-Class_Electric_Drive "Mercedes-Benz B-Class Electric Drive") used a Tesla-supplied battery pack.[285]

[Daimler](https://en.wikipedia.org/wiki/Daimler_AG "Daimler AG") and Tesla began working together in late 2007. On May 19, 2009, Daimler bought a stake of less than 10% in Tesla for a reported $50 million.[286][287] As part of the collaboration, Herbert Kohler, vice-president of E-Drive and Future Mobility at Daimler, took a Tesla board seat.[288] On July 13, 2009, Daimler sold 40% of its acquisition to Aabar, an investment company controlled by the [International Petroleum Investment Company](https://en.wikipedia.org/wiki/International_Petroleum_Investment_Company "International Petroleum Investment Company") owned by the government of [Abu Dhabi](https://en.wikipedia.org/wiki/Abu_Dhabi "Abu Dhabi").[289] In October 2014, Daimler sold its remaining holdings for a reported $780 million.[290]

Tesla supplied battery packs for [Freightliner Trucks](https://en.wikipedia.org/wiki/Freightliner_Trucks "Freightliner Trucks") in 2010.[291][292] The company also built electric-powertrain components for the [Mercedes-Benz A-Class E-Cell](https://en.wikipedia.org/wiki/Mercedes-Benz_A-Class_E-Cell "Mercedes-Benz A-Class E-Cell"), with 500 cars planned to be built for trial in Europe beginning in September 2011.[293][294] Tesla produced and co-developed the [Mercedes-Benz B250e](https://en.wikipedia.org/wiki/Mercedes-Benz_B-Class_Electric_Drive "Mercedes-Benz B-Class Electric Drive")'s powertrain, which ended production in 2017.[295] The electric motor was rated 134 hp (100 kW) and 230 pound force-feet (310 N⋅m), with a 36 kWh (130 MJ) battery. The vehicle had a driving range of 200 km (124 mi) with a top speed of 150 km/h (93 mph).[296] Daimler division [Smart](https://en.wikipedia.org/wiki/Smart_\(marque\) "Smart \(marque\)") produced the [Smart ED2](https://en.wikipedia.org/wiki/Smart_electric_drive "Smart electric drive") cars from 2009 to 2012 which had a 14-kilowatt-hour (50 MJ) [lithium-ion battery](https://en.wikipedia.org/wiki/Lithium-ion_battery "Lithium-ion battery") from Tesla.[297][298]

#### Toyota

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=54 "Edit section: Toyota")]

[](https://en.wikipedia.org/wiki/File:Toyota_RAV4_EV_WAS_2012_0791.JPG)[Toyota RAV4 EV](https://en.wikipedia.org/wiki/Toyota_RAV4_EV "Toyota RAV4 EV"), which used a Tesla-supplied battery and [powertrain](https://en.wikipedia.org/wiki/Powertrain "Powertrain") components

In May 2010, Tesla and [Toyota](https://en.wikipedia.org/wiki/Toyota "Toyota") announced a deal in which Tesla purchased the former [NUMMI](https://en.wikipedia.org/wiki/NUMMI "NUMMI") factory from Toyota for $42 million, Toyota purchased $50 million in Tesla stock, and the two companies collaborated on an electric vehicle.[31]

In July 2010, the companies announced they would work together on a second generation [Toyota RAV4 EV](https://en.wikipedia.org/wiki/Toyota_RAV4_EV "Toyota RAV4 EV").[299] The vehicle was unveiled at the October 2010 [Los Angeles Auto Show](https://en.wikipedia.org/wiki/Los_Angeles_Auto_Show "Los Angeles Auto Show") and 35 pilot vehicles were built for a demonstration and evaluation program that ran through 2011. Tesla supplied the lithium metal-oxide battery and other powertrain components[300][301] based on components from the Roadster.[302]

The production version was unveiled in August 2012, using battery pack, electronics and powertrain components from the Tesla Model S sedan (also launched in 2012).[303] The RAV4 EV had a limited production run which resulted in just under 3,000 vehicles being produced, before it was discontinued in 2014.[304][305]

According to Bloomberg News, the partnership between Tesla and Toyota was "marred by clashes between engineers".[306] Toyota engineers rejected designs that Tesla had proposed for an enclosure to protect the RAV4 EV's battery pack. Toyota took over responsibility for the enclosure's design and strengthened it. In 2014, Tesla ended up adding a titanium plate to protect the Model S sedan's battery after some debris-related crashes led to cars catching fire.[306][176] On June 5, 2017, Toyota announced that it had sold all of its shares in Tesla and halted the partnership.[307][308]

#### Mobileye

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=55 "Edit section: Mobileye")]

Initial versions of Autopilot were developed in partnership with [Mobileye](https://en.wikipedia.org/wiki/Mobileye "Mobileye") beginning in 2014.[309] Mobileye ended the partnership on July 26, 2016, citing "disagreements about how the technology was deployed".[310]

## Lawsuits and controversies

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=56 "Edit section: Lawsuits and controversies")]

Main articles: [List of lawsuits involving Tesla, Inc.](https://en.wikipedia.org/wiki/List_of_lawsuits_involving_Tesla,_Inc. "List of lawsuits involving Tesla, Inc.") and [Criticism of Tesla, Inc.](https://en.wikipedia.org/wiki/Criticism_of_Tesla,_Inc. "Criticism of Tesla, Inc.")

### Failed lawsuit alleging unfair review

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=57 "Edit section: Failed lawsuit alleging unfair review")]

In 2008 Tesla provided two Roadsters to British entertainment car-testing television programme _[Top Gear](https://en.wikipedia.org/wiki/Top_Gear_\(2002_TV_series\) "Top Gear \(2002 TV series\)")_ for testing, apparently confusing it with another programme, and suggesting "Matt and Alex could even take the Tesla for a spin and test it out, reaffirming its virtues?" Top Gear tested the car as a performance sports car—in a format designed for humour and entertainment, not an unbiased review about its intended purpose—and made some criticisms of its use as such, though full of praise for its performance and handling on the track. Tesla were so incensed about what it considered unfair criticism (for example, the test affirmed that _in the performance tests they ran_ the battery charge would have lasted for 55 miles, a figure that actually came from Tesla. Tesla said that the figure was untrue and libellous as the car's mileage _in normal use_ was claimed to be 211) that they sued, and lost.[311][312]

In 2025, when Tesla was the subject of protests and vandalism in response to Elon Musk's political activities with the Trump government, former Top Gear presenter [Jeremy Clarkson](https://en.wikipedia.org/wiki/Jeremy_Clarkson "Jeremy Clarkson") published gloating remarks, saying that the vandalism was "not funny. But also, it's kinda hilarious. Especially if you're me."[313]

### Sexual harassment

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=58 "Edit section: Sexual harassment")]

In 2021, seven women came forward with claims of having faced sexual harassment and discrimination while working at Tesla's Fremont factory.[314] They accused the company of facilitating a culture of rampant sexual harassment. The women said they were consistently subjected to catcalling, unwanted advances, unwanted touching, and discrimination while at work. "I was so tired of the unwanted attention and the males gawking at me I proceeded to create barriers around me just so I could get some relief," Brooks told _[The Washington Post](https://en.wikipedia.org/wiki/The_Washington_Post "The Washington Post")_. "That was something I felt necessary just so I can do my job." Stories range from intimate groping to being called out to the parking lot for sex.[315]

Women feared calling Human Resources for help, as their supervisors were often participants.[316] Musk himself is not indicted, but most of the women pressing charges believe their abuse is connected to the behavior of CEO Elon Musk. They cite his crude remarks about women's bodies, wisecracks about starting a university that abbreviated to "T.IT.S", and his generally dismissive attitude towards reporting sexual harassment.[317] "What we're addressing for each of the lawsuits is just a shocking pattern of rampant harassment that exists at Tesla," said attorney David A. Lowe.[316] In 2017, another woman had accused Tesla of very similar behavior and was subsequently fired. In a statement to the Guardian, Tesla confirmed the company had terminated her employment, saying it had thoroughly investigated the employee's allegations with the help of "a neutral, third-party expert" and concluded her complaints were unmerited.[318]

In May 2022, a California judge ruled that the sexual harassment lawsuit could move to court, rejecting Tesla's request for closed-door arbitration.[319]

### Labor disputes

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=59 "Edit section: Labor disputes")]

Main article: [Tesla and unions](https://en.wikipedia.org/wiki/Tesla_and_unions "Tesla and unions")

This section is an excerpt from [Elon Musk and trade unions § Tesla](https://en.wikipedia.org/wiki/Elon_Musk_and_trade_unions#Tesla "Elon Musk and trade unions").[[edit](https://en.wikipedia.org/w/index.php?title=Elon_Musk_and_trade_unions&action=edit)]

[](https://en.wikipedia.org/wiki/File:Tesla_workers_united.jpg)[Valentine's Day](https://en.wikipedia.org/wiki/Valentine%27s_Day "Valentine's Day") union drive at [Giga New York](https://en.wikipedia.org/wiki/Gigafactory_New_York "Gigafactory New York") in 2023 [Tesla has had labor disputes](https://en.wikipedia.org/wiki/Tesla_and_unions "Tesla and unions") in the United States, Germany and Sweden, including an ongoing [strike](https://en.wikipedia.org/wiki/Strike_action "Strike action") in Sweden. Tesla, Inc., an American [electric car](https://en.wikipedia.org/wiki/Electric_car "Electric car") and [solar panel](https://en.wikipedia.org/wiki/Solar_panel "Solar panel") manufacturer, has more than 140,000 workers employed across its global operations as of January 2024.[320] Tesla CEO [Elon Musk](https://en.wikipedia.org/wiki/Elon_Musk "Elon Musk") has expressed his [opposition to unions](https://en.wikipedia.org/wiki/Union_busting "Union busting") on [Twitter](https://en.wikipedia.org/wiki/Twitter "Twitter") (now called X). The [National Labor Relations Board](https://en.wikipedia.org/wiki/National_Labor_Relations_Board "National Labor Relations Board") held that one [tweet](https://en.wikipedia.org/wiki/Tweet_\(social_media\) "Tweet \(social media\)") was unlawful, but was overturned by a [federal appeals court](https://en.wikipedia.org/wiki/United_States_Court_of_Appeals_for_the_Fifth_Circuit "United States Court of Appeals for the Fifth Circuit").[321] All [unionization](https://en.wikipedia.org/wiki/Unionization "Unionization") efforts at the [Tesla Fremont Factory](https://en.wikipedia.org/wiki/Tesla_Fremont_Factory "Tesla Fremont Factory") and [Gigafactory New York](https://en.wikipedia.org/wiki/Gigafactory_New_York "Gigafactory New York") in the United States have been unsuccessful.[322] In Germany, [Gigafactory Berlin-Brandenburg](https://en.wikipedia.org/wiki/Gigafactory_Berlin-Brandenburg "Gigafactory Berlin-Brandenburg") and [Tesla Automation](https://en.wikipedia.org/wiki/Tesla_Automation "Tesla Automation") have elected [works councils](https://en.wikipedia.org/wiki/Works_council "Works council"), but they have not signed [collective bargaining agreements](https://en.wikipedia.org/wiki/Collective_agreement "Collective agreement") with the German trade union [IG Metall](https://en.wikipedia.org/wiki/IG_Metall "IG Metall").[323][324] The Gigafactory Berlin-Brandenburg works council is divided into pro-union and anti-union factions.[325] In Sweden, mechanics who are members of the trade union [IF Metall](https://en.wikipedia.org/wiki/IF_Metall "IF Metall") have been on strike since October 27, 2023, making it the longest strike in Sweden since 1938.[326] The strike has since spread, with other [Swedish, Danish and Norwegian unions](https://en.wikipedia.org/wiki/Council_of_Nordic_Trade_Unions "Council of Nordic Trade Unions") calling for [solidarity strikes](https://en.wikipedia.org/wiki/Solidarity_action "Solidarity action").[327]

### Accidents, repairs and safety violations

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=60 "Edit section: Accidents, repairs and safety violations")]

In June 2016, the [National Highway Traffic Safety Administration](https://en.wikipedia.org/wiki/National_Highway_Traffic_Safety_Administration "National Highway Traffic Safety Administration") (NHTSA) took issue with Tesla's use of [nondisclosure agreements](https://en.wikipedia.org/wiki/Non-disclosure_agreement "Non-disclosure agreement") (NDAs) regarding customer repairs[328] and, in October 2021, the NHTSA formally asked Tesla to explain its NDA policy regarding customers invited into the FSD Beta.[329] Tesla has used NDAs on multiple occasions with both employees[330] and customers[331] to allegedly prevent possible negative coverage.[332][333]

From 2014 to 2018, Tesla's Fremont Factory had three times as many [Occupational Safety and Health Administration](https://en.wikipedia.org/wiki/Occupational_Safety_and_Health_Administration "Occupational Safety and Health Administration") (OSHA) violations as the ten largest U.S. auto plants combined.[334] An investigation by the [Reveal](https://en.wikipedia.org/wiki/Reveal_\(podcast\) "Reveal \(podcast\)") podcast alleged that Tesla "failed to report some of its serious injuries on legally mandated reports" to downplay the extent of injuries.[335]

In January 2019, former Tesla security manager Sean Gouthro filed a whistleblower complaint alleging that the company had hacked employees' phones and spied on them, while also failing to report illegal activities to the authorities and shareholders.[336][337][338] Several legal cases have revolved around alleged whistleblower retaliation by Tesla. These include the dismissal of Tesla safety official Carlos Ramirez[339][340] and Tesla security employee Karl Hansen.[341] In 2020, the court ordered Hansen's case to [arbitration](https://en.wikipedia.org/wiki/Arbitration_in_the_United_States "Arbitration in the United States").[342] In June 2022, the arbitrator filed an [unopposed motion](https://en.wikipedia.org/wiki/Motion_\(legal\) "Motion \(legal\)") with the court stating Hansen "has failed to establish the claims… Accordingly, his claims are denied, and he shall take nothing".[343]

The California Civil Rights Department filed a suit in 2022 alleging "a pattern of racial harassment and bias" at the Tesla Fremont factory. As of April 2023,[[update]](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit) the department is also conducting a probe of the factory based on a 2021 complaint and claims that Tesla has been obstructing the investigation.[344]

According to _Jalopnik_ , a 2024 study by _iSeeCars_ found that Tesla had the highest fatal crash rate of any automaker in the United States.[345]

A 2025 lawsuit in California alleged that Tesla odometers were falsely exaggerating their readings to prematurely void their warranty.[346]

### Fraud allegations

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=61 "Edit section: Fraud allegations")]

There have been numerous concerns about Tesla's financial reporting. In 2013, _Bloomberg News_ questioned whether Tesla's financial reporting violated [Generally Accepted Accounting Principles](https://en.wikipedia.org/wiki/Generally_Accepted_Accounting_Principles_\(United_States\) "Generally Accepted Accounting Principles \(United States\)") (GAAP) reporting standards.[347] _[Fortune](https://en.wikipedia.org/wiki/Fortune_\(magazine\) "Fortune \(magazine\)")_ accused Tesla in 2016 of using [creative accounting](https://en.wikipedia.org/wiki/Creative_accounting "Creative accounting") to show positive cash flow and quarterly profits.[348] In 2018, analysts expressed concerns over Tesla's accounts receivable balance.[349] In September 2019, the SEC questioned Tesla CFO [Zach Kirkhorn](https://en.wikipedia.org/wiki/Zach_Kirkhorn "Zach Kirkhorn") about Tesla's warranty reserves and lease accounting.[350] In a letter to his clients, hedge fund manager [David Einhorn](https://en.wikipedia.org/wiki/David_Einhorn_\(hedge_fund_manager\) "David Einhorn \(hedge fund manager\)"), whose firm suffered losses from its short position against Tesla that quarter, accused Elon Musk in November 2019 of "significant fraud",[351][352] and publicly questioned Tesla's accounting practices, telling Musk that he was "beginning to wonder whether your accounts receivable exist."[353]

From 2012 to 2014, Tesla earned more than $295 million in [Zero Emission Vehicle](https://en.wikipedia.org/wiki/Zero-emissions_vehicle "Zero-emissions vehicle") credits for a battery-swapping technology that was never made available to customers.[354] Staff at [California Air Resources Board](https://en.wikipedia.org/wiki/California_Air_Resources_Board "California Air Resources Board") were concerned that Tesla was "gaming" the battery swap subsidies and in 2013 recommended eliminating the credits.[355]

A consolidated shareholders lawsuit alleges that Musk knew SolarCity was going broke before the acquisition, that he and the Tesla board overpaid for SolarCity, ignored their conflicts of interest and breached their fiduciary duties in connection with the deal, and failed to disclose "troubling facts" essential to an analysis of the proposed acquisition.[356] The members of the board settled in 2020, leaving Musk as the only defendant.[357] In April 2022, the [Delaware Court of Chancery](https://en.wikipedia.org/wiki/Delaware_Court_of_Chancery "Delaware Court of Chancery") ruled in favor of Musk,[358][359] and its ruling was upheld by the [Delaware Supreme Court](https://en.wikipedia.org/wiki/Delaware_Supreme_Court "Delaware Supreme Court") in June 2023.[360]

In August 2018, Elon Musk tweeted, "Am considering taking Tesla private at $420. Funding secured."[361] The tweet caused the stock to initially rise, but then drop when it was revealed to be false.[362][363][364] Musk settled fraud charges with the US [Securities and Exchange Commission](https://en.wikipedia.org/wiki/Securities_and_Exchange_Commission "Securities and Exchange Commission") (SEC) over his false statements in September 2018. According to the terms of the settlement, Musk agreed to have his tweets reviewed by Tesla's in-house counsel, he was removed from his chairman role at Tesla temporarily, and two new independent directors were appointed to the company's board.[365] Tesla and Musk also paid civil penalties of $20 million each.[365] A civil [class-action](https://en.wikipedia.org/wiki/Class_action "Class action") shareholder lawsuit over Musk's statements and other derivative lawsuits were also filed against Musk and the members of Tesla's board of directors, as then constituted, regarding claims and actions made that were associated with potentially going private.[366][367] In February 2023, a California jury unanimously found Musk and Tesla not liable in the class-action lawsuit.[368]

In September 2018, Tesla disclosed that it was under investigation by the US [Federal Bureau of Investigation](https://en.wikipedia.org/wiki/Federal_Bureau_of_Investigation "Federal Bureau of Investigation") (FBI) regarding its Model 3 production figures.[369] Authorities were investigating whether the company misled investors and made projections about its Model 3 production that it knew would be impossible to meet.[369] A stockholder class action lawsuit against Tesla related to Model 3 production numbers (unrelated to the FBI investigation) was dismissed in March 2019.[370][371][372]

In May 2024, Reuters reported that US federal prosecutors were investigating whether the company committed securities or wire fraud by "misleading investors and consumers" about Autopilot and Full Self-Driving.[373]

### Tesla US dealership disputes

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=62 "Edit section: Tesla US dealership disputes")]

See also: [Tesla US dealership disputes](https://en.wikipedia.org/wiki/Tesla_US_dealership_disputes "Tesla US dealership disputes")

Unlike other automakers, Tesla does not rely on franchised [auto dealerships](https://en.wikipedia.org/wiki/Auto_dealership "Auto dealership") to sell vehicles and instead directly sells vehicles through its website and a network of company-owned stores. In some areas, Tesla operates locations called "galleries" which "educate and inform customers about our products, but such locations do not actually transact in the sale of vehicles."[158] This is because some jurisdictions, particularly in the United States, prohibit auto manufacturers from directly selling vehicles to consumers. Dealership associations have filed lawsuits to prevent direct sales. These associations argued that the franchise system protects consumers by encouraging dealers to compete, lowering the price a customer pays. They also claimed that direct sales would allow manufacturers to undersell their dealers.[161] The United States [Federal Trade Commission](https://en.wikipedia.org/wiki/Federal_Trade_Commission "Federal Trade Commission") ultimately disproved the associations' claims and recommended allowing direct manufacturer sale, which they concluded would save consumers 8% in average vehicle price.[374][375][376]

Tesla has also lobbied state governments for the right to directly sell cars.[377] The company has argued that directly operating stores improves consumer education about electric vehicles,[158] because dealerships would sell both Tesla and gas-powered vehicles. Doing this, according to the company, would then set up a [conflict of interest](https://en.wikipedia.org/wiki/Conflict_of_interest "Conflict of interest") for the dealers since properly advertising the benefits of an electric car would disparage the gas-powered vehicles, creating a disincentive to dealership EV sales.[161] Musk himself further contended that dealers would have a disincentive to sell electric vehicles because they require less maintenance and therefore would reduce after-sales service revenue, a large profit center for most dealerships.[129]

### Intellectual property

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=63 "Edit section: Intellectual property")]

Tesla has sued former employees for stealing company information, including those who left to work for a rival such as [XPeng](https://en.wikipedia.org/wiki/XPeng "XPeng") and [Zoox](https://en.wikipedia.org/wiki/Zoox_\(company\) "Zoox \(company\)").[378] For example, Guangzhi Cao, a Tesla engineer, was accused of uploading [Tesla Autopilot](https://en.wikipedia.org/wiki/Tesla_Autopilot "Tesla Autopilot") source code to his [iCloud](https://en.wikipedia.org/wiki/ICloud "ICloud") account,[379] and Alex Khatilov was accused of downloading files related to its Warp Drive software to his personal [Dropbox](https://en.wikipedia.org/wiki/Dropbox_\(service\) "Dropbox \(service\)") account.[380]

### Misappropriation

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=64 "Edit section: Misappropriation")]

In 2018, a [class action](https://en.wikipedia.org/wiki/Class_action "Class action") was filed against Musk and the members of Tesla's board alleging they breached their [ fiduciary](https://en.wikipedia.org/wiki/Fiduciary "Fiduciary") duties by approving Musk's stock-based compensation plan.[367] Musk received the first portion of his stock options payout, worth more than $700 million in May 2020.[381]

In July 2023, Tesla board members returned $735 million to the company to settle a claim from a 2020 lawsuit alleging misappropriation of 11 million stock options granted to Elon Musk, Kimbal Musk, Larry Ellison, and others from 2017 to 2020.[382]

### Environmental violations

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=65 "Edit section: Environmental violations")]

In 2019, The [United States Environmental Protection Agency](https://en.wikipedia.org/wiki/United_States_Environmental_Protection_Agency "United States Environmental Protection Agency") fined Tesla for hazardous waste violations that occurred in 2017.[383] In June 2019, Tesla began negotiating penalties for 19 environmental violations from the [Bay Area Air Quality Management District](https://en.wikipedia.org/wiki/Bay_Area_Air_Quality_Management_District "Bay Area Air Quality Management District");[384] the violations took place around Tesla Fremont's paint shop, where there had been at least four fires between 2014 and 2019.[385] Environmental violations and permit deviations at Tesla's Fremont Factory increased from 2018 to 2019 with the production ramp of the Model 3.[386]

In June 2018, Tesla employee [Martin Tripp](https://en.wikipedia.org/wiki/Martin_Tripp "Martin Tripp") leaked information that Tesla was scrapping or reworking up to 40% of its raw materials at the Nevada Gigafactory.[387] After Tesla fired him for the leak, Tripp filed a lawsuit and claimed Tesla's security team gave police a false tip that he was planning a mass shooting at the Nevada factory.[388][336] The court ruled in Tesla's favor on September 17, 2020.[389][390]

In January 2024, 25 California counties sued Tesla, accusing the company of violating state health and safety codes by illegally disposing of hazardous waste. Later that week, the case was settled on the conditions that Tesla pay US$1.5 million and admit to acting "intentionally" and "negligent". Moreover, Tesla also agreed to train its employees on hazardous waste disposal and to have 10 percent of Tesla's facilities audited for waste disposal for the next 5 years.[391][392][393]

### Property damage

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=66 "Edit section: Property damage")]

In August 2019, [Walmart](https://en.wikipedia.org/wiki/Walmart "Walmart") filed a multi-million-dollar lawsuit against Tesla, claiming that Tesla's "negligent installation and maintenance" of solar panels caused roof fires at seven Walmart stores dating back to 2012.[394] Walmart reached a settlement with Tesla in November 2019; the terms of the settlement were not disclosed.[395]

In April 2021, a Norwegian judge found Tesla guilty of throttling charging speed through a 2019 over-the-air software update, after they failed to respond to the lawsuit. The 30 customers who were part of the lawsuit were awarded 136,000 Norwegian kroner each ($16,000).[396][397]

### Racism

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=67 "Edit section: Racism")]

Tesla has faced numerous complaints regarding workplace harassment and racial discrimination,[398][399] with one former Tesla worker who attempted to sue the employer describing it as "a hotbed of racist behavior."[400] As of December 2021, three percent of leadership at the company were African-American.[401] A former Black worker described the work environment at Tesla's [Buffalo](https://en.wikipedia.org/wiki/Buffalo,_New_York "Buffalo, New York") plant as a "very racist place."[402] Tesla and SpaceX's treatment of [Juneteenth](https://en.wikipedia.org/wiki/Juneteenth "Juneteenth") in 2020 also came under fire.[403] Approximately 100 former employees have submitted signed statements alleging that the company discriminates specifically against African Americans and "allows a racist environment in its factories."[404]

Few of these cases against Tesla ever make it to trial as most employees are made to sign [arbitration agreements](https://en.wikipedia.org/wiki/Arbitration "Arbitration").[405] Employees are afterwards required to resolve such disputes out of court, and behind closed doors. 

#### Fremont, California, plant

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=68 "Edit section: Fremont, California, plant")]

According to the state's Department of Fair Employment and Housing, the [Fremont](https://en.wikipedia.org/wiki/Fremont,_California "Fremont, California") factory is a racially segregated place where Black employees claim they are given the most menial[398] and physically demanding work.[406] The accusations of racism culminated in February 2022 with the California Department of Fair Employment and Housing suing Tesla for "discriminating against its Black workers."[407]

In July 2021, former employee Melvin Berry received $1 million in his discrimination case in arbitration against the company after he claimed he was referred to by the [n-word](https://en.wikipedia.org/wiki/N-word "N-word") and forced to work longer hours at the Fremont plant.[408]

In October 2021, a jury verdict in the _[Owen Diaz vs. Tesla](https://en.wikipedia.org/wiki/Owen_Diaz_vs._Tesla "Owen Diaz vs. Tesla")_ trial awarded the plaintiff $137 million in damages after he had faced racial harassment at Tesla's Fremont facility during 2015–2016.[409][410] In a blog, Tesla stressed that Diaz was never "really" a Tesla worker, and that most uttering of the n-word were expressed in a friendly manner.[411][412] In April 2022, federal judge [William Orrick](https://en.wikipedia.org/wiki/William_Orrick_III "William Orrick III") upheld the jury finding of Tesla's liability but reduced the total damage down to $15 million.[413] Diaz was given a two-week deadline to decide if he would collect the damages. In June 2022, Diaz announced that he would be rejecting the $15 million award, opening the door for a new trial.[414] In April 2023, Diaz was awarded $3.2 million in the new trial.[415]

### COVID-19 pandemic

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=69 "Edit section: COVID-19 pandemic")]

Tesla's initial response to the [COVID-19 pandemic in the United States](https://en.wikipedia.org/wiki/COVID-19_pandemic_in_the_United_States "COVID-19 pandemic in the United States") has been the subject of considerable criticism. Musk had sought to exempt the Tesla Fremont factory in Alameda County, California from the government's stay-at-home orders. In an [earnings call](https://en.wikipedia.org/wiki/Earnings_call "Earnings call") in April, he was heard calling the public health orders "fascist".[416] He had also called the public's response to the pandemic "dumb" and had said online that there would be zero cases by April.[417] In May 2020, while [Alameda County](https://en.wikipedia.org/wiki/Alameda_County,_California "Alameda County, California") officials were negotiating with the company to reopen the [Fremont Factory](https://en.wikipedia.org/wiki/Tesla_Fremont_Factory "Tesla Fremont Factory") on the 18th, Musk defied local government orders by restarting production on the 11th.[418][419][420] Tesla also sued Alameda County, questioning the legality of the orders, but backed down after the Fremont Factory was given approval to reopen.[421][422] In June 2020, Tesla published a detailed plan for bringing employees back to work and keeping them safe,[423] however some employees still expressed concern for their health.[424]

In May 2020, Musk told workers that they could stay home if they felt uncomfortable coming back to work.[425] But in June, Tesla dismissed an employee who criticized the company for taking inadequate safety measures to protect workers from the coronavirus at the Fremont Factory.[426] Three more employees at Tesla's Fremont Factory claimed they were laid off for staying home out of fear of catching COVID-19. This was subsequently denied by Tesla, which even stated that the employees were still on the payroll.[427] COVID-19 cases at the factory grew from 10 in May 2020 to 125 in December 2020, with about 450 total cases in that time out of the approximately 10,000 workers at the plant (4.5%).[416][428]

In China, Tesla had what one executive described as "not a green light from the government to get back to work – but a flashing-sirens police escort."[429] Tesla enjoyed special treatment and strong government support in China, including [tax breaks](https://en.wikipedia.org/wiki/Tax_break "Tax break"), cheap financing, and assistance in building its [Giga Shanghai](https://en.wikipedia.org/wiki/Giga_Shanghai "Giga Shanghai") factory at breakneck speeds.[429] Musk has praised China's way of doing things, a controversial stance due to deteriorating [U.S.–Chinese relations](https://en.wikipedia.org/wiki/U.S.%E2%80%93Chinese_relations "U.S.–Chinese relations"), the [Persecution of Uyghurs in China](https://en.wikipedia.org/wiki/Persecution_of_Uyghurs_in_China "Persecution of Uyghurs in China"), and alleged human rights abuses in Hong Kong.[429]

### Right to repair

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=70 "Edit section: Right to repair")]

In March 2023, a class action antitrust lawsuit was filed against Tesla by Virginia M. Lambrix in San Francisco, alleging that the company unlawfully monopolized the market for maintenance and repair of its vehicles in violation of the [Sherman Act](https://en.wikipedia.org/wiki/Sherman_Antitrust_Act "Sherman Antitrust Act") and California [ antitrust law](https://en.wikipedia.org/wiki/United_States_antitrust_law "United States antitrust law"), as a result of which owners were "forced to pay supracompetitive prices and suffer exorbitant wait times" for maintenance services and repair parts.[430][431] The lawsuit was later combined with four other similar suits.[432][433]

While six out of eight alleged antitrust violations were dismissed, in June 2024 US District Judge [Trina Thompson](https://en.wikipedia.org/wiki/Trina_Thompson "Trina Thompson") allowed two claims to proceed, including alleged violations of California's Cartwright Act and Unfair Competition Law (UCL), with the court finding evidence of a repairs monopoly in Tesla's designing of its vehicles to require diagnostic and software updates that only the company could provide, and evidence of a parts monopoly in Tesla's restricting [original equipment manufacturers](https://en.wikipedia.org/wiki/Original_equipment_manufacturer "Original equipment manufacturer") from selling "to anyone other than Tesla."[432][433]

### TeslaTakedown protests

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=71 "Edit section: TeslaTakedown protests")]

Main article: [Tesla Takedown](https://en.wikipedia.org/wiki/Tesla_Takedown "Tesla Takedown")

In February 2025, numerous planned protests occurred outside of Tesla showrooms and service centers throughout the US. The protests were in response to Musk's contentious actions as leader of the [Department of Government Efficiency](https://en.wikipedia.org/wiki/Department_of_Government_Efficiency "Department of Government Efficiency") (DOGE) at the beginning of Donald Trump's second term as US president.[434][435]

### Lawsuits against Tesla critics and vehicle malfunction complainants in China

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=72 "Edit section: Lawsuits against Tesla critics and vehicle malfunction complainants in China")]

On February 12, 2025, the [Associated Press](https://en.wikipedia.org/wiki/Associated_Press "Associated Press") reported that in China, Tesla had sued six car owners, six or more bloggers, and two media outlets for [defamation](https://en.wikipedia.org/wiki/Defamation "Defamation") since 2021. The car owners had complained publicly about Tesla quality and accidents caused by major mechanical malfunctions, like brake failure. The bloggers and media outlets had written critically about the company. As of the date of publication, the Associated Press reported that Tesla had won 11 verdicts, two judgments were on appeal, and one had been settled outside court.[436]

## Criticism

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=73 "Edit section: Criticism")]

Main article: [Criticism of Tesla, Inc.](https://en.wikipedia.org/wiki/Criticism_of_Tesla,_Inc. "Criticism of Tesla, Inc.")

### Data privacy

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=74 "Edit section: Data privacy")]

A Tesla vehicle was only the second product ever reviewed by the [Mozilla Foundation](https://en.wikipedia.org/wiki/Mozilla_Foundation "Mozilla Foundation") that failed all of their privacy criteria.[437][438]

A Tesla owner filed a lawsuit in 2023 following a [Reuters](https://en.wikipedia.org/wiki/Reuters "Reuters") report that Tesla employees shared "highly invasive videos and images recorded by customers' car cameras" with one another.[439]

Internal data troves shared with various international government agencies and news organizations by former employee and [whistleblower](https://en.wikipedia.org/wiki/Whistleblower "Whistleblower") Lukasz Krupski in late 2023 implicated Tesla in "serious data protection lapse[s]."[440] The data Krupski retrieved included "information about current and former Tesla staff, including passport numbers, medical details and salaries" and was readily available on internal systems that most employees had access to.[441] As of November 2023, the Data Protection Authority in the Netherlands was investigating whether Tesla's alleged lack of internal security violated privacy laws.[442]

### Short sellers

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=75 "Edit section: Short sellers")]

[TSLAQ](https://en.wikipedia.org/wiki/TSLAQ "TSLAQ") is a collective of Tesla skeptics and[ short sellers](https://en.wikipedia.org/wiki/Short_\(finance\) "Short \(finance\)") who are vocally critically of Tesla and aim to "shape [the] perception [of Tesla] and move its stock."[443] In January 2020, 20% of Tesla stock was shorted, the highest at that time of any stock in the U.S. equity markets.[444] By early 2021, according to CNN, short sellers had lost $40 billion during 2020 as the stock price climbed much higher.[445] [Michael Burry](https://en.wikipedia.org/wiki/Michael_Burry "Michael Burry"), a short seller portrayed in _[The Big Short](https://en.wikipedia.org/wiki/The_Big_Short_\(film\) "The Big Short \(film\)")_ , had shorted Tesla previously via his firm [Scion Asset Management](https://en.wikipedia.org/wiki/Scion_Asset_Management "Scion Asset Management"), but removed his position in October 2021.[446]

### Tesla's mission

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=76 "Edit section: Tesla's mission")]

According to automotive journalist Jamie Kitman, when multiple CEOs of major automotive manufacturers approached Tesla for EV technology that Musk had claimed the company was willing to share, they instead were offered the opportunity to buy regulatory credits from the company. This suggested, according to Kitman, that "the company may not be as eager for the electric revolution to occur as it claims."[447]

### Giga New York audit

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=77 "Edit section: Giga New York audit")]

In 2020, the [New York State Comptroller](https://en.wikipedia.org/wiki/New_York_State_Comptroller "New York State Comptroller") released an audit of the [Giga New York](https://en.wikipedia.org/wiki/Giga_New_York "Giga New York") factory project, concluding that it presented many red flags, including lack of basic [due diligence](https://en.wikipedia.org/wiki/Due_diligence "Due diligence") and that the factory itself produced only $0.54 in economic benefits for every $1 spent by the state.[448][449][450]

### Delays

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=78 "Edit section: Delays")]

Musk has been criticized for repeated pushing out both production and release dates of products.[451][452] By one count in 2016, Musk had missed 20 projections.[453] In October 2017, Musk predicted that Model 3 production would be 5,000 units per week by December.[454] A month later, he revised that target to "sometime in March" 2018.[455] Delivery dates for the Model 3 were delayed as well.[456] Other projects like converting supercharger stations to be [solar-powered](https://en.wikipedia.org/wiki/Solar-power "Solar-power") have also lagged projections.[457] Musk responded in late 2018: "punctuality is not my strong suit...I never made a mass-produced car. How am I supposed to know with precision when it's gonna get done?"[458]

## Vehicle product issues

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=79 "Edit section: Vehicle product issues")]

### Recalls

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=80 "Edit section: Recalls")]

On April 20, 2017, Tesla issued a worldwide recall of 53,000 (~70%) of the 76,000 vehicles it sold in 2016 due to faulty parking brakes which could become stuck and "prevent the vehicles from moving".[459][460] On March 29, 2018, Tesla issued a worldwide recall of 123,000 Model S cars built before April 2016 due to [corrosion](https://en.wikipedia.org/wiki/Corrosion "Corrosion")-susceptible [power steering](https://en.wikipedia.org/wiki/Power_steering "Power steering") bolts, which could fail and require the driver to use "increased force" to control the vehicle.[461]

In October 2020, Tesla initiated a recall of nearly 50,000 Model X and Y vehicles throughout China for suspension issues.[462] Soon after in November, the [NHTSA](https://en.wikipedia.org/wiki/NHTSA "NHTSA") announced it had opened its investigation into 115,000 Tesla cars regarding "front suspension safety issues", citing specifically 2015–2017 Model S and 2016–2017 Model X years. Cases of the "whompy wheel" phenomenon, which also included Model X and the occasional Model 3 cars, have been documented through 2020.[463][464]

In February 2021, Tesla was required by the NHTSA to recall 135,000 Model S and Model X vehicles built from 2012 to 2018 due to using a [flash memory](https://en.wikipedia.org/wiki/Flash_memory "Flash memory") device that was rated to last only 5 to 6 years.[465] The problem was related to touchscreen failures that could possibly affect the rearview camera, safety systems, Autopilot and other features.[466][467] The underlying technical reason is that the car writes a large amount of [syslog](https://en.wikipedia.org/wiki/Syslog "Syslog") content to the device, wearing it out prematurely.[468]

Also in February 2021, the German Federal Motor Transport Authority (KBA) ordered Tesla to recall 12,300 Model X cars because of "body mouldings problems".[469][470]

In June 2021, Tesla recalled 5,974 electric vehicles due to worries that brake caliper bolts might become loose, which could lead to loss of tire pressure, increasing the chance of a crash.[471]

On December 30, 2021, Tesla announced that they are recalling more than 475,000 US model vehicles. This included 356,309 Model 3 Tesla vehicles from 2017 to 2020 due to rear-view camera issues and a further 119,009 Tesla Model S vehicles due to potential problems with the trunk or boot. The Model S recall includes vehicles manufactured between 2014 and 2021. Around 1% of recalled Model 3s may have a defective rear-view camera, and around 14% of recalled model S' may have the defect. The recall was not linked to a contemporaneous issue regarding the "Passenger Play" feature, which allowed games to be played on the touchscreen while the car is in motion.[472] After an investigation was launched by the NHTSA covering 585,000 vehicles, Tesla agreed to make changes where the feature would be locked and unusable while the car is moving.[473]

In September 2022, Tesla announced that they are recalling almost 1.1 million US model vehicles because the automatic window reversal system might not react correctly after detecting an obstruction, increasing the risk of injury.[474][475] In response,[_[clarification needed](https://en.wikipedia.org/wiki/Wikipedia:Please_clarify "Wikipedia:Please clarify")_]} Tesla announced an over-the-air software fix.[475]

In February 2023, Tesla recalled its FSD software following a recommendation from NHTSA; the recall applied to approximately 360,000 cars.[476] NHTSA found that FSD caused "unreasonable risk" when used on city streets.[477] In March 2023, about 3,500 Model Y Teslas were recalled for a bolting issue concerning the cars' second-row seats.[478]

In December 2023, following a 2-year investigation by the [NHTSA](https://en.wikipedia.org/wiki/NHTSA "NHTSA"),[479] Tesla issued a wider recall on all vehicles equipped with any version of Autosteer, including 2012–2023 Model S; 2016–2023 Model X; 2017–2023 Model 3; and 2020–2023 Model Y, covering 2,031,220 vehicles in total.[480] The NHTSA concluded that Autosteer's controls were not sufficient to prevent misuse and did not ensure that the drivers maintained "continuous and sustained responsibility for vehicle operation" and states that affected vehicles will receive an over-the-air software remedy.[480][481]

### Fires

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=81 "Edit section: Fires")]

See also: [Plug-in electric vehicle fire incidents § Tesla](https://en.wikipedia.org/wiki/Plug-in_electric_vehicle_fire_incidents#Tesla "Plug-in electric vehicle fire incidents")

Tesla customers have reported the company as being "slow" to address how their cars can ignite.[482] In 2013, a Model S caught fire after the vehicle hit metal debris on a highway in [Kent, Washington](https://en.wikipedia.org/wiki/Kent,_Washington "Kent, Washington"). Tesla confirmed the fire began in the battery pack and was caused by the impact of an object.[483] As a result of this and other incidents, Tesla announced its decision to extend its current vehicle warranty to cover fire damage.[484] In March 2014, the NHTSA announced that it had closed the investigation into whether the Model S was prone to catch fire, after Tesla said it would provide more protection to its battery packs.[485] All Model S cars manufactured after March 6, 2014, have had the 0.25-inch (6.4 mm) aluminum shield over the battery pack replaced with a new three-layer shield.[486] In October 2019, the NHTSA opened an investigation into possible battery defects in Tesla's Model S and X vehicles from 2012 to 2019 that could cause "non-crash" fires.[487][488][489]

### Autopilot crashes

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=82 "Edit section: Autopilot crashes")]

See also: [List of Tesla Autopilot crashes](https://en.wikipedia.org/wiki/List_of_Tesla_Autopilot_crashes "List of Tesla Autopilot crashes")

A Model S driver died in a collision with a tractor-trailer in 2016, while the vehicle was in Autopilot mode; the driver is believed to be the first person to have died in a Tesla vehicle in Autopilot.[490][491] The NHTSA investigated the accident but found no safety-related defect trend.[492] In March 2018, a driver of a Tesla Model X was killed in a crash. Investigators say that the driver of the vehicle had his car in 'self-driving' mode and was using his phone to play games when the vehicle collided with the barrier in the middle of the freeway. Through investigation, the NTSB found that the Tesla malfunctioned due to the system being confused by an exit on the freeway.[493]

According to a document released in June 2021, the NHTSA has initiated at least 30 investigations into Tesla crashes that were believed to involve the use of Autopilot, with some [involving fatalities](https://en.wikipedia.org/wiki/Tesla_Autopilot#Fatal_crashes "Tesla Autopilot").[494][495] In early September 2021, the NHTSA updated the list with an additional fatality incident[496] and ordered Tesla to hand over all extensive[_[clarification needed](https://en.wikipedia.org/wiki/Wikipedia:Please_clarify "Wikipedia:Please clarify")_] data pertaining to US cars with Autopilot to determine if there is a safety defect that leads Tesla cars to collide with first-responder vehicles.[496][497][498] In late September 2021, Tesla released an over-the-air software update to detect emergency lights at night.[499] In October 2021, the NHTSA asked Tesla why it did not issue a recall when it sent out that update.[500] In June 2022, the NHTSA said it would expand its probe, extending it to 830,000 cars from all current Tesla models. The probe was moved up from the _Preliminary Evaluation_ level to _Engineering Analysis_. The regulator cited the reason for the expansion as the need to "explore the degree to which Autopilot and associated Tesla systems may exacerbate human factors or behavioral safety risks by undermining the effectiveness of the driver's supervision."[501]

A safety test conducted by the Dawn Project in August 2022 demonstrated that a test driver using the beta version of Full Self-Driving repeatedly hit a child-sized mannequin in its path,[502] but there has been controversy over its conclusions.[503] Several Tesla owners responded by conducting their own, independent tests using children; NHTSA released a statement warning against the practice.[504]

In 2025, the California DMV filed a lawsuit against Tesla, claiming they had misled drivers about its vehicles' self-driving capabilities. The DMV sought to suspend Tesla's sales and manufacturing in the state of California for a minimum of 30 days.[505][506]

### Software hacking

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=83 "Edit section: Software hacking")]

In August 2015, two researchers said they were able to take control of a Tesla Model S by hacking into the car's entertainment system.[507] The hack required the researchers to physically access the car.[508] Tesla issued a [security update](https://en.wikipedia.org/wiki/Patch_\(computing\) "Patch \(computing\)") for the Model S the day after the exploit was announced.[509]

In September 2016, researchers at [Tencent](https://en.wikipedia.org/wiki/Tencent "Tencent")'s Keen Security Lab demonstrated a remote attack on a Tesla Model S and controlled the vehicle in both Parking and Driving Mode without physical access. They were able to compromise the automotive networking bus ([CAN bus](https://en.wikipedia.org/wiki/CAN_bus "CAN bus")) when the vehicle's web browser was used while the vehicle was connected to a malicious Wi-Fi hotspot.[510] This was the first case of a remote control exploit demonstrated on a Tesla. The vulnerability was disclosed to Tesla under their [bug bounty program](https://en.wikipedia.org/wiki/Bug_bounty_program "Bug bounty program") and patched within 10 days, before the exploit was made public.[511] Tencent also hacked the doors of a Model X in 2017.[512]

In January 2018, security researchers informed Tesla that an [Amazon Web Services](https://en.wikipedia.org/wiki/Amazon_Web_Services "Amazon Web Services") account of theirs could be accessed directly from the Internet and that the account had been exploited for [cryptocurrency mining](https://en.wikipedia.org/wiki/Cryptocurrency_mining "Cryptocurrency mining"). Tesla responded by securing the compromised system, rewarding the security researchers financially via their bug bounty program, and stating that the compromise did not violate customer privacy, nor vehicle safety or security.[513][514] Later in 2019, Tesla awarded a car and $375,000 to [ethical hackers](https://en.wikipedia.org/wiki/White_hat_\(computer_security\) "White hat \(computer security\)") during a [Pwn2Own](https://en.wikipedia.org/wiki/Pwn2Own "Pwn2Own") Model 3 hacking event.[515]

In June 2022, Martin Herfurt, a security researcher in Austria, discovered that changes made to make Tesla vehicles easier to start with [NFC](https://en.wikipedia.org/wiki/Near-field_communication "Near-field communication") cards also allowed for pairing new keys to the vehicle, allowing an attacker to enroll their keys to a vehicle.[516]

### Phantom braking

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=84 "Edit section: Phantom braking")]

In February 2022, Tesla drivers have reported a surge in "phantom braking" events when using Tesla Autopilot which coincides with the automaker's removal of radar as a supplemental sensor in May 2021.[517] In response, NHTSA opened an investigation the same month.[518] In August 2022, a consumer class action was filed alleging that the Autopilot system in Tesla cars "contains a hazardous defect which causes the vehicle to suddenly and unintentionally brake".[519][520] In November 2024, Tesla failed to persuade a [U.S. District Judge](https://en.wikipedia.org/wiki/U.S._District_Judge "U.S. District Judge") to dismiss the lawsuit.[521] The next court date is January 7, 2025.[521]

In May 2023, German business newspaper _[Handelsblatt](https://en.wikipedia.org/wiki/Handelsblatt "Handelsblatt")_ published a series of articles based on a trove of internal Tesla data submitted to them from informants.[522] The 100 gigabytes of data "contain[ed] over 1,000 accident reports involving phantom braking or unintended acceleration" as well as complaints about Tesla Autopilot.[523] [Dutch authorities](https://en.wikipedia.org/wiki/Dutch_Data_Protection_Authority "Dutch Data Protection Authority") responded by saying they were investigating the company for possible data privacy violations.[524]

In February 2025, a German court declared Tesla Autopilot as defective for normal use due to phantom braking issues.[525]

In February, 2025, a group of Australian buyers sued Tesla with a [class action](https://en.wikipedia.org/wiki/Class_action "Class action") over multiple issues like battery range, driving assistance and phantom braking.[526]

### Driving range performance

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=85 "Edit section: Driving range performance")]

Tesla has received thousands of complaints from owners that the driving ranges of their vehicles did not meet the ranges advertised by Tesla or the projections of in-dash range meters. When service centers were overwhelmed with appointments to take care of these issues, Tesla established a diversion team to cancel as many appointments as possible. Customers were told that remote diagnostics had determined there was no problem and their appointments were canceled. The company has been fined by South Korean regulators for its exaggerated range estimates.[527]

## Vehicle sales

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=86 "Edit section: Vehicle sales")]

In 2024, Tesla ranked as the world's best-selling [battery electric](https://en.wikipedia.org/wiki/Battery_electric_vehicle "Battery electric vehicle") passenger car manufacturer, with a market share of 17.6%.[528] Tesla reported 2023 vehicle deliveries of 1.8 million units, up 38% from 2022.[529][530] In March 2024, Tesla produced its six millionth car.[531] In Q4 2023, BYD took over the top spot for EVs shipped, but Tesla regained the title in Q1 2024.[532]

Tesla sells the most vehicles in the US (674,000+ in 2023) and has received $11.4 billion in regulatory credits from federal and state governments. The federal government previously provided a $7500 rebate to customers.[533][534][535] Tesla's second largest market is China. It sold over 603,000 vehicles in 2023 and received around $426 million from the Chinese government for cars manufactured at its Shanghai facility.[536][537] Germany was Tesla's third largest market in 2023, selling 63,682 units. Germany ended its EV subsidy program in December 2023. Under the program, a €6,750 discount was provided with Tesla paying €2,250. Tesla announced it would cover the entire discount going forward.[538] In 2023, over 63,000 Teslas were sold in France with customers receiving up to €7,000 via the bonus écologique. The government decreased the budget for EV aid from €1.5 billion to €700 million in 2024.[539] Canada was in Tesla's top five markets in 2023 with over 52,000 vehicles sold. In 2025, Tesla was accused of having "a run on the bank" when it sold an historic 8,600 vehicles in 72 hours before Canada's EV rebate program's funds were depleted. Tesla claimed 4000 vehicles alone were sold over one weekend in Quebec City.[540][541] The Government of Canada announced on March 25, 2025 that it had suspended the rebates and was investigating. The government also declared Tesla would be ineligible for any future rebates as long as the US tariffs on Canadian products continued.[542] Elon Musk has stated he is against government subsidies but will take them if they are available.[543]

Sales have been negatively impacted by Elon Musk's actions during the [second Trump presidency](https://en.wikipedia.org/wiki/Second_presidency_of_Donald_Trump "Second presidency of Donald Trump").[544][545] Tesla sales have decreased worldwide (the most in countries like France, Germany, the UK, Sweden, Norway, and the Netherlands)[546][547] with his vocal support for far-right parties such as [Reform UK](https://en.wikipedia.org/wiki/Reform_UK "Reform UK")[548] and [AfD](https://en.wikipedia.org/wiki/Alternative_for_Germany "Alternative for Germany")[549] in Germany along with his [Nazi salute controversy](https://en.wikipedia.org/wiki/Elon_Musk_salute_controversy "Elon Musk salute controversy") leading to boycotts of his products and Tesla vehicles being nicknamed [Swasticars](https://en.wikipedia.org/wiki/Swastika "Swastika").[555] German sales fell 76% in February 2025 compared to February 2024.[556][557]

Tesla's shares dropped over 9 percent after its sales in the EU and UK declined by nearly half in January 2025, bringing its valuation below $1 trillion for the first time since November 2024. While European electric car sales surged, Tesla struggled against rising competition, particularly from Chinese automakers.[558] In May 2025, it was reported that the company's sales in Europe declined every month since the start of the year, including in April where sales dropped 49 percent to 7,261 from 14,228 in April 2024.[559] As of May 2025, the company has a 0.7 percent market share in Europe, down from 1.3 percent a year ago.[559]

### Production and sales by quarter

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=87 "Edit section: Production and sales by quarter")]

See also: [History of Tesla, Inc. § Timeline of production and sales](https://en.wikipedia.org/wiki/History_of_Tesla,_Inc.#Timeline_of_production_and_sales "History of Tesla, Inc.")

100,000

200,000

300,000

400,000

500,000

600,000

3 2012

4

1

2 2013

3

4

1

2 2014

3

4

1

2 2015

3

4

1

2 2016

3

4

1

2 2017

3

4

1

2 2018

3

4

1

2 2019

3

4

1

2 2020

3

4

1

2 2021

3

4

1

2 2022

3

4

1

2 2023

3

4

1

2 2024

3

4

1

2 2025

  * Model S
  * Model X
  * Model S/X
  * Models Other Than 3/Y
  * Model 3
  * Model 3/Y



  
Tesla deliveries vary significantly by month due to regional issues such as availability of [car carriers](https://en.wikipedia.org/wiki/Car_carrier "Car carrier") and registration. On March 9, 2020, the company produced its 1 millionth electric car, becoming the first [auto manufacturer](https://en.wikipedia.org/wiki/Auto_manufacturer "Auto manufacturer") to achieve such a milestone.[560] In the third quarter of 2021, Tesla sold its 2 millionth electric car, becoming the first auto manufacturer to achieve such a milestone.[561] In the first quarter of 2023, the Model Y became the world's best-selling car, surpassing the Toyota Corolla.[562]

## Finances

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=88 "Edit section: Finances")]

| This graph was using the [legacy Graph extension](https://www.mediawiki.org/wiki/Extension:Graph "mw:Extension:Graph"), which is no longer supported. It needs to be converted to the [new Chart extension](https://www.mediawiki.org/wiki/Extension:Chart "mw:Extension:Chart").  
---|---  
  
Tesla financial performance

For the fiscal (and calendar) year 2021, Tesla reported a net income of $5.52 billion.[563] The annual revenue was $53.8 billion, an increase of 71% over the previous fiscal year.[563]

Sales by business (2024)[564] Business  | Sales in billion $  | Share   
---|---|---  
Automotive  | 87.6  | 89.7%   
Energy Generation and Storage  | 10.1  | 10.3%   
Sales by region (2024)[564] Region  | Sales in billion $  | Share   
---|---|---  
United States  | 47.7  | 48.9%   
Other countries  | 29.0  | 29.7%   
China  | 20.9  | 21.4%   
  
Of the revenue number in 2021, $314 million came from selling regulatory credits to other automakers to meet government pollution standards. That number has been a smaller percentage of revenue for multiple quarters.[563]

Tesla ended 2021 with $17.6 billion of cash on hand, down $1.8 billion from the end of 2020.[158]: 31

In February 2021, a [10-K](https://en.wikipedia.org/wiki/Form_10-K "Form 10-K") filing revealed that Tesla had invested some $1.5 billion in the cryptocurrency bitcoin, and the company indicated it would soon accept bitcoin as a form of payment.[80] Critics then pointed out how investing in cryptocurrency can run counter to Tesla's environmental goals.[565][566] Tesla made more profit from the 2021 investment than the profit from selling cars in 2020, due to the Bitcoin price increase after the investment was announced.[567][568]

The quarter ending June 2021 was the first time Tesla made a profit independent of Bitcoin and regulatory credits.[569]

The key trends for Tesla are (as at the financial year ending December 31): 

Year  | Revenue  
(US$ m)  | Net income  
(US$ m)  | Total assets  
(US$ m)  | Employees  | Sources   
---|---|---|---|---|---  
2005  | 0  | −12  | 8  |  |   
2006  | 0  | −30  | 44  | 70  | [570][571]  
2007  | 0  | −78  | 34  | 268  |   
2008  | 15  | −83  | 52  | 252  |   
2009  | 112  | −56  | 130  | 514  |   
2010  | 117  | −154  | 386  | 899  | [571]  
2011  | 204  | −254  | 713  | 1,417  | [571]  
2012  | 413  | −396  | 1,114  | 2,914  | [571]  
2013  | 2,013  | −74  | 2,417  | 5,859  | [571]  
2014  | 3,198  | −294  | 5,831  | 10,161  | [571]  
2015  | 4,046  | −889  | 8,068  | 13,058  | [571]  
2016  | 7,000  | −675  | 22,664  | 17,782  | [571]  
2017  | 11,759  | −1,962  | 28,655  | 37,543  | [571]  
2018  | 21,461  | −976  | 29,740  | 48,817  | [571]  
2019  | 24,578  | −862  | 34,309  | 48,016  | [571]  
2020  | 31,536  | 721  | 52,148  | 70,757  | [571]  
2021  | 53,823  | 5,519  | 62,131  | 99,290  | [571]  
2022  | 81,462  | 12,556  | 82,338  | 127,855  | [571]  
2023  | 96,773  | 14,997  | 106,618  | 140,473  | [571]  
2024  | 97,690  | 7,091  | 122,070  | 125,665  | [571]  
  
## Corporate affairs

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=89 "Edit section: Corporate affairs")]

### List of chief executives

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=90 "Edit section: List of chief executives")]

  1. Martin Eberhard (2004–2007)
  2. Ze'ev Drori (2007–2008)[572][573]
  3. Elon Musk (since October 2008)[574]



### List of board chairs

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=91 "Edit section: List of board chairs")]

  1. Elon Musk (2004–2018)[575]
  2. [Robyn Denholm](https://en.wikipedia.org/wiki/Robyn_Denholm "Robyn Denholm") (since November 2018)[574]



### Board of directors

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=92 "Edit section: Board of directors")]

Tesla has received criticism that its board lacks enough independent directors. In an April 2017 public letter, a group of influential Tesla investors, including the [California State Teachers' Retirement System](https://en.wikipedia.org/wiki/California_State_Teachers%27_Retirement_System "California State Teachers' Retirement System"), asked Tesla to add two new independent directors to its board "who do not have any ties with chief executive Elon Musk".[576] The investors wrote that "five of six current non-executive directors have professional or personal ties to Mr. Musk that could put at risk their ability to exercise independent judgement."[577] Tesla's directors at the time included Brad Buss, who served as chief financial officer at SolarCity; [Steve Jurvetson](https://en.wikipedia.org/wiki/Steve_Jurvetson "Steve Jurvetson"), a venture capitalist who also sits on the board of SpaceX;[578] Elon Musk's brother, [Kimbal](https://en.wikipedia.org/wiki/Kimbal_Musk "Kimbal Musk"); and [Ira Ehrenpreis](https://en.wikipedia.org/w/index.php?title=Ira_Ehrenpreis&action=edit&redlink=1 "Ira Ehrenpreis \(page does not exist\)") and [Antonio Gracias](https://en.wikipedia.org/wiki/Antonio_Gracias "Antonio Gracias"), both of whom also invested in SpaceX.[579] The letter called for a more independent board that could put a check on [groupthink](https://en.wikipedia.org/wiki/Groupthink "Groupthink").[577] At first Musk responded on Twitter, writing that the investors "should buy Ford stock" because "their governance is amazing."[577] Two days later, he promised he would add two independent board members;[580] [Kathleen Wilson-Thompson](https://en.wikipedia.org/wiki/Kathleen_Wilson-Thompson "Kathleen Wilson-Thompson") and [Larry Ellison](https://en.wikipedia.org/wiki/Larry_Ellison "Larry Ellison") were added at the end of 2018.[581] Ellison stepped down in August 2022.[582] Former Tesla CTO [J. B. Straubel](https://en.wikipedia.org/wiki/J._B._Straubel "J. B. Straubel") who left the company in 2019, was elected to the board in 2023.[583]

Another criticism of the board composition is that most of the independent directors lack automotive industry experience.[584] The exception is Robyn Denholm who served in finance and corporate reporting roles at [Toyota Australia](https://en.wikipedia.org/wiki/Toyota_Australia "Toyota Australia") from 1989 to 1996.[585]

Other previous board members include businessman [Steve Westly](https://en.wikipedia.org/wiki/Steve_Westly "Steve Westly"); Daimler executive Herbert Kohler;[288] CEO and Chairman of [Johnson Publishing Company](https://en.wikipedia.org/wiki/Johnson_Publishing_Company "Johnson Publishing Company") Linda Johnson Rice;[586] and [United Nations Special Envoy](https://en.wikipedia.org/wiki/Special_Envoy_of_the_Secretary-General "Special Envoy of the Secretary-General") on Innovative Finance and Sustainable Investments [Hiromichi Mizuno](https://en.wikipedia.org/wiki/Hiromichi_Mizuno "Hiromichi Mizuno").[587][588]

As of May 2023[[update]](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit), the board members are:[589]

Joined  | Name  | Titles  | [Independent](https://en.wikipedia.org/wiki/Independent_director "Independent director")  
---|---|---|---  
2014[590] | [Robyn Denholm](https://en.wikipedia.org/wiki/Robyn_Denholm "Robyn Denholm") | Chair (since November 2018); former CFO and Head of Strategy at [Telstra](https://en.wikipedia.org/wiki/Telstra "Telstra")[585] | Yes   
2004[14] | [Elon Musk](https://en.wikipedia.org/wiki/Elon_Musk "Elon Musk") | CEO, [product architect](https://en.wikipedia.org/wiki/Chief_product_officer "Chief product officer"), former chairman; founder, CEO and CTO of SpaceX  | No   
2004[591] | [Kimbal Musk](https://en.wikipedia.org/wiki/Kimbal_Musk "Kimbal Musk") | SpaceX board member[592] | No   
2007[593] | [Ira Ehrenpreis](https://en.wikipedia.org/w/index.php?title=Ira_Ehrenpreis&action=edit&redlink=1 "Ira Ehrenpreis \(page does not exist\)") | General Partner at Technology Partners[586] | Disputed[576]  
2017[586] | [James Murdoch](https://en.wikipedia.org/wiki/James_Murdoch "James Murdoch") | Former CEO of [21st Century Fox](https://en.wikipedia.org/wiki/21st_Century_Fox "21st Century Fox")[586] | Yes   
2018[578] | [Kathleen Wilson-Thompson](https://en.wikipedia.org/wiki/Kathleen_Wilson-Thompson "Kathleen Wilson-Thompson") | Global head of Human Resources of [Walgreens Boots Alliance](https://en.wikipedia.org/wiki/Walgreens_Boots_Alliance "Walgreens Boots Alliance")[578] | Yes   
2022[594] | [Joe Gebbia](https://en.wikipedia.org/wiki/Joe_Gebbia "Joe Gebbia") | Co-founder, board member and advisor of [Airbnb](https://en.wikipedia.org/wiki/Airbnb "Airbnb")[595] | Yes   
2023[583] | [J. B. Straubel](https://en.wikipedia.org/wiki/J._B._Straubel "J. B. Straubel") | Founder and CEO of [Redwood Materials](https://en.wikipedia.org/wiki/Redwood_Materials "Redwood Materials"); former CTO of Tesla[583] | Disputed[583][596]  
  
### Ownership structure

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=93 "Edit section: Ownership structure")]

The 10 largest shareholders of Tesla in March 2024 were:[564]

Shareholder name  | Percentage   
---|---  
[Elon Musk](https://en.wikipedia.org/wiki/Elon_Musk "Elon Musk") | 12.9%   
[The Vanguard Group](https://en.wikipedia.org/wiki/The_Vanguard_Group "The Vanguard Group") | 7.2%   
[BlackRock](https://en.wikipedia.org/wiki/BlackRock "BlackRock") | 4.5%   
[State Street Corporation](https://en.wikipedia.org/wiki/State_Street_Corporation "State Street Corporation") | 3.4%   
[Geode Capital Management](https://en.wikipedia.org/wiki/Geode_Capital_Management "Geode Capital Management") | 1.7%   
[Capital Research & Management (World Investors)](https://en.wikipedia.org/wiki/Capital_Group_Companies "Capital Group Companies") | 1.3%   
[BlackRock Life](https://en.wikipedia.org/wiki/BlackRock "BlackRock") | 1.2%   
[Eaton Vance](https://en.wikipedia.org/wiki/Eaton_Vance "Eaton Vance") | 1.0%   
[Norges Bank](https://en.wikipedia.org/wiki/Norges_Bank "Norges Bank") | 1.0%   
[Fidelity Investments](https://en.wikipedia.org/wiki/Fidelity_Investments "Fidelity Investments") | 0.9%   
Others  | 64.9%   
  
## See also

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=94 "Edit section: See also")]

  * [List of automobile manufacturers of the United States](https://en.wikipedia.org/wiki/List_of_automobile_manufacturers_of_the_United_States "List of automobile manufacturers of the United States")
  * [List of Easter eggs in Tesla products](https://en.wikipedia.org/wiki/List_of_Easter_eggs_in_Tesla_products "List of Easter eggs in Tesla products")
  * [List of production battery electric vehicles](https://en.wikipedia.org/wiki/List_of_production_battery_electric_vehicles "List of production battery electric vehicles")
  * [Plug-in electric vehicles in California](https://en.wikipedia.org/wiki/Plug-in_electric_vehicles_in_California "Plug-in electric vehicles in California")
  * [Plug-in electric vehicles in the United States](https://en.wikipedia.org/wiki/Plug-in_electric_vehicles_in_the_United_States "Plug-in electric vehicles in the United States")



## Notes

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=95 "Edit section: Notes")]

  1. **^** According to company representatives, both pronunciations are correct,[4] though [Nikola Tesla](https://en.wikipedia.org/wiki/Nikola_Tesla "Nikola Tesla")'s surname is properly pronounced _TESS-lə_.



## References

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=96 "Edit section: References")]

  1. **^** ["Supercharger deployment charts"](https://supercharge.info/charts). _Supercharge.info_. Retrieved January 27, 2025.
  2. **^** Ewing, Jack (January 16, 2024). ["Musk Demands Bigger Stake in Tesla as Price for A.I. Work"](https://www.nytimes.com/2024/01/16/business/tesla-elon-musk-stock.html). _[The New York Times](https://en.wikipedia.org/wiki/The_New_York_Times "The New York Times")_. [Archived](https://web.archive.org/web/20240229082500/https://www.nytimes.com/2024/01/16/business/tesla-elon-musk-stock.html) from the original on February 29, 2024. Retrieved March 6, 2024.
  3. **^** ["Annual report Form 10-K 2024 Tesla Inc"](https://ir.tesla.com/_flysystem/s3/sec/000162828025003063/tsla-20241231-gen.pdf) (PDF). January 29, 2025. Retrieved February 23, 2025.
  4. **^** ["What's the correct way to pronounce 'Tesla'? We asked"](https://finance.yahoo.com/news/whats-correct-way-pronounce-tesla-asked-181021888.html). Yahoo! Finance. July 13, 2017. [Archived](https://web.archive.org/web/20211006032013/https://finance.yahoo.com/news/whats-correct-way-pronounce-tesla-asked-181021888.html) from the original on October 6, 2021. Retrieved October 6, 2021.
  5. **^** Cunningham, Doug (November 8, 2024). ["Tesla regains $1 trillion in market capitalization in post-election surge"](https://www.upi.com/Top_News/US/2024/11/08/Tesla-trillion-dollar-Musk-Trump/6721731085202/). _UPI_. Retrieved November 8, 2024.
  6. **^** Dey, Esha (February 26, 2025). ["Telsa Market Value Slips Below $1 Trillion as Europe Sales Sink"](https://www.bloomberg.com/news/articles/2025-02-25/tesla-market-value-slips-below-1-trillion-as-europe-sales-sink). _Bloomberg_. Retrieved February 26, 2024.
  7. **^** Schifrin, Matt; Murphy, Andrea (June 6, 2024). ["The Global 2000 2023"](https://www.forbes.com/lists/global2000/?sh=51d599675ac0). _Forbes_. [Archived](https://web.archive.org/web/20240129031905/https://www.forbes.com/lists/global2000/?sh=4f5ab07e5ac0) from the original on January 29, 2024. Retrieved February 7, 2024.
  8. ^ _**a**_ _**b**_ Vance 2015, p. 152.
  9. **^** Kay, Grace. ["Ousted Tesla cofounder Martin Eberhard sounds off on Elon Musk, how the company has changed, and the EV wars"](https://www.businessinsider.com/tesla-cofounder-martin-eberhard-interview-history-elon-musk-ev-market-2023-2). _Business Insider_. [Archived](https://web.archive.org/web/20230728202545/https://www.businessinsider.com/tesla-cofounder-martin-eberhard-interview-history-elon-musk-ev-market-2023-2) from the original on July 28, 2023. Retrieved July 28, 2023.
  10. ^ _**a**_ _**b**_ Reed, Eric (February 4, 2020). ["History of Tesla: Timeline and Facts"](https://www.thestreet.com/technology/history-of-tesla-15088992). _[TheStreet.com](https://en.wikipedia.org/wiki/TheStreet.com "TheStreet.com")_. [Archived](https://web.archive.org/web/20211220161321/https://www.thestreet.com/technology/history-of-tesla-15088992) from the original on December 20, 2021. Retrieved January 27, 2021.
  11. **^** ["Tesla: A Carmaker With Silicon Valley Spark"](https://www.bloomberg.com/news/articles/2007-07-29/tesla-a-carmaker-with-silicon-valley-spark). _Bloomberg.com_. July 30, 2007. Retrieved August 31, 2020.
  12. **^** Vance 2015, p. 154.
  13. ^ _**a**_ _**b**_ Burns, Matt (October 8, 2014). ["A Brief History of Tesla"](https://web.archive.org/web/20150717064829/https://techcrunch.com/gallery/a-brief-history-of-tesla/#/slide2). _[TechCrunch](https://en.wikipedia.org/wiki/TechCrunch "TechCrunch")_. Archived from [the original](https://techcrunch.com/gallery/a-brief-history-of-tesla/) on July 17, 2015. Retrieved June 11, 2015. "Tesla was founded not by Elon Musk, but rather by Martin Eberhard and Marc Tarpenning in July 2003. The two bootstrapped the fledgling auto company until [Elon Musk](https://en.wikipedia.org/wiki/Elon_Musk "Elon Musk") led the company's US$7.5 million Series A financing round in February 2004."
  14. ^ _**a**_ _**b**_ Vance 2015, p. 155.
  15. **^** LaMonica, Martin (September 21, 2009). ["Tesla Motors founders: Now there are five"](https://www.cnet.com/news/tesla-motors-founders-now-there-are-five/). [CNET](https://en.wikipedia.org/wiki/CNET "CNET"). [Archived](https://web.archive.org/web/20201115134134/https://www.cnet.com/news/tesla-motors-founders-now-there-are-five/) from the original on November 15, 2020. Retrieved October 6, 2016.
  16. **^** Mitchell, Russ (July 30, 2021). ["Review: A deep new history of Tesla takes the shine off Elon Musk"](https://www.latimes.com/entertainment-arts/books/story/2021-07-30/power-play-tesla-book-review). _[Los Angeles Times](https://en.wikipedia.org/wiki/Los_Angeles_Times "Los Angeles Times")_. [Archived](https://web.archive.org/web/20210802080819/https://www.latimes.com/entertainment-arts/books/story/2021-07-30/power-play-tesla-book-review) from the original on August 2, 2021. Retrieved July 30, 2021.
  17. ^ _**a**_ _**b**_ Musk, Elon (August 2, 2006). ["The Secret Tesla Motors Master Plan (just between you and me) No. 124"](https://www.tesla.com/blog/secret-tesla-motors-master-plan-just-between-you-and-me). _tesla.com_. Tesla Motors. [Archived](https://web.archive.org/web/20100802142703/http://www.teslamotors.com/blog/secret-tesla-motors-master-plan-just-between-you-and-me) from the original on August 2, 2010. [_[self-published source](https://en.wikipedia.org/wiki/Wikipedia:Verifiability#Self-published_sources "Wikipedia:Verifiability")_]
  18. **^** Weinstock, Suzanne (January 5, 2013). ["Venture-backed Tesla cuts costs"](https://www.privateequityinternational.com/venture-backed-tesla-cuts-costs/). _Private Equity International_. [Archived](https://web.archive.org/web/20210816094912/https://www.privateequityinternational.com/venture-backed-tesla-cuts-costs/) from the original on August 16, 2021. Retrieved August 16, 2021.
  19. ^ _**a**_ _**b**_ Williams, E. Freya (2015). _Green Giants: How Smart Companies Turn Sustainability into Billion-Dollar Businesses_. New York: [Amacom](https://en.wikipedia.org/wiki/HarperCollins#Amacom "HarperCollins"). [ISBN](https://en.wikipedia.org/wiki/ISBN_\(identifier\) "ISBN \(identifier\)") [978-0-8144-3614-1](https://en.wikipedia.org/wiki/Special:BookSources/978-0-8144-3614-1 "Special:BookSources/978-0-8144-3614-1"). [OCLC](https://en.wikipedia.org/wiki/OCLC_\(identifier\) "OCLC \(identifier\)") [911179554](https://search.worldcat.org/oclc/911179554).
  20. **^** Kanellos, Michael (August 13, 2007). ["Tesla CEO steps down as possible delays loom"](https://www.cnet.com/news/tesla-ceo-steps-down-as-possible-delays-loom/). [CNET](https://en.wikipedia.org/wiki/CNET "CNET"). [Archived](https://web.archive.org/web/20210821142628/https://www.cnet.com/news/tesla-ceo-steps-down-as-possible-delays-loom/) from the original on August 21, 2021. Retrieved August 21, 2021.
  21. **^** ["Martin Eberhard and Marc Tarpenning | American entrepreneurs"](https://www.britannica.com/biography/Martin-Eberhard-and-Marc-Tarpenning). _[Encyclopædia Britannica](https://en.wikipedia.org/wiki/Encyclop%C3%A6dia_Britannica "Encyclopædia Britannica")_. September 18, 2023. [Archived](https://web.archive.org/web/20190218075931/https://www.britannica.com/biography/Martin-Eberhard-and-Marc-Tarpenning) from the original on February 18, 2019. Retrieved April 7, 2021.
  22. ^ _**a**_ _**b**_ Baer, Drake (November 11, 2014). ["The Making Of Tesla: Invention, Betrayal, And The Birth Of The Roadster"](https://www.businessinsider.com/tesla-the-origin-story-2014-10). _[Business Insider](https://en.wikipedia.org/wiki/Business_Insider "Business Insider")_. [Archived](https://web.archive.org/web/20180711152847/http://www.businessinsider.com/tesla-the-origin-story-2014-10) from the original on July 11, 2018. Retrieved October 3, 2018.
  23. **^** Squatriglia, Chuck (June 11, 2009). ["Tesla's Founder Sues Tesla's CEO"](https://www.wired.com/2009/06/eberhard/). _[Wired](https://en.wikipedia.org/wiki/Wired_\(magazine\) "Wired \(magazine\)")_.
  24. **^** ["Case CIV484400; Martin Eberhard vs. Elon Musk"](https://odyportal-ext.sanmateocourt.org/Portal-External/). _Superior Court of California, County of San Mateo_. [Archived](https://web.archive.org/web/20240103174133/https://odyportal-ext.sanmateocourt.org/portal-external) from the original on January 3, 2024. Retrieved March 29, 2024.
  25. **^** Ramey, Jay (November 27, 2017). ["The first Tesla Roadster: A look back at the early adopter's electric car"](https://www.autoweek.com/news/green-cars/a1835876/first-tesla-roadster-look-back-early-adopters-electric-car/). _[Autoweek](https://en.wikipedia.org/wiki/Autoweek "Autoweek")_. [Archived](https://web.archive.org/web/20210806001336/https://www.autoweek.com/news/green-cars/a1835876/first-tesla-roadster-look-back-early-adopters-electric-car/) from the original on August 6, 2021. Retrieved August 21, 2021.
  26. **^** Gulker, Chris (September 28, 2010). ["Menlo Park's only auto factory assembles $100,000 cars"](https://inmenlo.com/2010/09/28/menlo-parks-only-auto-factory-assembles-100000-cars/). _InMenlo_. [Archived](https://web.archive.org/web/20210802184111/https://inmenlo.com/2010/09/28/menlo-parks-only-auto-factory-assembles-100000-cars/) from the original on August 2, 2021. Retrieved August 13, 2023.
  27. **^** ["Elon Musk's Life Story: Tesla CEO's Early Years, Career"](https://www.businessinsider.com/the-rise-of-elon-musk-2016-7). _[Business Insider](https://en.wikipedia.org/wiki/Business_Insider "Business Insider")_. June 28, 2021. [Archived](https://web.archive.org/web/20200323155539/https://www.businessinsider.com/the-rise-of-elon-musk-2016-7) from the original on March 23, 2020. Retrieved August 21, 2021.
  28. **^** Riddell, Lindsay (June 24, 2009). ["Tesla gets long-awaited government loan"](http://www.bizjournals.com/pacific/stories/2009/06/22/daily33.html). _[American City Business Journals](https://en.wikipedia.org/wiki/American_City_Business_Journals "American City Business Journals")_. [Archived](https://web.archive.org/web/20160505192107/http://www.bizjournals.com/pacific/stories/2009/06/22/daily33.html) from the original on May 5, 2016.
  29. **^** Cole, Jay (May 22, 2013). ["Tesla Repays Entire DoE Loan, Taxpayers Make $12 Million on the Deal"](https://insideevs.com/news/317959/tesla-repays-entire-doe-loan-taxpayers-make-26-million-on-the-deal-updated/). _[InsideEVs](https://en.wikipedia.org/wiki/InsideEVs "InsideEVs")_. [Archived](https://web.archive.org/web/20160506051437/http://insideevs.com/tesla-repays-entire-doe-loan-taxpayers-make-12-million-on-the-deal/) from the original on May 6, 2016.
  30. **^** Isidore, Chris (May 22, 2013). ["Tesla repays federal loan nearly 10 years early"](https://money.cnn.com/2013/05/22/autos/tesla-loan-repayment/index.html). [CNN](https://en.wikipedia.org/wiki/CNN "CNN"). [Archived](https://web.archive.org/web/20210821193050/https://money.cnn.com/2013/05/22/autos/tesla-loan-repayment/index.html) from the original on August 21, 2021. Retrieved August 21, 2021.
  31. ^ _**a**_ _**b**_ _**c**_ _**d**_ Davis, Joshua (September 27, 2010). ["How Elon Musk Turned Tesla into the Car Company of the Future"](https://www.wired.com/2010/09/ff-tesla/). _[Wired](https://en.wikipedia.org/wiki/Wired_\(magazine\) "Wired \(magazine\)")_. Vol. 18, no. 10\. [Archived](https://web.archive.org/web/20160624053602/http://www.wired.com/2010/09/ff_tesla/) from the original on June 24, 2016.
  32. **^** Andrejczak, Matt (June 28, 2010). ["Tesla Motors revs up $244 million IPO"](https://www.marketwatch.com/story/tesla-motors-revs-up-244-million-ipo-2010-06-28). _[MarketWatch](https://en.wikipedia.org/wiki/MarketWatch "MarketWatch")_. [Archived](https://web.archive.org/web/20201217101107/https://www.marketwatch.com/story/tesla-motors-revs-up-244-million-ipo-2010-06-28) from the original on December 17, 2020. Retrieved October 6, 2020.
  33. **^** Scholer, Kristen; Spears, Lee (June 29, 2010). ["Tesla Posts Second-Biggest Rally for 2010 U.S. IPO"](https://www.bloomberg.com/news/articles/2010-06-29/tesla-motors-raises-226-million-in-first-ipo-of-u-s-carmaker-in-54-years). Bloomberg L.P. [Archived](https://web.archive.org/web/20210821142620/https://www.bloomberg.com/news/articles/2010-06-29/tesla-motors-raises-226-million-in-first-ipo-of-u-s-carmaker-in-54-years) from the original on August 21, 2021. Retrieved August 21, 2021.
  34. **^** Squatriglia, Chuck (October 20, 2010). ["Tesla's Got the Factory, Now It Needs to Fill It"](https://www.wired.com/2010/10/teslas-got-the-factory-now-it-needs-to-fill-it/). _[Wired](https://en.wikipedia.org/wiki/Wired_\(magazine\) "Wired \(magazine\)")_. [Archived](https://web.archive.org/web/20200718011303/https://www.wired.com/2010/10/teslas-got-the-factory-now-it-needs-to-fill-it/) from the original on July 18, 2020. Retrieved December 18, 2020.
  35. **^** ["Tesla Motors begins delivering Model S electric cars in a Silicon Valley milestone"](https://www.mercurynews.com/2012/06/22/tesla-motors-begins-delivering-model-s-electric-cars-in-a-silicon-valley-milestone-2/). _[The Mercury News](https://en.wikipedia.org/wiki/The_Mercury_News "The Mercury News")_. June 22, 2012. [Archived](https://web.archive.org/web/20210103140724/https://www.mercurynews.com/2012/06/22/tesla-motors-begins-delivering-model-s-electric-cars-in-a-silicon-valley-milestone-2/) from the original on January 3, 2021. Retrieved September 9, 2020.
  36. **^** MacKenzie, Angus (December 10, 2012). ["Model S Motor Trend Car of the Year Award 2013"](https://www.motortrend.com/news/2013-motor-trend-car-of-the-year-tesla-model-s/). _[Motor Trend](https://en.wikipedia.org/wiki/Motor_Trend "Motor Trend")_. [Archived](https://web.archive.org/web/20211004211420/https://www.motortrend.com/news/2013-motor-trend-car-of-the-year-tesla-model-s/) from the original on October 4, 2021. Retrieved August 21, 2021.
  37. **^** Voelcker, John (October 1, 2013). ["Tesla Model S Was Best-Selling Car in Norway For September"](https://www.greencarreports.com/news/1087346_tesla-model-s-was-best-selling-car-in-norway-for-september). _Green Car Reports_. [Archived](https://web.archive.org/web/20210926134226/https://www.greencarreports.com/news/1087346_tesla-model-s-was-best-selling-car-in-norway-for-september) from the original on September 26, 2021. Retrieved August 21, 2021.
  38. **^** Cobb, Jeff (January 26, 2017). ["Tesla Model S Is World's Best-Selling Plug-in Car For Second Year in a Row"](http://www.hybridcars.com/tesla-model-s-is-worlds-best-selling-plug-in-car-for-second-year-in-a-row/). _HybridCars.com_. [Archived](https://web.archive.org/web/20170126165815/http://www.hybridcars.com/tesla-model-s-is-worlds-best-selling-plug-in-car-for-second-year-in-a-row/) from the original on January 26, 2017. Retrieved January 31, 2017.
  39. **^** T[esla Motors Joins NASDAQ-100 index beginning July 15, 2013](https://www.nasdaq.com/about/press-center/tesla-motors-inc-join-nasdaq-100-index-beginning-july-15-2013)
  40. **^** Maanyu, K. Nived; Raj, D Goutham; Krishna, R Vamsi; Choubey, Shruthi Bhargava (May 2020). ["A Study on Tesla Autopilot"](https://www.ijspr.com/citations/v71n1/IJSPR_7101_30573.pdf) (PDF). [Sreenidhi Institute of Science and Technology](https://en.wikipedia.org/wiki/Sreenidhi_Institute_of_Science_and_Technology "Sreenidhi Institute of Science and Technology"). [Archived](https://web.archive.org/web/20210626141413/https://www.ijspr.com/citations/v71n1/IJSPR_7101_30573.pdf) (PDF) from the original on June 26, 2021. Retrieved April 7, 2021.
  41. **^** Berzon, Alexandra; Sweet, Cassandra (May 1, 2015). ["Tesla CEO Elon Musk Unveils Line of Home and Industrial Battery Packs"](https://www.wsj.com/articles/tesla-ceo-elon-musk-unveils-line-of-home-and-industrial-battery-packs-1430461622). _[The Wall Street Journal](https://en.wikipedia.org/wiki/The_Wall_Street_Journal "The Wall Street Journal")_. [Archived](https://web.archive.org/web/20170121024723/http://www.wsj.com/articles/tesla-ceo-elon-musk-unveils-line-of-home-and-industrial-battery-packs-1430461622) from the original on January 21, 2017. Retrieved March 11, 2017.
  42. **^** Randall, Tom (May 8, 2015). ["Tesla's Battery Grabbed $800 Million in Its First Week"](https://www.bloomberg.com/news/articles/2015-05-08/tesla-s-battery-grabbed-800-million-in-its-first-week). Bloomberg L.P. [Archived](https://web.archive.org/web/20180612160406/https://www.bloomberg.com/news/articles/2015-05-08/tesla-s-battery-grabbed-800-million-in-its-first-week) from the original on June 12, 2018. Retrieved March 11, 2017.
  43. **^** Logan, Bryan (September 29, 2015). ["Here's Tesla's first SUV, the all-electric Model X"](https://www.businessinsider.com/here-comes-the-tesla-model-x-launch-2015-9). _[Business Insider](https://en.wikipedia.org/wiki/Business_Insider "Business Insider")_. [Archived](https://web.archive.org/web/20190207094433/https://www.businessinsider.com/here-comes-the-tesla-model-x-launch-2015-9) from the original on February 7, 2019. Retrieved September 9, 2020.
  44. **^** Valdes-Dapena, Peter (September 29, 2015). ["Tesla has delivered the first Model X SUVs"](https://money.cnn.com/2015/09/29/autos/tesla-model-x/). [CNN](https://en.wikipedia.org/wiki/CNN "CNN"). [Archived](https://web.archive.org/web/20210821142620/https://money.cnn.com/2015/09/29/autos/tesla-model-x/) from the original on August 21, 2021. Retrieved August 21, 2021.
  45. **^** Hawkins, Andrew J. (November 21, 2016). ["Tesla completes its $2.6 billion acquisition of SolarCity"](https://www.theverge.com/2016/11/21/13698314/tesla-completes-acquisition-solarcity-elon-musk). _[The Verge](https://en.wikipedia.org/wiki/The_Verge "The Verge")_. [Archived](https://web.archive.org/web/20210521112009/https://www.theverge.com/2016/11/21/13698314/tesla-completes-acquisition-solarcity-elon-musk) from the original on May 21, 2021. Retrieved September 14, 2020.
  46. **^** Hamilton, Isobel Asher (July 30, 2021). ["Elon Musk is defending Tesla's acquisition of SolarCity against angry shareholders. This is the story of how it was transformed into Tesla Energy"](https://www.businessinsider.com/solarcity-tesla-energy-beleaguered-history-elon-musk-2021-7). _[Business Insider](https://en.wikipedia.org/wiki/Business_Insider "Business Insider")_. [Archived](https://web.archive.org/web/20210724010854/https://www.businessinsider.com/solarcity-tesla-energy-beleaguered-history-elon-musk-2021-7) from the original on July 24, 2021. Retrieved July 24, 2021.
  47. **^** Kolodny, Lora (October 28, 2019). ["Tesla's Elon Musk knew SolarCity faced a 'liquidity crisis' at time of 2016 deal, legal documents show"](https://web.archive.org/web/20200815214348/https://www.cnbc.com/2019/10/28/musk-deposition-stockholders-v-tesla-solarcity.html). [CNBC](https://en.wikipedia.org/wiki/CNBC "CNBC"). Archived from [the original](https://www.cnbc.com/2019/10/28/musk-deposition-stockholders-v-tesla-solarcity.html) on August 15, 2020.
  48. **^** Ferris, Robert (February 1, 2017). ["Tesla is following in the footsteps of Apple and is changing its name"](https://www.nbcnews.com/tech/tech-news/elon-musk-s-tesla-motors-changing-its-name-n715476). [CNBC](https://en.wikipedia.org/wiki/CNBC "CNBC"). [Archived](https://web.archive.org/web/20210115160623/https://www.nbcnews.com/tech/tech-news/elon-musk-s-tesla-motors-changing-its-name-n715476) from the original on January 15, 2021. Retrieved February 10, 2021.
  49. ^ _**a**_ _**b**_ Hull, Dana (April 7, 2016). ["Tesla Says It Received More Than 325,000 Model 3 Reservations"](https://www.bloomberg.com/news/articles/2016-04-07/tesla-says-model-3-pre-orders-surge-to-325-000-in-first-week). [Bloomberg L.P.](https://en.wikipedia.org/wiki/Bloomberg_L.P. "Bloomberg L.P.") [Archived](https://web.archive.org/web/20160407192859/http://www.bloomberg.com/news/articles/2016-04-07/tesla-says-model-3-pre-orders-surge-to-325-000-in-first-week) from the original on April 7, 2016. Retrieved April 16, 2016.
  50. ^ _**a**_ _**b**_ [King, Gayle](https://en.wikipedia.org/wiki/Gayle_King "Gayle King") (April 13, 2018). ["Tesla CEO Elon Musk, stressed but "optimistic," predicts big increase in Model 3 production"](https://www.cbsnews.com/news/elon-musk-tesla-model-3-problems-interview-today-2018-04-13/). _[CBS News](https://en.wikipedia.org/wiki/CBS_News "CBS News")_. [Archived](https://web.archive.org/web/20230727004533/https://www.cbsnews.com/news/elon-musk-tesla-model-3-problems-interview-today-2018-04-13/) from the original on July 27, 2023. Retrieved July 26, 2023.
  51. **^** Isidore, Chris. ["Tesla will start working 24/7 to crank out Model 3s"](https://money.cnn.com/2018/04/18/news/companies/elon-musk-tesla-model-3-production/index.html). _CNNMoney_. [Archived](https://web.archive.org/web/20180418204050/http://money.cnn.com/2018/04/18/news/companies/elon-musk-tesla-model-3-production/index.html) from the original on April 18, 2018. Retrieved April 18, 2018.
  52. **^** Boudette, Neal E. (April 3, 2018). ["For Tesla, 'Production Hell' Looks Like the Reality of the Car Business"](https://www.nytimes.com/2018/04/03/business/tesla-model-3.html). _The New York Times_. [ISSN](https://en.wikipedia.org/wiki/ISSN_\(identifier\) "ISSN \(identifier\)") [0362-4331](https://search.worldcat.org/issn/0362-4331). [Archived](https://web.archive.org/web/20230602065302/https://www.nytimes.com/2018/04/03/business/tesla-model-3.html) from the original on June 2, 2023. Retrieved June 2, 2023.
  53. **^** Randall, Tom (April 3, 2018). ["Tesla's Model 3 Is Now America's Best-Selling Electric Car"](https://www.bloomberg.com/news/articles/2018-04-03/tesla-s-model-3-is-the-best-selling-electric-car-in-the-u-s). _Bloomberg Hyperdrive_. [Archived](https://web.archive.org/web/20231116211737/https://www.bloomberg.com/news/articles/2018-04-03/tesla-s-model-3-is-the-best-selling-electric-car-in-the-u-s) from the original on November 16, 2023. Retrieved June 2, 2023.
  54. **^** O'Kane, Sean (February 22, 2019). ["Tesla's Model 3 was the best-selling EV in the world last year"](https://www.theverge.com/2019/2/22/18236707/tesla-model-3-2018-best-selling-ev-global). _[The Verge](https://en.wikipedia.org/wiki/The_Verge "The Verge")_. [Archived](https://web.archive.org/web/20191019133330/https://www.theverge.com/2019/2/22/18236707/tesla-model-3-2018-best-selling-ev-global) from the original on October 19, 2019. Retrieved February 1, 2021.
  55. **^** Dickey, Megan Rose (August 8, 2018). ["Elon Musk tweets he's thinking about taking Tesla private"](https://techcrunch.com/2018/08/07/elon-musk-tesla-private-tweet/). _[TechCrunch](https://en.wikipedia.org/wiki/TechCrunch "TechCrunch")_. [Archived](https://web.archive.org/web/20210801101246/https://techcrunch.com/2018/08/07/elon-musk-tesla-private-tweet/) from the original on August 1, 2021. Retrieved August 21, 2021.
  56. **^** Huddleston, Tom Jr. (August 8, 2018). ["Elon Musk says he wants to take Tesla private at over $70 billion – here's what that means"](https://www.cnbc.com/2018/08/08/elon-musk-wants-to-take-tesla-private--heres-what-it-means.html). [CNBC](https://en.wikipedia.org/wiki/CNBC "CNBC"). [Archived](https://web.archive.org/web/20210821183045/https://www.cnbc.com/2018/08/08/elon-musk-wants-to-take-tesla-private--heres-what-it-means.html) from the original on August 21, 2021. Retrieved August 21, 2021.
  57. **^** Kolodny, Lora (July 22, 2020). ["Tesla reports fourth straight quarter of profits"](https://www.cnbc.com/2020/07/22/tesla-tsla-earnings-q2-2020.html). CNBC. [Archived](https://web.archive.org/web/20210128195702/https://www.cnbc.com/2020/07/22/tesla-tsla-earnings-q2-2020.html) from the original on January 28, 2021. Retrieved July 23, 2020.
  58. **^** Vlastelica, Ryan (January 6, 2021). ["Tesla Eyes Another Milestone as Valuation Nears Facebook's"](https://finance.yahoo.com/news/tesla-eyes-another-milestone-valuation-170917947.html). Yahoo! Finance. [Archived](https://web.archive.org/web/20210107034326/https://finance.yahoo.com/news/tesla-eyes-another-milestone-valuation-170917947.html) from the original on January 7, 2021. Retrieved January 7, 2021.
  59. **^** Wayland, Michael; Kolodny, Lora (December 14, 2020). ["Tesla's market cap tops the 9 largest automakers combined – Experts disagree about if that can last"](https://www.cnbc.com/2020/12/14/tesla-valuation-more-than-nine-largest-carmakers-combined-why.html). CNBC. [Archived](https://web.archive.org/web/20210811165742/https://www.cnbc.com/2020/12/14/tesla-valuation-more-than-nine-largest-carmakers-combined-why.html) from the original on August 11, 2021. Retrieved January 7, 2021.
  60. **^** Root, Al (December 7, 2020). ["Tesla Becomes Only the Sixth Company to Top $600 Billion in Value"](https://www.barrons.com/articles/tesla-becomes-only-the-sixth-company-to-top-600-billion-in-value-51607377650). _Barron's_. [Archived](https://web.archive.org/web/20240330002814/https://www.barrons.com/articles/tesla-becomes-only-the-sixth-company-to-top-600-billion-in-value-51607377650) from the original on March 30, 2024. Retrieved March 29, 2024.
  61. ^ _**a**_ _**b**_ Davies, Rob (December 21, 2020). ["Tesla joins Wall Street's S&P 500 share index"](https://www.theguardian.com/technology/2020/dec/21/tesla-joins-wall-streets-sp-500-share-index). _The Guardian_. [Archived](https://web.archive.org/web/20210111211551/https://www.theguardian.com/technology/2020/dec/21/tesla-joins-wall-streets-sp-500-share-index) from the original on January 11, 2021. Retrieved December 22, 2020.
  62. **^** Bursztynsky, Jessica (January 8, 2021). ["Tesla closes day as fifth most valuable U.S. company, passing Facebook"](https://www.cnbc.com/2021/01/07/tesla-passes-facebook-to-become-fifth-most-valuable-us-company.html). CNBC. [Archived](https://web.archive.org/web/20210225193446/https://www.cnbc.com/2021/01/07/tesla-passes-facebook-to-become-fifth-most-valuable-us-company.html) from the original on February 25, 2021. Retrieved March 31, 2021.
  63. ^ _**a**_ _**b**_ Lambert, Fred (March 15, 2019). ["Tesla unveils Model Y electric SUV with 300 miles range and 7-seats"](https://electrek.co/2019/03/14/tesla-model-y-electric-suv-unveil/). _[Electrek](https://en.wikipedia.org/wiki/Electrek "Electrek")_. [Archived](https://web.archive.org/web/20190315055454/https://electrek.co/2019/03/14/tesla-model-y-electric-suv-unveil/) from the original on March 15, 2019. Retrieved March 15, 2019.
  64. ^ _**a**_ _**b**_ Lambert, Fred (April 8, 2020). ["Tesla Model Y teardown: shows some great improvements over Model 3 despite sharing 75% of parts"](https://electrek.co/2020/04/08/tesla-model-y-teardown-improvements-over-model-3-sharing-parts/). _Electrek_. [Archived](https://web.archive.org/web/20210205060739/https://electrek.co/2020/04/08/tesla-model-y-teardown-improvements-over-model-3-sharing-parts/) from the original on February 5, 2021. Retrieved October 28, 2020.
  65. ^ _**a**_ _**b**_ Dow, Jameson (March 13, 2020). ["Tesla Model Y specs: we finally know how big it is"](https://electrek.co/2020/03/13/tesla-model-y-specs-we-finally-know-how-big-it-is/). _electrek.co_. [Archived](https://web.archive.org/web/20201022020842/https://electrek.co/2020/03/13/tesla-model-y-specs-we-finally-know-how-big-it-is/) from the original on October 22, 2020. Retrieved March 14, 2020.
  66. **^** Ohnsman, Alan (January 7, 2019). ["Elon Musk Accelerates Tesla's China Strategy With Shanghai Gigafactory Groundbreaking"](https://www.forbes.com/sites/alanohnsman/2019/01/07/elon-musk-accelerates-teslas-china-strategy-with-shanghai-gigafactory-groundbreaking/). _Forbes_. [Archived](https://web.archive.org/web/20230820195138/https://www.forbes.com/sites/alanohnsman/2019/01/07/elon-musk-accelerates-teslas-china-strategy-with-shanghai-gigafactory-groundbreaking/) from the original on August 20, 2023. Retrieved August 20, 2023.
  67. **^** ["Tesla delivers its first 'Made in China' cars"](https://www.bbc.com/news/business-50921729). _BBC News_. December 30, 2019. [Archived](https://web.archive.org/web/20230820195139/https://www.bbc.com/news/business-50921729) from the original on August 20, 2023. Retrieved August 20, 2023.
  68. **^** ["German court says Tesla can clear trees to build car factory"](https://www.theguardian.com/technology/2020/feb/21/german-court-tesla-can-clear-trees-car-factory). _The Guardian_. February 21, 2020. [Archived](https://web.archive.org/web/20200702041145/https://www.theguardian.com/technology/2020/feb/21/german-court-tesla-can-clear-trees-car-factory) from the original on July 2, 2020. Retrieved July 2, 2020.
  69. **^** ["Tesla opens Giga Berlin factory in Germany"](https://www.dw.com/en/tesla-opens-giga-berlin-factory-in-germany/av-61210668). Deutsche Welle. March 22, 2022. Retrieved March 22, 2022.[_[permanent dead link](https://en.wikipedia.org/wiki/Wikipedia:Link_rot "Wikipedia:Link rot")_]
  70. **^** Lambert, Fred (July 25, 2020). ["Watch Tesla start construction work at Gigafactory Texas in drone video"](https://electrek.co/2020/07/25/tesla-starts-construction-gigafactory-texas-drone-video/). _Electrek_. [Archived](https://web.archive.org/web/20210309232909/https://electrek.co/2020/07/25/tesla-starts-construction-gigafactory-texas-drone-video/) from the original on March 9, 2021. Retrieved August 24, 2020.
  71. ^ _**a**_ _**b**_ Carlson, Kara (April 7, 2022). ["Inside Elon's big, weird Austin party: Music, robots – and even a petting zoo"](https://www.statesman.com/story/business/2022/04/07/tesla-fans-show-up-cyber-rodeo-celebration-austin/9504356002/). Austin American-Statesman. [Archived](https://web.archive.org/web/20220409035616/https://www.statesman.com/story/business/2022/04/07/tesla-fans-show-up-cyber-rodeo-celebration-austin/9504356002/) from the original on April 9, 2022. Retrieved April 9, 2022.
  72. **^** Shakir, Umar (March 1, 2023). ["Tesla confirms its next Gigafactory will be in Mexico"](https://www.theverge.com/2023/3/1/23571725/tesla-gigafactory-monterrey-mexico-announce-investor-day). _The Verge_. [Archived](https://web.archive.org/web/20230303013849/https://www.theverge.com/2023/3/1/23571725/tesla-gigafactory-monterrey-mexico-announce-investor-day) from the original on March 3, 2023. Retrieved March 3, 2023.
  73. **^** ["Tesla shuts down Fremont plant after mounting criticism"](https://www.cnet.com/roadshow/news/tesla-fremont-factory-shut-musk-coronavirus-covid19-pandemic/). _CNET_. March 24, 2020. [Archived](https://web.archive.org/web/20240202234830/https://www.cnet.com/roadshow/news/tesla-fremont-factory-shut-musk-coronavirus-covid19-pandemic/) from the original on February 2, 2024. Retrieved February 2, 2024.
  74. **^** Wong, Julia Carrie (May 11, 2020). ["Elon Musk reopens California Tesla factory in defiance of lockdown order"](https://www.theguardian.com/technology/2020/may/11/tesla-factory-reopening-elon-musk-california-lockdown). _The Guardian_. [ISSN](https://en.wikipedia.org/wiki/ISSN_\(identifier\) "ISSN \(identifier\)") [0261-3077](https://search.worldcat.org/issn/0261-3077). [Archived](https://web.archive.org/web/20240202234830/https://www.theguardian.com/technology/2020/may/11/tesla-factory-reopening-elon-musk-california-lockdown) from the original on February 2, 2024. Retrieved February 2, 2024.
  75. **^** Sandler, Rachel (May 20, 2020). ["Tesla Drops Lawsuit Against Alameda County Over Factory Reopening"](https://www.forbes.com/sites/rachelsandler/2020/05/20/tesla-drops-lawsuit-against-alameda-county-over-factory-reopening/). _Forbes_. [Archived](https://web.archive.org/web/20240202234830/https://www.forbes.com/sites/rachelsandler/2020/05/20/tesla-drops-lawsuit-against-alameda-county-over-factory-reopening/) from the original on February 2, 2024. Retrieved February 2, 2024.
  76. **^** Hull, Dana (December 1, 2021). ["Tesla Makes It Official, Marking Headquarters Move to Texas"](https://www.bloomberg.com/news/articles/2021-12-01/tesla-makes-it-official-completing-headquarters-move-to-texas). Bloomberg L.P. [Archived](https://web.archive.org/web/20211201221909/https://www.bloomberg.com/news/articles/2021-12-01/tesla-makes-it-official-completing-headquarters-move-to-texas) from the original on December 1, 2021. Retrieved December 2, 2021.
  77. **^** del Castillo, Amanda (October 7, 2021). ["Elon Musk says Tesla will move Palo Alto headquarters to Austin"](https://abc7news.com/11096209/). _ABC7 San Francisco_. [Archived](https://web.archive.org/web/20211007230244/https://abc7news.com/11096209/) from the original on October 7, 2021. Retrieved October 13, 2021.
  78. **^** Alamalhodaei, Aria (September 23, 2021). ["Tesla's battery-manufacturing 'Megafactory' breaks ground in California"](https://techcrunch.com/2021/09/23/teslas-battery-manufacturing-megafactory-breaks-ground-in-california/). _TechCrunch_. Retrieved October 22, 2021.
  79. **^** Hull, Dana; Breslau, Karen (February 23, 2023). ["Newsom, Musk dedicate former HP headquarters in Palo Alto to Tesla engineers"](https://www.latimes.com/business/technology/story/2023-02-22/tesla-to-open-engineering-headquarters-in-palo-alto-california). _Los Angeles Times_. Bloomberg. [Archived](https://web.archive.org/web/20230226064644/https://www.latimes.com/business/technology/story/2023-02-22/tesla-to-open-engineering-headquarters-in-palo-alto-california) from the original on February 26, 2023. Retrieved February 26, 2023.
  80. ^ _**a**_ _**b**_ ["Tesla Bets on Bitcoin in Blue-Chip Boost to Cryptocurrency"](https://www.bloomberg.com/news/articles/2021-02-08/tesla-bets-1-5-billion-on-bitcoin-in-new-policy-crypto-surges). Bloomberg L.P. February 8, 2021. Retrieved February 8, 2021.
  81. **^** Porter, Jon (March 24, 2021). ["You can now buy a Tesla with bitcoin in the US"](https://www.theverge.com/2021/3/24/22347905/tesla-bitcoin-payment-us-cryptocurrency-elon-musk). _The Verge_. [Archived](https://web.archive.org/web/20210514052408/https://www.theverge.com/2021/3/24/22347905/tesla-bitcoin-payment-us-cryptocurrency-elon-musk) from the original on May 14, 2021. Retrieved May 14, 2021.
  82. **^** ["Tesla will no longer accept Bitcoin over climate concerns, says Musk"](https://www.bbc.com/news/business-57096305). _[BBC News](https://en.wikipedia.org/wiki/BBC_News "BBC News")_. May 13, 2021. [Archived](https://web.archive.org/web/20220531102821/https://www.bbc.com/news/business-57096305) from the original on May 31, 2022. Retrieved May 13, 2021.
  83. **^** Iyengar, Rish (May 13, 2021). ["Bitcoin drops around 12% after Elon Musk tweets that Tesla will not accept it as payment"](https://www.cnn.com/2021/05/12/tech/elon-musk-tesla-bitcoin/index.html). CNN. [Archived](https://web.archive.org/web/20210513025226/https://www.cnn.com/2021/05/12/tech/elon-musk-tesla-bitcoin/index.html) from the original on May 13, 2021. Retrieved May 13, 2021.
  84. **^** Newburger, Emma (June 13, 2021). ["Musk says Tesla will accept bitcoin again as crypto miners use more clean energy"](https://www.cnbc.com/2021/06/13/musk-tesla-will-accept-bitcoin-when-miners-use-clean-energy.html). _CNBC_. Retrieved July 24, 2024.
  85. **^** ["Elon Musk's Tesla sells most of its Bitcoin holdings"](https://www.bbc.com/news/business-62246367). _BBC News_. July 21, 2022. [Archived](https://web.archive.org/web/20220721002059/https://www.bbc.com/news/business-62246367) from the original on July 21, 2022. Retrieved July 21, 2022.
  86. **^** Marshall, Aarian (February 13, 2024). ["Tesla Wins EV Charging! Now What?"](https://www.wired.com/story/tesla-wins-ev-charging-now-what/). _Wired_. [ISSN](https://en.wikipedia.org/wiki/ISSN_\(identifier\) "ISSN \(identifier\)") [1059-1028](https://search.worldcat.org/issn/1059-1028). [Archived](https://web.archive.org/web/20240403143405/https://www.wired.com/story/tesla-wins-ev-charging-now-what/) from the original on April 3, 2024. Retrieved April 3, 2024.
  87. **^** Hawkins, Andrew J. (November 30, 2023). ["Tesla Cybertruck delivery event: Elon Musk hands over the first trucks to customers"](https://www.theverge.com/2023/11/30/23980053/tesla-cybertruck-delivery-event-elon-musk-live). _The Verge_. [Archived](https://web.archive.org/web/20240203163415/https://www.theverge.com/2023/11/30/23980053/tesla-cybertruck-delivery-event-elon-musk-live) from the original on February 3, 2024. Retrieved February 3, 2024.
  88. **^** Dow, Jameson (April 15, 2024). ["Tesla lays off 'more than 10%' of its global workforce"](https://electrek.co/2024/04/15/tesla-lays-off-more-than-10-of-its-global-workforce/). _Electrek_. [Archived](https://web.archive.org/web/20240415140553/https://electrek.co/2024/04/15/tesla-lays-off-more-than-10-of-its-global-workforce/) from the original on April 15, 2024. Retrieved April 15, 2024.
  89. **^** ["Your Evening Briefing: Elon Musk Officially Shifts Tesla's Incorporation to Texas After Vote"](https://www.bloomberg.com/news/newsletters/2024-06-14/bloomberg-evening-briefing-elon-musk-moves-tesla-incorporation-to-texas). _Bloomberg.com_. June 14, 2024. Retrieved September 24, 2024.
  90. **^** ["Musk Shows Tesla Cybercab, Sees Sub-$30,000 Cost and 2026 Production"](https://www.msn.com/en-us/autos/other/musk-shows-tesla-cybercab-sees-sub-30000-cost-and-2026-production/ar-AA1s4ROh). _[Bloomberg News](https://en.wikipedia.org/wiki/Bloomberg_News "Bloomberg News")_. October 11, 2024. Retrieved October 11, 2024.
  91. ^ _**a**_ _**b**_ Hawkins, Andrew J. (October 10, 2024). ["Tesla's robovan is the surprise of the night"](https://www.theverge.com/2024/10/10/24267158/tesla-van-robotaxi-autonomous-price-release-date). _[The Verge](https://en.wikipedia.org/wiki/The_Verge "The Verge")_.
  92. **^** Alvarez, Simon (April 22, 2019). ["Tesla outlines plan for 'Robotaxi' ride-sharing service"](https://www.teslarati.com/tesla-network-robotaxi-fleet-details-elon-musk/). _[Car and Driver](https://en.wikipedia.org/wiki/Car_and_Driver "Car and Driver")_. Retrieved November 4, 2024.
  93. **^** Josefs, Adam (December 9, 2024). ["Delaware court rejects Elon Musk's pay package"](https://www.killerstartups.com/delaware-court-rejects-elon-musks-pay-package/). _KillerStartups_. Retrieved December 10, 2024.
  94. **^** Eddy, Melissa; Ewing, Jack (March 5, 2025). ["Tesla Sales Slump in Europe as Anger Toward Elon Musk Grows"](https://www.nytimes.com/2025/03/05/business/tesla-germany-sales-elon-musk.html). _The New York Times_. [ISSN](https://en.wikipedia.org/wiki/ISSN_\(identifier\) "ISSN \(identifier\)") [0362-4331](https://search.worldcat.org/issn/0362-4331). [Archived](https://web.archive.org/web/20250307003916/https://www.nytimes.com/2025/03/05/business/tesla-germany-sales-elon-musk.html) from the original on March 7, 2025. Retrieved March 8, 2025.
  95. **^** Hawkins, Eleanor (February 27, 2025). ["Investors question whether Musk's X and DOGE ties are hurting Tesla"](https://www.axios.com/2025/02/27/tesla-stock-musk-doge-social-presence). _Axios_. Retrieved March 8, 2025.
  96. **^** Cerullo, Megan (February 26, 2025). Picchi, Aimee (ed.). ["As Elon Musk takes a leading role with DOGE, some Tesla customers are rethinking their allegiance"](https://www.cbsnews.com/news/tesla-sales-elon-musk-reputation-brand-damage-trump-doge/). _CBS News_. [Archived](https://web.archive.org/web/20250301042408/https://www.cbsnews.com/news/tesla-sales-elon-musk-reputation-brand-damage-trump-doge/) from the original on March 1, 2025. Retrieved March 8, 2025.
  97. **^** Lavietes, Matt (March 8, 2025). ["Tesla facilities face wave of attacks as Elon Musk delves into politics"](https://www.nbcnews.com/news/crime-courts/tesla-facilities-face-wave-attacks-elon-musk-delves-politics-rcna195458). _NBC News_. Retrieved March 8, 2025.
  98. **^** Verma, Pranshu; Thadani, Trisha (March 8, 2025). ["Anger at Elon Musk turns violent with molotov cocktails and gunfire at Tesla lots"](https://www.washingtonpost.com/technology/2025/03/08/elon-musk-tesla-protest-violence-vandalism/). _The Washington Post_. [ISSN](https://en.wikipedia.org/wiki/ISSN_\(identifier\) "ISSN \(identifier\)") [0190-8286](https://search.worldcat.org/issn/0190-8286). Retrieved March 8, 2025.
  99. **^** Kolodny, Lora (March 7, 2025). ["Tesla shares have declined every week since Elon Musk went to Washington"](https://www.cnbc.com/2025/03/07/tesla-shares-declined-every-week-since-elon-musk-went-to-washington.html). _CNBC_. Retrieved March 29, 2025.
  100. **^** Kerr, Dara; Helmore, Edward (March 29, 2025). ["Protests hit Tesla dealerships across the world in challenge to Elon Musk"](https://www.theguardian.com/world/2025/mar/29/tesla-protests-elon-musk-doge). _The Guardian_. Retrieved March 29, 2025.
  101. **^** ["Great, Grok is in cars now too"](https://www.engadget.com/ai/great-grok-is-in-cars-now-too-202153874.html). _Engadget_. July 13, 2025. Retrieved July 14, 2025.
  102. **^** ["Tesla opens first showroom in India after years of negotiations"](https://timesofindia.indiatimes.com/auto/news/tesla-opens-first-showroom-in-india-after-years-of-negotiations-marking-strategic-entry-starting-with-rs-60-lakh/articleshow/122484028.cms). _Times of India_. July 15, 2025.
  103. **^** ["Tesla debuts in India with upscale showroom launch in Mumbai"](https://apnews.com/article/5699547c6b70fefa9cc19b3f90c85217). _AP News_. July 15, 2025.
  104. **^** ["Tesla Signature series Model X to begin delivery September 29"](https://www.cnbc.com/2015/09/03/tesla-signature-series-model-x-to-begin-delivery-september-29.html). CNBC. [Reuters](https://en.wikipedia.org/wiki/Reuters "Reuters"). September 3, 2015. [Archived](https://web.archive.org/web/20191009130801/https://www.cnbc.com/2015/09/03/tesla-signature-series-model-x-to-begin-delivery-september-29.html) from the original on October 9, 2019. Retrieved September 4, 2015.
  105. **^** ["Tesla Fourth Quarter & Full Year 2016 Update"](https://web.archive.org/web/20170223212145/http://files.shareholder.com/downloads/ABEA-4CW8X0/3853068125x0x929284/22C29259-6C19-41AC-9CAB-899D148F323D/TSLA_Update_Letter_2016_4Q.pdf) (PDF). _Tesla Inc_. Palo Alto. February 22, 2017. Archived from [the original](http://files.shareholder.com/downloads/ABEA-4CW8X0/3853068125x0x929284/22C29259-6C19-41AC-9CAB-899D148F323D/TSLA_Update_Letter_2016_4Q.pdf) (PDF) on February 23, 2017. Retrieved February 22, 2017. _Production totaled 24,882 vehicles in 4Q 2016 and vehicle deliveries totaled 22,252 units. No breakdown by model was provided._
  106. **^** Jose, Pontes (February 2, 2021). ["Global Top 20 – December 2020"](http://ev-sales.blogspot.com/2021/02/global-top-20-december-2020.html). EVSales.com. [Archived](https://web.archive.org/web/20210202225647/http://ev-sales.blogspot.com/2021/02/global-top-20-december-2020.html) from the original on February 2, 2021. Retrieved February 3, 2021. "Global sales totaled 3,124,793 plug-in passenger cars in 2020, with a BEV to PHEV ratio of 69:31, and a global market share of 4%. The world's top selling plug-in car was the Tesla Model 3 with 365,240 units delivered, and Tesla was the top selling manufacturer of plug-in passenger cars in 2019 with 499,535 units, followed by VW with 220,220."
  107. **^** O'Kane, Sean (February 22, 2019). ["Tesla's Model 3 was the best-selling EV in the world last year"](https://www.theverge.com/2019/2/22/18236707/tesla-model-3-2018-best-selling-ev-global). _The Verge_. [Archived](https://web.archive.org/web/20191019133330/https://www.theverge.com/2019/2/22/18236707/tesla-model-3-2018-best-selling-ev-global) from the original on October 19, 2019. Retrieved February 1, 2021.
  108. **^** Jose, Pontes (January 30, 2022). ["World EV Sales – Tesla Model 3 Wins 4th Consecutive Best Seller Title in Record Year"](https://cleantechnica.com/2022/01/30/world-ev-sales-tesla-model-3-wins-4th-consecutive-best-seller-title-in-record-year/). CleanTechnica. [Archived](https://web.archive.org/web/20220206013050/https://cleantechnica.com/2022/01/30/world-ev-sales-tesla-model-3-wins-4th-consecutive-best-seller-title-in-record-year/) from the original on February 6, 2022. Retrieved February 5, 2022. "The top 3 global best selling plug-in electric cars in 2021 were the Tesla Model 3 (500,713), the Wuling Hongguang Mini EV (424,138), and the Tesla Model Y (410,517)"
  109. **^** Shahan, Zachary (August 26, 2021). ["Tesla Model 3 Has Passed 1 Million Sales"](https://cleantechnica.com/2021/08/26/tesla-model-3-has-passed-1-million-sales/). CleanTechnica. [Archived](https://web.archive.org/web/20210904115030/https://cleantechnica.com/2021/08/26/tesla-model-3-has-passed-1-million-sales/) from the original on September 4, 2021. Retrieved August 26, 2021.
  110. **^** Munoz, Juan Felipe (May 25, 2023). ["Tesla Model Y Was The World's Best-Selling Car In Q1 2023"](https://www.motor1.com/news/669135/tesla-model-y-worlds-best-selling-car-q1-2023/). _Motor1_. [Archived](https://web.archive.org/web/20230525191111/https://www.motor1.com/news/669135/tesla-model-y-worlds-best-selling-car-q1-2023/) from the original on May 25, 2023. Retrieved May 26, 2023.
  111. **^** Subramanian, Pras (December 2, 2022). ["Tesla Semi unveiled with tri-motor setup, megawatt charging tech"](https://finance.yahoo.com/news/tesla-semi-unveiled-with-tri-motor-setup-megawatt-charging-tech-121619247.html). Yahoo! Finance. [Archived](https://web.archive.org/web/20221204233255/https://finance.yahoo.com/news/tesla-semi-unveiled-with-tri-motor-setup-megawatt-charging-tech-121619247.html) from the original on December 4, 2022. Retrieved December 5, 2022.
  112. **^** Tarantola, A. (December 1, 2022). ["Tesla finally delivers its first production Semi"](https://www.engadget.com/tesla-finally-delivers-its-first-production-semi-to-pepsi-014214184.html). _Engadget_. [Archived](https://web.archive.org/web/20230131081212/https://www.engadget.com/tesla-finally-delivers-its-first-production-semi-to-pepsi-014214184.html) from the original on January 31, 2023. Retrieved December 2, 2022.
  113. **^** Greenhalgh, Keiron (April 25, 2024). ["Tesla Promises Semi Truck Production to Begin in Late 2025"](https://www.ttnews.com/articles/tesla-semi-truck-2025). _Transport Topics_. Retrieved May 1, 2024.
  114. **^** ["Shattered glass: Futuristic design questioned after Tesla Cybertruck launch"](https://www.reuters.com/article/us-tesla-truck-windows-idUSKBN1XW1CU). _Reuters_. November 22, 2019. [Archived](https://web.archive.org/web/20200808120546/https://www.reuters.com/article/us-tesla-truck-windows-idUSKBN1XW1CU) from the original on August 8, 2020. Retrieved May 13, 2020.
  115. **^** Ricker, Thomas (November 22, 2019). ["Elon Musk's Cybertruck is here, and so are the jokes"](https://www.theverge.com/2019/11/22/20975725/love-hate-telsa-cybertruck-design). _The Verge_. [Archived](https://web.archive.org/web/20201111224641/https://www.theverge.com/2019/11/22/20975725/love-hate-telsa-cybertruck-design) from the original on November 11, 2020. Retrieved May 13, 2020.
  116. **^** McFarland, Matt (November 22, 2019). ["Tesla's Cybertruck has become the butt of every internet joke"](https://www.cnn.com/2019/11/22/tech/cybertruck-tesla-pickup-truck-jokes/index.html). CNN. [Archived](https://web.archive.org/web/20201112023715/https://edition.cnn.com/2019/11/22/tech/cybertruck-tesla-pickup-truck-jokes/index.html) from the original on November 12, 2020. Retrieved May 13, 2020.
  117. **^** Wilkins, Catherine (July 17, 2025). ["New Tesla report reveals staggering extent of automaker's problems with Cybertruck: 'The final product was disappointing'"](https://www.thecooldown.com/green-business/tesla-cybertruck-sales-analysis-shrinking/). _The Cool Down_. Retrieved July 19, 2025.
  118. **^** ["Tesla Roadster is back: 0–60 in 1.9 seconds, 620-mile range"](https://www.greencarreports.com/news/1113862_tesla-roadster-is-back-0-60-in-1-9-seconds-620-mile-range). _Green Car Reports_. November 17, 2017. [Archived](https://web.archive.org/web/20201001075817/https://www.greencarreports.com/news/1113862_tesla-roadster-is-back-0-60-in-1-9-seconds-620-mile-range) from the original on October 1, 2020. Retrieved April 8, 2020.
  119. ^ _**a**_ _**b**_ _**c**_ Gibbs, Samuel (November 17, 2017). ["Tesla Roadster: nine things we know about the 'smackdown to gasoline cars'"](https://www.theguardian.com/technology/2017/nov/17/tesla-roadster-electric-supercar-elon-musk-fast). _The Guardian_. UK. [Archived](https://web.archive.org/web/20171117132834/https://www.theguardian.com/technology/2017/nov/17/tesla-roadster-electric-supercar-elon-musk-fast) from the original on November 17, 2017. Retrieved June 23, 2018.
  120. ^ _**a**_ _**b**_ Lassa, Todd (July 24, 2024). ["Tesla Still Talking Up Roadster, a Cheaper Model, and Robotaxis"](https://www.autoweek.com/news/a61687963/tesla-robotaxi-coming-roadster-entry-model/). _[Autoweek](https://en.wikipedia.org/wiki/Autoweek "Autoweek")_. Retrieved July 25, 2024.
  121. **^** Morris, James (January 7, 2023). ["Tesla Next Generation Platform: Everything We Know So Far"](https://www.forbes.com/sites/jamesmorris/2023/01/07/tesla-next-generation-platform-everything-we-know-so-far/?sh=5470a0d827aa). _[Forbes](https://en.wikipedia.org/wiki/Forbes "Forbes")_. [Archived](https://web.archive.org/web/20230313230415/https://www.forbes.com/sites/jamesmorris/2023/01/07/tesla-next-generation-platform-everything-we-know-so-far/?sh=5470a0d827aa) from the original on March 13, 2023. Retrieved March 13, 2023.
  122. **^** ["Elon Musk unveils the Robovan: the biggest surprise from Tesla's We, Robot event"](https://www.msn.com/en-us/news/technology/elon-musk-unveils-the-robovan-the-biggest-surprise-from-tesla-s-we-robot-event/ar-AA1s4hW3). _TechCrunch_. October 10, 2024. Retrieved October 24, 2024.
  123. **^** Woodyard, Chris (August 3, 2011). ["Tesla boasts about electric car deliveries, plans for sedan"](http://content.usatoday.com/communities/driveon/post/2011/08/tesla-boasts-about-electric-car-deliveries-plans-for-sedan/1). _[USA Today](https://en.wikipedia.org/wiki/USA_Today "USA Today")_. [Archived](https://web.archive.org/web/20140309053119/http://content.usatoday.com/communities/driveon/post/2011/08/tesla-boasts-about-electric-car-deliveries-plans-for-sedan/1) from the original on March 9, 2014. Retrieved October 4, 2011.
  124. **^** ["Supply Agreement for Products and Services – Lotus Cars Limited"](https://www.sec.gov/Archives/edgar/data/1318605/000119312510017054/dex1023.htm). _sec.gov_. July 11, 2005. [Archived](https://web.archive.org/web/20210108100543/https://www.sec.gov/Archives/edgar/data/1318605/000119312510017054/dex1023.htm) from the original on January 8, 2021. Retrieved February 2, 2020.
  125. **^** Lambert, Fred (December 7, 2019). ["Tesla starts charging $10 a month for its 'premium connectivity' features"](https://electrek.co/2019/12/07/tesla-starts-chargin-month-premium-connectivity-features/). _Electrek_. [Archived](https://web.archive.org/web/20210202143553/https://electrek.co/2019/12/07/tesla-starts-chargin-month-premium-connectivity-features/) from the original on February 2, 2021. Retrieved January 26, 2021.
  126. **^** ["Tesla Service Struggles To Keep Up With Sales Volume"](https://cleantechnica.com/2019/03/21/tesla-service-struggles-to-keep-up-with-sales-volume/). _CleanTechnica_. March 21, 2019. [Archived](https://web.archive.org/web/20201112032817/https://cleantechnica.com/2019/03/21/tesla-service-struggles-to-keep-up-with-sales-volume/) from the original on November 12, 2020. Retrieved September 1, 2020.
  127. **^** ["Tesla Mobile Service"](https://electrek.co/guides/tesla-mobile-service/). _Electrek_. [Archived](https://web.archive.org/web/20210122154032/https://electrek.co/guides/tesla-mobile-service/) from the original on January 22, 2021. Retrieved September 1, 2020.
  128. **^** ["Q3 2024 Shareholder Deck"](https://digitalassets.tesla.com/tesla-contents/image/upload/IR/TSLA-Q3-2024-Update.pdf) (PDF). _Tesla, Inc_. October 23, 2024. Retrieved November 10, 2024.
  129. ^ _**a**_ _**b**_ Yarow, Jay (March 12, 2014). ["Watch Elon Musk Make An Emotional Speech About How Auto Dealers Are 'Perverting Democracy' To Destroy Tesla And Hurt Customers"](https://www.businessinsider.com/elon-musk-on-teslas-auto-dealer-model-2014-3). _Business Insider_. [Archived](https://web.archive.org/web/20210721174226/https://www.businessinsider.com/elon-musk-on-teslas-auto-dealer-model-2014-3) from the original on July 21, 2021. Retrieved July 21, 2021.
  130. **^** Dent, Steve (March 22, 2019). ["Tesla drops annual servicing for 'as needed' repair model"](https://www.engadget.com/2019/03/22/tesla-annual-servicing-now-as-needed/). _Engadget_. US. [Archived](https://web.archive.org/web/20201108013009/https://www.engadget.com/2019-03-22-tesla-annual-servicing-now-as-needed.html) from the original on November 8, 2020. Retrieved August 18, 2019.
  131. **^** Lambert, Fred (August 2, 2022). ["Tesla enables paid charging at Destination Chargers, but there's a catch"](https://electrek.co/2022/08/02/tesla-enables-paid-charging-destination-chargers-catch/). _Electrek_. [Archived](https://web.archive.org/web/20220804033551/https://electrek.co/2022/08/02/tesla-enables-paid-charging-destination-chargers-catch/) from the original on August 4, 2022. Retrieved August 4, 2022.
  132. **^** Laing, Keith (September 7, 2023). ["Tesla to Supply Hilton Hotels With 20,000 EV Chargers by 2025"](https://www.bnnbloomberg.ca/tesla-to-supply-hilton-hotels-with-20-000-ev-chargers-by-2025-1.1968375). _Bloomberg News_. [Archived](https://web.archive.org/web/20231005181456/https://www.bnnbloomberg.ca/tesla-to-supply-hilton-hotels-with-20-000-ev-chargers-by-2025-1.1968375) from the original on October 5, 2023. Retrieved October 5, 2023.
  133. **^** Glon, Ronan (June 4, 2017). ["AAA raising insurance rates for Tesla owners"](https://web.archive.org/web/20170606123925/http://www.leftlanenews.com/aaa-raising-insurance-costs-for-tesla-owners-96345.html). Left Lane News. Archived from [the original](http://www.leftlanenews.com/aaa-raising-insurance-costs-for-tesla-owners-96345.html) on June 6, 2017. Retrieved June 7, 2017.
  134. **^** Barlyn, Suzanne; Mathias, Tamara (August 28, 2019). ["Tesla rolls out insurance in California"](https://www.reuters.com/article/us-tesla-markel-insurance/tesla-rolls-out-insurance-in-california-idUSKCN1VI2QZ). _Reuters_. [Archived](https://web.archive.org/web/20201107234740/https://www.reuters.com/article/us-tesla-markel-insurance/tesla-rolls-out-insurance-in-california-idUSKCN1VI2QZ) from the original on November 7, 2020. Retrieved November 29, 2020.
  135. ^ _**a**_ _**b**_ ["Tesla Insurance Support"](https://www.tesla.com/support/insurance). _Tesla, Inc_. [Archived](https://web.archive.org/web/20230818053623/https://www.tesla.com/support/insurance) from the original on August 18, 2023. Retrieved August 18, 2023.
  136. **^** Sully, Evan (October 21, 2020). ["Experts say Tesla's unique data-tracking abilities give it an advantage as Elon Musk looks to build a 'major insurance company' for Tesla owners"](https://www.businessinsider.com/tesla-vehicle-data-advantage-car-insurance-ambitions-elon-musk-analysts-2020-10). _Business Insider_. [Archived](https://web.archive.org/web/20201201201859/https://www.businessinsider.com/tesla-vehicle-data-advantage-car-insurance-ambitions-elon-musk-analysts-2020-10) from the original on December 1, 2020. Retrieved November 29, 2020.
  137. **^** ["Tesla expands its own insurance based on real-time driver data to two more states – now in 10 states"](https://electrek.co/2022/07/26/tesla-expands-insurance-based-on-real-time-driver-data-two-more-states/). _electrek.co_. [Archived](https://web.archive.org/web/20220727033341/https://electrek.co/2022/07/26/tesla-expands-insurance-based-on-real-time-driver-data-two-more-states/) from the original on July 27, 2022. Retrieved January 2, 2022.
  138. **^** Desjardins, Jeff (April 28, 2018). ["Here's what the future of Tesla could look like"](https://www.businessinsider.com/heres-what-the-future-of-tesla-could-look-like-2018-4). _[Business Insider](https://en.wikipedia.org/wiki/Business_Insider "Business Insider")_. [Archived](https://web.archive.org/web/20180501203935/https://www.businessinsider.com/heres-what-the-future-of-tesla-could-look-like-2018-4) from the original on May 1, 2018. Retrieved January 22, 2020.
  139. **^** Weber, Harri (January 24, 2024). ["Tesla's solar installs drop, but battery business is booming"](https://techcrunch.com/2024/01/24/teslas-solar-installs-drop-but-battery-business-is-booming/). _TechCrunch_. [Archived](https://web.archive.org/web/20240203160053/https://techcrunch.com/2024/01/24/teslas-solar-installs-drop-but-battery-business-is-booming/) from the original on February 3, 2024. Retrieved February 3, 2024.
  140. **^** Hanley, Steve (August 12, 2016). ["Elon Musk & SolarCity CTO Peter Rive Announce 'Solar Roof' (Not 'Solar on the Roof')"](https://cleantechnica.com/2016/08/12/elon-musk-solarcity-cto-peter-rive-announce-solar-roof-not-solar-roof/). _CleanTechnica_. [Archived](https://web.archive.org/web/20210512174107/https://cleantechnica.com/2016/08/12/elon-musk-solarcity-cto-peter-rive-announce-solar-roof-not-solar-roof/) from the original on May 12, 2021. Retrieved May 12, 2021.
  141. **^** ["Complete review of Tesla solar panels: are they worth it?"](https://www.solarreviews.com/blog/are-tesla-solar-panels-worth-it). _Solar Reviews_. May 9, 2021. [Archived](https://web.archive.org/web/20210512150310/https://www.solarreviews.com/blog/are-tesla-solar-panels-worth-it) from the original on May 12, 2021. Retrieved May 12, 2021.
  142. **^** Debord, Matthew (May 1, 2015). ["Elon Musk's big announcement: it's called 'Tesla Energy'"](http://www.businessinsider.com/here-comes-teslas-missing-piece-battery-announcement-2015-4). _[Business Insider](https://en.wikipedia.org/wiki/Business_Insider "Business Insider")_. [Archived](https://web.archive.org/web/20150505021205/http://www.businessinsider.com/here-comes-teslas-missing-piece-battery-announcement-2015-4) from the original on May 5, 2015. Retrieved June 11, 2015.
  143. **^** Lambert, Fred (May 3, 2020). ["Tesla has a new product: Autobidder, a step toward becoming an electric utility"](https://electrek.co/2020/05/03/tesla-autobidder-new-product-electric-utility/). _Electrek_. [Archived](https://web.archive.org/web/20200624190611/https://electrek.co/2020/05/03/tesla-autobidder-new-product-electric-utility/) from the original on June 24, 2020. Retrieved June 26, 2020.
  144. **^** Hampton, Liz; Jin, Hyunjoo (September 8, 2021). ["Tesla plans energy trading team as company expands battery projects"](https://www.reuters.com/business/energy/tesla-plans-begin-trading-solar-wind-battery-storage-energy-2021-09-08/). _Reuters_. [Archived](https://web.archive.org/web/20220202213125/https://www.reuters.com/business/energy/tesla-plans-begin-trading-solar-wind-battery-storage-energy-2021-09-08/) from the original on February 2, 2022. Retrieved February 3, 2022.
  145. **^** Delbert, Caroline (May 4, 2020). ["Elon Musk Would Like to Help You Trade Energy"](https://www.popularmechanics.com/science/a30153019/elon-musk-tesla-energy-autobidder/). _Popular Mechanics_. [Archived](https://web.archive.org/web/20220202170328/https://www.popularmechanics.com/science/a30153019/elon-musk-tesla-energy-autobidder/) from the original on February 2, 2022. Retrieved February 3, 2022.
  146. **^** ["Tesla Battery Backup Systems Manage Over 1.2GWh of Energy Storage Via Autobidder"](https://finance.yahoo.com/news/tesla-battery-backup-systems-manage-142235080.html). _finance.yahoo.com_. March 16, 2021. [Archived](https://web.archive.org/web/20230820094909/https://finance.yahoo.com/news/tesla-battery-backup-systems-manage-142235080.html) from the original on August 20, 2023. Retrieved August 20, 2023.
  147. **^** ["Tesla Electric"](https://www.tesla.com/electric). _Tesla_. [Archived](https://web.archive.org/web/20230627234252/https://www.tesla.com/electric) from the original on June 27, 2023. Retrieved June 27, 2023.
  148. **^** Lambert, Fred (August 17, 2023). ["Tesla Electric customers came out of Texas heatwave with an extra $100 in their pockets"](https://electrek.co/2023/08/17/tesla-electric-customers-texas-heatwave-extra-100/). _Electrek_. [Archived](https://web.archive.org/web/20230818093050/https://electrek.co/2023/08/17/tesla-electric-customers-texas-heatwave-extra-100/) from the original on August 18, 2023. Retrieved August 20, 2023.
  149. **^** Wilson, Kevin A. (March 15, 2018). ["Worth the Watt: A Brief History of the Electric Car, 1830 to Present"](https://www.caranddriver.com/features/g15378765/worth-the-watt-a-brief-history-of-the-electric-car-1830-to-present). _Car and Driver_. [Archived](https://web.archive.org/web/20210317062346/https://www.caranddriver.com/features/g15378765/worth-the-watt-a-brief-history-of-the-electric-car-1830-to-present/) from the original on March 17, 2021. Retrieved July 10, 2021.
  150. **^** Welch, David (July 30, 2007). ["Tesla: A Carmaker With Silicon Valley Spark"](https://web.archive.org/web/20140914195549/http://www.businessweek.com/stories/2007-07-29/tesla-a-carmaker-with-silicon-valley-spark). BloombergBusinessweek. Archived from [the original](http://www.businessweek.com/stories/2007-07-29/tesla-a-carmaker-with-silicon-valley-spark) on September 14, 2014. Retrieved March 13, 2014.
  151. **^** Crothers, Brooke (April 7, 2024). ["Tesla Model Y And Model 3 Price Cuts Continue As Owners Feel The Pain"](https://www.forbes.com/sites/brookecrothers/2024/04/07/tesla-model-y-model-3-price-cuts-continue-feel-the-pain/). _[Forbes](https://en.wikipedia.org/wiki/Forbes "Forbes")_.
  152. **^** Vaughan, Adam (October 25, 2013). ["12 interesting things we learned from Tesla's Elon Musk this week"](https://www.theguardian.com/environment/2013/oct/25/things-learned-tesla-elon-musk-electric-car). _[The Guardian](https://en.wikipedia.org/wiki/The_Guardian "The Guardian")_. [Archived](https://web.archive.org/web/20201126023450/https://www.theguardian.com/environment/2013/oct/25/things-learned-tesla-elon-musk-electric-car) from the original on November 26, 2020. Retrieved December 13, 2016.
  153. **^** ["Tesla Introduced A Business Model The World Has Not Seen Before"](https://cleantechnica.com/2020/08/29/tesla-introduced-a-business-model-the-world-has-not-seen-before/). _[CleanTechnica](https://en.wikipedia.org/wiki/CleanTechnica "CleanTechnica")_. August 29, 2020. [Archived](https://web.archive.org/web/20210118080229/https://cleantechnica.com/2020/08/29/tesla-introduced-a-business-model-the-world-has-not-seen-before/) from the original on January 18, 2021. Retrieved September 1, 2020.
  154. **^** Read, Richard. ["GM Follows Tesla's Lead, Plans To Sell Directly To Online Shoppers"](https://www.thecarconnection.com/news/1087492_gm-follows-teslas-lead-plans-to-sell-directly-to-online-shoppers). _The Car Connection_. [Archived](https://web.archive.org/web/20200924043030/https://www.thecarconnection.com/news/1087492_gm-follows-teslas-lead-plans-to-sell-directly-to-online-shoppers) from the original on September 24, 2020. Retrieved February 2, 2020.
  155. ^ _**a**_ _**b**_ ["7 Reasons Why Tesla Insists on Selling its Own Cars"](https://fortune.com/2016/01/19/why-tesla-sells-directly/). _[Fortune](https://en.wikipedia.org/wiki/Fortune_\(magazine\) "Fortune \(magazine\)")_. January 19, 2016. [Archived](https://web.archive.org/web/20201108115213/https://fortune.com/2016/01/19/why-tesla-sells-directly/) from the original on November 8, 2020. Retrieved September 1, 2020.
  156. **^** O'Toole, James (July 2, 2013). ["Tesla direct-sales petition hits 100,000 signatures"](https://money.cnn.com/2013/07/02/autos/tesla-petition/). [CNN](https://en.wikipedia.org/wiki/CNN "CNN"). [Archived](https://web.archive.org/web/20201112021607/https://money.cnn.com/2013/07/02/autos/tesla-petition/) from the original on November 12, 2020. Retrieved August 3, 2020.
  157. **^** Shipley, Lou (February 28, 2020). ["How Tesla Sets Itself Apart"](https://hbr.org/2020/02/how-tesla-sets-itself-apart). _[Harvard Business Review](https://en.wikipedia.org/wiki/Harvard_Business_Review "Harvard Business Review")_. [ISSN](https://en.wikipedia.org/wiki/ISSN_\(identifier\) "ISSN \(identifier\)") [0017-8012](https://search.worldcat.org/issn/0017-8012). [Archived](https://web.archive.org/web/20201224070209/https://hbr.org/2020/02/how-tesla-sets-itself-apart) from the original on December 24, 2020. Retrieved September 5, 2020.
  158. ^ _**a**_ _**b**_ _**c**_ _**d**_ ["Annual report Form 10-K 2021 Tesla Inc"](https://www.sec.gov/Archives/edgar/data/1318605/000095017022000796/tsla-20211231.htm). [U.S. Securities and Exchange Commission](https://en.wikipedia.org/wiki/U.S._Securities_and_Exchange_Commission "U.S. Securities and Exchange Commission"). [Archived](https://web.archive.org/web/20220217081012/https://www.sec.gov/Archives/edgar/data/1318605/000095017022000796/tsla-20211231.htm) from the original on February 17, 2022. Retrieved February 7, 2022.
  159. **^** Chapman, Steve (June 20, 2013). ["Car buyers get hijacked"](https://www.chicagotribune.com/2013/06/20/car-buyers-get-hijacked/). _Chicago Tribune_. [Archived](https://web.archive.org/web/20150420035811/http://articles.chicagotribune.com/2013-06-20/news/ct-oped-0620-chapman-20130620_1_tesla-motors-car-dealers-car-costs) from the original on April 20, 2015. Retrieved April 12, 2015.
  160. **^** ["Tesla Stores"](https://www.tesla.com/en_EU/findus/list). _Tesla_. April 2, 2021. [Archived](https://web.archive.org/web/20210410114330/https://www.tesla.com/en_EU/findus/list) from the original on April 10, 2021. Retrieved April 2, 2021.
  161. ^ _**a**_ _**b**_ _**c**_ Borchers, Callum (November 20, 2013). ["Automaker Tesla looks to bypass car dealers"](https://www.bostonglobe.com/business/2013/11/20/tesla-battles-auto-dealers-direct-sales-consumers/3f1xBFN21xH8QqQc3jijTP/story.html). _[The Boston Globe](https://en.wikipedia.org/wiki/The_Boston_Globe "The Boston Globe")_. [Archived](https://web.archive.org/web/20201028151135/https://www.bostonglobe.com/business/2013/11/20/tesla-battles-auto-dealers-direct-sales-consumers/3f1xBFN21xH8QqQc3jijTP/story.html) from the original on October 28, 2020. Retrieved June 22, 2017.
  162. **^** ["Tesla Has Altered The Car Dealership Model for the Better"](https://insideevs.com/features/394452/tesla-transforming-car-dealership/). _[InsideEVs](https://en.wikipedia.org/wiki/InsideEVs "InsideEVs")_. January 23, 2020. [Archived](https://web.archive.org/web/20210821183552/https://insideevs.com/features/394452/tesla-transforming-car-dealership/) from the original on August 21, 2021. Retrieved August 21, 2021.
  163. **^** Valdes-Dapena, Peter (May 20, 2013). ["Tesla's fight with America's car dealers"](https://money.cnn.com/2013/05/20/autos/telsa-car-dealers/). [CNN](https://en.wikipedia.org/wiki/CNN "CNN"). [Archived](https://web.archive.org/web/20210821183550/https://money.cnn.com/2013/05/20/autos/telsa-car-dealers/) from the original on August 21, 2021. Retrieved August 21, 2021.
  164. ^ _**a**_ _**b**_ Lambert, Fred (February 26, 2016). ["Tesla is now ~80% vertically integrated, says Goldman Sachs after a Tesla Factory visit"](http://electrek.co/2016/02/26/tesla-vertically-integrated/). _Electrek_. [Archived](https://web.archive.org/web/20201201130705/https://electrek.co/2016/02/26/tesla-vertically-integrated/) from the original on December 1, 2020. Retrieved March 31, 2016.
  165. **^** McAssey, Pat (October 13, 2016). ["Volkswagen CEO 'Annoyed Beyond Measure' That DHL Made Electric Van"](https://web.archive.org/web/20200809020608/https://nesn.com/nesn-fuel/). _NESN Fuel_. Archived from [the original](http://nesnfuel.com/2016/10/13/volkswagen-ceo-annoyed-beyond-measure-that-dhl-made-electric-van/) on August 9, 2020. Retrieved October 20, 2016.
  166. **^** ["Alternative Fuels Data Center: Developing Infrastructure to Charge Plug-In Electric Vehicles"](http://www.afdc.energy.gov/fuels/electricity_infrastructure.html). _afdc.energy.gov_. [United States Department of Energy](https://en.wikipedia.org/wiki/United_States_Department_of_Energy "United States Department of Energy"). [Archived](https://web.archive.org/web/20210122200700/https://afdc.energy.gov/fuels/electricity_infrastructure.html) from the original on January 22, 2021. Retrieved April 10, 2016.
  167. **^** Stringham, Edward Peter; Miller, Jennifer Kelly; Clark, J.R. (2015). ["Overcoming Barriers to Entry in an Established Industry: Tesla Motors"](http://journals.sagepub.com/doi/10.1525/cmr.2015.57.4.85). _California Management Review_. **57** (4): 85–103\. [doi](https://en.wikipedia.org/wiki/Doi_\(identifier\) "Doi \(identifier\)"):[10.1525/cmr.2015.57.4.85](https://doi.org/10.1525%2Fcmr.2015.57.4.85). [ISSN](https://en.wikipedia.org/wiki/ISSN_\(identifier\) "ISSN \(identifier\)") [0008-1256](https://search.worldcat.org/issn/0008-1256). [S2CID](https://en.wikipedia.org/wiki/S2CID_\(identifier\) "S2CID \(identifier\)") [155655599](https://api.semanticscholar.org/CorpusID:155655599). [Archived](https://web.archive.org/web/20220606103701/https://journals.sagepub.com/doi/10.1525/cmr.2015.57.4.85) from the original on June 6, 2022. Retrieved April 16, 2022.
  168. **^** Korzeniewski, Jeremy (July 29, 2020). ["Elon Musk: Tesla would share batteries, technology with competitors"](https://www.autoblog.com/2020/07/29/tesla-supplying-batteries-tech-competitors/). _Autoblog_. [Archived](https://web.archive.org/web/20210114140552/https://www.autoblog.com/2020/07/29/tesla-supplying-batteries-tech-competitors/) from the original on January 14, 2021. Retrieved October 11, 2020.
  169. **^** Blattberg, Eric (June 14, 2014). ["Here's what Tesla's 'good faith' patent stance actually means"](https://venturebeat.com/2014/06/14/heres-what-teslas-good-faith-patent-stance-actually-means/). _VentureBeat_. [Archived](https://web.archive.org/web/20201201130003/https://venturebeat.com/2014/06/14/heres-what-teslas-good-faith-patent-stance-actually-means/) from the original on December 1, 2020. Retrieved April 12, 2015.
  170. **^** Vance, Ashlee (June 13, 2014). ["Why Elon Musk Just Opened Tesla's Patents to His Biggest Rivals"](https://www.bloomberg.com/news/articles/2014-06-12/why-elon-musk-just-opened-teslas-patents-to-his-biggest-rivals). _[Bloomberg News](https://en.wikipedia.org/wiki/Bloomberg_News "Bloomberg News")_.
  171. **^** Roberts, Jeff John (April 15, 2024). ["Tesla inks semiconductor deal with Tata electronics for global operations"](https://web.archive.org/web/20240415040331/https://www.business-standard.com/companies/news/tesla-inks-semiconductor-deal-with-tata-electronics-for-global-operations-124041500087_1.html). Business Standard. Archived from [the original](https://www.business-standard.com/companies/news/tesla-inks-semiconductor-deal-with-tata-electronics-for-global-operations-124041500087_1.html/) on April 15, 2024. Retrieved April 15, 2024.
  172. **^** ["Exclusive: Tesla retreats from next-generation 'gigacasting' manufacturing process"](https://www.reuters.com/business/autos-transportation/tesla-retreats-next-generation-gigacasting-manufacturing-process-2024-05-01/). _Reuters_. May 2, 2024. Retrieved May 2, 2024.
  173. ^ _**a**_ _**b**_ _**c**_ _**d**_ _**e**_ _**f**_ Kane, Mark (May 23, 2022). ["What Batteries Are Tesla Using In Its Electric Cars?"](https://insideevs.com/news/587455/batteries-tesla-using-electric-cars/). _InsideEVs_. [Archived](https://web.archive.org/web/20230803204018/https://insideevs.com/news/587455/batteries-tesla-using-electric-cars/) from the original on August 3, 2023. Retrieved August 17, 2023.
  174. **^** ["Tesla Now Has Multiple Battery Options: Which One Should You Choose?"](https://insideevs.com/news/575956/tesla-battery-chemistries-explained/). _InsideEVs_. [Archived](https://web.archive.org/web/20230620013539/https://insideevs.com/news/575956/tesla-battery-chemistries-explained/) from the original on June 20, 2023. Retrieved June 20, 2023.
  175. **^** Merano, Maria (October 11, 2021). ["Tesla Model S Plaid battery pack shows that 18650 cell innovations are not over yet"](https://www.teslarati.com/tesla-model-s-plaid-battery-swan-song/). _TESLARATI_. [Archived](https://web.archive.org/web/20230620013543/https://www.teslarati.com/tesla-model-s-plaid-battery-swan-song/) from the original on June 20, 2023. Retrieved June 20, 2023.
  176. ^ _**a**_ _**b**_ Prince, Max (March 28, 2014). ["Meet the up-armored, titanium-shielded Tesla Model S"](https://www.roadandtrack.com/go/news/tesla-model-s-titanium-underbody-shield). _Road & Track_. [Archived](https://web.archive.org/web/20210105192234/https://www.roadandtrack.com/new-cars/news/a7483/tesla-model-s-titanium-underbody-shield/) from the original on January 5, 2021. Retrieved October 6, 2020.
  177. **^** Lee, Timothy (December 13, 2021). ["Why battery costs have plunged 89 percent since 2010"](https://fullstackeconomics.com/untitled-2/). _Full Stack Economics_. [Archived](https://web.archive.org/web/20220130153727/https://fullstackeconomics.com/untitled-2/) from the original on January 30, 2022. Retrieved January 30, 2022. "Tesla hasn't shared its exact battery costs with BloombergNEF, but the group estimates Tesla spends $112 per kWh – 15 percent below the industry average of $132."
  178. **^** Fisher, Thomas (June 11, 2013). ["What Goes into A Tesla Model S Battery – And What It May Cost"](https://web.archive.org/web/20190503210702/https://www.greencarreports.com/news/1084682_what-goes-into-a-tesla-model-s-battery--and-what-it-may-cost). _Green Car Reports_. Archived from [the original](http://www.greencarreports.com/news/1084682_what-goes-into-a-tesla-model-s-battery--and-what-it-may-cost) on May 3, 2019. Retrieved February 11, 2014.
  179. **^** ["Panasonic to expand battery capacity at Tesla Gigafactory"](https://techcrunch.com/2020/09/08/panasonic-to-expand-battery-capacity-at-tesla-gigafactory/). _TechCrunch_. September 8, 2020. Retrieved April 7, 2021.
  180. **^** Gurskiy, Denis (January 27, 2021). ["Tesla Factory: Stats, Production, History, and Delivery Numbers by Gigafactory"](https://evbite.com/tesla-factory-stats-production-history-and-delivery-numbers-by-gigafactory/). _EVBite_. [Archived](https://web.archive.org/web/20210127153104/https://evbite.com/tesla-factory-stats-production-history-and-delivery-numbers-by-gigafactory/) from the original on January 27, 2021. Retrieved February 1, 2021.
  181. **^** Lambert, Fred (January 19, 2021). ["First look at Tesla's new structural battery pack that will power its future electric cars"](https://electrek.co/2021/01/19/tesla-structural-battery-pack-first-picture/). _Electrek_. [Archived](https://web.archive.org/web/20230603102219/https://electrek.co/2021/01/19/tesla-structural-battery-pack-first-picture/) from the original on June 3, 2023. Retrieved June 14, 2023.
  182. **^** Hawkins, Andrew J. (September 22, 2020). ["Tesla announces "tabless" battery cells that will improve range of its electric cars"](https://www.theverge.com/2020/9/22/21449238/tesla-electric-car-battery-tabless-cells-day-elon-musk). _[The Verge](https://en.wikipedia.org/wiki/The_Verge "The Verge")_. [Archived](https://web.archive.org/web/20210118165249/https://www.theverge.com/2020/9/22/21449238/tesla-electric-car-battery-tabless-cells-day-elon-musk) from the original on January 18, 2021. Retrieved September 23, 2020.
  183. **^** Sanderson, Henry (February 19, 2020). ["Tesla's choice of cheaper lithium batteries hits cobalt miners"](https://www.ft.com/content/7264bdda-5310-11ea-90ad-25e377c0ee1f). _Financial Times_. [Archived](https://ghostarchive.org/archive/20221211231233/https://www.ft.com/content/7264bdda-5310-11ea-90ad-25e377c0ee1f) from the original on December 11, 2022. Retrieved May 6, 2020.
  184. **^** Lambert, Fred (April 22, 2022). ["Tesla is already using cobalt-free LFP batteries in half of its new cars produced"](https://electrek.co/2022/04/22/tesla-using-cobalt-free-lfp-batteries-in-half-new-cars-produced/). _[Electrek](https://en.wikipedia.org/wiki/Electrek "Electrek")_. [Archived](https://web.archive.org/web/20221112003108/https://electrek.co/2022/04/22/tesla-using-cobalt-free-lfp-batteries-in-half-new-cars-produced/) from the original on November 12, 2022. Retrieved November 12, 2022.
  185. **^** Hanley, Steve (May 11, 2021). ["Tesla Transitions To LFP Battery Cells For Megapack Installations"](https://cleantechnica.com/2021/05/11/tesla-transitions-to-lfp-battery-cells-for-megapack-installations/). _CleanTechnica_. [Archived](https://web.archive.org/web/20230818053632/https://cleantechnica.com/2021/05/11/tesla-transitions-to-lfp-battery-cells-for-megapack-installations/) from the original on August 18, 2023. Retrieved August 18, 2023.
  186. **^**

     * MacQueen, Caitlyn (January 19, 2021). ["Advanced Battery Scientists Join Excludive Tesla Partnership at DAL in Research Chair Roles"](https://www.dal.ca/news/2021/01/19/advanced-battery-scientists-join-exclusive-tesla-partnership-at-.html). _[Dalhousie University](https://en.wikipedia.org/wiki/Dalhousie_University "Dalhousie University")_. [Archived](https://web.archive.org/web/20210821182257/https://www.dal.ca/news/2021/01/19/advanced-battery-scientists-join-exclusive-tesla-partnership-at-.html) from the original on August 21, 2021. Retrieved August 21, 2021.
     * ["Tesla Motors signs first Canadian university research agreement with Dalhousie University"](https://www.dal.ca/news/media/media-releases/2015/06/17/tesla_motors_sign_first_canadian_university_research_agreement_with_dalhousie_university.html) (Press release). [Dalhousie University](https://en.wikipedia.org/wiki/Dalhousie_University "Dalhousie University"). June 17, 2015. [Archived](https://web.archive.org/web/20210821182256/https://www.dal.ca/news/media/media-releases/2015/06/17/tesla_motors_sign_first_canadian_university_research_agreement_with_dalhousie_university.html) from the original on August 21, 2021. Retrieved August 21, 2021.
     * Shirouzu, Norihiko; Lienert, Paul (May 14, 2020). ["How Tesla tapped a tiny Canadian lab for battery breakthroughs"](https://www.reuters.com/article/us-autos-tesla-batteries-laboratory/how-tesla-tapped-a-tiny-canadian-lab-for-battery-breakthroughs-idUSKBN22Q1WK). _[Reuters](https://en.wikipedia.org/wiki/Reuters "Reuters")_. [Archived](https://web.archive.org/web/20210821182255/https://www.reuters.com/article/us-autos-tesla-batteries-laboratory/how-tesla-tapped-a-tiny-canadian-lab-for-battery-breakthroughs-idUSKBN22Q1WK) from the original on August 21, 2021. Retrieved August 21, 2021.

  187. **^** Lambert, Fred (January 21, 2020). ["Elon Musk: Tesla acquisition of Maxwell is going to have a very big impact on batteries"](https://electrek.co/2020/01/21/tesla-acquisition-maxwell-big-impact-battery-elon-musk/). _[Electrek](https://en.wikipedia.org/wiki/Electrek "Electrek")_. [Archived](https://web.archive.org/web/20210523051630/https://electrek.co/2020/01/21/tesla-acquisition-maxwell-big-impact-battery-elon-musk/) from the original on May 23, 2021. Retrieved August 21, 2021.
  188. **^** Lambert, Fred (July 21, 2021). ["Tesla (TSLA) sells back Maxwell Technology's ultracapacitor business to former executives"](https://electrek.co/2021/07/21/tesla-tsla-sells-back-maxwell-technology-ultracapacitor-business-to-former-executives/). _[Electrek](https://en.wikipedia.org/wiki/Electrek "Electrek")_. [Archived](https://web.archive.org/web/20210821202834/https://electrek.co/2021/07/21/tesla-tsla-sells-back-maxwell-technology-ultracapacitor-business-to-former-executives/) from the original on August 21, 2021. Retrieved August 21, 2021.
  189. **^** ["Tesla Adds Hibar Systems To Its List Of Acquisitions"](https://cleantechnica.com/2019/10/06/tesla-adds-hibar-systems-to-its-list-of-acquisitions/). _[CleanTechnica](https://en.wikipedia.org/wiki/CleanTechnica "CleanTechnica")_. October 6, 2019. [Archived](https://web.archive.org/web/20201120044143/https://cleantechnica.com/2019/10/06/tesla-adds-hibar-systems-to-its-list-of-acquisitions/) from the original on November 20, 2020. Retrieved February 9, 2020.
  190. **^** Palmer, Annie (October 7, 2019). ["Tesla reportedly bought a company that specializes in high-speed battery manufacturing"](https://www.cnbc.com/2019/10/07/tesla-reportedly-bought-a-company-for-high-speed-battery-manufacturing.html). [CNBC](https://en.wikipedia.org/wiki/CNBC "CNBC"). [Archived](https://web.archive.org/web/20200930200815/https://www.cnbc.com/2019/10/07/tesla-reportedly-bought-a-company-for-high-speed-battery-manufacturing.html) from the original on September 30, 2020. Retrieved February 9, 2020.
  191. **^** ["Tesla taps tiny startup's tech to build cheaper, cleaner batteries"](https://techcrunch.com/2021/05/04/tesla-taps-tiny-startups-tech-to-build-cheaper-cleaner-batteries/). _[TechCrunch](https://en.wikipedia.org/wiki/TechCrunch "TechCrunch")_. May 4, 2021. [Archived](https://web.archive.org/web/20210821183046/https://techcrunch.com/2021/05/04/tesla-taps-tiny-startups-tech-to-build-cheaper-cleaner-batteries/) from the original on August 21, 2021. Retrieved August 21, 2021.
  192. **^** Bonifacic, Igor (May 4, 2021). ["Tesla may have paid $3 to buy patents for making cleaner EV batteries"](https://www.engadget.com/tesla-springpower-international-patents-190100400.html). _[Engadget](https://en.wikipedia.org/wiki/Engadget "Engadget")_. [Archived](https://web.archive.org/web/20210821191754/https://www.engadget.com/tesla-springpower-international-patents-190100400.html) from the original on August 21, 2021. Retrieved August 21, 2021.
  193. ^ _**a**_ _**b**_ Rogers, Chase (May 8, 2023). ["Abbott, Musk share stage for Tesla lithium refinery groundbreaking in South Texas"](https://www.caller.com/story/news/local/2023/05/08/abbott-musk-attend-tesla-lithium-refinery-groundbreaking-in-south-texas/70194611007/). _Corpus Christi Caller-Times_. Retrieved January 30, 2025.
  194. **^** Carlson, Kara (January 7, 2025). ["Musk's Massive Tesla Lithium Plant Hunts for Water in Drought-Hit Texas"](https://financialpost.com/pmn/business-pmn/musks-massive-tesla-lithium-plant-hunts-for-water-in-drought-hit-texas). _Financial Post_. Retrieved January 30, 2025.
  195. **^** Lambert, Fred (July 19, 2017). ["Tesla's over-the-air software updates make other vehicles 'highly vulnerable to obsolescence', says analyst"](https://electrek.co/2017/07/19/tesla-software-updates-vs-auto-industry/). _Electrek_. [Archived](https://web.archive.org/web/20201109023738/https://electrek.co/2017/07/19/tesla-software-updates-vs-auto-industry/) from the original on November 9, 2020. Retrieved December 18, 2020.
  196. **^** Lambert, Fred (October 3, 2023). ["Tesla just made your car safer after a crash through a software update"](https://electrek.co/2023/10/03/tesla-made-car-safer-after-crash-through-software-update/). _Electrek_. [Archived](https://web.archive.org/web/20231003191544/https://electrek.co/2023/10/03/tesla-made-car-safer-after-crash-through-software-update/) from the original on October 3, 2023. Retrieved October 3, 2023.
  197. **^** Lambert, Fred (February 15, 2020). ["Tesla starts selling rear-heated seats on Model 3 SR and SR Plus as $300 OTA upgrade"](https://electrek.co/2020/02/15/tesla-rear-heated-seats-model-3-ota-upgrades/). _Electrek_. [Archived](https://web.archive.org/web/20201222144719/https://electrek.co/2020/02/15/tesla-rear-heated-seats-model-3-ota-upgrades/) from the original on December 22, 2020. Retrieved September 1, 2020.
  198. **^** Baldwin, Roberto (June 22, 2020). ["Musk Announces Tesla Basic Autopilot Deal"](https://www.caranddriver.com/news/a32935630/musk-tesla-autopilot-upgrade-deal-june/). _Car and Driver_. [Archived](https://web.archive.org/web/20201023114841/https://www.caranddriver.com/news/a32935630/musk-tesla-autopilot-upgrade-deal-june/) from the original on October 23, 2020. Retrieved August 26, 2020.
  199. **^** ["Tesla's 3-Phase 4-Pole AC Induction Motor – Why Nikola Tesla's 19th Century Induction Motor Is The Ideal Choice for the 21st Century Electric Car"](https://cleantechnica.com/2016/05/30/nikola-teslas-19th-century-induction-motor-ideal-choice-21st-century-electric-car/). _CleanTechnica_. May 30, 2016. [Archived](https://web.archive.org/web/20210130224642/https://cleantechnica.com/2016/05/30/nikola-teslas-19th-century-induction-motor-ideal-choice-21st-century-electric-car/) from the original on January 30, 2021. Retrieved January 26, 2021.
  200. **^** ["Tesla Model 3's IPM-SynRM electric motor explained"](https://uk.motor1.com/news/462107/video-tesla-model-3-electric-motor-explained/). _Motor1_. Retrieved July 9, 2024.
  201. **^** Gaddam, Yogeshwari S. (January 8, 2021). ["Tesla Model 3's IPM-SynRM electric motor"](https://www.lesics.com/tesla-model-3_s-ipm-synrm-electric-motor.html). _Lesics_. Retrieved July 9, 2024.
  202. **^** ["Motor technology from Model 3 helps Tesla boost Model S range 10%"](https://arstechnica.com/cars/2019/04/motor-technology-from-model-3-helps-tesla-boost-model-s-range-10/). _[ArsTechnica](https://en.wikipedia.org/wiki/ArsTechnica "ArsTechnica")_. April 24, 2019. [Archived](https://web.archive.org/web/20210121072656/https://arstechnica.com/cars/2019/04/motor-technology-from-model-3-helps-tesla-boost-model-s-range-10/) from the original on January 21, 2021. Retrieved November 3, 2019.
  203. **^** Lambert, Fred (October 19, 2023). ["Toyota signs deal with Tesla for NACS and Supercharger access"](https://electrek.co/2023/10/19/toyota-signs-deal-tesla-nacs-supercharger-access/). _Electrek_. [Archived](https://web.archive.org/web/20231020085747/https://electrek.co/2023/10/19/toyota-signs-deal-tesla-nacs-supercharger-access/) from the original on October 20, 2023. Retrieved October 19, 2023.
  204. **^** ["Tesla Autopilot – The Ultimate Guide"](https://www.findmyelectric.com/tesla-autopilot-ultimate-guide/). _Find My Electric_. Retrieved May 17, 2025.
  205. **^** Lee, Timothy B. (May 7, 2021). ["Tesla Autopilot director contradicts Musk's self-driving timeline"](https://arstechnica.com/cars/2021/05/tesla-autopilot-director-contradicts-musks-self-driving-timeline/). _Ars Technica_. Retrieved July 22, 2021.
  206. **^** Torchinsky, Jason (July 7, 2023). ["Elon Musk Predicts Level 4 Or 5 Full Self-Driving 'Later This Year' For the Tenth Year In A Row"](https://www.theautopian.com/elon-musk-predicts-level-4-or-5-full-self-driving-later-this-year-for-the-tenth-year-in-a-row/). _The Autopian_. Retrieved July 9, 2023.
  207. **^** ["Autopilot vs. Autonomous | RoboticsTomorrow"](https://www.roboticstomorrow.com/article/2016/12/autopilot-vs-autonomous/9251). _roboticstomorrow.com_. Retrieved May 16, 2025.
  208. **^** Williams, Elliot (March 4, 2019). ["Does Tesla's Autosteer Make Cars Less Safe?"](https://hackaday.com/2019/03/04/does-teslas-autosteer-make-cars-less-safe/). _Hackaday_. Retrieved March 5, 2019.
  209. **^** ["Tesla Vehicle Safety Report"](https://www.tesla.com/VehicleSafetyReport). _tesla.com_. Retrieved April 29, 2024. "In the 4th quarter [of 2023], we recorded one crash for every 5.39 million miles driven in which drivers were using Autopilot technology. For drivers who were not using Autopilot technology, we recorded one crash for every 1.00 million miles driven. By comparison, the most recent data available from NHTSA and FHWA (from 2022) shows that in the United States there was an automobile crash approximately every 670,000 miles."
  210. **^** Shepardson, David (March 18, 2021). ["U.S. safety agency reviewing 23 Tesla crashes, three from recent weeks"](https://www.reuters.com/article/us-tesla-crash-idUSKBN2BA2ML). _Reuters_. Retrieved May 19, 2021.
  211. **^** Hawkins, Andrew J. (October 22, 2020). ["Tesla's "Full Self-Driving" beta is here, and it looks scary as hell"](https://www.theverge.com/2020/10/22/21528508/tesla-full-self-driving-beta-first-reaction-video). _The Verge_. Retrieved January 2, 2021.
  212. **^** Mitrache, Vlad (October 26, 2020). ["Full Self-Driving Beta Release Is Tesla's Most Irresponsible Move so Far"](https://www.autoevolution.com/news/full-self-driving-beta-release-is-tesla-s-most-irresponsible-move-so-far-150642.html). _autoevolution_. Retrieved January 2, 2021.
  213. **^** ["Should Tesla be 'beta testing' autopilot if there is a chance someone might die?"](https://www.theguardian.com/technology/2016/jul/06/tesla-autopilot-fatal-crash-public-beta-testing). _The Guardian_. July 6, 2016. Retrieved January 2, 2021.
  214. **^** Widen, William H.; Koopman, Philip (September 27, 2021). "Autonomous Vehicle Regulation, Does Tesla's Full Self-Driving Beta Release Comply with Law?". [SSRN](https://en.wikipedia.org/wiki/SSRN_\(identifier\) "SSRN \(identifier\)") [3931341](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=3931341).
  215. **^** Muoio, Danielle (November 1, 2016). ["Elon Musk: Tesla is developing a special kind of glass for its Model 3"](https://www.businessinsider.com/elon-musk-tesla-model-3-will-use-glass-from-solar-roof-2016-11). _[Business Insider](https://en.wikipedia.org/wiki/Business_Insider "Business Insider")_. [Archived](https://web.archive.org/web/20210821182254/https://www.businessinsider.com/elon-musk-tesla-model-3-will-use-glass-from-solar-roof-2016-11) from the original on August 21, 2021. Retrieved August 21, 2021.
  216. **^** Lambert, Fred (June 4, 2020). ["Tesla quietly acquired automated manufacturing firm to design factories"](https://electrek.co/2020/06/04/tesla-acquired-automated-manufacturing-firm-compass-automation/). _Electrek_. [Archived](https://web.archive.org/web/20210316163336/https://electrek.co/2020/06/04/tesla-acquired-automated-manufacturing-firm-compass-automation/) from the original on March 16, 2021. Retrieved March 28, 2021.
  217. **^** ["Tesla just bought an automation company to help it build the factory of the future — here's what we know about it"](https://www.businessinsider.com/tesla-buys-perbix-facts-details-2017-11). _Business Insider_. November 7, 2017. [Archived](https://web.archive.org/web/20200627200348/https://www.businessinsider.com/tesla-buys-perbix-facts-details-2017-11) from the original on June 27, 2020. Retrieved March 28, 2021.
  218. **^** Bomey, Nathan. ["A first in Michigan: Tesla buys Grand Rapids auto supplier"](https://www.freep.com/story/money/cars/2015/05/06/tesla-motors-acquisition-riviera-tool-grand-rapids/70916758/). _Detroit Free Press_. [Archived](https://web.archive.org/web/20211028172934/https://www.freep.com/story/money/cars/2015/05/06/tesla-motors-acquisition-riviera-tool-grand-rapids/70916758/) from the original on October 28, 2021. Retrieved October 11, 2021.
  219. **^** ["Giga Presses – the giant die casts that are reshaping car manufacturing"](https://europe.autonews.com/suppliers/giga-presses-help-toyota-volvo-hyundai-cut-production-costs). US. Reuters. February 10, 2023. [Archived](https://web.archive.org/web/20230228212541/https://europe.autonews.com/suppliers/giga-presses-help-toyota-volvo-hyundai-cut-production-costs) from the original on February 28, 2023. Retrieved March 1, 2023 – via Automotive News.
  220. **^** Inagaki, Kana; Campbell, Peter; Keohane, David (November 6, 2023). ["Toyota takes on Tesla's gigacasting in battle for carmaking's future"](https://www.ft.com/content/08048b42-ce72-4b64-9e0e-d15fbc98a9da). _Financial Times_. [Archived](https://web.archive.org/web/20231111213609/https://www.ft.com/content/08048b42-ce72-4b64-9e0e-d15fbc98a9da) from the original on November 11, 2023. Retrieved November 11, 2023.
  221. **^** ["Why are other automakers chasing Tesla's 'Gigacasting'?"](https://www.reuters.com/technology/why-are-other-automakers-chasing-teslas-gigacasting-2023-06-14/). _Reuters_. June 14, 2023. Retrieved May 26, 2024.
  222. **^** Bobrowsky, Meghan; Elliott, Rebecca (September 30, 2022). ["Elon Musk Unveils Prototype of Tesla's Humanoid Robot Optimus, Says It Will Cost Less Than a Car"](https://www.wsj.com/articles/tesla-ai-day-2022-elon-musk-11664536415). _The Wall Street Journal_. [ISSN](https://en.wikipedia.org/wiki/ISSN_\(identifier\) "ISSN \(identifier\)") [0099-9660](https://search.worldcat.org/issn/0099-9660). [Archived](https://web.archive.org/web/20221016133724/https://www.wsj.com/articles/tesla-ai-day-2022-elon-musk-11664536415) from the original on October 16, 2022. Retrieved October 2, 2022.
  223. **^** ["Q4 2023 Shareholder Deck"](https://digitalassets.tesla.com/tesla-contents/image/upload/IR/TSLA-Q4-2024-Update.pdf) (PDF). _Tesla, Inc_. January 29, 2025. Retrieved January 29, 2025.
  224. **^** Korosec, Kirsten. ["Tesla to reduce on-site staff at Nevada gigafactory by 75%"](https://techcrunch.com/2020/03/27/tesla-to-reduce-on-site-staff-at-nevada-gigafactory-by-75/). _TechCrunch_. [Archived](https://web.archive.org/web/20201212115420/https://techcrunch.com/2020/03/27/tesla-to-reduce-on-site-staff-at-nevada-gigafactory-by-75/) from the original on December 12, 2020. Retrieved February 2, 2021.
  225. **^** ["Tesla Factory"](https://www.tesla.com/factory). _Tesla_. March 28, 2022. [Archived](https://web.archive.org/web/20220628005146/https://www.tesla.com/factory) from the original on June 28, 2022.
  226. ^ _**a**_ _**b**_ Johnston, Adam (January 8, 2016). ["Tesla Starts Off 2016 By Producing & Delivering Powerwall"](https://cleantechnica.com/2016/01/08/tesla-starts-off-2016-producing-delivering-powerwall/). _CleanTechnica_. [Archived](https://web.archive.org/web/20170108003136/https://cleantechnica.com/2016/01/08/tesla-starts-off-2016-producing-delivering-powerwall/) from the original on January 8, 2017. Retrieved January 6, 2017.
  227. **^** Damon, Anjeanette. ["Worker injuries, 911 calls, housing crisis: Recruiting Tesla exacts a price"](https://www.usatoday.com/in-depth/news/investigations/2019/11/12/tesla-gigafactory-brings-nevada-jobs-and-housing-woes-worker-injuries-strained-ems/2452396001/). _USA Today_. [Archived](https://web.archive.org/web/20210131182404/https://www.usatoday.com/in-depth/news/investigations/2019/11/12/tesla-gigafactory-brings-nevada-jobs-and-housing-woes-worker-injuries-strained-ems/2452396001/) from the original on January 31, 2021. Retrieved February 2, 2021.
  228. **^** Loveday, Steven (March 31, 2021). ["New Nevada Tesla Semi Production Line May Build 5 Trucks Per Week"](https://insideevs.com/news/498003/tesla-semi-production-nevada-assembly-line/). _[InsideEVs](https://en.wikipedia.org/wiki/InsideEVs "InsideEVs")_. [Archived](https://web.archive.org/web/20211004224656/https://insideevs.com/news/498003/tesla-semi-production-nevada-assembly-line/) from the original on October 4, 2021. Retrieved April 5, 2021.
  229. ^ _**a**_ _**b**_ _**c**_ Ayre, James (September 7, 2017). ["Solar Roof Tile Production at Tesla's Buffalo "Gigafactory" Now Up & Running"](https://cleantechnica.com/2017/09/07/solar-roof-tile-production-teslas-buffalo-facility-now-running). _Clean Technica_. [Archived](https://web.archive.org/web/20210112210929/https://cleantechnica.com/2017/09/07/solar-roof-tile-production-teslas-buffalo-facility-now-running/) from the original on January 12, 2021. Retrieved September 8, 2017.
  230. **^** Hanley, Steve (February 28, 2020). ["Tesla Now Has 1,800 Employees in New York, Panasonic Quits Gigafactory 2 In Buffalo (The Solar One)"](https://cleantechnica.com/2020/02/28/tesla-now-has-1800-employees-in-new-york-panasonic-quits-gigafactory-2-in-buffalo-the-solar-one/). _CleanTechnica_. [Archived](https://web.archive.org/web/20200319001838/https://cleantechnica.com/2020/02/28/tesla-now-has-1800-employees-in-new-york-panasonic-quits-gigafactory-2-in-buffalo-the-solar-one/) from the original on March 19, 2020. Retrieved February 2, 2021.
  231. ^ _**a**_ _**b**_ Kolodny, Lora (October 23, 2019). ["Tesla shares soar after crushing third-quarter earnings"](https://www.cnbc.com/2019/10/23/tesla-tsla-earnings-q3-2019.html). [CNBC](https://en.wikipedia.org/wiki/CNBC "CNBC"). [Archived](https://web.archive.org/web/20201101122515/https://www.cnbc.com/2019/10/23/tesla-tsla-earnings-q3-2019.html) from the original on November 1, 2020. Retrieved October 23, 2019.
  232. **^** ["Tesla to Spend $188 Million to Expand Shanghai Factory"](https://www.thestreet.com/investing/tesla-invests-188-miin-shanghai-facility). _The Street_. November 26, 2021. [Archived](https://web.archive.org/web/20220718124525/https://www.thestreet.com/investing/tesla-invests-188-miin-shanghai-facility) from the original on July 18, 2022. Retrieved December 1, 2021.
  233. **^** Lambert, Fred (March 22, 2022). ["Elon Musk personally delivers first made-in Germany Tesla Model Y at Gigafactory Berlin"](https://electrek.co/2022/03/22/elon-musk-delivers-firs-germany-tesla-model-y-gigafactory-berlin/). _electrek.co_. [Archived](https://web.archive.org/web/20220718124525/https://electrek.co/2022/03/22/elon-musk-delivers-firs-germany-tesla-model-y-gigafactory-berlin/) from the original on July 18, 2022. Retrieved March 22, 2022.
  234. **^** ["New Tesla factory near Berlin to create 'up to 10,000 jobs'"](https://www.thelocal.de/20191113/up-to-10000-jobs-to-be-created-in-elon-musks-new-tesla-factory-near-berlin). _The Local Germany_. November 13, 2019. [Archived](https://web.archive.org/web/20210101214805/https://www.thelocal.de/20191113/up-to-10000-jobs-to-be-created-in-elon-musks-new-tesla-factory-near-berlin) from the original on January 1, 2021. Retrieved September 1, 2020.
  235. **^** Kane, Mark. ["Tesla's Elon Musk Shows Off Huge Progress at Giga Berlin"](https://insideevs.com/news/485334/tesla-elon-musk-giga-berlin-progress/). _InsideEVs_. [Archived](https://web.archive.org/web/20220714211215/https://insideevs.com/news/485334/tesla-elon-musk-giga-berlin-progress/) from the original on July 14, 2022. Retrieved February 3, 2021.
  236. **^** ["Elon Musk opens Tesla's Texas gigafactory with an all-night, neon-light 'Cyber Rodeo'"](https://fortune.com/2022/04/07/tesla-cyber-rodeo-texas-austin-gigafactory-elon-musk/). _Fortune_. April 9, 2022. [Archived](https://web.archive.org/web/20220718125258/https://fortune.com/2022/04/07/tesla-cyber-rodeo-texas-austin-gigafactory-elon-musk/) from the original on July 18, 2022. Retrieved April 9, 2022.
  237. **^** Vorrath, Sophie (July 22, 2020). ["Giga Texas! Austin to build Tesla's new Cybertruck and Tesla Semi"](https://thedriven.io/2020/07/23/giga-texas-austin-revealed-as-location-for-teslas-new-us-gigafactory/). _The Driven_. [Archived](https://web.archive.org/web/20210101213306/https://thedriven.io/2020/07/23/giga-texas-austin-revealed-as-location-for-teslas-new-us-gigafactory/) from the original on January 1, 2021. Retrieved August 26, 2020.
  238. **^** ["Elon Must: Over 10,000 people are needed for Giga Texas just through 2022!"](https://twitter.com/elonmusk/status/1377340744340361216). [Archived](https://web.archive.org/web/20220714223027/https://twitter.com/elonmusk/status/1377340744340361216) from the original on July 14, 2022. Retrieved August 21, 2021 – via Twitter.
  239. **^** Marshall, Matt (June 2, 2016). ["2006: San Carlos start-up Tesla seeks sexier electric car"](http://www.mercurynews.com/business/ci_26147213/from-archives-san-carlos-start-up-tesla-seeks). _Mercury News_. San Jose, California. [Archived](https://web.archive.org/web/20160608012754/http://www.mercurynews.com/business/ci_26147213/from-archives-san-carlos-start-up-tesla-seeks) from the original on June 8, 2016. Retrieved June 7, 2016.
  240. **^** Gulker, Chris (September 28, 2010). ["Menlo Park's only auto factory assembles $100,000 cars"](https://inmenlo.com/2010/09/28/menlo-parks-only-auto-factory-assembles-100000-cars/). _InMenlo_. [Archived](https://web.archive.org/web/20210802184111/https://inmenlo.com/2010/09/28/menlo-parks-only-auto-factory-assembles-100000-cars/) from the original on August 2, 2021. Retrieved August 13, 2023.
  241. **^** Behrens, Zach (May 2, 2008). ["Tesla Opens First Dealership in Los Angeles"](https://web.archive.org/web/20080505050516/http://laist.com/2008/05/02/tesla_opens_fir.php). _LAist_. Archived from [the original](https://laist.com/2008/05/02/tesla_opens_fir.php) on May 5, 2008. Retrieved April 8, 2020.
  242. **^** ["Tesla moving headquarters and powertrain operations to Palo Alto"](http://www.mercurynews.com/news/ci_13146859?source=rss). _Mercury News_. August 17, 2009. [Archived](https://web.archive.org/web/20110629071830/http://www.mercurynews.com/news/ci_13146859?source=rss) from the original on June 29, 2011. Retrieved September 14, 2009.
  243. **^** Kiley, David (April 2, 2010). ["Goodbye, NUMMI: How a Plant Changed the Culture of Car-Making"](https://www.popularmechanics.com/cars/a5514/4350856/). _Popular Mechanics_. [Archived](https://web.archive.org/web/20210307211326/https://www.popularmechanics.com/cars/a5514/4350856/) from the original on March 7, 2021. Retrieved February 15, 2021.
  244. **^** Randall, Tom (January 4, 2017). ["Tesla Flips the Switch on the Gigafactory"](https://www.bloomberg.com/news/articles/2017-01-04/tesla-flips-the-switch-on-the-gigafactory). _Bloomberg.com_. [Archived](https://web.archive.org/web/20170104232426/https://www.bloomberg.com/news/articles/2017-01-04/tesla-flips-the-switch-on-the-gigafactory) from the original on January 4, 2017. Retrieved January 4, 2017.
  245. **^** Lambert, Fred (January 3, 2018). ["Tesla increases hiring effort at Gigafactory 1 to reach goal of 35 GWh of battery production"](https://electrek.co/2018/01/03/tesla-gigafactory-hiring-effort-battery-production/). _electrek.co_. [Archived](https://web.archive.org/web/20180713143504/https://electrek.co/2018/01/03/tesla-gigafactory-hiring-effort-battery-production/) from the original on July 13, 2018. Retrieved July 13, 2018.
  246. ^ _**a**_ _**b**_ Hidalgo, Jason (March 2, 2023). ["Nevada approves $330 million in tax incentives for Tesla electric semi facility. What we know"](https://www.rgj.com/story/news/money/business/2023/03/02/tesla-gets-330-million-tax-incentives-for-electric-semi-facility-nevada/69963517007/). _[Reno Gazette Journal](https://en.wikipedia.org/wiki/Reno_Gazette_Journal "Reno Gazette Journal")_. Retrieved August 17, 2023.
  247. **^** Damon, Anjeanette (September 16, 2014). ["Inside Nevada's $1.25 billion Tesla tax deal"](http://www.rgj.com/story/news/2014/09/04/nevada-strikes-billion-tax-break-deal-tesla/15096777/). _[Reno Gazette Journal](https://en.wikipedia.org/wiki/Reno_Gazette_Journal "Reno Gazette Journal")_. Retrieved November 3, 2016. "the company must invest a minimum of $3.5 billion in manufacturing equipment and real property in the state. Five other states charge no sales tax at all and 34 states, including Arizona and Texas, don't charges sales tax on manufacturing equipment."
  248. **^** Robinson, David (August 31, 2017). ["6 things to watch as Panasonic gears up to start production"](https://buffalonews.com/2017/08/31/6-things-watch-panasonic-gears-start-production/). The Buffalo News. [Archived](https://web.archive.org/web/20170903003031/http://buffalonews.com/2017/08/31/6-things-watch-panasonic-gears-start-production/) from the original on September 3, 2017. Retrieved March 7, 2020.
  249. **^** Christmann, Samantha (December 27, 2016). ["Panasonic will invest in Tesla's South Buffalo solar plant"](https://buffalonews.com/2016/12/27/panasonic-will-invest-solarcity/). _The Buffalo News_. [Archived](https://web.archive.org/web/20161228143931/http://buffalonews.com/2016/12/27/panasonic-will-invest-solarcity/) from the original on December 28, 2016. Retrieved December 27, 2016.
  250. **^** Bykowicz, Julie; Mann, Ted (July 6, 2023). ["New York State Built Elon Musk a $1 Billion Factory. 'It Was a Bad Deal.'"](https://www.wsj.com/articles/elon-musk-tesla-buffalo-new-york-solar-plant-1b634b9e). _[The Wall Street Journal](https://en.wikipedia.org/wiki/The_Wall_Street_Journal "The Wall Street Journal")_. [ISSN](https://en.wikipedia.org/wiki/ISSN_\(identifier\) "ISSN \(identifier\)") [0099-9660](https://search.worldcat.org/issn/0099-9660). [Archived](https://web.archive.org/web/20230807034155/https://www.wsj.com/articles/elon-musk-tesla-buffalo-new-york-solar-plant-1b634b9e) from the original on August 7, 2023. Retrieved August 7, 2023.
  251. **^** ["Tesla's newest factory is inside a fabric building"](https://www.clearspan.com/news/teslas-newest-factory-is-inside-a-fabric-building/) (Press release). US: ClearSpan. June 21, 2018. [Archived](https://web.archive.org/web/20240226234929/https://www.clearspan.com/news/teslas-newest-factory-is-inside-a-fabric-building/) from the original on February 26, 2024. Retrieved April 1, 2024.
  252. **^** Lambert, Fred (January 5, 2021). ["Tesla Gigafactory Texas hits hyperspeed with giant building coming up, new job postings"](https://electrek.co/2021/01/05/tesla-gigafactory-texas-hits-hyperspeed-giant-building-new-job-postings/). _Electrek_. [Archived](https://web.archive.org/web/20210207160702/https://electrek.co/2021/01/05/tesla-gigafactory-texas-hits-hyperspeed-giant-building-new-job-postings/) from the original on February 7, 2021. Retrieved February 3, 2021.
  253. **^** Lambert, Fred (December 1, 2021). ["Tesla announces it has officially moved its headquarters next to Gigafactory Texas"](https://electrek.co/2021/12/01/tesla-officially-moved-headquarters-gigafactory-texas/). _Electrek_. [Archived](https://web.archive.org/web/20211206114534/https://electrek.co/2021/12/01/tesla-officially-moved-headquarters-gigafactory-texas/) from the original on December 6, 2021. Retrieved December 6, 2021.
  254. **^** Campbell, Jason (September 23, 2021). ["Lathrop Lands Tesla Mega Battery Plant"](https://www.mantecabulletin.com/news/local-news/lathrop-lands-tesla-mega-battery-plant/). _Mantica / Ripon Bulletin_. [Archived](https://web.archive.org/web/20210924191338/https://www.mantecabulletin.com/news/local-news/lathrop-lands-tesla-mega-battery-plant/) from the original on September 24, 2021. Retrieved September 25, 2021.
  255. **^** Lee-Jones, Sarah (September 22, 2021). ["New Tesla Megafactory Breaks Ground in Lathrop, California"](https://teslanorth.com/2021/09/22/new-tesla-megafactory-breaks-ground-in-lathrop-california/). _Tesla North_. [Archived](https://web.archive.org/web/20210923033903/https://teslanorth.com/2021/09/22/new-tesla-megafactory-breaks-ground-in-lathrop-california/) from the original on September 23, 2021. Retrieved September 22, 2021.
  256. **^** Hull, Dana; Breslau, Karen (February 23, 2023). ["Newsom, Musk dedicate former HP headquarters in Palo Alto to Tesla engineers"](https://www.latimes.com/business/technology/story/2023-02-22/tesla-to-open-engineering-headquarters-in-palo-alto-california). _[Los Angeles Times](https://en.wikipedia.org/wiki/Los_Angeles_Times "Los Angeles Times")_. [Archived](https://web.archive.org/web/20230226064644/https://www.latimes.com/business/technology/story/2023-02-22/tesla-to-open-engineering-headquarters-in-palo-alto-california) from the original on February 26, 2023. Retrieved February 16, 2024.
  257. **^** Revell, Eric (July 23, 2024). ["Musk says Tesla's Mexico factory on pause over Trump's tariff pledge"](https://www.foxbusiness.com/politics/musk-says-teslas-mexico-factory-pause-over-trumps-tariff-pledge). _[Fox Business](https://en.wikipedia.org/wiki/Fox_Business "Fox Business")_. Retrieved July 25, 2024.
  258. **^** Leeds, Samson (June 28, 2009). ["Tesla opens Flagship Euro Store in London"](https://web.archive.org/web/20220122220141/http://www.sablogzone.com/carzone/?p=4686). _Top Car Zone_. Sablog zone. Archived from [the original](http://www.sablogzone.com/carzone/?p=4686) on January 22, 2022. Retrieved October 25, 2009.
  259. **^** Boston, William; Higgins, Tim (July 30, 2018). ["Tesla Explores Building Major Factory in Europe"](https://www.wsj.com/articles/tesla-explores-building-major-factory-in-europe-1532961828). _[The Wall Street Journal](https://en.wikipedia.org/wiki/The_Wall_Street_Journal "The Wall Street Journal")_. [Archived](https://web.archive.org/web/20210131043537/https://www.wsj.com/articles/tesla-explores-building-major-factory-in-europe-1532961828) from the original on January 31, 2021. Retrieved January 26, 2021.
  260. **^** Kane, Mark. ["Tesla's New Tilburg Factory Now Open"](https://insideevs.com/teslas-new-tilburg-factory-now-open/). InsideEVs. [Archived](https://web.archive.org/web/20160517153058/http://insideevs.com/teslas-new-tilburg-factory-now-open/) from the original on May 17, 2016. Retrieved January 5, 2017. "re-assembled after leaving Tesla's Fremont factory in California in order to meet domestic manufacturing/regulatory standards and to avoid extra EU taxation/import tariff rules. The 'final assembly' process reportedly takes about 2–3 hours per vehicle, but saves about ~10% worth of fees added to the EVs' pricing."
  261. **^** Tredway, Gareth (November 8, 2016). ["Tesla buys automated manufacturing specialist Grohmann"](http://automotivelogistics.media/news/tesla-buys-automated-manufacturing-specialist-grohmann-engineering). _Automotive Logistics_. [Archived](https://web.archive.org/web/20161220194319/http://automotivelogistics.media/news/tesla-buys-automated-manufacturing-specialist-grohmann-engineering) from the original on December 20, 2016. Retrieved December 20, 2016.
  262. **^** Linden, Fritz-Peter (April 6, 2017). ["Demnächst nur noch ein einziger Kunde für Tesla Grohmann in Prüm"](http://www.volksfreund.de/nachrichten/region/pruem/aktuell/Heute-in-der-Pruemer-Zeitung-Demnaechst-nur-noch-ein-einziger-Kunde-fuer-Tesla-Grohmann-in-Pruem;art8111,4623412) [Next, only a single customer for Tesla Grohmann in Prüm] (in German). Volksfreund.de. [Archived](https://web.archive.org/web/20180811195440/https://www.volksfreund.de/region/pruem/demnaechst-nur-noch-ein-einziger-kunde-fuer-tesla-grohmann-in-pruem_aid-4918484) from the original on August 11, 2018. Retrieved April 8, 2017. "We need all capacities in Prüm to drive the production of the Model 3 in large numbers. "a fast and smooth transfer of current customers to other suppliers" is being carried out."
  263. **^** Lambert, Fred (November 8, 2016). ["Tesla plans to choose location for 'Gigafactory 2' in Europe next year, will produce both batteries and cars"](https://electrek.co/2016/11/08/tesla-location-gigafactory-2-europe-2017-both-batteries-and-cars/). _electrek.co_. [Archived](https://web.archive.org/web/20170812210958/https://electrek.co/2016/11/08/tesla-location-gigafactory-2-europe-2017-both-batteries-and-cars/) from the original on August 12, 2017. Retrieved December 11, 2016.
  264. **^** Lambert, Fred (January 8, 2017). ["The race to get 'Tesla Gigafactory 2' heats up, French Minister visits Fremont factory"](https://electrek.co/2017/01/08/tesla-gigafactory-2-french-minister/). _Electrek_. [Archived](https://web.archive.org/web/20210121011523/https://electrek.co/2017/01/08/tesla-gigafactory-2-french-minister/) from the original on January 21, 2021. Retrieved January 8, 2017.
  265. **^** Remondini, Chiara; Rauwald, Christoph (November 12, 2019). ["Tesla Plans to Build Next Factory in Berlin, Elon Musk Says"](https://www.bloomberg.com/amp/news/articles/2019-11-12/tesla-plans-to-build-next-factory-in-berlin-musk-says). Bloomberg L.P. [Archived](https://web.archive.org/web/20200817215412/https://www.bloomberg.com/amp/news/articles/2019-11-12/tesla-plans-to-build-next-factory-in-berlin-musk-says) from the original on August 17, 2020. Retrieved November 12, 2019.
  266. **^** ["Mandatory Musk: Tesla is building a factory in Brandenburg"](http://www.german-times.com/mandatory-musk-tesla-is-building-a-factory-in-brandenburg/). _The German Times_. October 1, 2020. [Archived](https://web.archive.org/web/20220529001953/https://www.german-times.com/mandatory-musk-tesla-is-building-a-factory-in-brandenburg/) from the original on May 29, 2022. Retrieved May 30, 2022.
  267. ^ _**a**_ _**b**_ ["Tesla's first European Gigafactory opens near Berlin"](https://www.dw.com/en/teslas-first-european-gigafactory-opens-near-berlin/a-60006610). Deutsche Welle. March 22, 2022. [Archived](https://web.archive.org/web/20220326000521/https://www.dw.com/en/teslas-first-european-gigafactory-opens-near-berlin/a-60006610) from the original on March 26, 2022. Retrieved March 26, 2022.
  268. **^** Dawson, Chester; Takahashi, Yoshio (November 15, 2010). ["Tesla Plans Japan Push"](https://www.wsj.com/articles/SB10001424052748703305404575610031563449408). _[The Wall Street Journal](https://en.wikipedia.org/wiki/The_Wall_Street_Journal "The Wall Street Journal")_. [Archived](https://web.archive.org/web/20160407124159/http://www.wsj.com/articles/SB10001424052748703305404575610031563449408) from the original on April 7, 2016. Retrieved February 15, 2021.
  269. **^** Dawson, Chester; Takahashi, Yoshio (November 15, 2010). ["Tesla Plans Japan Push"](https://web.archive.org/web/20160407124159/http://www.wsj.com/articles/SB10001424052748703305404575610031563449408). _[The Wall Street Journal](https://en.wikipedia.org/wiki/The_Wall_Street_Journal "The Wall Street Journal")_. Archived from [the original](https://www.wsj.com/articles/SB10001424052748703305404575610031563449408) on April 7, 2016. Retrieved June 26, 2013.
  270. **^** ["Tesla to build factory in Shanghai"](https://www.bbc.com/news/business-44789823). _BBC News_. July 11, 2018. [Archived](https://web.archive.org/web/20200808174607/https://www.bbc.com/news/business-44789823) from the original on August 8, 2020. Retrieved August 6, 2018.
  271. **^** ["Annual Report On Form 10-K For The Year Ended December 31, 2024"](https://www.sec.gov/ix?doc=/Archives/edgar/data/1318605/000162828025003063/tsla-20241231.htm). _United States Securities and Exchange Commission_. January 22, 2025. p. 90. Retrieved February 6, 2024.
  272. **^** ["Tesla's China sales hit record high in 2024, bucking global decline"](https://www.reuters.com/business/autos-transportation/teslas-china-sales-rise-record-high-83000-december-2025-01-03/). _[Reuters](https://en.wikipedia.org/wiki/Reuters "Reuters")_. January 3, 2025. Retrieved February 6, 2025.
  273. **^** Klender, Joey (April 12, 2023). ["Elon Musk reignites Tesla Gigafactory India rumors with one simple move"](https://www.teslarati.com/elon-musk-reignites-tesla-gigafactory-india-rumors/). _TESLARATI_. [Archived](https://web.archive.org/web/20230807193521/https://www.teslarati.com/elon-musk-reignites-tesla-gigafactory-india-rumors/) from the original on August 7, 2023. Retrieved August 7, 2023.
  274. **^** Das, Mehul Reuben (August 3, 2023). ["Tesla In India: Elon Musk's EV company sets up shop in India, leases office space in Pune"](https://www.firstpost.com/tech/news-analysis/tesla-in-india-elon-musks-ev-company-sets-up-shop-in-india-leases-office-space-in-pune-12954172.html). _Firstpost_. [Archived](https://web.archive.org/web/20230807063618/https://www.firstpost.com/tech/news-analysis/tesla-in-india-elon-musks-ev-company-sets-up-shop-in-india-leases-office-space-in-pune-12954172.html) from the original on August 7, 2023. Retrieved August 6, 2023.
  275. **^** Gupta, Poornima (January 7, 2010). ["Tesla, Panasonic partner on electric car batteries"](https://www.reuters.com/article/idUSN0721766720100107). _[Reuters](https://en.wikipedia.org/wiki/Reuters "Reuters")_. [Archived](https://web.archive.org/web/20210116065020/https://www.reuters.com/article/idUSN0721766720100107) from the original on January 16, 2021. Retrieved April 12, 2015.
  276. **^** ["Panasonic invests $30m in Tesla"](http://www.newstatesman.com/energy-and-clean-tech/2010/11/panasonic-battery-tesla). _[New Statesman](https://en.wikipedia.org/wiki/New_Statesman "New Statesman")_. [Archived](https://web.archive.org/web/20200925033711/https://www.newstatesman.com/energy-and-clean-tech/2010/11/panasonic-battery-tesla) from the original on September 25, 2020. Retrieved November 16, 2010.
  277. **^** ["Panasonic, Tesla agree to partnership for US car battery plant"](http://asia.nikkei.com/Business/Deals/Panasonic-Tesla-agree-to-partnership-for-US-car-battery-plant). _[Nikkei Inc](https://en.wikipedia.org/wiki/Nihon_Keizai_Shimbun "Nihon Keizai Shimbun")_. July 29, 2014. [Archived](https://web.archive.org/web/20201112025622/https://asia.nikkei.com/Business/Deals/Panasonic-Tesla-agree-to-partnership-for-US-car-battery-plant) from the original on November 12, 2020. Retrieved August 1, 2014.
  278. **^** Kolodny, Lora (February 26, 2020). ["Tesla, Panasonic will reportedly stop joint solar cell production at Gigafactory 2 in Buffalo"](https://web.archive.org/web/20200808130532/https://www.cnbc.com/2020/02/26/tesla-panasonic-said-to-end-joint-solar-cell-production-at-buffalo-factory.html). CNBC. Archived from [the original](https://www.cnbc.com/2020/02/26/tesla-panasonic-said-to-end-joint-solar-cell-production-at-buffalo-factory.html) on August 8, 2020. Retrieved September 1, 2020.
  279. **^** ["Japan's Panasonic to end solar panel production – domestic media"](https://www.reuters.com/article/panasonic-solar-idUSL1N2K6036). _Reuters_. January 31, 2021. [Archived](https://web.archive.org/web/20210208060850/https://www.reuters.com/article/panasonic-solar-idUSL1N2K6036) from the original on February 8, 2021. Retrieved February 8, 2021.
  280. **^** Inagaki, Kana (March 14, 2021). ["Panasonic to reduce Tesla reliance as battery tie-up evolves"](https://www.ft.com/content/e2949f70-6bae-4dcf-a9c3-5536c7d247eb). _[Financial Times](https://en.wikipedia.org/wiki/Financial_Times "Financial Times")_. [Archived](https://ghostarchive.org/archive/20221211231249/https://www.ft.com/content/e2949f70-6bae-4dcf-a9c3-5536c7d247eb) from the original on December 11, 2022. Retrieved March 17, 2021.
  281. **^** Jamasmie, Cecilia (September 28, 2020). ["Piedmont Lithium stock soars on confirmed Tesla deal"](https://www.mining.com/piedmont-lithium-soars-after-confirming-tesla-deal/). _mining.com_. [Archived](https://web.archive.org/web/20210316204716/https://www.mining.com/piedmont-lithium-soars-after-confirming-tesla-deal/) from the original on March 16, 2021. Retrieved March 13, 2021.
  282. **^** [Piedmont Lithium Signs Sales Agreement with Tesla](https://d1io3yog0oux5.cloudfront.net/_387a8ad2078edb4536fdea17b0a793e2/piedmontlithium/db/336/2620/pdf/2118399.pdf) [Archived](https://web.archive.org/web/20210126175847/https://d1io3yog0oux5.cloudfront.net/_387a8ad2078edb4536fdea17b0a793e2/piedmontlithium/db/336/2620/pdf/2118399.pdf) January 26, 2021, at the [Wayback Machine](https://en.wikipedia.org/wiki/Wayback_Machine "Wayback Machine"), September 28, 2020. Retrieved March 14, 2021.
  283. **^** ["NT opens first lithium mine, supplying Tesla"](https://www.pv-magazine-australia.com/2022/10/13/nt-opens-first-lithium-mine-supplying-tesla/). _PV Magazine_. October 13, 2022. [Archived](https://web.archive.org/web/20221015040012/https://www.pv-magazine-australia.com/2022/10/13/nt-opens-first-lithium-mine-supplying-tesla/) from the original on October 15, 2022. Retrieved October 15, 2022.
  284. **^** Kaufman, Alexander C. (August 24, 2015). ["Tesla Wants To Take Stress Out of Vacationing with an Electric Car"](https://www.huffingtonpost.com/entry/tesla-has-a-new-way-to-quell-range-anxiety_55db4405e4b04ae49703c584?kvcommref=mostpopular). _[HuffPost](https://en.wikipedia.org/wiki/HuffPost "HuffPost")_. [Archived](https://web.archive.org/web/20150914183518/http://www.huffingtonpost.com/entry/tesla-has-a-new-way-to-quell-range-anxiety_55db4405e4b04ae49703c584?kvcommref=mostpopular) from the original on September 14, 2015. Retrieved August 26, 2015.
  285. **^** Ross, Jeffrey N. (October 4, 2012). ["Mercedes B-Class headed to America... but only as an EV?"](http://www.autoblog.com/2012/10/04/mercedes-b-class-headed-to-america-but-only-as-an-ev/). [Autoblog.com](https://en.wikipedia.org/wiki/Autoblog.com "Autoblog.com"). [Archived](https://web.archive.org/web/20141106032216/http://www.autoblog.com/2012/10/04/mercedes-b-class-headed-to-america-but-only-as-an-ev/) from the original on November 6, 2014. Retrieved November 5, 2014.
  286. **^** Arrington, Michael (May 19, 2009). ["Tesla Worth More Than Half A Billion Dollars After Daimler Investment"](https://techcrunch.com/2009/05/19/tesla-worth-a-half-billion-dollars-after-daimler-investment/). Techcrunch.com. [Archived](https://web.archive.org/web/20090803173400/http://www.techcrunch.com/2009/05/19/tesla-worth-a-half-billion-dollars-after-daimler-investment/) from the original on August 3, 2009. Retrieved August 1, 2009.
  287. **^** Godske, Bjørn (May 21, 2010). ["Toyota buys $50mio stake in Tesla"](https://web.archive.org/web/20100523070707/http://ing.dk/artikel/109051-toyota-koeber-for-300-mio-kroner-aktier-i-tesla). _Ing.dk_. Archived from [the original](https://ing.dk/artikel/109051-toyota-koeber-for-300-mio-kroner-aktier-i-tesla) on May 23, 2010. Retrieved May 21, 2010.
  288. ^ _**a**_ _**b**_ ["Daimler changes Tesla board member in shift to hyrids and EV's"](https://www.torquenews.com/1075/daimler-changes-tesla-board-member-shift-hyrids-and-evs). _Torque News_. [Archived](https://web.archive.org/web/20210205043048/https://www.torquenews.com/1075/daimler-changes-tesla-board-member-shift-hyrids-and-evs) from the original on February 5, 2021. Retrieved January 26, 2021.
  289. **^** Atkins, Thomas (July 13, 2009). ["UAE'S Aabar buys 40 pct of Daimler's Tesla stake"](https://www.reuters.com/article/earningsSeason/idUSLD56046820090713). _[Reuters](https://en.wikipedia.org/wiki/Reuters "Reuters")_. [Archived](https://web.archive.org/web/20210319144413/https://www.reuters.com/article/earningsSeason/idUSLD56046820090713) from the original on March 19, 2021. Retrieved April 12, 2015.
  290. **^** Ramsey, Mike (October 21, 2014). ["Daimler sells Tesla stake for $780 Million"](https://www.marketwatch.com/story/daimler-sells-tesla-stake-for-780-million-2014-10-21). _[MarketWatch](https://en.wikipedia.org/wiki/MarketWatch "MarketWatch")_. [Archived](https://web.archive.org/web/20210821182255/https://www.marketwatch.com/story/daimler-sells-tesla-stake-for-780-million-2014-10-21) from the original on August 21, 2021. Retrieved August 21, 2021.
  291. **^** Peterson, Andrew (March 12, 2010). ["Tesla Motors to Provide Batteries for Freightliner Custom Chassis Electric Van"](https://www.motortrend.com/news/tesla-motors-to-provide-batteries-for-freightliner-custom-chassis-electric-van-6980/). _[Motor Trend](https://en.wikipedia.org/wiki/Motor_Trend "Motor Trend")_. [Archived](https://web.archive.org/web/20210821192750/https://www.motortrend.com/news/tesla-motors-to-provide-batteries-for-freightliner-custom-chassis-electric-van-6980/) from the original on August 21, 2021. Retrieved August 21, 2021.
  292. **^** ["10-K"](https://www.sec.gov/Archives/edgar/data/1318605/000119312514069681/d668062d10k.htm). _sec.gov_. [Archived](https://web.archive.org/web/20201021194954/https://www.sec.gov/Archives/edgar/data/1318605/000119312514069681/d668062d10k.htm) from the original on October 21, 2020. Retrieved August 24, 2020.
  293. **^** ["Mercedes-Benz Introduces the Battery-Powered A-Class E-CELL; Production Run of 500"](http://www.greencarcongress.com/2010/09/aclassecell-20100915.html). _Green Car Congress_. September 15, 2010. [Archived](https://archive.today/20130103080524/http://www.greencarcongress.com/2010/09/aclassecell-20100915.html) from the original on January 3, 2013. Retrieved May 4, 2011.
  294. **^** Masson, Laurent J (March 29, 2011). ["Quick Drive: Electric Mercedes A-Class E-Cell"](http://www.plugincars.com/short-test-electric-mercedes-class-e-cell-106984.html). _Plugin Cars_. [Archived](https://web.archive.org/web/20201020022934/https://www.plugincars.com/short-test-electric-mercedes-class-e-cell-106984.html) from the original on October 20, 2020. Retrieved May 4, 2011.
  295. **^** Halvorson, Bengt (August 7, 2017). ["Bye-Bye Baby B: Mercedes Spikes Its Electric Subcompact, Eyes More Mainstream EVs"](https://www.caranddriver.com/news/a15340139/bye-bye-baby-b-mercedes-spikes-its-electric-subcompact-eyes-more-mainstream-evs). _Car and Driver_. [Archived](https://web.archive.org/web/20190209180457/https://www.caranddriver.com/news/a15340139/bye-bye-baby-b-mercedes-spikes-its-electric-subcompact-eyes-more-mainstream-evs/) from the original on February 9, 2019. Retrieved February 9, 2019.
  296. **^** ["Mercedes-Benz B Class Electric Coming To U.S.: Report (Compliance Car Watch)"](http://www.greencarreports.com/news/1079700_mercedes-benz-b-class-electric-coming-to-u-s-report-compliance-car-watch). _Green Car Reports_. October 9, 2012. [Archived](https://web.archive.org/web/20230406034229/https://www.greencarreports.com/news/1079700_mercedes-benz-b-class-electric-coming-to-u-s-report-compliance-car-watch) from the original on April 6, 2023. Retrieved April 19, 2016.
  297. **^** Gordon-Bloomfield, Nikki (September 16, 2015). ["Report: Next-Generation Smart ForTwo Electric Drive Will Feature Renault-Made Motors"](https://web.archive.org/web/20170802032518/https://transportevolved.com/2015/09/16/report-next-generation-smart-fortwo-electric-drive-will-feature-renault-made-motors/). _Transport Evolved_. Archived from [the original](https://transportevolved.com/2015/09/16/report-next-generation-smart-fortwo-electric-drive-will-feature-renault-made-motors/) on August 2, 2017. Retrieved August 29, 2017.
  298. **^** Squatriglia, Chuck (January 13, 2009). ["Tesla Motors Joins Daimler On a Smart EV"](https://www.wired.com/2009/01/tesla-motors-jo/). _[Wired](https://en.wikipedia.org/wiki/Wired_\(magazine\) "Wired \(magazine\)")_. [ISSN](https://en.wikipedia.org/wiki/ISSN_\(identifier\) "ISSN \(identifier\)") [1059-1028](https://search.worldcat.org/issn/1059-1028). [Archived](https://web.archive.org/web/20210625015644/https://www.wired.com/2009/01/tesla-motors-jo/) from the original on June 25, 2021. Retrieved July 10, 2021.
  299. **^** Abuelsamid, Sam (July 16, 2010). ["Breaking: Tesla and Toyota to develop RAV4 EV, hope to launch in 2012"](https://www.autoblog.com/2010/07/16/breaking-tesla-and-toyota-to-develop-rav4-ev-hope-to-launch-in/). _[Weblogs, Inc.](https://en.wikipedia.org/wiki/Weblogs,_Inc. "Weblogs, Inc.")_ [Archived](https://web.archive.org/web/20210821190307/https://www.autoblog.com/2010/07/16/breaking-tesla-and-toyota-to-develop-rav4-ev-hope-to-launch-in/) from the original on August 21, 2021. Retrieved August 21, 2021.
  300. **^** ["Toyota unveils RAV4 EV demonstration vehicle; targeting fully-engineered version in 2012 for market"](https://www.greencarcongress.com/2010/11/rav4ev-20101117.html). _Green Car Congress_. November 17, 2010. [Archived](https://web.archive.org/web/20210821202843/https://www.greencarcongress.com/2010/11/rav4ev-20101117.html) from the original on August 21, 2021. Retrieved August 21, 2021.
  301. **^** Tellem, Tori (November 17, 2010). ["2012 Toyota RAV4-EV: Take Two"](https://wheels.blogs.nytimes.com/2010/11/17/2012-toyota-rav4-ev-take-two/?ref=automobiles). _[The New York Times](https://en.wikipedia.org/wiki/The_New_York_Times "The New York Times")_. [Archived](https://web.archive.org/web/20210821193312/https://wheels.blogs.nytimes.com/2010/11/17/2012-toyota-rav4-ev-take-two/?ref=automobiles) from the original on August 21, 2021. Retrieved August 21, 2021.
  302. **^** Garrett, Jerry (August 3, 2012). ["Toyota and Tesla Trot Out the RAV4 EV"](https://wheels.blogs.nytimes.com/2012/08/03/toyota-and-tesla-trot-out-the-rav4-ev/). _[The New York Times](https://en.wikipedia.org/wiki/The_New_York_Times "The New York Times")_. [Archived](https://web.archive.org/web/20210507091758/https://wheels.blogs.nytimes.com/2012/08/03/toyota-and-tesla-trot-out-the-rav4-ev/) from the original on May 7, 2021. Retrieved August 21, 2021.
  303. **^** ["Toyota RAV4 EV key for meeting California ZEV requirements; Tesla powertrain uses Model S components"](https://www.greencarcongress.com/2012/08/rav4ev-20120803.html). _Green Car Congress_. August 10, 2012. [Archived](https://web.archive.org/web/20210801205740/https://www.greencarcongress.com/2012/08/rav4ev-20120803.html) from the original on August 1, 2021. Retrieved August 21, 2021.
  304. **^** ["Toyota Wraps Up Production of RAV4 EV"](http://www.plugincars.com/toyota-wraps-production-rav4-ev-130150.html). _PluginCars.com_. September 29, 2014. [Archived](https://web.archive.org/web/20150702212253/http://www.plugincars.com/toyota-wraps-production-rav4-ev-130150.html) from the original on July 2, 2015. Retrieved August 28, 2018.
  305. **^** ["Don't look for a Toyota RAV4 EV successor anytime soon"](https://www.cnet.com/roadshow/news/toyota-rav4-ev-sounds-unlikely-hybrid-double-down/). _Roadshow_. April 3, 2018. [Archived](https://web.archive.org/web/20180828170359/https://www.cnet.com/roadshow/news/toyota-rav4-ev-sounds-unlikely-hybrid-double-down/) from the original on August 28, 2018. Retrieved August 28, 2018.
  306. ^ _**a**_ _**b**_ Trudell, Craig; Ohnsman, Alan (August 7, 2014). ["Why the Tesla-Toyota Partnership Short-Circuited"](https://www.bloomberg.com/news/articles/2014-08-07/tesla-toyota-deal-to-develop-electric-suv-fizzles). Bloomberg L.P. [Archived](https://web.archive.org/web/20160312053258/https://www.bloomberg.com/news/articles/2014-08-07/tesla-toyota-deal-to-develop-electric-suv-fizzles) from the original on March 12, 2016. Retrieved July 28, 2021.
  307. **^** Tajitsu, Naomi (June 5, 2017). ["Toyota dumps all its shares in Tesla as their tie-up ends"](https://www.businessinsider.com/tesla-stock-price-toyota-sells-all-shares-2017-6). _[Business Insider](https://en.wikipedia.org/wiki/Business_Insider "Business Insider")_. [Reuters](https://en.wikipedia.org/wiki/Reuters "Reuters"). [Archived](https://web.archive.org/web/20210821190307/https://www.businessinsider.com/tesla-stock-price-toyota-sells-all-shares-2017-6) from the original on August 21, 2021. Retrieved August 21, 2021.
  308. **^** Tajitsu, Naomi (June 3, 2017). ["Toyota sells all shares in Tesla as their tie-up ends"](https://www.reuters.com/article/us-toyota-tesla/toyota-sells-all-shares-in-tesla-as-their-tie-up-ends-idUSKBN18U05E). _[Reuters](https://en.wikipedia.org/wiki/Reuters "Reuters")_. [Archived](https://web.archive.org/web/20210821190308/https://www.reuters.com/article/us-toyota-tesla/toyota-sells-all-shares-in-tesla-as-their-tie-up-ends-idUSKBN18U05E) from the original on August 21, 2021. Retrieved August 21, 2021.
  309. **^** Lienert, Paul; Shirouzu, Norihiko; Taylor, Edward (September 22, 2020). ["The Musk Method: Learn from partners then go it alone"](https://www.reuters.com/article/us-tesla-batteryday-technology/the-musk-method-learn-from-partners-then-go-it-alone-idUKKBN2680IU). _[Reuters](https://en.wikipedia.org/wiki/Reuters "Reuters")_. [Archived](https://web.archive.org/web/20220602124001/https://www.reuters.com/article/us-tesla-batteryday-technology/the-musk-method-learn-from-partners-then-go-it-alone-idUKKBN2680IU) from the original on June 2, 2022. Retrieved February 15, 2021.
  310. **^** Ramsey, Mike (July 26, 2016). ["Mobileye Ends Partnership With Tesla"](https://www.wsj.com/articles/mobileye-ends-partnership-with-tesla-1469544028). _[The Wall Street Journal](https://en.wikipedia.org/wiki/The_Wall_Street_Journal "The Wall Street Journal")_. [ISSN](https://en.wikipedia.org/wiki/ISSN_\(identifier\) "ISSN \(identifier\)") [0099-9660](https://search.worldcat.org/issn/0099-9660). [Archived](https://web.archive.org/web/20201108142324/https://www.wsj.com/articles/mobileye-ends-partnership-with-tesla-1469544028) from the original on November 8, 2020. Retrieved February 3, 2020.
  311. **^** Wilmnan, Andy (April 2, 2011). ["Tesla Vs Top Gear: Executive Producer Andy Wilman Speaks"](https://www.jalopnik.com/tesla-vs-top-gear-executive-producer-andy-wilman-speak-5788297/). Jalopnik.
  312. **^** ["Tesla Motors Ltd & Anor v British Broadcasting Corporation (BBC), [2013] EWCA Civ 152"](https://www.casemine.com/judgement/uk/5a8ff7b460d03e7f57eb15c6). Casemine. March 5, 2013.
  313. **^** Stanley-Smith, Joe (March 23, 2025). ["Jeremy Clarkson taunts 'idiot' Elon Musk over Tesla vandalism"](https://www.politico.eu/article/jeremy-clarkson-idiot-elon-musk-tesla-vandalism/). POLITICO.
  314. **^** Bonifacic, Igor (December 15, 2021). ["Six more women sue Tesla over workplace sexual harassment"](https://techcrunch.com/2021/12/14/six-more-women-sue-tesla-over-workplace-sexual-harassment/). _TechCrunch_. US. Retrieved February 18, 2022.
  315. **^** Dillon, Nancy (December 15, 2021). ["Six Women Sue Tesla Alleging 'Rampant Sexual Harassment' at California Facilities"](https://www.rollingstone.com/culture/culture-news/tesla-sexual-harassment-lawsuits-1271824/). _Rolling Stone_. US. Retrieved February 18, 2022.
  316. ^ _**a**_ _**b**_ Siddiqui, Faiz (December 14, 2021). ["Six Tesla workers file additional lawsuits alleging sexual harassment"](https://www.washingtonpost.com/technology/2021/12/14/tesla-sexual-harassment/). _The Washington Post_. [ISSN](https://en.wikipedia.org/wiki/ISSN_\(identifier\) "ISSN \(identifier\)") [0190-8286](https://search.worldcat.org/issn/0190-8286). Retrieved February 18, 2022.
  317. **^** Ohnsman, Alan (December 14, 2021). ["Tesla Hit By 6 More Sexual Harassment Claims"](https://www.forbes.com/sites/alanohnsman/2021/12/14/tesla-hit-by-6-more-sexual-harassment-claims/). _Forbes_. US. Retrieved February 18, 2022.
  318. **^** Levin, Sam (June 1, 2017). ["Tesla fires female engineer who alleged sexual harassment"](https://www.theguardian.com/technology/2017/jun/01/tesla-fires-aj-vandermeyden-lawsuit-sexual-harrassment). _The Guardian_. UK. Retrieved February 18, 2022.
  319. **^** Trop, Jaclyn (May 25, 2022). ["Tesla sexual harassment suit can proceed in court"](https://techcrunch.com/2022/05/24/judge-rules-that-tesla-sexual-harassment-suit-can-proceed-in-court/). _TechCrunch_. US. Retrieved June 22, 2022.
  320. **^** ["Annual Report Form 10-K 2023 Tesla Inc"](https://www.sec.gov/Archives/edgar/data/1318605/000162828024002390/tsla-20231231.htm). [U.S. Securities and Exchange Commission](https://en.wikipedia.org/wiki/U.S._Securities_and_Exchange_Commission "U.S. Securities and Exchange Commission"). January 26, 2024. [Archived](https://web.archive.org/web/20240129182309/https://www.sec.gov/Archives/edgar/data/1318605/000162828024002390/tsla-20231231.htm) from the original on January 29, 2024. Retrieved February 8, 2024.
  321. **^** McGill, Kevin (October 29, 2024). ["Elon Musk wins a court battle over a 2018 post during a labor dispute"](https://apnews.com/article/elon-musk-uaw-twitter-nlrb-tesla-5fab2a42b9543f7702deed80d79c7d02). [Associated Press](https://en.wikipedia.org/wiki/Associated_Press "Associated Press"). [Archived](https://web.archive.org/web/20241029221955/https://apnews.com/article/elon-musk-uaw-twitter-nlrb-tesla-5fab2a42b9543f7702deed80d79c7d02) from the original on October 29, 2024. Retrieved October 30, 2024.
  322. **^** Budman, Scott (September 3, 2024). ["Protesters Call out Tesla for Being Only American Car Company Not Represented by Union"](https://www.nbcbayarea.com/news/local/east-bay/fremont-tesla-labor-day-protest/3641285/). _[NBC Bay Area](https://en.wikipedia.org/wiki/NBC_Bay_Area "NBC Bay Area")_. [Archived](https://web.archive.org/web/20240909115258/https://www.nbcbayarea.com/news/local/east-bay/fremont-tesla-labor-day-protest/3641285/) from the original on September 9, 2024. Retrieved September 15, 2024.
  323. **^** ["Tesla Lehnt Tarifverträge Für Beschäftigte Weiterhin Ab"](https://www.rbb24.de/wirtschaft/beitrag/2024/01/brandenburg-gruenheide-tesla-gegen-tarifbindung.html) [Tesla continues to reject collective agreements for workers]. _[Rundfunk Berlin-Brandenburg](https://en.wikipedia.org/wiki/Rundfunk_Berlin-Brandenburg "Rundfunk Berlin-Brandenburg")_ (in German). January 7, 2024. [Archived](https://web.archive.org/web/20240120172559/https://www.rbb24.de/wirtschaft/beitrag/2024/01/brandenburg-gruenheide-tesla-gegen-tarifbindung.html) from the original on January 20, 2024. Retrieved September 15, 2024.
  324. **^** Vetter, Philipp (October 18, 2017). ["Deutschland: Tesla Einigt Sich Auf Deutliche Gehaltssteigerung"](https://www.welt.de/wirtschaft/article169760888/Tesla-gibt-deutschen-Mitarbeitern-deutliche-Gehaltserhoehung.html) [Germany: Tesla agrees on significant salary increase]. _[Die Welt](https://en.wikipedia.org/wiki/Die_Welt "Die Welt")_ (in German). [Archived](https://web.archive.org/web/20180831121644/https://www.welt.de/wirtschaft/article169760888/Tesla-gibt-deutschen-Mitarbeitern-deutliche-Gehaltserhoehung.html) from the original on August 31, 2018. Retrieved December 24, 2021.
  325. **^** ["Neugewählter Tesla-Betriebsrat Bestätigt Michaela Schmitz Als Vorsitzende"](https://www.rbb24.de/wirtschaft/beitrag/2024/04/brandenburg-gruenheide-tesla-betriebsrat-konstituierende-sitzung-schmitz-vorsitzende.html) [Newly elected Tesla works council confirms Michaela Schmitz as chairwoman]. _[Rundfunk Berlin-Brandenburg](https://en.wikipedia.org/wiki/Rundfunk_Berlin-Brandenburg "Rundfunk Berlin-Brandenburg")_ (in German). April 5, 2024. [Archived](https://web.archive.org/web/20240825144831/https://www.rbb24.de/wirtschaft/beitrag/2024/04/brandenburg-gruenheide-tesla-betriebsrat-konstituierende-sitzung-schmitz-vorsitzende.html) from the original on August 25, 2024. Retrieved August 25, 2024.
  326. **^** [Kuhn, Gabriel](https://en.wikipedia.org/wiki/Gabriel_Kuhn "Gabriel Kuhn") (May 8, 2024). ["The Head of SAC: Tesla Faces a 'Stalemate' in Sweden as the Longest Strike in Decades Grinds On"](https://turningpointmag.org/2024/05/08/the-head-of-sac-tesla-faces-a-stalemate-in-sweden-as-the-longest-strike-in-decades-grinds-on/). _Turning Point_. [Archived](https://web.archive.org/web/20240614234038/https://turningpointmag.org/2024/05/08/the-head-of-sac-tesla-faces-a-stalemate-in-sweden-as-the-longest-strike-in-decades-grinds-on/) from the original on June 14, 2024. Retrieved June 14, 2024.
  327. **^** ["Tesla Labour Dispute Ignites Nordic Sympathy Strikes"](https://www.reuters.com/business/autos-transportation/tesla-labour-dispute-ignites-nordic-sympathy-strikes-2023-12-07/). _[Reuters](https://en.wikipedia.org/wiki/Reuters "Reuters")_. December 7, 2023. [Archived](https://web.archive.org/web/20231210062434/https://www.reuters.com/business/autos-transportation/tesla-labour-dispute-ignites-nordic-sympathy-strikes-2023-12-07/) from the original on December 10, 2023. Retrieved September 27, 2024.
  328. **^** ["Tesla revises nondisclosure clause as Musk accuses customers of 'fraud' on suspension claims"](https://www.cnbc.com/2016/06/11/tesla-revises-nondisclosure-clause-as-musk-accuses-customers-of-fraud-on-suspension-claims.html). CNBC. June 11, 2016. Retrieved October 19, 2021.
  329. **^** Kolodny, Lora (October 13, 2021). ["NHTSA asks Tesla why it didn't initiate a recall when it pushed safety-related software update"](https://www.cnbc.com/2021/10/13/nhtsa-asks-tesla-why-it-didnt-initiate-a-recall-after-safety-related-software-update.html). CNBC. Retrieved October 19, 2021.
  330. **^** Burgess, Christopher (August 30, 2018). ["Tesla insider with expired NDA spills the tech beans"](https://www.csoonline.com/article/3301290/tesla-insider-with-expired-nda-spills-the-tech-beans.html). _CSO Online_. Retrieved October 19, 2021.
  331. **^** Kolodny, Lora (October 12, 2021). ["Tesla invites more drivers to 'Full Self-Driving Beta' program – read the email here"](https://www.cnbc.com/2021/10/12/tesla-invites-drivers-to-fsd-beta-10point2-nda-restrictions-not-included-.html). CNBC. Retrieved October 19, 2021.
  332. **^** ["Tesla NDA Warns 'Self Driving' Beta Testers 'People Want Tesla to Fail'"](https://www.vice.com/en/article/how-teslas-self-driving-beta-testers-protect-the-company-from-critics/). _Vice (magazine)_. September 27, 2021. Retrieved October 19, 2021.
  333. **^** ["Tech workers at Tesla, Intel say NDAs have 'silenced' them"](https://www.hrdive.com/news/tech-workers-at-tesla-intel-say-ndas-have-silenced-them/532024/). _HR Dive_. Retrieved October 19, 2021.
  334. **^** Stumpf, Rob (March 3, 2019). ["Tesla Had 3 Times as Many OSHA Violations as the 10 Largest US Plants Combined"](https://www.thedrive.com/news/26727/tesla-had-3-times-as-many-osha-violations-as-the-10-largest-us-plants-combined). _The Drive_. Retrieved May 15, 2020.
  335. **^** ["Tesla says its factory is safer. But it left injuries off the books"](https://www.revealnews.org/article/tesla-says-its-factory-is-safer-but-it-left-injuries-off-the-books/). _Reveal_. April 16, 2018. Retrieved May 15, 2020.
  336. ^ _**a**_ _**b**_ O'Kane, Sean (March 13, 2019). ["Tesla allegedly hacked, spied on, and followed Gigafactory whistleblower: report"](https://www.theverge.com/2019/3/13/18263757/tesla-elon-musk-employee-hack-spying-whistleblower-gigafactory-martin-tripp-drugs). _The Verge_. Retrieved May 11, 2020.
  337. **^** O'Kane, Sean (March 11, 2019). ["Another former Tesla security manager says the company spied on employees"](https://www.theverge.com/2019/3/11/18259786/tesla-spying-employees-security-manager-whistleblower-gigafactory-nevada). _The Verge_. Retrieved May 11, 2020.
  338. **^** ["Gouthro v. Tesla Motors Inc :: Nevada District Court :: Federal Civil Lawsuit No. 2:20-cv-00286-GMN-BNW, Judge Gloria M. Navarro presiding"](https://www.plainsite.org/dockets/475n48t5b/nevada-district-court/gouthro-v-tesla-motors-inc/). _plainsite.org_. Retrieved September 20, 2020.
  339. **^** Evans, Will (June 11, 2018). ["Tesla fired safety official for reporting unsafe conditions, lawsuit says"](https://www.revealnews.org/blog/tesla-fired-safety-official-for-reporting-unsafe-conditions-lawsuit-says/). _Reveal_. Retrieved May 11, 2020.
  340. **^** ["Ramirez v. Tesla, Inc. :: Superior Court of California, County of Alameda :: State Civil Lawsuit No. RG18908005"](https://www.plainsite.org/dockets/3erxp7g38/superior-court-of-california-county-of-alameda/ramirez-v-tesla-inc/). _plainsite.org_. Retrieved September 20, 2020.
  341. **^** Spillman, Benjamin. ["Tesla whistleblower claims rampant theft, drug dealing at Nevada Gigafactory"](https://www.rgj.com/story/news/2018/08/16/tesla-ex-employee-alleges-theft-spying-drug-dealing-gigafactory-nevada/1011626002/). _Reno Gazette Journal_. Retrieved May 11, 2020.
  342. **^** ["Case 3:19-cv-00413-LRH-WGC Document 55"](https://www.plainsite.org/dockets/download.html?id=291214068&z=7bf3e87d). _PlainSite_. July 15, 2020. Retrieved August 7, 2022.
  343. **^** Hoffman, Bill (June 17, 2022). ["JAMS Arbitration Case Reference No. 1260005897"](https://www.plainsite.org/dockets/download.html?id=308199976&a=1&z=db789cce). _PlainSite_. Retrieved August 7, 2022. "Claimant has failed to establish the claims contained in his demand for arbitration. Accordingly, his claims are denied, and he shall take nothing."
  344. **^** ["Tesla Obstructed Probe of Worker Discrimination, California Says"](https://www.bloomberg.com/news/articles/2023-04-13/tesla-obstructed-probe-of-worker-discrimination-california-says). _Bloomberg.com_. April 13, 2023. Retrieved April 14, 2023.
  345. **^** Kalmowitz, Andy (November 15, 2024). ["Teslas Are The Most Fatal Cars On The Road, Study Finds"](https://www.jalopnik.com/teslas-are-the-most-fatal-cars-on-the-road-study-finds-1851700691/). _Jalopnik_. Retrieved June 30, 2025.
  346. **^** Gitlin, Jonathan M. (April 17, 2025). ["Tesla odometer uses "predictive algorithms" to void warranty, lawsuit claims"](https://arstechnica.com/cars/2025/04/tesla-makes-its-cars-lie-about-their-mileage-lawsuit-claims/). _Ars Technica_. Retrieved April 21, 2025.
  347. **^** Hiltzikh, Michael (November 18, 2013). ["The air starts leaking out of Tesla's tires"](https://www.latimes.com/business/hiltzik/la-fi-mh-tesla-20131118-story.html). _[Los Angeles Times](https://en.wikipedia.org/wiki/Los_Angeles_Times "Los Angeles Times")_.
  348. **^** Tully, Shawn (September 2, 2016). ["Why Tesla's Cash Crunch May Be Worse Than You Think"](https://fortune.com/2016/09/02/elon-musk-tesla-cash-crunch-worse-than-you-think/). _Fortune_. Retrieved May 26, 2020.
  349. **^** Mitchell, Russ (November 2, 2018). ["Three takeaways from the 10-Q report that Tesla just filed"](https://www.latimes.com/business/autos/la-fi-hy-tesla-10q-takeaways-20181102-story.html). _Los Angeles Times_. Retrieved May 26, 2020.
  350. **^** Owens, Jeremy C. (November 28, 2019). ["The SEC recently quizzed Tesla about its accounting, filings show"](https://www.marketwatch.com/story/the-sec-recently-quizzed-tesla-about-its-accounting-filings-show-2019-11-27). _MarketWatch_. Retrieved May 26, 2020.
  351. **^** Powell, Jamie; Jones, Claire (December 18, 2019). ["The question of Tesla's cash to be collected"](https://www.ft.com/content/1c5c83c0-43fe-4043-82ae-22b4865ae99c). _[Financial Times](https://en.wikipedia.org/wiki/Financial_Times "Financial Times")_. [Archived](https://ghostarchive.org/archive/20221211231205/https://www.ft.com/content/1c5c83c0-43fe-4043-82ae-22b4865ae99c) from the original on December 11, 2022. Retrieved February 15, 2021.
  352. **^** Cox, Jeff (November 8, 2019). ["Elon Musk gloats to hedge fund adversary over Tesla surge, calling David Einhorn 'Mr. Unicorn'"](https://www.cnbc.com/2019/11/08/elon-musk-gloats-to-his-hedge-fund-adversary-over-tesla-surge-calling-david-einhorn-mr-unicorn.html). _CNBC_.
  353. **^** Querolo, Nic; Trudell, Craig (April 30, 2020). ["Tesla Declines After Einhorn Questions Musk's Accounting"](https://www.bloomberg.com/news/articles/2020-04-30/tesla-pares-gains-after-einhorn-questions-accounts-receivable). Bloomberg L.P. Retrieved May 26, 2020.
  354. **^** Richards, Tori (July 23, 2015). ["Tesla got $295M in subsidies for technology it didn't offer"](https://web.archive.org/web/20181010015457/https://www.watchdog.org/california/tesla-got-m-in-subsidies-for-technology-it-didn-t/article_16c9814c-8176-5a03-a595-8cf0e2119455.html). _Watchdog.org_. Archived from [the original](https://www.watchdog.org/california/tesla-got-m-in-subsidies-for-technology-it-didn-t/article_16c9814c-8176-5a03-a595-8cf0e2119455.html) on October 10, 2018.
  355. **^** Niedermeyer, Edward (June 23, 2015). ["Tesla Battery Swap: CARB's Bridge To Nowhere"](https://web.archive.org/web/20150623233619/https://dailykanban.com/2015/06/tesla-battery-swap-carbs-bridge-to-nowhere/). _DailyKanban_. Archived from [the original](https://dailykanban.com/2015/06/tesla-battery-swap-carbs-bridge-to-nowhere/) on June 23, 2015. Retrieved May 18, 2020.
  356. **^** ["Elon Musk knew SolarCity was going broke before merger with Tesla, lawsuit alleges"](https://www.latimes.com/business/story/2019-09-23/solarcity-tesla-merger-shareholder-lawsuit). _Los Angeles Times_. September 24, 2019. Retrieved May 19, 2020.
  357. **^** Hals, Tom (January 31, 2020). ["Tesla directors settle, isolating Musk as SolarCity trial looms"](https://www.reuters.com/article/us-tesla-solarcity-lawsuit/tesla-directors-settle-isolating-musk-as-solarcity-trial-looms-idINKBN1ZT2HF). _Reuters_. Retrieved February 4, 2021.
  358. **^** ["Elon Musk wins $13B suit over Solar City deal Tesla shareholders called a 'bailout'"](https://techcrunch.com/2022/04/27/elon-musk-wins-13b-suit-over-solar-city-deal-tesla-shareholders-called-a-bailout/). _techcrunch.com_. April 27, 2022. Retrieved April 27, 2022.
  359. **^** ["Elon Musk wins shareholder lawsuit over Tesla's $2.6 billion SolarCity acquisition"](https://www.cnbc.com/2022/04/27/elon-musk-wins-shareholder-lawsuit-over-the-companys-2point6-billion-solarcity-acquisition.html). CNBC. April 27, 2022. Retrieved April 27, 2022.
  360. **^** Hals, Tom (June 6, 2023). ["Court upholds Musk's win in $13 bln lawsuit over Tesla-SolarCity deal"](https://www.reuters.com/legal/court-upholds-ruling-musk-over-tesla-solarcity-deal-2023-06-06/). [Reuters](https://en.wikipedia.org/wiki/Reuters "Reuters").
  361. **^** O'Kane, Sean (August 7, 2019). ["The lesson from Elon Musk's 'funding secured' mess is to never tweet"](https://www.theverge.com/tldr/2019/8/7/20758944/elon-musk-twitter-tesla-funding-secured-private-420). _[The Verge](https://en.wikipedia.org/wiki/The_Verge "The Verge")_.
  362. **^** Rosenblatt, Joel (April 15, 2020). ["Tesla Can't Duck Lawsuit Over Musk's Take-Private Tweet"](https://www.bloomberg.com/news/articles/2020-04-15/tesla-can-t-duck-investor-suit-over-musk-s-take-private-tweet). Bloomberg L.P.
  363. **^** Osborne, Charlie. ["The $40 million tweet: Elon Musk settles with SEC, Tesla bears the brunt"](https://www.zdnet.com/article/the-40-million-tweet-elon-musk-settles-with-sec-tesla-bears-the-brunt/). ZDNet. Retrieved May 26, 2020.
  364. **^** Mitchell, Russ (April 15, 2020). ["Judge deems Musk's 'funding secured' tweet false and misleading. A trial awaits"](https://www.latimes.com/business/story/2020-04-15/tesla-musk-funding-secured-trial). _Los Angeles Times_. Retrieved May 26, 2020.
  365. ^ _**a**_ _**b**_ Wayland, Michael (August 8, 2019). ["Tesla's chaotic year after Musk's 'funding secured' tweet"](https://www.cnbc.com/2019/08/08/teslas-chaotic-year-after-musks-funding-secured-tweet.html). [CNBC](https://en.wikipedia.org/wiki/CNBC "CNBC").
  366. **^** ["Tesla, Elon Musk must face shareholder lawsuit over going-private tweet"](https://www.autoblog.com/2020/04/15/tesla-elon-musk-going-private-tweet-shareholder-lawsuit/). _Autoblog_. Retrieved May 26, 2020.
  367. ^ _**a**_ _**b**_ ["Tesla 10-K Files with SEC"](https://www.sec.gov/Archives/edgar/data/1318605/000156459019003165/tsla-10k_20181231.htm). _sec.gov_. February 19, 2019. Retrieved March 3, 2019. This article incorporates text from this source, which is in the [public domain](https://en.wikipedia.org/wiki/Public_domain "Public domain").
  368. **^** Godoy, Jody; Jin, Hyunjoo (February 3, 2023). ["Tesla's Elon Musk found not liable in trial over 2018 'funding secured' tweets"](https://www.reuters.com/legal/securities-fraud-trial-over-elon-musks-2018-tweets-draws-close-2023-02-03/). _Reuters_. Retrieved February 4, 2023.
  369. ^ _**a**_ _**b**_ Cimilluca, Dana; Pulliam, Susan; Viswanatha, Aruna (October 26, 2018). ["Tesla Faces Deepening Criminal Probe Over Whether It Misstated Production Figures"](https://www.wsj.com/articles/tesla-faces-deepening-criminal-probe-over-whether-it-misstated-production-figures-1540576636). _[The Wall Street Journal](https://en.wikipedia.org/wiki/The_Wall_Street_Journal "The Wall Street Journal")_. [ISSN](https://en.wikipedia.org/wiki/ISSN_\(identifier\) "ISSN \(identifier\)") [0099-9660](https://search.worldcat.org/issn/0099-9660). Retrieved May 19, 2020.
  370. **^** Sage, Alexandria. ["Tesla, Elon Musk win dismissal of lawsuit over Model 3 production"](https://www.reuters.com/article/instant-article/idINKCN1R62JC). _Reuters_. Retrieved February 9, 2021.
  371. **^** Kelleher, Kevin (March 25, 2019). ["Federal Judge Dismisses Tesla Shareholders' Lawsuit on Model 3 Production – Again"](https://fortune.com/2019/03/25/federal-court-dismisses-shareholder-lawsuit-tesla-misled-model-3-production/). Retrieved December 17, 2019.
  372. **^** ["Elon Musk's Big Lie About Tesla Is Finally Exposed"](https://www.rollingstone.com/culture/culture-commentary/elon-musk-tesla-crash-1234930544/). Rolling Stone. December 17, 2023.
  373. **^** Spector, Mike; Prentice, Chris (May 8, 2024). ["In Tesla Autopilot probe, US prosecutors focus on securities, wire fraud"](https://www.reuters.com/business/autos-transportation/tesla-autopilot-probe-us-prosecutors-focus-securities-wire-fraud-2024-05-08/). _Reuters_. Retrieved May 17, 2024.
  374. **^** ["Direct-to-consumer auto sales: It's not just about Tesla"](https://www.ftc.gov/news-events/blogs/competition-matters/2015/05/direct-consumer-auto-sales-its-not-just-about-tesla). May 11, 2015.
  375. **^** Read, Richard (May 13, 2015). ["Can The FTC Persuade Michigan & Other States To Open Their Doors To Tesla?"](https://web.archive.org/web/20210727220517/https://www.thecarconnection.com/news/1098273_can-the-ftc-persuade-michigan-other-states-to-open-their-doors-to-tesla). _The Car Connection_. Archived from [the original](https://www.thecarconnection.com/news/1098273_can-the-ftc-persuade-michigan-other-states-to-open-their-doors-to-tesla) on July 27, 2021. Retrieved August 21, 2021.
  376. **^** Bodisch, Gerald R. (May 2009). ["Economic Effects Of State Bans On Direct Manufacturer Sales To Car Buyers"](https://www.justice.gov/atr/public/eag/246374.htm). US: Department of Justice. Retrieved July 9, 2019.
  377. **^** ["EV rivals Tesla, Rivian unite to target direct sales legislation"](https://techcrunch.com/2021/03/03/ev-rivals-tesla-rivian-unite-to-target-direct-sales-legislation/). _TechCrunch_. March 3, 2021. Retrieved August 9, 2021.
  378. **^** ["Tesla settles with ex-engineer accused of stealing trade secrets"](https://www.thestatesman.com/technology/tesla-settles-ex-engineer-accused-stealing-trade-secrets-1502963166.html). _The Statesman_. April 17, 2021.
  379. **^** ["Tesla sues former employees for allegedly stealing data, Autopilot source code"](https://www.reuters.com/article/us-tesla-lawsuit-idUSKCN1R21P9). _Reuters_. March 21, 2019. Retrieved January 25, 2021.
  380. **^** Lyons, Kim (January 24, 2021). ["Tesla sues former employee for allegedly stealing software"](https://www.theverge.com/2021/1/24/22247095/tesla-sues-former-employee-software-dropbox). _The Verge_. Retrieved January 25, 2021.
  381. **^** Feiner, Lauren; Kolodny, Lora (May 28, 2020). ["Elon Musk earns first performance-based payout from Tesla, worth more than $700 million"](https://www.cnbc.com/2020/05/28/musk-gets-first-tranche-of-multimillion-dollar-tesla-incentive-payout.html). [CNBC](https://en.wikipedia.org/wiki/CNBC "CNBC").
  382. **^** Bellan, Rebecca (July 17, 2023). ["Tesla directors pay $735M to settle claims they overpaid themselves"](https://techcrunch.com/2023/07/17/tesla-directors-pay-735m-to-settle-claims-they-overpaid-themselves/). _TechCrunch_. Retrieved July 21, 2023.
  383. **^** O'Kane, Sean (April 1, 2019). ["Tesla penalized for violating hazardous waste law at California factory"](https://www.theverge.com/2019/4/1/18291091/tesla-epa-fine-hazardous-waste-fremont-factory). _The Verge_. Retrieved May 18, 2020.
  384. **^** Niedermeyer, Edward (June 6, 2019). ["Tesla in Settlement Proceedings Over 19 Air Quality Violations As Investigation Continues"](https://www.thedrive.com/tech/28413/tesla-in-settlement-proceedings-over-19-air-quality-violations-as-investigation-continues). _The Drive_. Retrieved May 18, 2020.
  385. **^** Niedermeyer, Edward (June 3, 2019). ["Tesla Air Quality Compliance Violations Center On Troubled Paint Shop"](https://www.thedrive.com/tech/28339/tesla-air-quality-compliance-violations-center-on-troubled-paint-shop). _The Drive_. Retrieved May 18, 2020.
  386. **^** Niedermeyer, Edward (June 3, 2019). ["Documents Show Persistent Air Quality Non-Compliance at Tesla Factory"](https://www.thedrive.com/tech/28338/documents-show-persistent-air-quality-non-compliance-at-tesla-factory). _The Drive_. Retrieved May 18, 2020.
  387. **^** Robinson, Matt; Faux, Zeke (March 13, 2019). ["When Elon Musk Tried to Destroy a Tesla Whistleblower"](https://www.bloomberg.com/news/features/2019-03-13/when-elon-musk-tried-to-destroy-tesla-whistleblower-martin-tripp). _Bloomberg Businessweek_.
  388. **^** Klippenstein, Matthew (July 21, 2019). ["Tesla Enters 'Whistleblower Hell'"](https://www.thedrive.com/tech/29089/tesla-enters-whistleblower-hell). _The Drive_. Retrieved May 11, 2020.
  389. **^** Szymkowski, Sean (September 18, 2020). ["Tesla wins lawsuit against whistleblower accused of hacks"](https://www.cnet.com/roadshow/news/tesla-lawsuit-whistleblower-hacks/). _Roadshow_. Retrieved September 20, 2020.
  390. **^** ["Tesla, Inc. v. Tripp :: Nevada District Court :: Federal Civil Lawsuit No. 3:18-cv-00296-MMD-CLB, Judge Miranda M. Du presiding"](https://www.plainsite.org/dockets/3br5tkwuj/nevada-district-court/tesla-inc-v-tripp/). _plainsite.org_. Retrieved September 20, 2020.
  391. **^** ["Tesla, Inc. Settles Environmental Enforcement Action Brought by California District Attorneys"](https://www.sjgov.org/department/da/news/press-release/2024/02/02/tesla-inc.-settles-environmental-enforcement-action-brought-by-california-district-attorneys). _sjgov.org_. Retrieved March 25, 2024.
  392. **^** Calma, Justine (February 3, 2024). ["How bad is Tesla's hazardous waste problem in California?"](https://www.theverge.com/2024/2/3/24058476/tesla-hazardous-waste-suit-settlement-california). _The Verge_. Retrieved February 5, 2024.
  393. **^** ["Tesla ordered to pay $1.5 million over alleged hazardous waste violations in California"](https://apnews.com/article/tesla-california-hazardous-waste-settlement-ea1eb742720b8a1fefe38a45407a8a9b). _AP News_. February 3, 2024. Retrieved February 5, 2024.
  394. **^** Lee, Timothy. ["After seven roof fires, Walmart sues Tesla over solar panel flaws"](https://arstechnica.com/tech-policy/2019/08/after-seven-roof-fires-walmart-sues-tesla-over-solar-panel-flaws/). _Ars Technica_. Retrieved June 12, 2020.
  395. **^** Korosec, Kirsten (November 5, 2019). ["Walmart reaches settlement with Tesla over solar panel fires, drops lawsuit"](https://techcrunch.com/2019/11/05/walmart-reaches-settlement-with-tesla-over-solar-panel-fires-drops-lawsuit/). _TechCrunch_. [Archived](https://web.archive.org/web/20200924212601/https://techcrunch.com/2019/11/05/walmart-reaches-settlement-with-tesla-over-solar-panel-fires-drops-lawsuit/) from the original on September 24, 2020. Retrieved June 12, 2020.
  396. **^** Porterfield, Carlie (May 24, 2021). ["Tesla Found Guilty Of Throttling Battery Life, Charging Speed in Norway"](https://www.forbes.com/sites/carlieporterfield/2021/05/24/tesla-found-guilty-of-throttling-battery-life-charging-speed-in-norway/). _Forbes_.
  397. **^** Hawkins, Andrew J. (May 24, 2021). ["Tesla faces a huge fine in Norway for throttling battery charging speeds"](https://www.theverge.com/2021/5/24/22451101/tesla-fine-norway-throttle-battery-charging-speed). _The Verge_. Retrieved March 25, 2024.
  398. ^ _**a**_ _**b**_ Hepler, Lauren (November 30, 2018). ["Menial Tasks, Slurs and Swastikas: Many Black Workers at Tesla Say They Faced Racism"](https://www.nytimes.com/2018/11/30/business/tesla-factory-racism.html). _The New York Times_. [ISSN](https://en.wikipedia.org/wiki/ISSN_\(identifier\) "ISSN \(identifier\)") [0362-4331](https://search.worldcat.org/issn/0362-4331). Retrieved April 9, 2021.
  399. **^** ["Former Tesla employee who said supervisors called him the N-word awarded $1 million"](https://www.cbsnews.com/news/tesla-million-melvin-berry-fremont-california-n-word-racial-discrimination/). CBS News. August 6, 2021. Retrieved February 19, 2022.
  400. **^** ["Lawsuit calls Tesla factory a hotbed of racism; Tesla calls lawsuit a 'hotbed of misinformation'"](https://www.latimes.com/business/autos/la-fi-hy-tesla-racism-lawsuit-20171115-story.html). _Los Angeles Times_. November 15, 2017. Retrieved April 25, 2021.
  401. **^** ["Tesla Impact Report 2021"](https://www.tesla.com/ns_videos/2021-tesla-impact-report.pdf) (PDF). _Tesla_. June 26, 2022. Retrieved August 3, 2022.
  402. **^** ["Former Tesla workers describe hostile workplace at Buffalo facility"](https://web.archive.org/web/20210411124108/https://www.wivb.com/news/former-tesla-workers-describe-hostile-workplace-at-buffalo-facility/). _News 4 Buffalo_. November 25, 2019. Archived from [the original](https://www.wivb.com/news/former-tesla-workers-describe-hostile-workplace-at-buffalo-facility/) on April 11, 2021. Retrieved April 9, 2021.
  403. **^** Koren, Marina (June 21, 2020). ["Elon Musk's Lesson in How Not to Celebrate Diversity"](https://www.theatlantic.com/science/archive/2020/06/elon-musk-juneteenth-spacex-tesla/613330/). _The Atlantic_. Retrieved April 25, 2021.
  404. **^** Wille, Matt (July 6, 2021). ["Tesla Fremont employees allege widespread racism on the factory floor"](https://www.inputmag.com/culture/tesla-fremont-employees-allege-widespread-racism-on-the-factory-floor). _Input_. [Archived](https://web.archive.org/web/20220804142413/https://www.inputmag.com/culture/tesla-fremont-employees-allege-widespread-racism-on-the-factory-floor) from the original on August 4, 2022. Retrieved September 9, 2021.
  405. **^** ["Tesla Racism Verdict of $137 Million Could Be Cut if Appealed"](https://time.com/6104336/tesla-racism-verdict-appeal/). _Time_. Retrieved February 19, 2022.
  406. **^** Wiessner, Daniel; Jin, Hyunjoo (February 11, 2022). ["California sues Tesla over Black workers' allegations of discrimination"](https://www.reuters.com/business/california-agency-sues-tesla-over-alleged-discrimination-harassment-wsj-2022-02-10/). _Reuters_. Retrieved February 19, 2022.
  407. **^** ["California Sues Tesla, Alleging Racial Discrimination and Harassment"](https://news.justia.com/california-sues-tesla-alleging-racial-discrimination-and-harassment/). _news.justia.com_. February 10, 2022. Retrieved February 11, 2022.
  408. **^** ["Ex-Tesla Employee Called Racial Slur Wins Rare $1 Million Award"](https://www.bloomberg.com/news/articles/2021-08-05/ex-tesla-employee-called-racial-slur-wins-rare-1-million-award). _Bloomberg.com_. August 5, 2021. Retrieved September 9, 2021.
  409. **^** ["Black ex-Tesla worker who claimed racial abuse awarded $137M"](https://apnews.com/article/business-san-francisco-race-and-ethnicity-tesla-inc-african-americans-d74d7fc97fc5b0608c26015aa77d7c74). _AP NEWS_. October 5, 2021.
  410. **^** ["Tesla must face lawsuit claiming racism at California factory"](https://www.reuters.com/article/us-tesla-lawsuit-racism-idUSKBN1YZ18E). _Reuters_. December 31, 2019.
  411. **^** ["Regarding Today's Jury Verdict"](https://www.tesla.com/blog/regarding-todays-jury-verdict). _tesla.com_. October 4, 2021. Retrieved February 19, 2022.
  412. **^** Kolodny, Lora (October 5, 2021). ["Tesla must pay $137 million to ex-worker over hostile work environment, racism"](https://www.cnbc.com/2021/10/05/tesla-must-pay-137-million-to-ex-worker-over-hostile-work-environment-racism.html). CNBC. Retrieved February 19, 2022.
  413. **^** Stempel, Jonathan; Wiessner, Daniel (April 14, 2022). ["Judge finds Tesla liable to Black former worker who alleged bias, but slashes payout"](https://www.reuters.com/technology/us-judge-cuts-verdict-tesla-race-bias-case-15-mln-137-mln-2022-04-14/). _Reuters_.
  414. **^** ["Former Tesla worker rejects $15M payout in racial abuse lawsuit"](https://techcrunch.com/2022/06/21/former-tesla-worker-rejects-15m-payout-in-racial-abuse-lawsuit/). _TechCrunch_. June 21, 2022. Retrieved June 22, 2022.
  415. **^** Paul, Kari (April 3, 2023). ["Black former worker awarded $3.2m in Tesla factory racial-harassment suit"](https://www.theguardian.com/us-news/2023/apr/03/tesla-racial-harassment-lawsuit-award-california-factory). _The Guardian_. [ISSN](https://en.wikipedia.org/wiki/ISSN_\(identifier\) "ISSN \(identifier\)") [0261-3077](https://search.worldcat.org/issn/0261-3077). Retrieved April 28, 2023.
  416. ^ _**a**_ _**b**_ Siddiqui, Faiz (March 13, 2021). ["Hundreds of covid cases reported at Tesla plant following Musk's defiant reopening, county data shows"](https://www.washingtonpost.com/technology/2021/03/12/hundreds-covid-cases-reported-tesla-plant-following-musks-defiant-reopening-county-data-shows/). _[The Washington Post](https://en.wikipedia.org/wiki/The_Washington_Post "The Washington Post")_. Retrieved March 13, 2021.
  417. **^** Walsh, Joe. ["Elon Musk's False Covid Predictions: A Timeline"](https://www.forbes.com/sites/joewalsh/2021/03/13/elon-musks-false-covid-predictions-a-timeline/). _Forbes_. Retrieved February 17, 2022.
  418. **^** Marshall, Aarian. ["Elon Musk Defies Lockdown Orders and Reopens Tesla's Factory"](https://www.wired.com/story/elon-musk-defies-lockdown-orders-reopens-tesla-factory/). _Wired_. Retrieved June 12, 2020.
  419. **^** ["The dispute over reopening the Tesla factory may be over"](https://www.latimes.com/business/story/2020-05-13/dispute-over-reopening-tesla-factory-may-be-over). _Los Angeles Times_. May 13, 2020. Retrieved June 14, 2020.
  420. **^** Boudette, Neal E. (May 8, 2020). ["Tesla Tells Workers It Will Reopen California Factory Despite County Order"](https://www.nytimes.com/2020/05/08/business/economy/tesla-coronavirus-factory-alameda.html). _The New York Times_. [ISSN](https://en.wikipedia.org/wiki/ISSN_\(identifier\) "ISSN \(identifier\)") [0362-4331](https://search.worldcat.org/issn/0362-4331). [Archived](https://web.archive.org/web/20200508224003/https://www.nytimes.com/2020/05/08/business/economy/tesla-coronavirus-factory-alameda.html) from the original on May 8, 2020. Retrieved June 14, 2020.
  421. **^** Difeliciantonio, Chase (May 21, 2020). ["Tesla drops lawsuit against Alameda County after Fremont factory reopens"](https://www.sfchronicle.com/business/article/Tesla-drops-lawsuit-against-Alameda-County-after-15284242.php). _San Francisco Chronicle_. Retrieved June 14, 2020.
  422. **^** Bursztynsky, Jessica; Kolodny, Lora (May 20, 2020). ["Tesla drops lawsuit against California's Alameda County over coronavirus restrictions"](https://www.cnbc.com/2020/05/20/tesla-drops-suit-against-californias-alameda-county.html). CNBC. Retrieved June 13, 2020.
  423. **^** Newburger, Emma; Kolodny, Lora (May 10, 2020). ["Tesla says it will resume operations. Here is the company's plan to bring employees back to work"](https://www.cnbc.com/2020/05/10/coronavirus-teslas-plan-to-bring-employees-back-to-work.html). CNBC. Retrieved June 13, 2020.
  424. **^** Kolodny, Lora (June 12, 2020). ["Tesla safety boss tries to calm factory workers, some are concerned about lax coronavirus precautions"](https://www.cnbc.com/2020/06/12/tesla-laurie-shelby-email-on-covid-19-fremont-workers-worried.html). CNBC. Retrieved June 12, 2020.
  425. **^** Holmes, Aaron. ["More Tesla employees say they were fired for staying home over COVID-19 fears even though CEO Elon Musk said they could"](https://www.businessinsider.com/tesla-plant-firings-elon-musk-covid-19-staying-home-2020-7). _Business Insider_. Retrieved July 2, 2020.
  426. **^** ["Tesla worker who criticized coronavirus safety measures receives termination notice"](https://www.mercurynews.com/tesla-worker-who-criticized-coronavirus-safety-measures-receives-termination-notice). _The Mercury News_. June 18, 2020. Retrieved June 19, 2020.
  427. **^** ["Coronavirus: Elon Musk's Tesla denies firing employees who stayed home during lockdown"](https://news.sky.com/story/coronavirus-elon-musks-tesla-denies-firing-employees-who-stayed-home-during-lockdown-12020300). Sky News. Retrieved February 9, 2021.
  428. **^** Coleman, Justine (March 14, 2021). ["Hundreds of Tesla workers tested positive at reopened plant"](https://thehill.com/business-a-lobbying/business-a-lobbying/543157-hundreds-of-tesla-workers-tested-positive-at-reopened). _The Hill_. Retrieved March 15, 2021.
  429. ^ _**a**_ _**b**_ _**c**_ ["Elon Musk Loves China, and China Loves Him Back – for Now"](https://www.bloomberg.com/news/features/2021-01-13/china-loves-elon-musk-and-tesla-tsla-how-long-will-that-last). _Bloomberg.com_. January 13, 2021. Retrieved February 22, 2021.
  430. **^** Scarcella, Mike (March 15, 2023). ["Tesla hit with 'right to repair' antitrust class actions"](https://www.reuters.com/legal/tesla-hit-with-right-repair-antitrust-class-actions-2023-03-15/). _[Reuters](https://en.wikipedia.org/wiki/Reuters "Reuters")_. [Archived](https://web.archive.org/web/20230315182615/https://www.reuters.com/legal/tesla-hit-with-right-repair-antitrust-class-actions-2023-03-15/) from the original on March 15, 2023. Retrieved July 3, 2024.
  431. **^** Thompson, Michelle (March 17, 2023). ["Tesla accused in lawsuit of monopolizing parts, repairs"](https://www.repairerdrivennews.com/2023/03/17/tesla-accused-in-lawsuit-of-monopolizing-parts-repairs/). _Repairer Driven News_. [Archived](https://web.archive.org/web/20230317101511/https://www.repairerdrivennews.com/2023/03/17/tesla-accused-in-lawsuit-of-monopolizing-parts-repairs/) from the original on March 17, 2023. Retrieved July 3, 2024.
  432. ^ _**a**_ _**b**_ Stempel, Jonathan (June 18, 2024). ["Tesla must face owners' lawsuit claiming it monopolizes vehicle repairs and parts"](https://www.reuters.com/legal/tesla-must-face-owners-lawsuit-claiming-it-monopolizes-vehicle-repairs-parts-2024-06-18/). _[Reuters](https://en.wikipedia.org/wiki/Reuters "Reuters")_. Retrieved July 3, 2024.
  433. ^ _**a**_ _**b**_ Lowery, Lurah (June 26, 2024). ["Two of eight claims in Tesla anti-trust lawsuit will move forward"](https://www.repairerdrivennews.com/2024/06/26/two-of-eight-claims-in-tesla-anti-trust-lawsuit-will-move-forward/). _Repairer Driven News_. [Archived](https://web.archive.org/web/20240626224809/https://www.repairerdrivennews.com/2024/06/26/two-of-eight-claims-in-tesla-anti-trust-lawsuit-will-move-forward/) from the original on June 26, 2024. Retrieved July 3, 2024.
  434. **^** ["'Smash the broligarchy': Protest in Seattle joins national Musk backlash"](https://www.seattletimes.com/seattle-news/smash-the-broligarchy-protest-in-seattle-joins-national-musk-backlash/). _The Seattle Times_. February 15, 2025. Retrieved February 16, 2025.
  435. **^** Venegas, Natalie (February 15, 2025). ["'Unplug Mad King': Protesters Rally Against Elon Musk at Tesla Facilities"](https://www.newsweek.com/unplug-mad-king-protesters-rally-against-elon-musk-tesla-facilities-2031825). _[Newsweek](https://en.wikipedia.org/wiki/Newsweek "Newsweek")_. Retrieved February 16, 2025.
  436. **^** ["Her parents were injured in a Tesla crash. She ended up having to pay Tesla damages"](https://apnews.com/article/tesla-china-lawsuits-musk-investigation-58b10ccace488784fcc63646ab78b410). _AP News_. February 12, 2025. Retrieved February 18, 2025.
  437. **^** ["It's Official: Cars Are the Worst Product Category We Have Ever Reviewed for Privacy"](https://foundation.mozilla.org/en/privacynotincluded/articles/its-official-cars-are-the-worst-product-category-we-have-ever-reviewed-for-privacy/). _[Mozilla Foundation](https://en.wikipedia.org/wiki/Mozilla_Foundation "Mozilla Foundation")_. Retrieved September 13, 2023.
  438. **^** Jeong, Andrew (September 12, 2023). ["Carmakers can collect — and sell — too much data about you, watchdog says"](https://www.washingtonpost.com/business/2023/09/07/car-privacy-mozilla-report/). _Washington Post_. [ISSN](https://en.wikipedia.org/wiki/ISSN_\(identifier\) "ISSN \(identifier\)") [0190-8286](https://search.worldcat.org/issn/0190-8286). Retrieved September 13, 2023.
  439. **^** Jin, Hyunjoo; Scarcella, Mike (April 10, 2023). ["Tesla hit with class action lawsuit over alleged privacy intrusion"](https://www.reuters.com/business/autos-transportation/tesla-hit-with-class-action-lawsuit-over-alleged-privacy-intrusion-2023-04-08/). _Reuters_. Retrieved October 13, 2023.
  440. **^** ["Tesla recall vindicates whistleblower Lukasz Krupski"](https://www.blueprintforfreespeech.net/en/news/lukasz-krupski). _Blueprint for Free Speech_. Retrieved January 23, 2024.
  441. **^** Titcomb, James (December 11, 2023). ["China had access to Tesla employees' data, whistleblower claims"](https://www.telegraph.co.uk/business/2023/12/11/china-access-tesla-employees-data-whistleblower-privacy/). _The Telegraph_. [ISSN](https://en.wikipedia.org/wiki/ISSN_\(identifier\) "ISSN \(identifier\)") [0307-1235](https://search.worldcat.org/issn/0307-1235). Retrieved January 23, 2024.
  442. **^** Ewing, Jack (November 10, 2023). ["Man vs. Musk: A Whistleblower Creates Headaches for Tesla"](https://www.nytimes.com/2023/11/10/business/tesla-whistleblower-elon-musk.html). _[The New York Times](https://en.wikipedia.org/wiki/The_New_York_Times "The New York Times")_.
  443. **^** ["The Tesla Skeptics Who Bet Against Elon Musk"](https://www.bloomberg.com/news/features/2020-01-22/the-tesla-tslaq-skeptics-who-bet-against-elon-musk). _Bloomberg.com_. January 22, 2020. Retrieved February 8, 2021.
  444. **^** ["Electric Burn: Those Who Bet Against Elon Musk And Tesla Are Paying A Big Price"](https://www.npr.org/2020/01/16/796328145/electric-burn-those-who-bet-against-elon-musk-and-tesla-are-paying-a-big-price). _NPR.org_. January 16, 2020. Retrieved April 28, 2021.
  445. **^** Isidore, Chris (January 6, 2021). ["Tesla short sellers lost $40 billion in 2020. Elon Musk made more than triple that"](https://www.cnn.com/2021/01/06/investing/tesla-shorts-losses-elon-musk-win/index.html). CNN. Retrieved April 28, 2021.
  446. **^** ["'Big Short' investor Burry says he's no longer betting against Tesla – CNBC"](https://www.reuters.com/business/big-short-investor-burry-says-hes-no-longer-betting-against-tesla-cnbc-2021-10-15/). _Reuters_. October 15, 2021. Retrieved January 30, 2022.
  447. **^** ["Does Tesla actually want competitors to make electric cars?"](https://www.marketplace.org/2021/08/27/does-tesla-actually-want-competitors-to-make-electric-cars/). _Marketplace_. August 27, 2021. Retrieved August 28, 2021.
  448. **^** ["Scathing audit of high-tech projects slams ESD for lack of due diligence on Tesla at RiverBend"](https://web.archive.org/web/20201227141211/https://www.wivb.com/news/local-news/buffalo/scathing-audit-of-high-tech-projects-slams-esd-for-lack-of-due-diligence-on-tesla-at-riverbend/). _News 4 Buffalo_. August 21, 2020. Archived from [the original](https://www.wivb.com/news/local-news/buffalo/scathing-audit-of-high-tech-projects-slams-esd-for-lack-of-due-diligence-on-tesla-at-riverbend/) on December 27, 2020. Retrieved August 21, 2020.
  449. **^** Heaney, Jim (August 24, 2020). ["Buffalo Billion audit: shock and ugh"](https://www.investigativepost.org/2020/08/23/buffalo-billion-audit-shock-and-ugh/). _Investigative Post_. Retrieved February 9, 2021.
  450. **^** Hogan, Bernadette; Hicks, Nolan (August 21, 2020). ["More 'Buffalo Billion' woes as audit finds Cuomo boondoggle a waste of tax money"](https://nypost.com/2020/08/21/audit-finds-cuomos-buffalo-billions-a-waste-of-tax-money/). _New York Post_. Retrieved February 9, 2021.
  451. **^** Jones, Chuck. ["Tesla's Musk Is Overpromising Again On Self-Driving Cars"](https://www.forbes.com/sites/chuckjones/2019/10/22/teslas-musk-is-overpromising-again-on-self-driving-cars/). _Forbes_. Retrieved February 8, 2021.
  452. **^** DeBord, Matthew. ["This is why Tesla always overpromises and underdelivers"](https://www.businessinsider.com/why-tesla-always-overpromises-and-underdelivers-2015-10). _Business Insider_. Retrieved February 8, 2021.
  453. **^** Pulliam, Susan; Mike, Ramsey; Dugan, Ianthe Jeanne (August 15, 2016). ["Elon Musk Sets Ambitious Goals at Tesla – and Often Falls Short"](https://www.wsj.com/articles/elon-musk-sets-ambitious-goals-at-teslaand-often-falls-short-1471275436). _[The Wall Street Journal](https://en.wikipedia.org/wiki/The_Wall_Street_Journal "The Wall Street Journal")_. [ISSN](https://en.wikipedia.org/wiki/ISSN_\(identifier\) "ISSN \(identifier\)") [0099-9660](https://search.worldcat.org/issn/0099-9660). Retrieved February 8, 2021.
  454. **^** Holley, Peter (October 2, 2017). ["'We understand what needs to be fixed,' Tesla says after missing Model 3 production goals"](https://www.washingtonpost.com/news/innovations/wp/2017/10/02/we-understand-what-needs-to-be-fixed-tesla-says-after-bungling-model-3-production-goals/). _The Washington Post_. Retrieved November 5, 2017.
  455. **^** Holley, Peter (November 3, 2017). ["Analysis – Sleepless nights, broken robots and mounting pressure: Musk offers rare glimpse inside Tesla's 'production hell'"](https://www.washingtonpost.com/news/innovations/wp/2017/11/03/sleepless-nights-broken-robots-and-mounting-pressure-musk-offers-rare-glimpse-inside-teslas-production-hell). _The Washington Post_. Retrieved November 6, 2017.
  456. **^** Mitchell, Russ (January 9, 2018). ["Tesla Model 3 delivery delayed again"](https://www.chicagotribune.com/autos/sc-auto-tips-0111-model-3-delayed-again-20180109-story.html). _Chicago Tribune_. Retrieved February 8, 2021.
  457. **^** Lavrinc, Damon (December 17, 2014). ["What Will Tesla And Elon Musk Over Promise Next?"](https://jalopnik.com/what-will-tesla-and-elon-musk-over-promise-next-1672402636). _[Jalopnik](https://en.wikipedia.org/wiki/Jalopnik "Jalopnik")_.
  458. **^** Stahl, Lesley (December 9, 2018). ["Tesla CEO Elon Musk: The 60 Minutes Interview"](https://www.cbsnews.com/news/tesla-ceo-elon-musk-the-2018-60-minutes-interview/). CBS News. Retrieved April 27, 2021.
  459. **^** Lee, Timothy B. (June 10, 2016). ["Tesla's real problem isn't that its cars are expensive. It's that they're unreliable"](https://www.vox.com/2016/6/9/11880450/tesla-doomed). _[Vox](https://en.wikipedia.org/wiki/Vox_\(website\) "Vox \(website\)")_. Retrieved April 21, 2017.
  460. **^** Lee, Timothy B. (April 20, 2017). ["Tesla is recalling most of the cars it sold in 2016"](https://www.vox.com/new-money/2017/4/20/15374592/tesla-recall-53000-cars). _[Vox](https://en.wikipedia.org/wiki/Vox_\(website\) "Vox \(website\)")_. Retrieved April 21, 2017.
  461. **^** Wang, Christine (March 29, 2018). ["Tesla voluntarily recalls 123,000 Model S cars over faulty steering component"](https://www.cnbc.com/2018/03/29/tesla-recalls-123000-model-s-cars-over-potential-power-steering-failure-reports.html). CNBC. Retrieved March 31, 2018.
  462. **^** Kolodny, Lora (October 23, 2020). ["Tesla recalls nearly 50,000 Model S and X cars in China over faulty suspension"](https://www.cnbc.com/2020/10/23/tesla-recalls-model-s/x-cars-in-china-over-faulty-suspension.html). CNBC. Retrieved February 8, 2021.
  463. **^** Taylor, Thom (August 1, 2020). ["This Is Bad: "Whompy Wheel" Syndrome Causing Teslas To Crash"](https://www.motorbiscuit.com/this-is-bad-whompy-wheel-syndrom-causing-teslas-to-crash/). _MotorBiscuit_. Retrieved February 8, 2021.
  464. **^** Lopez, Linette. ["'Aladdin' star says a defect in his Tesla Model 3 led to his car wreck, and it comes from a problem area the company has known about for years"](https://www.businessinsider.com/aladdin-stars-problem-with-tesla-known-as-whompy-wheels-2019-6). _Business Insider_. Retrieved October 26, 2020.
  465. **^** ["Tesla agrees to recall 135,000 vehicles over touch screen failures after sparring with regulators"](https://www.washingtonpost.com/technology/2021/02/02/tesla-touch-screen-recall/). _[The Washington Post](https://en.wikipedia.org/wiki/The_Washington_Post "The Washington Post")_. February 2, 2021. Retrieved February 22, 2021.
  466. **^** Ewing, Steven. ["Tesla asked to recall Model S, Model X over touchscreen failures"](https://www.cnet.com/roadshow/news/tesla-model-s-model-x-touchscreen-recall/). _Roadshow_. Retrieved January 15, 2021.
  467. **^** ["Tesla recalls 158,000 Model S and Model X vehicles – report"](https://www.caradvice.com.au/916000/tesla-recalls-158000-model-s-and-model-x-vehicles-report/). _CarAdvice_. Retrieved January 15, 2021.
  468. **^** Quandt, Jeffrey (August 14, 2020). ["Confidential Business Information: Re: PE20-010 – Response to Information Request (First Submission)"](https://static.nhtsa.gov/odi/inv/2020/INRL-PE20010-80992P.pdf) (PDF). National Highway Traffic Safety Administration. Retrieved September 15, 2021.
  469. **^** ["Tesla must recall 12,300 Model X cars over faulty moulding – KBA"](https://www.reuters.com/article/us-tesla-recall-idUSKBN2AC0ZU). _Reuters_. February 12, 2021. Retrieved February 12, 2021.
  470. **^** Krok, Andrew. ["Tesla told to recall 12,300 Model X SUVs over trim adhesive"](https://www.cnet.com/roadshow/news/tesla-germany-model-x-recall-trim-adhesive/). _Roadshow_. Retrieved February 12, 2021.
  471. **^** Bursztynsky, Jessica (June 2, 2021). ["Tesla recalls 6,000 cars over risk of loose bolts"](https://www.cnbc.com/2021/06/02/tesla-recalls-nearly-6000-model-3-and-model-y-cars-over-loose-bolt-risk.html). CNBC. Retrieved June 3, 2021.
  472. **^** ["Tesla to recall 475,000 cars in the US"](https://www.bbc.com/news/technology-59818800). BBC. December 30, 2021.
  473. **^** Levy, Ari (December 24, 2021). ["Tesla locks access to video games in main display while car is in motion"](https://www.cnbc.com/2021/12/23/tesla-disables-passenger-play-video-games-while-vehicle-is-moving.html). CNBC. Retrieved December 31, 2021.
  474. **^** Shepardson, David (September 22, 2022). ["Tesla recalls nearly 1.1 million U.S. vehicles to update window reversing software"](https://www.reuters.com/business/autos-transportation/tesla-recalls-nearly-11-million-us-vehicles-update-window-reversing-software-2022-09-22/). _Reuters_. Retrieved September 23, 2022.
  475. ^ _**a**_ _**b**_ Root, Al (September 22, 2022). ["Tesla Recalls Another Million-Plus EVs. Elon Musk Goes After the Safety Patrol"](https://www.barrons.com/articles/tesla-stock-ev-recall-software-51663863560). _Barrons_. Retrieved September 23, 2022.
  476. **^** Isidore, Chris (February 16, 2023). ["Tesla recalling nearly 363,000 vehicles equipped with 'Full Self-Driving' | CNN Business"](https://www.cnn.com/2023/02/16/business/tesla-fsd-recall/index.html). _CNN_. Retrieved March 7, 2023.
  477. **^** Boudette, Neal E. (February 16, 2023). ["Tesla to Recall 362,000 Cars With Its 'Full Self Driving' System"](https://www.nytimes.com/2023/02/16/business/tesla-recall-full-self-driving.html). _The New York Times_. [ISSN](https://en.wikipedia.org/wiki/ISSN_\(identifier\) "ISSN \(identifier\)") [0362-4331](https://search.worldcat.org/issn/0362-4331). Retrieved March 7, 2023.
  478. **^** Mendoza, Jordan. ["Tesla recalls thousands of Model Y vehicles over loose bolts in seat back frames"](https://www.usatoday.com/story/money/cars/2023/03/05/tesla-recall-model-y-loose-bolts/11407892002/). _USA TODAY_. Retrieved March 7, 2023.
  479. **^** Krisher, Tom (December 13, 2023). ["Tesla recalls nearly all vehicles sold in US to fix system that monitors drivers using Autopilot"](https://apnews.com/article/tesla-autopilot-recall-driver-monitoring-system-8060508627a34e6af889feca46eb3002). _AP News_. Associated Press. Retrieved December 13, 2023.
  480. ^ _**a**_ _**b**_ [Part 573 Safety Recall Report](https://static.nhtsa.gov/odi/rcl/2023/RCLRPT-23V838-8276.PDF) (PDF) (Report). National Highway Traffic Safety Administration. December 12, 2023. 23V-838. Retrieved December 13, 2023. "At no cost to customers, affected vehicles will receive an over-the-air software remedy"
  481. **^** Laing, Keith (December 13, 2023). ["How Over-the-Air Updates Are Changing the Auto Recall Game"](https://www.bloomberg.com/news/articles/2023-12-13/car-recalls-how-over-the-air-updates-are-changing-the-game). _bloomberg.com_. Retrieved December 17, 2023.
  482. **^** ["The Detroit News"](https://www.detroitnews.com/restricted/?return=https%3A%2F%2Fwww.detroitnews.com%2Fin-depth%2Fbusiness%2Fautos%2F2022%2F03%2F18%2Fhow-gm-tesla-handled-their-electric-vehicle-fires%2F6107340001%2F). _detroitnews.com_. Retrieved November 14, 2022.
  483. **^** Jensen, Christopher (October 2, 2013). ["Tesla Says Car Fire Started in Battery"](https://wheels.blogs.nytimes.com/2013/10/02/highway-fire-of-tesla-model-s-included-its-lithium-battery/). _[The New York Times](https://en.wikipedia.org/wiki/The_New_York_Times "The New York Times")_.
  484. **^** Voelcker, John (November 19, 2013). ["Tesla Fires: NHTSA Will Probe, Warranty To Cover Fire Damage, Ride-Height Tweak"](https://www.greencarreports.com/news/1088588_tesla-fires-nhtsa-will-probe-warranty-to-cover-fire-damage-ride-height-tweak). _Green Car Reports_.
  485. **^** Ivory, Danielle (March 28, 2014). ["Federal Safety Agency Ends Its Investigation of Tesla Fires"](https://www.nytimes.com/2014/03/29/business/safety-agency-ends-investigation-of-tesla-fires.html). _[The New York Times](https://en.wikipedia.org/wiki/The_New_York_Times "The New York Times")_. [Archived](https://web.archive.org/web/20140329000140/http://www.nytimes.com/2014/03/29/business/safety-agency-ends-investigation-of-tesla-fires.html) from the original on March 29, 2014. Retrieved March 31, 2014.
  486. **^** George, Patrick (March 28, 2014). ["The Tesla Model S: Now With Road Debris-Crushing Titanium!"](https://jalopnik.com/the-tesla-model-s-now-with-road-debris-crushing-titani-1553544362). _[Jalopnik](https://en.wikipedia.org/wiki/Jalopnik "Jalopnik")_.
  487. **^** Siddiqui, Faiz; Duncan, Ian (November 1, 2019). ["Federal safety officials probe alleged Tesla battery defects"](https://www.washingtonpost.com/technology/2019/11/01/federal-safety-officials-probe-alleged-tesla-battery-defects/). _[The Washington Post](https://en.wikipedia.org/wiki/The_Washington_Post "The Washington Post")_.
  488. **^** Lopez, Linette (June 24, 2020). ["Tesla knew its Model S battery had a design flaw that could lead to leaks and, ultimately, fires starting in 2012. It sold the car anyway"](https://www.businessinsider.com/tesla-faulty-battery-cooling-systems-design-model-s-2012-2019-6). _Business Insider_.
  489. **^** Blanco, Sebastian (November 1, 2019). ["NHTSA, Investigating Tesla Fire Reports, Demands Data on Battery Software Changes"](https://www.caranddriver.com/news/a29666731/nhtsa-investigation-tesla-fire-reports/). _[Car and Driver](https://en.wikipedia.org/wiki/Car_and_Driver "Car and Driver")_.
  490. **^** Vlasic, Bill; Boudette, Neal E. (June 30, 2016). ["Self-Driving Tesla Was Involved in Fatal Crash, U.S. Says"](https://www.nytimes.com/2016/07/01/business/self-driving-tesla-fatal-crash-investigation.html). _[The New York Times](https://en.wikipedia.org/wiki/The_New_York_Times "The New York Times")_. [Archived](https://web.archive.org/web/20160630224122/http://www.nytimes.com/2016/07/01/business/self-driving-tesla-fatal-crash-investigation.html) from the original on June 30, 2016.
  491. **^** ["Preliminary Report, Highway HWY16FH018"](https://www.ntsb.gov/investigations/AccidentReports/Pages/HWY16FH018-preliminary.aspx). [NTSB](https://en.wikipedia.org/wiki/NTSB "NTSB"). July 26, 2016.
  492. **^** Steware, Jack (January 20, 2017). ["After Probing Tesla's Deadly Crash, Feds Say Yay to Self-Driving"](https://www.wired.com/2017/01/probing-teslas-deadly-crash-feds-say-yay-self-driving/). _[Wired](https://en.wikipedia.org/wiki/Wired_\(magazine\) "Wired \(magazine\)")_.
  493. **^** ["Apple engineer killed in Tesla SUV crash on Silicon Valley freeway was playing videogame: NTSB"](https://www.marketwatch.com/story/apple-engineer-killed-in-tesla-suv-crash-on-silicon-valley-freeway-was-playing-videogame-ntsb-2020-02-25). _[MarketWatch](https://en.wikipedia.org/wiki/MarketWatch "MarketWatch")_. February 25, 2020.
  494. **^** ["PlainSite :: Documents :: NHTSA Special Crash Investigations ADAS / ADS Case Spreadsheet"](https://www.plainsite.org/documents/2f8yuc/nhtsa-adas-ads-cases-spreadsheet/). _plainsite.org_. Retrieved June 23, 2021.
  495. **^** ["U.S. safety agency probes 10 Tesla crash deaths since 2016"](https://www.reuters.com/business/autos-transportation/us-safety-agency-says-it-has-opened-probes-into-10-tesla-crash-deaths-since-2016-2021-06-17/). _Reuters_. June 17, 2021. Retrieved June 23, 2021.
  496. ^ _**a**_ _**b**_ Kolodny, Lora (September 1, 2021). ["Tesla must deliver Autopilot crash data to federal auto safety watchdog by October 22"](https://www.cnbc.com/2021/09/01/tesla-must-deliver-autopilot-crash-data-to-nhtsa-by-october-22.html). CNBC. Retrieved September 4, 2021.
  497. **^** White, Annie (September 2, 2021). ["Tesla Must Send Autopilot Data to Feds by October 22"](https://www.caranddriver.com/news/a37465030/tesla-autopilot-data-nhtsa-october-22-investigation/). _Car and Driver_. Retrieved September 4, 2021.
  498. **^** Mitchell, Russ (September 2, 2021). ["'A very big deal': Federal safety regulator takes aim at Tesla Autopilot"](https://www.latimes.com/business/story/2021-09-02/safety-regulators-put-tesla-autopilot). _Los Angeles Times_. Retrieved September 4, 2021.
  499. **^** Holderith, Peter (September 27, 2021). ["Tesla Autopilot Will Now Recognize Emergency Lights, Reduce Speed: Report"](https://www.thedrive.com/news/42531/tesla-autopilot-will-now-recognize-emergency-lights-reduce-speed-report). _The Drive_. Retrieved September 7, 2022.
  500. **^** Shepardson, David (October 13, 2021). ["U.S. asks Tesla why it did not recall Autopilot after software changes"](https://www.reuters.com/business/autos-transportation/us-nhtsa-asks-tesla-why-it-did-not-recall-autopilot-system-ap-2021-10-13/). _Reuters_. Retrieved September 7, 2022.
  501. **^** Hetzner, Christiaan (June 12, 2022). ["Elon Musk's regulatory woes mount as U.S. moves closer to recalling Tesla's self-driving software"](https://fortune.com/2022/06/10/elon-musk-tesla-nhtsa-investigation-traffic-safety-autonomous-fsd-fatal-probe/). _Fortune_. Retrieved June 12, 2022.
  502. **^** Helmore, Edward (August 9, 2022). ["Tesla's self-driving technology fails to detect children in the road, tests find"](https://www.theguardian.com/technology/2022/aug/09/tesla-self-driving-technology-safety-children). _The Guardian_.
  503. **^** Stumpf, Rob (August 12, 2022). ["Controversy Erupts Over Video of FSD Tesla Striking Child Mannequin"](https://www.thedrive.com/news/controversy-erupts-over-video-of-fsd-tesla-striking-child-mannequin). _The Drive_. Retrieved August 17, 2022.
  504. **^** ["Don't Use Your Kids to Test Tesla's Safety Features, NHTSA Warns"](https://www.bloomberg.com/news/articles/2022-08-17/don-t-use-your-kids-to-test-tesla-s-safety-features-nhtsa-warns?sref=ExbtjcSG). _Bloomberg.com_. Bloomberg L.P. August 17, 2022. Retrieved September 9, 2022.
  505. **^** ["CA DMV claims Tesla misled drivers on Autopilot capabilities; seeks to suspend business for 30 days"](https://abc7news.com/post/california-dmv-claims-tesla-misled-drivers-driving-capabilities-looks-suspend-business-30-days/17234677/). _ABC7 San Francisco_. July 22, 2025. Retrieved July 23, 2025.
  506. **^** ["Tesla sales could be suspended in California"](https://www.newsweek.com/telsa-california-dmv-lawsuit-2102299). _Newsweek_. July 22, 2025. Retrieved July 23, 2025.
  507. **^** Masunaga, Samantha (August 6, 2015). ["Researchers hack a Tesla Model S, bring car to stop"](https://www.latimes.com/business/la-fi-hy-tesla-hack-20150806-story.html). _[Los Angeles Times](https://en.wikipedia.org/wiki/Los_Angeles_Times "Los Angeles Times")_. Retrieved August 10, 2015.
  508. **^** Mahaffey, Kevin (August 6, 2015). ["The new assembly line: 3 best practices for building (secure) connected cars"](https://blog.lookout.com/blog/2015/08/06/tesla-research/). _Lookout_. Retrieved August 13, 2015.
  509. **^** O'Connor, Fred (August 7, 2015). ["Tesla patches Model S after researchers hack car's software"](https://www.wired.com/2015/08/researchers-hacked-model-s-teslas-already/). _[Wired](https://en.wikipedia.org/wiki/Wired_\(magazine\) "Wired \(magazine\)")_. Retrieved August 11, 2015.
  510. **^** ["Car Hacking Research: Remote Attack Tesla Motors"](http://keenlab.tencent.com/en/2016/09/19/Keen-Security-Lab-of-Tencent-Car-Hacking-Research-Remote-Attack-to-Tesla-Cars/). September 19, 2016. Retrieved September 21, 2016.
  511. **^** Lambert, Fred (September 20, 2016). ["First Tesla Model S remotely controlled by hackers, Tesla already pushed a fix"](https://electrek.co/2016/09/20/first-tesla-model-s-remotely-controlled-hackers-tesla-pushed-a-fix/). _Electrek_. Retrieved September 21, 2016.
  512. **^** ["This Tesla Investor's Tech Team Just Hacked the Model X – Again"](http://fortune.com/2017/07/28/tesla-model-x-tencent/). _Fortune_. July 28, 2017. Retrieved October 15, 2017.
  513. **^** Hackett, Robert (February 20, 2018). ["Tesla Hackers Hijacked Amazon Cloud Account to Mine Cryptocurrency"](https://web.archive.org/web/20201210083754/https://fortune.com/2018/02/20/tesla-hack-amazon-cloud-cryptocurrency-mining/). _Fortune_. Archived from [the original](http://fortune.com/2018/02/20/tesla-hack-amazon-cloud-cryptocurrency-mining/) on December 10, 2020. Retrieved February 21, 2018.
  514. **^** Lambert, Fred (February 20, 2018). ["Tesla's cloud was 'hijacked' by hackers to mine cryptocurrencies"](https://electrek.co/2018/02/20/tesla-cloud-hijacked-hackers-mine-cryptocurrencies/). _electrek.co_. Retrieved February 21, 2018.
  515. **^** Villaruel, John Carlo A. (March 24, 2019). ["Hackers Who Cracked Tesla Model 3 Security in Competition Win Electric Car And $375K"](https://www.techtimes.com/articles/240177/20190324/hackers-who-cracked-tesla-model-3-security-in-competition-win-electric-car-and-375k.htm). _Tech Times_.
  516. **^** Goodin, Dan (June 8, 2022). ["Gone in 130 seconds: New Tesla hack gives thieves their own personal key"](https://arstechnica.com/information-technology/2022/06/hackers-out-to-steal-a-tesla-can-create-their-very-own-personal-key/). _Ars Technica_. Retrieved July 21, 2022.
  517. **^** ["Tesla drivers report a surge in 'phantom braking'"](https://www.washingtonpost.com/technology/2022/02/02/tesla-phantom-braking/). _[The Washington Post](https://en.wikipedia.org/wiki/The_Washington_Post "The Washington Post")_. [ISSN](https://en.wikipedia.org/wiki/ISSN_\(identifier\) "ISSN \(identifier\)") [0190-8286](https://search.worldcat.org/issn/0190-8286). Retrieved September 9, 2022.
  518. **^** ["Tesla investigated over 'phantom braking' problem"](https://www.bbc.com/news/technology-60432351). _BBC News_. February 18, 2022.
  519. **^** ["Law.com Radar"](https://web.archive.org/web/20220829064800/https://www.law.com/radar/card/alvarez-toledo-v-tesla-inc-45770852-0/). _www.law.com_. Archived from [the original](https://www.law.com/radar/card/alvarez-toledo-v-tesla-inc-45770852-0/) on August 29, 2022. Retrieved August 29, 2022.
  520. **^** Hals, Tom; Jin, Hyunjoo; Hals, Tom (August 29, 2022). ["Tesla hit with proposed class action over phantom braking issue"](https://www.reuters.com/legal/tesla-hit-with-proposed-class-action-over-phantom-braking-issue-2022-08-29/). _Reuters_. [Archived](https://web.archive.org/web/20220909092641/https://www.reuters.com/legal/tesla-hit-with-proposed-class-action-over-phantom-braking-issue-2022-08-29/) from the original on September 9, 2022. Retrieved September 9, 2022.
  521. ^ _**a**_ _**b**_ Nordlund, Scott. ["Tesla fails to get "phantom braking" case dismissed"](https://driveteslacanada.ca/news/tesla-fails-to-get-phantom-braking-case-dismissed/). _driveteslacanada.ca_. Retrieved January 6, 2025.
  522. **^** ["Elektromobilität : "Mein Autopilot hat mich fast umgebracht": Tesla-Files nähren Zweifel an Elon Musks Versprechen"](https://www.handelsblatt.com/unternehmen/industrie/elektromobilitaet-mein-autopilot-hat-mich-fast-umgebracht-tesla-files-naehren-zweifel-an-elon-musks-versprechen/29166564.html) ["My autopilot almost killed me": Tesla files cast doubt on Elon Musk's promises]. _Handelsblatt_ (in German). Retrieved May 26, 2023.
  523. **^** ["Whistleblower Drops 100 Gigabytes Of Tesla Secrets To German News Site: Report"](https://jalopnik.com/whistleblower-drops-100-gigabytes-of-tesla-secrets-to-g-1850476542). _Jalopnik_. May 25, 2023. Retrieved May 26, 2023.
  524. **^** Alkousaa, Riham; Sterling, Toby (May 26, 2023). ["Dutch watchdog looking into alleged Tesla data breach"](https://www.reuters.com/business/autos-transportation/german-authorities-looking-into-possible-data-protection-violations-by-tesla-2023-05-25/). _Reuters_. Retrieved May 26, 2023.
  525. **^** Anderson, Brad (February 19, 2025). ["Court Rules Tesla's Autopilot Defective For Normal Use After Phantom Braking"](https://www.carscoops.com/2025/02/german-court-finds-teslas-autopilot-defective-after-lawsuit/). _Carscoops_. Retrieved February 24, 2025.
  526. **^** Biggs, Tim (February 25, 2025). ["Tesla hit with Australian class action over 'phantom braking'"](https://www.smh.com.au/technology/tesla-hit-with-australian-class-action-over-phantom-braking-20250225-p5lex5.html). _Sydney Morning Herald_. Australia. Retrieved February 25, 2025.
  527. **^** Steve Stecklow and Norihiko Shirouzu. "[Tesla created secret team to suppress thousands of driving range complaints](https://www.reuters.com/investigates/special-report/tesla-batteries-range/)". _Reuters_ , July 27, 2023. Retrieved July 27, 2023.
  528. **^** ["Global NEV Sales Expected to Grow 18% in 2025, with US Market Facing Uncertainty, Says TrendForce"](https://www.trendforce.com/presscenter/news/20250220-12478.html). _TrendForce_. February 20, 2025. Retrieved July 15, 2025.
  529. **^** Kolodny, Lora (January 2, 2024). ["Tesla reported 485,000 deliveries for the fourth quarter, bringing 2023 total to 1.8 million"](https://www.cnbc.com/2024/01/02/tesla-tsla-q4-2023-vehicle-delivery-and-production-numbers.html). _CNBC_. Retrieved March 29, 2024.
  530. **^** ["Tesla Vehicle Production & Deliveries and Date for Financial Results & Webcast for Fourth Quarter 2023"](https://ir.tesla.com/press-release/tesla-vehicle-production-deliveries-and-date-financial-results-webcast-fourth-quarter-2023). _ir.tesla.com_. January 2, 2024. Retrieved March 29, 2024.
  531. **^** Nurman, Andy (March 29, 2024). ["Tesla's Electrifying 6 Million Production Milestone – The Road Ahead"](https://carlist.com/teslas-electrifying-6-million-production-milestone-the-road-ahead/). _Carlist_. Retrieved March 29, 2024.
  532. **^** ["BYD hands back top EV seller title to Tesla after Q1 sales decline"](https://finance.yahoo.com/news/byd-may-hand-back-top-063348429.html). _Yahoo Finance_. April 2, 2024. Retrieved April 3, 2024.
  533. **^** Butler, Desmond; Thadani, Trisha; Martinez, Emmanuel; Gregg, Aaron; Melgar, Luis; O'connell, Jonathan; Keating, Dan (February 18, 2025). ["Elon Musk's business empire is built on $38 billion in government funding"](https://www.washingtonpost.com/technology/interactive/2025/elon-musk-business-government-contracts-funding/). _Washington Post_. US. Retrieved March 9, 2025.
  534. **^** ["Elon Musk's business empire is built on $38 billion in government funding"](https://www.msn.com/en-us/money/companies/elon-musk-s-business-empire-is-built-on-38-billion-in-government-funding/ar-AA1zP6hC). _www.msn.com_. Retrieved March 9, 2025.
  535. **^** Zlatev, Daniel (October 3, 2024). ["US government reveals its spending on EV tax credit subsidies putting more Teslas on the road"](https://www.notebookcheck.net/US-government-reveals-its-spending-on-EV-tax-credit-subsidies-putting-more-Teslas-on-the-road.897281.0.html). _Notebookcheck_. Retrieved March 9, 2025.
  536. **^** ["Tesla Sales by Country 2024"](https://worldpopulationreview.com/country-rankings/tesla-sales-by-country). _worldpopulationreview.com_. Retrieved March 9, 2025.
  537. **^** ["China Paid Billions In Aid To Local EV Makers, Including Tesla, To Dominate The Market: Study"](https://insideevs.com/news/716063/china-ev-subsidies-byd-tesla-billions-study/). _InsideEVs_. Retrieved March 9, 2025.
  538. **^** Merano, Maria (December 19, 2023). ["Tesla compensates for the end of EV subsidies in Germany"](https://www.teslarati.com/tesla-germany-ev-subsidies-compensate/). _TESLARATI_. Retrieved March 9, 2025.
  539. **^** Lilley, Zane (November 27, 2024). ["Big drop in state aid for electric cars in France: what is still available?"](https://www.connexionfrance.com/practical/big-drop-in-state-aid-for-electric-cars-in-france-what-is-still-available/691851). _www.connexionfrance.com_. Retrieved March 9, 2025.
  540. **^** John, Darryn (March 6, 2025). ["Tesla sells over 8,600 cars in Canada during final days of iZEV rebate fund"](https://driveteslacanada.ca/news/tesla-sells-8600-cars-canada-final-days-of-canadas-izev/). _Drive Tesla_. Retrieved March 9, 2025.
  541. **^** CHOWN OVED, MARCO (March 7, 2025). ["Other dealers 'stiffed' as Tesla claimed rebates"](https://www.pressreader.com/canada/toronto-star/20250307/281509346946851). Retrieved March 9, 2025 – via PressReader.
  542. **^** ["Canada freezes rebate payments to Tesla, bars it from future programs due to tariffs"](https://www.cbc.ca/news/politics/canada-freezes-rebates-tesla-1.7493434). _CBC_. Canada. March 25, 2025. Retrieved March 28, 2025.
  543. **^** Kirkham, Chris (August 12, 2024). ["Musk embraces Trump and scorns subsidies. But Tesla still lobbies for US benefits"](https://www.reuters.com/world/us/musk-embraces-trump-scorns-subsidies-tesla-still-lobbies-us-benefits-2024-08-12/). _Reuters_.
  544. **^** Ohnsman, Alan. ["Elon Musk's Relentless Trolling Of Democrats Is Tarnishing Tesla"](https://www.forbes.com/sites/alanohnsman/2024/11/01/elon-musks-relentless-trolling-of-democrats-is-tarnishing-tesla/). _Forbes_.
  545. **^** ["'Few Democrats willing to buy a Tesla' after Elon Musk backs Trump, investor warns"](https://finance.yahoo.com/news/few-democrats-willing-to-buy-a-tesla-after-elon-musk-backs-trump-investor-warns-170050119.html). _finance.yahoo.com_.
  546. **^** Gitlin, Jonathan M. (February 5, 2025). ["Teslas turn toxic as sales crash in Europe and the UK"](https://arstechnica.com/cars/2025/02/tesla-sales-plummet-in-the-uk-france-and-germany/). _Ars Technica_. US. Retrieved February 6, 2025.
  547. **^** Morpurgo, Connor (February 4, 2025). ["Tesla sales plummet in European countries after Musk interferes with politics"](https://euroweeklynews.com/2025/02/03/tesla-sales-plummet-in-european-countries-after-musk-interferes-with-politics/). _Euro Weekly News_. Retrieved February 6, 2025.
  548. **^** Francis, Sam; Farley, Harry (January 5, 2025). ["Musk says Farage 'doesn't have what it takes' to be Reform UK leader"](https://www.bbc.co.uk/news/articles/c70ep8lp4jjo). _BBC News_. UK. Retrieved February 27, 2025.
  549. **^** Treisman, Rachel (January 27, 2025). ["Elon Musk faces criticism for encouraging Germans to move beyond 'past guilt'"](https://www.npr.org/2025/01/27/nx-s1-5276084/elon-musk-german-far-right-afd-holocaust). _NPR_. US. Retrieved February 27, 2025.
  550. **^** O'Mahony, Proinsias (March 2, 2025). ["'Don't buy a Swasticar': Elon Musk is wrecking Tesla's brand"](https://www.irishtimes.com/your-money/2025/03/02/dont-buy-a-swasticar-elon-musk-is-wrecking-teslas-brand/). _[The Irish Times](https://en.wikipedia.org/wiki/The_Irish_Times "The Irish Times")_. [Archived](https://web.archive.org/web/20250306050557/https://www.irishtimes.com/your-money/2025/03/02/dont-buy-a-swasticar-elon-musk-is-wrecking-teslas-brand/) from the original on March 6, 2025. Retrieved March 6, 2025.
  551. **^** Ford, Lily (February 26, 2025). ["Anti-Elon Musk Poster "Don't Buy a Swasticar" in London Goes Viral"](https://www.hollywoodreporter.com/news/general-news/elon-musk-poster-london-tesla-sales-drop-1236147614/). _[The Hollywood Reporter](https://en.wikipedia.org/wiki/The_Hollywood_Reporter "The Hollywood Reporter")_. [Archived](https://web.archive.org/web/20250228132605/https://www.hollywoodreporter.com/news/general-news/elon-musk-poster-london-tesla-sales-drop-1236147614/) from the original on February 28, 2025. Retrieved March 6, 2025.
  552. **^** Howe, Megan (February 28, 2025). ["Elon Musk mocked in huge advert for 'Swasticar' months after he was accused of giving a 'Nazi-style' salute"](https://www.standard.co.uk/news/politics/elon-musk-tesla-swasticar-nazi-salute-b1213170.html). _[Evening Standard](https://en.wikipedia.org/wiki/Evening_Standard "Evening Standard")_. [Archived](https://web.archive.org/web/20250228190313/https://www.standard.co.uk/news/politics/elon-musk-tesla-swasticar-nazi-salute-b1213170.html) from the original on February 28, 2025. Retrieved March 6, 2025.
  553. **^** Kassam, Ashifa (February 25, 2025). ["'I felt nothing but disgust': Tesla owners vent their anger at Elon Musk"](https://www.theguardian.com/business/2025/feb/25/i-felt-nothing-but-disgust-tesla-owners-vent-their-anger-at-elon-musk). _[The Guardian](https://en.wikipedia.org/wiki/The_Guardian "The Guardian")_. [Archived](https://web.archive.org/web/20250228135530/https://www.theguardian.com/business/2025/feb/25/i-felt-nothing-but-disgust-tesla-owners-vent-their-anger-at-elon-musk) from the original on February 28, 2025. Retrieved March 6, 2025.
  554. **^** Burman, Theo (February 25, 2025). ["Anti-Elon Musk 'Swasticar' Advert Viewed More Than 6 Million Times"](https://www.newsweek.com/anti-elon-musk-swasticar-advert-tiktok-2035950). _[Newsweek](https://en.wikipedia.org/wiki/Newsweek "Newsweek")_. US. Retrieved February 27, 2025.
  555. **^** According to multiple sources:[550][551][552][553][554]
  556. **^** ["Tesla Sales Fall In Germany After Musk Backs Far Right"](https://www.barrons.com/news/tesla-sales-in-germany-fall-76-in-february-9fb92a0d). _Agence France Presse_. US. March 5, 2025. Retrieved March 6, 2025 – via Barrons.
  557. **^** ["Tesla's German car sales continue their decline in February"](https://www.reuters.com/business/retail-consumer/teslas-german-car-sales-continue-their-decline-february-2025-03-05/). _Reuters_. US. March 5, 2025. Retrieved March 6, 2025.
  558. **^** Espiner, Tom (February 26, 2025). ["Tesla shares slump after European sales fall"](https://web.archive.org/web/20250226081814/https://www.bbc.com/news/articles/cvgd9v3r69qo). _BBC News_. Archived from [the original](https://www.bbc.com/news/articles/cvgd9v3r69qo) on February 26, 2025. Retrieved February 26, 2025.
  559. ^ _**a**_ _**b**_ Lea, Robert (May 27, 2025). ["Tesla sales down 49% across Europe in April"](https://www.thetimes.com/business-money/companies/article/tesla-sales-drop-38-percent-across-europe-nd5757f2b). _www.thetimes.com_. Retrieved May 28, 2025.
  560. **^** Lambert, Fred (March 9, 2020). ["Tesla produces its 1 millionth electric car"](https://electrek.co/2020/03/09/tesla-produces-1000000th-electric-car/). _Electrek_. Retrieved March 28, 2020.
  561. **^** Kane, Mark (October 21, 2021). ["Tesla Sold 2 Million Electric Cars: First Automaker To Reach Milestone"](https://insideevs.com/news/542197/tesla-sold-2000000-electric-cars/). _InsideEVs_. Retrieved November 15, 2021.
  562. **^** Weatherbed, Jess (May 26, 2023). ["The Tesla Model Y is now the world's bestselling car"](https://www.theverge.com/2023/5/26/23738581/tesla-model-y-ev-record-world-bestselling-car-electric). _The Verge_. Retrieved May 29, 2023.
  563. ^ _**a**_ _**b**_ _**c**_ ["Tesla Fourth Quarter & Full Year 2021 Update"](https://web.archive.org/web/20240113135400/https://tesla-cdn.thron.com/static/WIIG2L_TSLA_Q4_2021_Update_O7MYNE.pdf?xseo=&response-content-disposition=inline&filename=%22tsla-q4-and-fy-2021-update.pdf%22) (PDF). Palo Alto: Tesla. January 26, 2022. Archived from [the original](https://tesla-cdn.thron.com/static/WIIG2L_TSLA_Q4_2021_Update_O7MYNE.pdf?xseo=&response-content-disposition=inline%3Bfilename%3D%22tsla-q4-and-fy-2021-update.pdf%22) (PDF) on January 13, 2024. Retrieved January 27, 2022. See table "Operational Summary" pp. 7 and 8 for revised and final production and sales numbers.
  564. ^ _**a**_ _**b**_ _**c**_ ["Tesla, Inc.: Shareholders Board Members Managers and Company Profile | US88160R1014"](https://www.marketscreener.com/quote/stock/TESLA-INC-6344549/company/). _MarketScreener_. Retrieved March 6, 2024.
  565. **^** Jackson, Jon (February 8, 2021). ["Elon Musk's Bitcoin investment supports energy waste, some critics say"](https://www.newsweek.com/musk-bitcoin-environment-1567697). _[Newsweek](https://en.wikipedia.org/wiki/Newsweek "Newsweek")_. Retrieved February 13, 2021.
  566. **^** Shieber, Jonathan (February 8, 2021). ["Tesla's Bitcoin investment could be bad for the company's climate reputation and its bottom line"](https://au.finance.yahoo.com/news/teslas-bitcoin-investment-could-bad-010425356.html). [Yahoo! Finance](https://en.wikipedia.org/wiki/Yahoo!_Finance "Yahoo! Finance"). Retrieved February 13, 2021.
  567. **^** Dean, James. ["Tesla made more profit from bitcoin in a month than from selling cars last year"](https://www.thetimes.com/business-money/technology/article/tesla-makes-more-money-on-bitcoin-than-on-cars-5sfl6qd0h). _[The Times](https://en.wikipedia.org/wiki/The_Times "The Times")_. [ISSN](https://en.wikipedia.org/wiki/ISSN_\(identifier\) "ISSN \(identifier\)") [0140-0460](https://search.worldcat.org/issn/0140-0460). Retrieved February 19, 2021.
  568. **^** ["Tesla May Have Already Made More in Profits From Bitcoin Than Electric Vehicles"](https://finance.yahoo.com/news/tesla-may-already-made-more-224537165.html). Yahoo! Finance. February 16, 2021. Retrieved February 19, 2021.
  569. **^** O'Kane, Sean (July 26, 2021). ["Tesla finally made a profit without the help of emission credits"](https://www.theverge.com/2021/7/26/22594778/tesla-q2-2021-earnings-revenue-profit-credits-emissions-bitcoin). _The Verge_. Retrieved October 22, 2021.
  570. **^** ["2006: San Carlos start-up Tesla seeks sexier electric car"](https://www.mercurynews.com/2014/07/14/2006-san-carlos-start-up-tesla-seeks-sexier-electric-car/). July 14, 2014.
  571. ^ _**a**_ _**b**_ _**c**_ _**d**_ _**e**_ _**f**_ _**g**_ _**h**_ _**i**_ _**j**_ _**k**_ _**l**_ _**m**_ _**n**_ _**o**_ _**p**_ ["Tesla, Inc. TSLA on Nasdaq"](https://www.sec.gov/edgar/browse/?CIK=1318605). [U.S. Securities and Exchange Commission](https://en.wikipedia.org/wiki/U.S._Securities_and_Exchange_Commission "U.S. Securities and Exchange Commission").
  572. **^** ["History of Tesla: Timeline and Facts"](https://www.thestreet.com/technology/history-of-tesla-15088992). _TheStreet_. February 4, 2020. Retrieved May 21, 2023.
  573. **^** ["Tesla Motors Zaps Another C.E.O. and Lays Off Staff"](https://archive.nytimes.com/bits.blogs.nytimes.com/2008/10/15/tesla-motors-zaps-another-ceo-and-lays-off-staff/). _The New York Times_. October 15, 2008. Retrieved May 21, 2023.
  574. ^ _**a**_ _**b**_ ["Tesla Inc. Company Profile & Executives"](https://www.wsj.com/market-data/quotes/TSLA/company-people). _The Wall Street Journal_. Retrieved March 12, 2022.
  575. **^** ["Elon Musk's Job Duties Keep Growing: Here's a List of His Board Positions Over the Years"](https://www.wsj.com/articles/elon-musks-job-duties-keep-growing-heres-a-list-of-his-board-positions-over-the-years-11649190473). _The Wall Street Journal_. April 5, 2022. Retrieved May 21, 2023.
  576. ^ _**a**_ _**b**_ ["Tesla Seeks Independent Directors as Board's Musk Ties Eyed"](https://www.bloomberg.com/news/articles/2017-04-11/tesla-investors-press-for-more-board-members-without-musk-ties). _Bloomberg.com_. April 11, 2017. Retrieved December 31, 2020.
  577. ^ _**a**_ _**b**_ _**c**_ ["Elon Musk spars with investors who want independent Tesla board"](https://www.usatoday.com/story/money/cars/2017/04/12/elon-musk-ctw-investment-group/100393980/). _[USA Today](https://en.wikipedia.org/wiki/USA_Today "USA Today")_. Retrieved April 14, 2017.
  578. ^ _**a**_ _**b**_ _**c**_ Rapier, Graham. ["Tesla has named two new board members – here's the full list of company directors"](https://www.businessinsider.com/tesla-board-of-directors-full-list-elon-musk-chairman-replacement-2018-8). _Business Insider_. Retrieved January 26, 2021.
  579. **^** Waters, Richard (April 12, 2017). ["Tesla investors seek stronger boardroom controls"](https://www.ft.com/content/205aa80a-1f20-11e7-a454-ab04428977f9). _Financial Times_. [Archived](https://ghostarchive.org/archive/20221211231221/https://www.ft.com/content/205aa80a-1f20-11e7-a454-ab04428977f9) from the original on December 11, 2022. Retrieved February 21, 2020.
  580. **^** ["Musk Promises 2 New Directors for Tesla Amid Shareholder Criticism"](http://www.foxbusiness.com/features/2017/04/12/musk-promises-2-new-directors-for-tesla-amid-shareholder-criticism.html). _[Fox Business](https://en.wikipedia.org/wiki/Fox_Business "Fox Business")_. April 12, 2017. Retrieved April 14, 2017.
  581. **^** Rai, Sonam; Klayman, Ben (December 28, 2018). ["Tesla names close Musk friend Larry Ellison to board"](https://www.reuters.com/article/us-tesla-directors-idUSKCN1OR0ZY). _Reuters_. Retrieved September 5, 2022.
  582. **^** Ponciano, Jonathan (June 10, 2022). ["Tesla Files For Another Stock Split—Reveals Billionaire Larry Ellison To Leave Board"](https://www.forbes.com/sites/jonathanponciano/2022/06/10/tesla-files-for-another-stock-split-reveals-billionaire-larry-ellison-to-leave-board/). _Forbes_. Retrieved September 5, 2022.
  583. ^ _**a**_ _**b**_ _**c**_ _**d**_ Hull, Dana; O'Kane, Sean (May 16, 2023). ["Tesla Investors Elect Former Executive JB Straubel to Board"](https://www.bloomberg.com/news/articles/2023-05-16/tesla-shareholders-elect-former-executive-jb-straubel-to-board). _Bloomberg News_. [Archived](https://archive.today/20230516213453/https://www.bloomberg.com/news/articles/2023-05-16/tesla-shareholders-elect-former-executive-jb-straubel-to-board) from the original on May 16, 2023.
  584. **^** Hartmans, Avery. ["Tesla's biggest investor says the company's chairwoman gives Elon Musk 'emotional' support so he can focus on leading the company"](https://www.businessinsider.com/tesla-chairwoman-gives-elon-musk-emotional-support-baillie-gifford-2020-3). _Business Insider_. Retrieved March 9, 2020.
  585. ^ _**a**_ _**b**_ Holland, Maximilian (November 8, 2018). ["More Background On New Tesla Chair Robyn Denholm"](https://cleantechnica.com/2018/11/08/more-background-on-new-tesla-chair-robyn-denholm/). _CleanTechnica_. Retrieved March 9, 2020.
  586. ^ _**a**_ _**b**_ _**c**_ _**d**_ Wesoff, Eric (April 22, 2019). ["Tesla Announces Departure of 4 Board Members Ahead of a Really Big Week"](https://www.greentechmedia.com/articles/read/tesla-announces-departure-of-4-board-members-ahead-of-a-really-big-week). Greentech Media. Retrieved January 26, 2021.
  587. **^** Yoshida, Kaori (December 31, 2020). ["Tesla director Hiro Mizuno picked as UN sustainable investment envoy"](https://asia.nikkei.com/Business/Finance/Tesla-director-Hiro-Mizuno-picked-as-UN-sustainable-investment-envoy). _Nikkei Asia_. Retrieved December 31, 2020.
  588. **^** ["Tesla Changes Up Board With Nomination of Former Tech Chief"](https://www.bloomberg.com/news/articles/2023-04-06/tesla-nominates-former-tech-chief-straubel-to-board-of-directors). _Bloomberg.com_. April 6, 2023. Retrieved April 29, 2023.
  589. **^** ["TSLA | Tesla Inc. Company Profile & Executives – WSJ"](https://www.wsj.com/market-data/quotes/TSLA/company-people). _The Wall Street Journal_. Retrieved May 16, 2023.
  590. **^** Kolodny, Lora (November 8, 2018). ["Robyn Denholm replaces Elon Musk as Tesla's board chair"](https://www.cnbc.com/2018/11/08/robyn-denholm-will-replace-elon-musk-as-teslas-board-chair.html#:~:text=Robyn%20has%20served%20on%20the,Juniper%20Networks%2C%20and%20Sun%20Microsystems.). CNBC. Retrieved January 26, 2021.
  591. **^** ["Date of report (Date of earliest event reported): October 7, 2021"](https://www.sec.gov/ix?doc=/Archives/edgar/data/1318605/000156459021050737/tsla-8k_20211007.htm). _sec.gov_. Retrieved October 15, 2021.
  592. **^** ["Elon Musk is sticking with SpaceX board member Steve Jurvetson, shows new SEC filing"](https://techcrunch.com/2019/01/03/elon-musk-is-sticking-with-spacex-board-member-steve-jurveston-shows-new-sec-filing/). _TechCrunch_. Retrieved January 26, 2021.
  593. **^** Ohnsman, Alan. ["Elon's Enablers: Tesla's Submissive Board May Be As Big A Risk As An Erratic CEO"](https://www.forbes.com/sites/alanohnsman/2018/09/13/elons-enablers-teslas-submissive-board-may-be-as-big-a-risk-as-an-erratic-ceo/). _Forbes_. Retrieved February 22, 2020.
  594. **^** Nishant, Niket; Sriram, Akash (September 28, 2022). ["Tesla adds billionaire Airbnb co-founder Gebbia to board"](https://www.reuters.com/business/autos-transportation/tesla-adds-airbnb-co-founder-gebbia-board-2022-09-28/). _[Reuters](https://en.wikipedia.org/wiki/Reuters "Reuters")_.
  595. **^** Korosec, Kirsten (September 28, 2022). ["Tesla appoints Airbnb co-founder to board"](https://techcrunch.com/2022/09/28/tesla-appoints-airbnb-co-founder-to-board/). _[TechCrunch](https://en.wikipedia.org/wiki/TechCrunch "TechCrunch")_.
  596. **^** Jin, Hyunjoo (April 27, 2023). ["Glass Lewis recommends vote against Tesla board nominee JB Straubel"](https://www.reuters.com/business/autos-transportation/glass-lewis-recommends-tesla-investor-vote-against-board-nominee-jb-straubel-2023-04-27/). Reuters.



## Sources

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=97 "Edit section: Sources")]

  * [Vance, Ashlee](https://en.wikipedia.org/wiki/Ashlee_Vance "Ashlee Vance") (2015). _Elon Musk: Tesla, SpaceX, and the Quest for a Fantastic Future_. New York: HarperCollins. [ISBN](https://en.wikipedia.org/wiki/ISBN_\(identifier\) "ISBN \(identifier\)") [978-0062301239](https://en.wikipedia.org/wiki/Special:BookSources/978-0062301239 "Special:BookSources/978-0062301239"). [OCLC](https://en.wikipedia.org/wiki/OCLC_\(identifier\) "OCLC \(identifier\)") [881436803](https://search.worldcat.org/oclc/881436803).



## Further reading

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=98 "Edit section: Further reading")]

  * Higgins, Tim (2021). _[Power Play: Tesla, Elon Musk, and the Bet of the Century](https://en.wikipedia.org/wiki/Power_Play:_Tesla,_Elon_Musk,_and_the_Bet_of_the_Century "Power Play: Tesla, Elon Musk, and the Bet of the Century")_. Doubleday. [ISBN](https://en.wikipedia.org/wiki/ISBN_\(identifier\) "ISBN \(identifier\)") [978-0385545464](https://en.wikipedia.org/wiki/Special:BookSources/978-0385545464 "Special:BookSources/978-0385545464").
  * McKenzie, Hamish (2018). _Insane Mode: How Elon Musk's Tesla Sparked an Electric Revolution to End the Age of Oil_. New York: Dutton. [ISBN](https://en.wikipedia.org/wiki/ISBN_\(identifier\) "ISBN \(identifier\)") [978-1101985953](https://en.wikipedia.org/wiki/Special:BookSources/978-1101985953 "Special:BookSources/978-1101985953").
  * Niedermeyer, Edward (2019). _[Ludicrous: The Unvarnished Story of Tesla Motors](https://en.wikipedia.org/wiki/Ludicrous:_The_Unvarnished_Story_of_Tesla_Motors "Ludicrous: The Unvarnished Story of Tesla Motors")_. Dallas: BenBella. [ISBN](https://en.wikipedia.org/wiki/ISBN_\(identifier\) "ISBN \(identifier\)") [978-1948836326](https://en.wikipedia.org/wiki/Special:BookSources/978-1948836326 "Special:BookSources/978-1948836326"). [OCLC](https://en.wikipedia.org/wiki/OCLC_\(identifier\) "OCLC \(identifier\)") [1089841254](https://www.worldcat.org/oclc/1089841254).



## External links

[[edit](https://en.wikipedia.org/w/index.php?title=Tesla,_Inc.&action=edit&section=99 "Edit section: External links")]

[](https://en.wikipedia.org/wiki/File:Commons-logo.svg)

Wikimedia Commons has media related to [Tesla, Inc.](https://commons.wikimedia.org/wiki/Category:Tesla,_Inc. "commons:Category:Tesla, Inc.").

  * [Official website](https://www.tesla.com/) [](https://www.wikidata.org/wiki/Q478214#P856 "Edit this at Wikidata")
  * [Tesla, Inc.](https://www.opensecrets.org/orgs/summary?id=D000057516) on [OpenSecrets](https://en.wikipedia.org/wiki/OpenSecrets "OpenSecrets"), a website that tracks and publishes data on campaign finance and lobbying [](https://www.wikidata.org/wiki/Q478214#P4691 "Edit this at Wikidata")
  * Business data for Tesla, Inc.: 

    * [Google](https://www.google.com/finance/quote/TSLA)
    * [Reuters](https://www.reuters.com/markets/companies/TSLA.OQ)
    * [SEC filings](https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK=1318605)
    * [Yahoo!](https://finance.yahoo.com/quote/TSLA)




  * [v](https://en.wikipedia.org/wiki/Template:Tesla,_Inc. "Template:Tesla, Inc.")
  * [t](https://en.wikipedia.org/wiki/Template_talk:Tesla,_Inc. "Template talk:Tesla, Inc.")
  * [e](https://en.wikipedia.org/wiki/Special:EditPage/Template:Tesla,_Inc. "Special:EditPage/Template:Tesla, Inc.")

Tesla, Inc.  
---  
[Electric  
vehicles](https://en.wikipedia.org/wiki/Electric_vehicle "Electric vehicle")| | Current| 

  * [Cybertruck](https://en.wikipedia.org/wiki/Tesla_Cybertruck "Tesla Cybertruck")
  * [Model 3](https://en.wikipedia.org/wiki/Tesla_Model_3 "Tesla Model 3")
  * [Model S](https://en.wikipedia.org/wiki/Tesla_Model_S "Tesla Model S")
  * [Model X](https://en.wikipedia.org/wiki/Tesla_Model_X "Tesla Model X")
  * [Model Y](https://en.wikipedia.org/wiki/Tesla_Model_Y "Tesla Model Y")
  * [Semi](https://en.wikipedia.org/wiki/Tesla_Semi "Tesla Semi")

  
---|---  
Future| 

  * [Cybercab](https://en.wikipedia.org/wiki/Tesla_Cybercab "Tesla Cybercab")
  * [Cyberquad](https://en.wikipedia.org/wiki/Tesla_Cyberquad "Tesla Cyberquad")
  * [Roadster (2nd gen)](https://en.wikipedia.org/wiki/Tesla_Roadster_\(second_generation\) "Tesla Roadster \(second generation\)")
  * [Robovan](https://en.wikipedia.org/wiki/Tesla_Robovan "Tesla Robovan")
  * [Tesla next-generation vehicle platform](https://en.wikipedia.org/wiki/Tesla_next-generation_vehicle_platform "Tesla next-generation vehicle platform")

  
Discontinued| 

  * [Roadster (1st gen.)](https://en.wikipedia.org/wiki/Tesla_Roadster_\(first_generation\) "Tesla Roadster \(first generation\)")

  
Joint projects| 

  * [Cyberquad for Kids](https://en.wikipedia.org/wiki/Tesla_Cyberquad "Tesla Cyberquad")
  * Mercedes 
    * [A-Class](https://en.wikipedia.org/wiki/Mercedes_A-Class_E-Cell "Mercedes A-Class E-Cell")
    * [B-Class](https://en.wikipedia.org/wiki/Mercedes-Benz_B-Class_Electric_Drive "Mercedes-Benz B-Class Electric Drive")
    * [Smart Fortwo ED](https://en.wikipedia.org/wiki/Smart_electric_drive#Second_generation "Smart electric drive")
  * [Toyota RAV4 EV (2nd gen.)](https://en.wikipedia.org/wiki/Toyota_RAV4_EV#Second_generation "Toyota RAV4 EV")

  
  
[Tesla Energy](https://en.wikipedia.org/wiki/Tesla_Energy "Tesla Energy")| | Charging| 

  * [Megacharger](https://en.wikipedia.org/wiki/Tesla_Megacharger "Tesla Megacharger")
  * [North American Charging Standard](https://en.wikipedia.org/wiki/North_American_Charging_Standard "North American Charging Standard") (NACS)
  * [Supercharger](https://en.wikipedia.org/wiki/Tesla_Supercharger "Tesla Supercharger")

  
---|---  
Storage| 

  * [Megapack](https://en.wikipedia.org/wiki/Tesla_Megapack "Tesla Megapack")
  * [Powerwall](https://en.wikipedia.org/wiki/Tesla_Powerwall "Tesla Powerwall")
  * [Powerpack](https://en.wikipedia.org/wiki/Tesla_Powerpack "Tesla Powerpack") (discontinued)

  
Solar| 

  * [Solar panels](https://en.wikipedia.org/wiki/Tesla_solar_panels "Tesla solar panels")
  * [Solar Roof](https://en.wikipedia.org/wiki/Tesla_Solar_Roof "Tesla Solar Roof")

  
  
[Artificial  
intelligence](https://en.wikipedia.org/wiki/Artificial_intelligence "Artificial intelligence")| | [Robotics](https://en.wikipedia.org/wiki/Robotics "Robotics")| 

  * [Optimus (robot)](https://en.wikipedia.org/wiki/Optimus_\(robot\) "Optimus \(robot\)")

  
---|---  
[Supercomputers](https://en.wikipedia.org/wiki/Supercomputer "Supercomputer")| 

  * [Tesla Dojo](https://en.wikipedia.org/wiki/Tesla_Dojo "Tesla Dojo")

  
[ADAS](https://en.wikipedia.org/wiki/Advanced_driver-assistance_system "Advanced driver-assistance system")| 

  * [Tesla Autopilot](https://en.wikipedia.org/wiki/Tesla_Autopilot "Tesla Autopilot") ([Tesla Autopilot hardware](https://en.wikipedia.org/wiki/Tesla_Autopilot_hardware "Tesla Autopilot hardware"))

  
  
[Factories](https://en.wikipedia.org/wiki/List_of_Tesla_factories "List of Tesla factories")| 

  * [Berlin](https://en.wikipedia.org/wiki/Gigafactory_Berlin-Brandenburg "Gigafactory Berlin-Brandenburg")
  * [Fremont](https://en.wikipedia.org/wiki/Tesla_Fremont_Factory "Tesla Fremont Factory")
  * _[Mexico](https://en.wikipedia.org/wiki/Gigafactory_Mexico "Gigafactory Mexico") (future)_
  * [Nevada](https://en.wikipedia.org/wiki/Gigafactory_Nevada "Gigafactory Nevada")
  * [New York](https://en.wikipedia.org/wiki/Gigafactory_New_York "Gigafactory New York")
  * [Shanghai](https://en.wikipedia.org/wiki/Gigafactory_Shanghai "Gigafactory Shanghai")
  * [Texas](https://en.wikipedia.org/wiki/Gigafactory_Texas "Gigafactory Texas")
  * [Tilburg](https://en.wikipedia.org/wiki/Tesla_facilities_in_Tilburg "Tesla facilities in Tilburg")

  
People| | Executives| 

  * [Elon Musk](https://en.wikipedia.org/wiki/Elon_Musk "Elon Musk") (CEO)
  * [Robyn Denholm](https://en.wikipedia.org/wiki/Robyn_Denholm "Robyn Denholm") (Chair)
  * [Vaibhav Taneja](https://en.wikipedia.org/wiki/Vaibhav_Taneja "Vaibhav Taneja") (CFO)
  * [Tom Zhu](https://en.wikipedia.org/wiki/Tom_Zhu "Tom Zhu") (SVP of automotive)

  
---|---  
Others| 

  * [Franz von Holzhausen](https://en.wikipedia.org/wiki/Franz_von_Holzhausen "Franz von Holzhausen") (Chief Designer)
  * [Joe Gebbia](https://en.wikipedia.org/wiki/Joe_Gebbia "Joe Gebbia") (board)
  * [James Murdoch](https://en.wikipedia.org/wiki/James_Murdoch "James Murdoch") (board)
  * [Kimbal Musk](https://en.wikipedia.org/wiki/Kimbal_Musk "Kimbal Musk") (board)
  * [J. B. Straubel](https://en.wikipedia.org/wiki/J._B._Straubel "J. B. Straubel") (board, co-founder)
  * [Kathleen Wilson-Thompson](https://en.wikipedia.org/wiki/Kathleen_Wilson-Thompson "Kathleen Wilson-Thompson") (board)

  
Former| 

  * [Deepak Ahuja](https://en.wikipedia.org/wiki/Deepak_Ahuja "Deepak Ahuja") (two-time CFO)
  * [Drew Baglino](https://en.wikipedia.org/wiki/Drew_Baglino "Drew Baglino") (SVP of engineering)
  * [Ze'ev Drori](https://en.wikipedia.org/wiki/Ze%27ev_Drori "Ze'ev Drori") (CEO)
  * [Martin Eberhard](https://en.wikipedia.org/wiki/Martin_Eberhard "Martin Eberhard") (co-founder, CEO)
  * [Larry Ellison](https://en.wikipedia.org/wiki/Larry_Ellison "Larry Ellison") (board)
  * [Steve Jurvetson](https://en.wikipedia.org/wiki/Steve_Jurvetson "Steve Jurvetson") (board)
  * [Andrej Karpathy](https://en.wikipedia.org/wiki/Andrej_Karpathy "Andrej Karpathy") (AI)
  * [Zach Kirkhorn](https://en.wikipedia.org/wiki/Zach_Kirkhorn "Zach Kirkhorn") (CFO)
  * [Arnnon Geshuri](https://en.wikipedia.org/wiki/Arnnon_Geshuri "Arnnon Geshuri") (HR)
  * [Jérôme Guillen](https://en.wikipedia.org/wiki/Jerome_Guillen "Jerome Guillen") (automotive)
  * [Jim Keller](https://en.wikipedia.org/wiki/Jim_Keller_\(engineer\) "Jim Keller \(engineer\)") (Autopilot)
  * [Chris Lattner](https://en.wikipedia.org/wiki/Chris_Lattner "Chris Lattner") (Autopilot)
  * [Hiromichi Mizuno](https://en.wikipedia.org/wiki/Hiromichi_Mizuno "Hiromichi Mizuno") (board)
  * [Marc Tarpenning](https://en.wikipedia.org/wiki/Marc_Tarpenning "Marc Tarpenning") (co-founder, CFO)
  * [Jay Vijayan](https://en.wikipedia.org/wiki/Jay_Vijayan "Jay Vijayan") (CIO)

  
  
Controversies| 

  * [Criticism of Tesla](https://en.wikipedia.org/wiki/Criticism_of_Tesla,_Inc. "Criticism of Tesla, Inc.")
  * [Dealership disputes](https://en.wikipedia.org/wiki/Tesla_US_dealership_disputes "Tesla US dealership disputes")
  * [Everyone Hates Elon](https://en.wikipedia.org/wiki/Everyone_Hates_Elon "Everyone Hates Elon")
  * [Lawsuits (list)](https://en.wikipedia.org/wiki/List_of_lawsuits_involving_Tesla,_Inc. "List of lawsuits involving Tesla, Inc.")
    * _[Owen Diaz v. Tesla](https://en.wikipedia.org/wiki/Owen_Diaz_v._Tesla "Owen Diaz v. Tesla")_
  * [List of Tesla Autopilot crashes](https://en.wikipedia.org/wiki/List_of_Tesla_Autopilot_crashes "List of Tesla Autopilot crashes")
  * [Tesla Takedown](https://en.wikipedia.org/wiki/Tesla_Takedown "Tesla Takedown")
  * [TSLAQ](https://en.wikipedia.org/wiki/TSLAQ "TSLAQ")
  * [Vandalism](https://en.wikipedia.org/wiki/2025_Tesla_vandalism "2025 Tesla vandalism")

  
Related| 

  * [History](https://en.wikipedia.org/wiki/History_of_Tesla,_Inc. "History of Tesla, Inc.")
  * [DeepScale](https://en.wikipedia.org/wiki/DeepScale "DeepScale")
  * [Easter eggs in products](https://en.wikipedia.org/wiki/List_of_Easter_eggs_in_Tesla_products "List of Easter eggs in Tesla products")
  * [Giga Press](https://en.wikipedia.org/wiki/Giga_Press "Giga Press")
  * [Hibar Systems](https://en.wikipedia.org/wiki/Hibar_Systems "Hibar Systems")
  * [Hornsdale Power Reserve](https://en.wikipedia.org/wiki/Hornsdale_Power_Reserve "Hornsdale Power Reserve")
  * [Maxwell Technologies](https://en.wikipedia.org/wiki/Maxwell_Technologies "Maxwell Technologies") (divested)
  * [SolarCity](https://en.wikipedia.org/wiki/SolarCity "SolarCity") (merged into Tesla Energy)
  * [Tesla Automation](https://en.wikipedia.org/wiki/Tesla_Automation "Tesla Automation")
  * [Tesla battery station](https://en.wikipedia.org/wiki/Tesla_battery_station "Tesla battery station")
  * [Tesla Roadster in space](https://en.wikipedia.org/wiki/Elon_Musk%27s_Tesla_Roadster "Elon Musk's Tesla Roadster")
  * [Tesla Robotaxi](https://en.wikipedia.org/wiki/Tesla_Robotaxi "Tesla Robotaxi")
  * [Tesla and unions](https://en.wikipedia.org/wiki/Tesla_and_unions "Tesla and unions")
  * [Tesla Diner](https://en.wikipedia.org/wiki/Tesla_Diner "Tesla Diner")
  * [Zep Solar](https://en.wikipedia.org/wiki/Zep_Solar "Zep Solar") (merged into Tesla Energy)
  * [Nikola Tesla](https://en.wikipedia.org/wiki/Nikola_Tesla "Nikola Tesla") (namesake)

  
  
  * [Category](https://en.wikipedia.org/wiki/Category:Tesla,_Inc. "Category:Tesla, Inc.")

  
  
  * [v](https://en.wikipedia.org/wiki/Template:Tesla_timeline "Template:Tesla timeline")
  * [t](https://en.wikipedia.org/wiki/Template_talk:Tesla_timeline "Template talk:Tesla timeline")
  * [e](https://en.wikipedia.org/wiki/Special:EditPage/Template:Tesla_timeline "Special:EditPage/Template:Tesla timeline")

Tesla, car timeline, 2008–present  
---  
| **Type** | 2000s  | 2010s  | 2020s   
---|---|---|---  
8 | 9  | 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9  | 0 | 1 | 2 | 3 | 4 | 5   
[Sports car](https://en.wikipedia.org/wiki/Sports_car "Sports car") | [Roadster (Gen 1)](https://en.wikipedia.org/wiki/Tesla_Roadster_\(first_generation\) "Tesla Roadster \(first generation\)") |  |  | [Roadster (Gen 2)](https://en.wikipedia.org/wiki/Tesla_Roadster_\(second_generation\) "Tesla Roadster \(second generation\)")  
[Mid-size car](https://en.wikipedia.org/wiki/Mid-size_car "Mid-size car") |  |  | [Model 3](https://en.wikipedia.org/wiki/Tesla_Model_3 "Tesla Model 3") | [Model 3](https://en.wikipedia.org/wiki/Tesla_Model_3 "Tesla Model 3")  
[Full-size car](https://en.wikipedia.org/wiki/Full-size_car "Full-size car") |  |  | [Model S](https://en.wikipedia.org/wiki/Tesla_Model_S "Tesla Model S") | [Model S](https://en.wikipedia.org/wiki/Tesla_Model_S "Tesla Model S") | [Model S](https://en.wikipedia.org/wiki/Tesla_Model_S "Tesla Model S")  
[Crossover SUV](https://en.wikipedia.org/wiki/Crossover_\(automobile\) "Crossover \(automobile\)") |  |  | [Model X](https://en.wikipedia.org/wiki/Tesla_Model_X "Tesla Model X") | [Model X](https://en.wikipedia.org/wiki/Tesla_Model_X "Tesla Model X")  
|  | [Model Y](https://en.wikipedia.org/wiki/Tesla_Model_Y "Tesla Model Y") | [Model Y](https://en.wikipedia.org/wiki/Tesla_Model_Y "Tesla Model Y")  
[Pickup truck](https://en.wikipedia.org/wiki/Pickup_truck "Pickup truck") |  |  |  | [Cybertruck](https://en.wikipedia.org/wiki/Tesla_Cybertruck "Tesla Cybertruck")  
[Heavy-duty truck](https://en.wikipedia.org/wiki/Heavy-duty_truck "Heavy-duty truck") |  |  |  | [Semi](https://en.wikipedia.org/wiki/Tesla_Semi "Tesla Semi")  
  
  * [v](https://en.wikipedia.org/wiki/Template:Elon_Musk "Template:Elon Musk")
  * [t](https://en.wikipedia.org/wiki/Template_talk:Elon_Musk "Template talk:Elon Musk")
  * [e](https://en.wikipedia.org/wiki/Special:EditPage/Template:Elon_Musk "Special:EditPage/Template:Elon Musk")

[Elon Musk](https://en.wikipedia.org/wiki/Elon_Musk "Elon Musk")  
---  
Main| 

  * [Awards and honors](https://en.wikipedia.org/wiki/List_of_awards_and_honors_received_by_Elon_Musk "List of awards and honors received by Elon Musk")
  * [Business career](https://en.wikipedia.org/wiki/Business_career_of_Elon_Musk "Business career of Elon Musk")
  * [Filmography](https://en.wikipedia.org/wiki/Elon_Musk_filmography "Elon Musk filmography")
  * [Legal affairs](https://en.wikipedia.org/wiki/Legal_affairs_of_Elon_Musk "Legal affairs of Elon Musk")
  * [International relations](https://en.wikipedia.org/wiki/International_relations_of_Elon_Musk "International relations of Elon Musk")
  * [Political activities](https://en.wikipedia.org/wiki/Political_activities_of_Elon_Musk "Political activities of Elon Musk")
  * [Protests](https://en.wikipedia.org/wiki/Protests_against_Elon_Musk "Protests against Elon Musk")
    * [Tesla Takedown](https://en.wikipedia.org/wiki/Tesla_Takedown "Tesla Takedown")
  * [Public image](https://en.wikipedia.org/wiki/Public_image_of_Elon_Musk "Public image of Elon Musk")
  * [Views](https://en.wikipedia.org/wiki/Views_of_Elon_Musk "Views of Elon Musk")
  * [Wealth](https://en.wikipedia.org/wiki/Wealth_of_Elon_Musk "Wealth of Elon Musk")

| [](https://en.wikipedia.org/wiki/File:Elon_Musk_2015.jpg)  
Companies| 

  * [Zip2](https://en.wikipedia.org/wiki/Zip2 "Zip2")
  * [X.com](https://en.wikipedia.org/wiki/X.com_\(bank\) "X.com \(bank\)")
    * [PayPal](https://en.wikipedia.org/wiki/PayPal "PayPal")
  * [SpaceX](https://en.wikipedia.org/wiki/SpaceX "SpaceX") ([Starlink](https://en.wikipedia.org/wiki/Starlink "Starlink"))
  * Tesla, Inc.
    * [Tesla Energy](https://en.wikipedia.org/wiki/Tesla_Energy "Tesla Energy")
    * [Criticism](https://en.wikipedia.org/wiki/Criticism_of_Tesla,_Inc. "Criticism of Tesla, Inc.")
    * [TSLAQ](https://en.wikipedia.org/wiki/TSLAQ "TSLAQ")
    * [lawsuits](https://en.wikipedia.org/wiki/List_of_lawsuits_involving_Tesla,_Inc. "List of lawsuits involving Tesla, Inc.")
    * [trade unions](https://en.wikipedia.org/wiki/Tesla_and_trade_unions "Tesla and trade unions")
    * [vandalism](https://en.wikipedia.org/wiki/2025_Tesla_vandalism "2025 Tesla vandalism")
  * [OpenAI](https://en.wikipedia.org/wiki/OpenAI "OpenAI")
  * [Neuralink](https://en.wikipedia.org/wiki/Neuralink "Neuralink")
  * [The Boring Company](https://en.wikipedia.org/wiki/The_Boring_Company "The Boring Company")
    * [Boring test tunnel](https://en.wikipedia.org/wiki/Boring_test_tunnel "Boring test tunnel")
  * [SolarCity](https://en.wikipedia.org/wiki/SolarCity "SolarCity")
  * [Thud](https://en.wikipedia.org/wiki/Thud_\(media_company\) "Thud \(media company\)")
  * [X Corp.](https://en.wikipedia.org/wiki/X_Corp. "X Corp.")
    * [Twitter, Inc.](https://en.wikipedia.org/wiki/Twitter,_Inc. "Twitter, Inc.")
    * [Twitter](https://en.wikipedia.org/wiki/Twitter "Twitter")
    * [Twitter under Elon Musk](https://en.wikipedia.org/wiki/Twitter_under_Elon_Musk "Twitter under Elon Musk")
    * [Acquisition of Twitter](https://en.wikipedia.org/wiki/Acquisition_of_Twitter_by_Elon_Musk "Acquisition of Twitter by Elon Musk")
    * [Twitter Files](https://en.wikipedia.org/wiki/Twitter_Files "Twitter Files")
    * [December 2022 suspensions](https://en.wikipedia.org/wiki/December_2022_Twitter_suspensions "December 2022 Twitter suspensions")
    * [Block in Brazil](https://en.wikipedia.org/wiki/Blocking_of_Twitter_in_Brazil "Blocking of Twitter in Brazil")
    * [lawsuits](https://en.wikipedia.org/wiki/List_of_lawsuits_involving_X_Corp. "List of lawsuits involving X Corp.")
  * [xAI](https://en.wikipedia.org/wiki/XAI_\(company\) "XAI \(company\)")
    * [Grok](https://en.wikipedia.org/wiki/Grok_\(chatbot\) "Grok \(chatbot\)")
    * [Colossus](https://en.wikipedia.org/wiki/Colossus_\(supercomputer\) "Colossus \(supercomputer\)")

  
Organizations| 

  * [Department of Government Efficiency](https://en.wikipedia.org/wiki/Department_of_Government_Efficiency "Department of Government Efficiency")

  
Politics| 

  * [America PAC](https://en.wikipedia.org/wiki/America_PAC "America PAC")
  * [DOGE](https://en.wikipedia.org/wiki/Department_of_Government_Efficiency "Department of Government Efficiency")
    * [response](https://en.wikipedia.org/wiki/Response_to_the_Department_of_Government_Efficiency "Response to the Department of Government Efficiency")
  * ["Fork in the Road" memo](https://en.wikipedia.org/wiki/2025_U.S._federal_deferred_resignation_program "2025 U.S. federal deferred resignation program")
  * [Salute controversy](https://en.wikipedia.org/wiki/Elon_Musk_salute_controversy "Elon Musk salute controversy")
  * [RBG PAC](https://en.wikipedia.org/wiki/RBG_PAC "RBG PAC")
  * [Views on trade unions](https://en.wikipedia.org/wiki/Elon_Musk_and_trade_unions "Elon Musk and trade unions")
  * [City of Starbase Incorporation election](https://en.wikipedia.org/wiki/City_of_Starbase_Incorporation_election "City of Starbase Incorporation election")
  * [Donald Trump feud](https://en.wikipedia.org/wiki/Trump%E2%80%93Musk_feud "Trump–Musk feud")
  * [America Party](https://en.wikipedia.org/wiki/America_Party "America Party")

  
Depictions| 

  * _[Elon Musk: Tesla, SpaceX, and the Quest for a Fantastic Future](https://en.wikipedia.org/wiki/Elon_Musk:_Tesla,_SpaceX,_and_the_Quest_for_a_Fantastic_Future "Elon Musk: Tesla, SpaceX, and the Quest for a Fantastic Future")_ (2015)
  * "[The Musk Who Fell to Earth](https://en.wikipedia.org/wiki/The_Musk_Who_Fell_to_Earth "The Musk Who Fell to Earth")" (2015)
  * _[The Space Barons](https://en.wikipedia.org/wiki/The_Space_Barons "The Space Barons")_ (2018)
  * _[Ludicrous: The Unvarnished Story of Tesla Motors](https://en.wikipedia.org/wiki/Ludicrous:_The_Unvarnished_Story_of_Tesla_Motors "Ludicrous: The Unvarnished Story of Tesla Motors")_ (2019)
  * "[One Crew over the Crewcoo's Morty](https://en.wikipedia.org/wiki/One_Crew_over_the_Crewcoo%27s_Morty "One Crew over the Crewcoo's Morty")" (2019)
  * _[Power Play: Tesla, Elon Musk, and the Bet of the Century](https://en.wikipedia.org/wiki/Power_Play:_Tesla,_Elon_Musk,_and_the_Bet_of_the_Century "Power Play: Tesla, Elon Musk, and the Bet of the Century")_ (2021)
  * _[Return to Space](https://en.wikipedia.org/wiki/Return_to_Space "Return to Space")_ (2022)
  * _[Elon Musk's Crash Course](https://en.wikipedia.org/wiki/Elon_Musk%27s_Crash_Course "Elon Musk's Crash Course")_ (2022)
  * [_Elon Musk_](https://en.wikipedia.org/wiki/Elon_Musk_\(Isaacson_book\) "Elon Musk \(Isaacson book\)") (2023)
  * _[Musk](https://en.wikipedia.org/wiki/Musk_\(film\) "Musk \(film\)")_ (TBA)

  
People| | [Family](https://en.wikipedia.org/wiki/Musk_family "Musk family")| 

  * [Justine Musk](https://en.wikipedia.org/wiki/Justine_Musk "Justine Musk") (first wife)
  * [Talulah Riley](https://en.wikipedia.org/wiki/Talulah_Riley "Talulah Riley") (second wife)
  * [Vivian Wilson](https://en.wikipedia.org/wiki/Vivian_Wilson "Vivian Wilson") (daughter)
  * [Maye Musk](https://en.wikipedia.org/wiki/Maye_Musk "Maye Musk") (mother)
  * [Errol Musk](https://en.wikipedia.org/wiki/Errol_Musk "Errol Musk") (father)
  * [Kimbal Musk](https://en.wikipedia.org/wiki/Kimbal_Musk "Kimbal Musk") (brother)
  * [Tosca Musk](https://en.wikipedia.org/wiki/Tosca_Musk "Tosca Musk") (sister)
  * [Joshua N. Haldeman](https://en.wikipedia.org/wiki/Joshua_N._Haldeman "Joshua N. Haldeman") (grandfather)
  * [Lyndon Rive](https://en.wikipedia.org/wiki/Lyndon_Rive "Lyndon Rive") (cousin)

  
---|---  
Partners| 

  * [Grimes](https://en.wikipedia.org/wiki/Grimes "Grimes")
  * [Shivon Zilis](https://en.wikipedia.org/wiki/Shivon_Zilis "Shivon Zilis")

  
  
Related| 

  * [Astra Nova School](https://en.wikipedia.org/wiki/Astra_Nova_School "Astra Nova School")
  * [Billionaire space race](https://en.wikipedia.org/wiki/Billionaire_space_race "Billionaire space race")
  * [Elon Musk's Tesla Roadster](https://en.wikipedia.org/wiki/Elon_Musk%27s_Tesla_Roadster "Elon Musk's Tesla Roadster")
  * [ElonJet](https://en.wikipedia.org/wiki/ElonJet "ElonJet")
  * [Hyperloop](https://en.wikipedia.org/wiki/Hyperloop "Hyperloop")
  * [Snailbrook, Texas](https://en.wikipedia.org/wiki/Snailbrook,_Texas "Snailbrook, Texas")
  * [Texas Institute of Technology and Science](https://en.wikipedia.org/wiki/Texas_Institute_of_Technology_and_Science "Texas Institute of Technology and Science")
  * [Starbase, Texas](https://en.wikipedia.org/wiki/Starbase,_Texas "Starbase, Texas")

  
  
  * [Category](https://en.wikipedia.org/wiki/Category:Elon_Musk "Category:Elon Musk")

  
  
  * [v](https://en.wikipedia.org/wiki/Template:Nikola_Tesla "Template:Nikola Tesla")
  * [t](https://en.wikipedia.org/wiki/Template_talk:Nikola_Tesla "Template talk:Nikola Tesla")
  * [e](https://en.wikipedia.org/wiki/Special:EditPage/Template:Nikola_Tesla "Special:EditPage/Template:Nikola Tesla")

[Nikola Tesla](https://en.wikipedia.org/wiki/Nikola_Tesla "Nikola Tesla")  
---  
Career and  
inventions| 

  * [Patents](https://en.wikipedia.org/wiki/List_of_Nikola_Tesla_patents "List of Nikola Tesla patents")
  * [Plasma lamp](https://en.wikipedia.org/wiki/Plasma_lamp "Plasma lamp")
    * [plasma globe](https://en.wikipedia.org/wiki/Plasma_globe "Plasma globe")
  * [Polyphase system](https://en.wikipedia.org/wiki/Polyphase_system "Polyphase system")
  * [Alternating-current commutatorless induction motor](https://en.wikipedia.org/wiki/Induction_motor "Induction motor")
  * [Tesla Experimental Station](https://en.wikipedia.org/wiki/Tesla_Experimental_Station "Tesla Experimental Station")
  * [Teleforce](https://en.wikipedia.org/wiki/Teleforce "Teleforce")
  * [Telegeodynamics](https://en.wikipedia.org/wiki/Telegeodynamics "Telegeodynamics")
  * [Tesla coil](https://en.wikipedia.org/wiki/Tesla_coil "Tesla coil")
    * [history](https://en.wikipedia.org/wiki/History_of_the_Tesla_coil "History of the Tesla coil")
    * [Wireless power](https://en.wikipedia.org/wiki/Wireless_power_transfer#Tesla "Wireless power transfer")
    * [Resonant inductive coupling](https://en.wikipedia.org/wiki/Resonant_inductive_coupling#History "Resonant inductive coupling")
  * [Radio control](https://en.wikipedia.org/wiki/Radio_control "Radio control")
  * [Tesla turbine](https://en.wikipedia.org/wiki/Tesla_turbine "Tesla turbine")
  * [Tesla's oscillator](https://en.wikipedia.org/wiki/Tesla%27s_oscillator "Tesla's oscillator")
  * [Tesla valve](https://en.wikipedia.org/wiki/Tesla_valve "Tesla valve")
  * [Three-phase electric power](https://en.wikipedia.org/wiki/Three-phase_electric_power "Three-phase electric power")
  * [Wardenclyffe Tower](https://en.wikipedia.org/wiki/Wardenclyffe_Tower "Wardenclyffe Tower")
  * [Tesla Electric Light and Manufacturing](https://en.wikipedia.org/wiki/Tesla_Electric_Light_and_Manufacturing "Tesla Electric Light and Manufacturing")

| [](https://en.wikipedia.org/wiki/File:Tesla-bulb.jpg)  
Writings| 

  * _[The Inventions, Researches, and Writings of Nikola Tesla](https://en.wikipedia.org/wiki/The_Inventions,_Researches,_and_Writings_of_Nikola_Tesla "The Inventions, Researches, and Writings of Nikola Tesla")_
  * _[Colorado Springs Notes, 1899–1900](https://en.wikipedia.org/wiki/Colorado_Springs_Notes,_1899%E2%80%931900 "Colorado Springs Notes, 1899–1900")_
  * "[Fragments of Olympian Gossip](https://en.wikipedia.org/wiki/Fragments_of_Olympian_Gossip "Fragments of Olympian Gossip")"
  * _[My Inventions: The Autobiography of Nikola Tesla](https://en.wikipedia.org/wiki/My_Inventions:_The_Autobiography_of_Nikola_Tesla "My Inventions: The Autobiography of Nikola Tesla")_

  
Other| 

  * [War of the currents](https://en.wikipedia.org/wiki/War_of_the_currents "War of the currents")
  * [Westinghouse Electric](https://en.wikipedia.org/wiki/Westinghouse_Electric_Corporation "Westinghouse Electric Corporation")
  * [World Wireless System](https://en.wikipedia.org/wiki/World_Wireless_System "World Wireless System")
  * [In popular culture](https://en.wikipedia.org/wiki/Nikola_Tesla_in_popular_culture "Nikola Tesla in popular culture")

  
Related| 

  * [Nikola Tesla Museum](https://en.wikipedia.org/wiki/Nikola_Tesla_Museum "Nikola Tesla Museum")
  * [Nikola Tesla Memorial Center](https://en.wikipedia.org/wiki/Nikola_Tesla_Memorial_Center "Nikola Tesla Memorial Center")
  * [Tesla Science Center at Wardenclyffe](https://en.wikipedia.org/wiki/Tesla_Science_Center_at_Wardenclyffe "Tesla Science Center at Wardenclyffe")
  * [IEEE Nikola Tesla Award](https://en.wikipedia.org/wiki/IEEE_Nikola_Tesla_Award "IEEE Nikola Tesla Award")
  * [Tesla Satellite Award](https://en.wikipedia.org/wiki/Nikola_Tesla_Satellite_Award "Nikola Tesla Satellite Award")
  * [Belgrade Nikola Tesla Airport](https://en.wikipedia.org/wiki/Belgrade_Nikola_Tesla_Airport "Belgrade Nikola Tesla Airport")
  * [New Yorker Hotel](https://en.wikipedia.org/wiki/Wyndham_New_Yorker_Hotel "Wyndham New Yorker Hotel")
  * _[The Secret of Nikola Tesla](https://en.wikipedia.org/wiki/The_Secret_of_Nikola_Tesla "The Secret of Nikola Tesla")_ (1980 film)
  * _[Tesla – Lightning in His Hand](https://en.wikipedia.org/wiki/Tesla_%E2%80%93_Lightning_in_His_Hand "Tesla – Lightning in His Hand")_ (2003 opera)
  * _[The Prestige](https://en.wikipedia.org/wiki/The_Prestige_\(film\) "The Prestige \(film\)")_ (2006 film)
  * _[Tower to the People](https://en.wikipedia.org/wiki/Tower_to_the_People "Tower to the People")_ (2015 documentary)
  * _[Tesla](https://en.wikipedia.org/wiki/Tesla_\(2016_film\) "Tesla \(2016 film\)")_ (2016 film)
  * _[The Tesla World Light](https://en.wikipedia.org/wiki/The_Tesla_World_Light "The Tesla World Light")_ (2017 film)
  * _[The Current War](https://en.wikipedia.org/wiki/The_Current_War "The Current War")_ (2019 film)
  * "[Nikola Tesla's Night of Terror](https://en.wikipedia.org/wiki/Nikola_Tesla%27s_Night_of_Terror "Nikola Tesla's Night of Terror")" (2020 TV episode)
  * _[Tesla](https://en.wikipedia.org/wiki/Tesla_\(2020_film\) "Tesla \(2020 film\)")_ (2020 film)
  * _[Tesla: Man Out of Time](https://en.wikipedia.org/wiki/Tesla:_Man_Out_of_Time "Tesla: Man Out of Time")_
  * _[Wizard: The Life and Times of Nikola Tesla](https://en.wikipedia.org/wiki/Wizard:_The_Life_and_Times_of_Nikola_Tesla "Wizard: The Life and Times of Nikola Tesla")_
  * _[The Man Who Invented the Twentieth Century](https://en.wikipedia.org/wiki/The_Man_Who_Invented_the_Twentieth_Century "The Man Who Invented the Twentieth Century")_
  * [Boat](https://en.wikipedia.org/wiki/Maid_of_the_Mist "Maid of the Mist")
  * [SI derived unit](https://en.wikipedia.org/wiki/Tesla_\(unit\) "Tesla \(unit\)")
  * [Lunar crater](https://en.wikipedia.org/wiki/Tesla_\(crater\) "Tesla \(crater\)")
  * [Asteroid](https://en.wikipedia.org/wiki/2244_Tesla "2244 Tesla")
  * Tesla, Inc.

  
  
  * [v](https://en.wikipedia.org/wiki/Template:Silicon_Valley "Template:Silicon Valley")
  * [t](https://en.wikipedia.org/wiki/Template_talk:Silicon_Valley "Template talk:Silicon Valley")
  * [e](https://en.wikipedia.org/wiki/Special:EditPage/Template:Silicon_Valley "Special:EditPage/Template:Silicon Valley")

[Silicon Valley](https://en.wikipedia.org/wiki/Silicon_Valley "Silicon Valley")  
---  
Cities| 

  * [Belmont](https://en.wikipedia.org/wiki/Belmont,_California "Belmont, California")
  * [Campbell](https://en.wikipedia.org/wiki/Campbell,_California "Campbell, California")
  * [Cupertino](https://en.wikipedia.org/wiki/Cupertino,_California "Cupertino, California")
  * [East Palo Alto](https://en.wikipedia.org/wiki/East_Palo_Alto,_California "East Palo Alto, California")
  * [Fremont](https://en.wikipedia.org/wiki/Fremont,_California "Fremont, California")
  * [Los Altos](https://en.wikipedia.org/wiki/Los_Altos,_California "Los Altos, California")
  * [Los Altos Hills](https://en.wikipedia.org/wiki/Los_Altos_Hills,_California "Los Altos Hills, California")
  * [Los Gatos](https://en.wikipedia.org/wiki/Los_Gatos,_California "Los Gatos, California")
  * [Menlo Park](https://en.wikipedia.org/wiki/Menlo_Park,_California "Menlo Park, California")
  * [Milpitas](https://en.wikipedia.org/wiki/Milpitas,_California "Milpitas, California")
  * [Morgan Hill](https://en.wikipedia.org/wiki/Morgan_Hill,_California "Morgan Hill, California")
  * [Mountain View](https://en.wikipedia.org/wiki/Mountain_View,_California "Mountain View, California")
  * [Newark](https://en.wikipedia.org/wiki/Newark,_California "Newark, California")
  * [Palo Alto](https://en.wikipedia.org/wiki/Palo_Alto,_California "Palo Alto, California")
  * [Redwood City](https://en.wikipedia.org/wiki/Redwood_City,_California "Redwood City, California")
  * [San Carlos](https://en.wikipedia.org/wiki/San_Carlos,_California "San Carlos, California")
  * [San Jose](https://en.wikipedia.org/wiki/San_Jose,_California "San Jose, California")
  * [San Mateo](https://en.wikipedia.org/wiki/San_Mateo,_California "San Mateo, California")
  * [Santa Clara](https://en.wikipedia.org/wiki/Santa_Clara,_California "Santa Clara, California")
  * [Saratoga](https://en.wikipedia.org/wiki/Saratoga,_California "Saratoga, California")
  * [Sunnyvale](https://en.wikipedia.org/wiki/Sunnyvale,_California "Sunnyvale, California")
  * [Woodside](https://en.wikipedia.org/wiki/Woodside,_California "Woodside, California")

  
Higher education| 

  * [Cañada](https://en.wikipedia.org/wiki/Ca%C3%B1ada_College "Cañada College")
  * [Carnegie Mellon](https://en.wikipedia.org/wiki/Carnegie_Mellon_Silicon_Valley "Carnegie Mellon Silicon Valley")
  * [De Anza](https://en.wikipedia.org/wiki/De_Anza_College "De Anza College")
  * [Evergreen Valley](https://en.wikipedia.org/wiki/Evergreen_Valley_College "Evergreen Valley College")
  * [Foothill](https://en.wikipedia.org/wiki/Foothill_College "Foothill College")
  * [International Technological](https://en.wikipedia.org/wiki/International_Technological_University "International Technological University")
  * [Menlo](https://en.wikipedia.org/wiki/Menlo_College "Menlo College")
  * [Mission](https://en.wikipedia.org/wiki/Mission_College_\(California\) "Mission College \(California\)")
  * [Hispanic](https://en.wikipedia.org/wiki/National_Hispanic_University "National Hispanic University")
  * [Northwestern Poly](https://en.wikipedia.org/wiki/Northwestern_Polytechnic_University "Northwestern Polytechnic University")
  * [Ohlone](https://en.wikipedia.org/wiki/Ohlone_College "Ohlone College")
  * [San José City](https://en.wikipedia.org/wiki/San_Jos%C3%A9_City_College "San José City College")
  * [San José State](https://en.wikipedia.org/wiki/San_Jos%C3%A9_State_University "San José State University")
  * [Santa Clara](https://en.wikipedia.org/wiki/Santa_Clara_University "Santa Clara University")
  * [Silicon Valley University](https://en.wikipedia.org/wiki/Silicon_Valley_University "Silicon Valley University")
  * [Stanford](https://en.wikipedia.org/wiki/Stanford_University "Stanford University")
  * [University of Silicon Valley](https://en.wikipedia.org/wiki/University_of_Silicon_Valley "University of Silicon Valley")

  
[Companies](https://en.wikipedia.org/wiki/Category:Companies_based_in_Silicon_Valley "Category:Companies based in Silicon Valley")  
(including subsidiaries  
and defunct companies)| 

  * [3Com](https://en.wikipedia.org/wiki/3Com "3Com")
  * [Access Systems Americas](https://en.wikipedia.org/wiki/Access_Systems_Americas "Access Systems Americas")
  * [Actuate](https://en.wikipedia.org/wiki/Actuate_Corporation "Actuate Corporation")
  * [Adaptec](https://en.wikipedia.org/wiki/Adaptec "Adaptec")
  * [Adobe](https://en.wikipedia.org/wiki/Adobe_Inc "Adobe Inc")
  * [AMD](https://en.wikipedia.org/wiki/Advanced_Micro_Devices "Advanced Micro Devices")
  * [Agilent Technologies](https://en.wikipedia.org/wiki/Agilent_Technologies "Agilent Technologies")
  * [Altera](https://en.wikipedia.org/wiki/Altera "Altera")
  * [Amdahl](https://en.wikipedia.org/wiki/Amdahl_Corporation "Amdahl Corporation")
  * [Ampex](https://en.wikipedia.org/wiki/Ampex "Ampex")
  * [Apple](https://en.wikipedia.org/wiki/Apple_Inc "Apple Inc")
  * [Applied Materials](https://en.wikipedia.org/wiki/Applied_Materials "Applied Materials")
  * [Aricent](https://en.wikipedia.org/wiki/Aricent "Aricent")
  * [Asus](https://en.wikipedia.org/wiki/Asus "Asus")
  * [Atari](https://en.wikipedia.org/wiki/Atari "Atari")
  * [Atmel](https://en.wikipedia.org/wiki/Atmel "Atmel")
  * [Autodesk](https://en.wikipedia.org/wiki/Autodesk "Autodesk")
  * [Avaya](https://en.wikipedia.org/wiki/Avaya "Avaya")
  * [BEA Systems](https://en.wikipedia.org/wiki/BEA_Systems "BEA Systems")
  * [Box](https://en.wikipedia.org/wiki/Box_\(company\) "Box \(company\)")
  * [Brocade](https://en.wikipedia.org/wiki/Brocade_Communications_Systems "Brocade Communications Systems")
  * [BusinessObjects](https://en.wikipedia.org/wiki/BusinessObjects "BusinessObjects")
  * [Capcom](https://en.wikipedia.org/wiki/Capcom "Capcom")
  * [Cisco](https://en.wikipedia.org/wiki/Cisco "Cisco")
  * [Computer Literacy Bookshops](https://en.wikipedia.org/wiki/Computer_Literacy_Bookshops "Computer Literacy Bookshops")
  * [Cypress Semiconductor](https://en.wikipedia.org/wiki/Cypress_Semiconductor "Cypress Semiconductor")
  * [eBay](https://en.wikipedia.org/wiki/EBay "EBay")
  * [Electronic Arts](https://en.wikipedia.org/wiki/Electronic_Arts "Electronic Arts")
  * [Facebook](https://en.wikipedia.org/wiki/Facebook "Facebook")
  * [Foundry Networks](https://en.wikipedia.org/wiki/Foundry_Networks "Foundry Networks")
  * [Fry's Electronics](https://en.wikipedia.org/wiki/Fry%27s_Electronics "Fry's Electronics")
  * [Fujitsu](https://en.wikipedia.org/wiki/Fujitsu "Fujitsu")
  * [Gaia Online](https://en.wikipedia.org/wiki/Gaia_Online "Gaia Online")
  * [Geeknet](https://en.wikipedia.org/wiki/Geeknet "Geeknet")
  * [Google](https://en.wikipedia.org/wiki/Google "Google")
  * [Hewlett-Packard](https://en.wikipedia.org/wiki/Hewlett-Packard "Hewlett-Packard")
  * [HGST](https://en.wikipedia.org/wiki/HGST "HGST")
  * [IETF](https://en.wikipedia.org/wiki/Internet_Engineering_Task_Force "Internet Engineering Task Force")
  * [Intel](https://en.wikipedia.org/wiki/Intel "Intel")
  * [Internet Systems Consortium](https://en.wikipedia.org/wiki/Internet_Systems_Consortium "Internet Systems Consortium")
  * [Intuit](https://en.wikipedia.org/wiki/Intuit "Intuit")
  * [Juniper Networks](https://en.wikipedia.org/wiki/Juniper_Networks "Juniper Networks")
  * [Knight Ridder](https://en.wikipedia.org/wiki/Knight_Ridder "Knight Ridder")
  * [LinkedIn](https://en.wikipedia.org/wiki/LinkedIn "LinkedIn")
  * [Logitech](https://en.wikipedia.org/wiki/Logitech "Logitech")
  * [LSI Corporation](https://en.wikipedia.org/wiki/LSI_Corporation "LSI Corporation")
  * [Magellan Navigation](https://en.wikipedia.org/wiki/Magellan_Navigation "Magellan Navigation")
  * [Marvell Technology Group](https://en.wikipedia.org/wiki/Marvell_Technology_Group "Marvell Technology Group")
  * [Maxtor](https://en.wikipedia.org/wiki/Maxtor "Maxtor")
  * [McAfee](https://en.wikipedia.org/wiki/McAfee "McAfee")
  * [Memorex](https://en.wikipedia.org/wiki/Memorex "Memorex")
  * [Microsoft](https://en.wikipedia.org/wiki/Microsoft "Microsoft")
  * [Mozilla Corporation](https://en.wikipedia.org/wiki/Mozilla_Corporation "Mozilla Corporation")
  * [National Semiconductor](https://en.wikipedia.org/wiki/National_Semiconductor "National Semiconductor")
  * [Netscape](https://en.wikipedia.org/wiki/Netscape "Netscape")
  * [NetApp](https://en.wikipedia.org/wiki/NetApp "NetApp")
  * [Netflix](https://en.wikipedia.org/wiki/Netflix,_Inc. "Netflix, Inc.")
  * [NeXT](https://en.wikipedia.org/wiki/NeXT "NeXT")
  * [Nintendo of America](https://en.wikipedia.org/wiki/Nintendo#Offices_and_locations "Nintendo")
  * [Nortel](https://en.wikipedia.org/wiki/Nortel "Nortel")
  * [Nvidia](https://en.wikipedia.org/wiki/Nvidia "Nvidia")
  * [Opera Software](https://en.wikipedia.org/wiki/Opera_Software "Opera Software")
  * [Oppo Digital](https://en.wikipedia.org/wiki/Oppo_Digital "Oppo Digital")
  * [Oracle Corporation](https://en.wikipedia.org/wiki/Oracle_Corporation "Oracle Corporation")
  * [Palm](https://en.wikipedia.org/wiki/Palm_Inc "Palm Inc")
  * [Palo Alto Networks](https://en.wikipedia.org/wiki/Palo_Alto_Networks "Palo Alto Networks")
  * [PayPal](https://en.wikipedia.org/wiki/PayPal "PayPal")
  * [Pinterest](https://en.wikipedia.org/wiki/Pinterest "Pinterest")
  * [Playdom](https://en.wikipedia.org/wiki/Playdom "Playdom")
  * [Rambus](https://en.wikipedia.org/wiki/Rambus "Rambus")
  * [Redback Networks](https://en.wikipedia.org/wiki/Redback_Networks "Redback Networks")
  * [Reputation.com](https://en.wikipedia.org/wiki/Reputation.com "Reputation.com")
  * [Roku](https://en.wikipedia.org/wiki/Roku,_Inc "Roku, Inc")
  * [SAP](https://en.wikipedia.org/wiki/SAP "SAP")
  * [SanDisk](https://en.wikipedia.org/wiki/SanDisk "SanDisk")
  * [Silicon Graphics](https://en.wikipedia.org/wiki/Silicon_Graphics "Silicon Graphics")
  * [Silicon Image](https://en.wikipedia.org/wiki/Silicon_Image "Silicon Image")
  * [Solectron](https://en.wikipedia.org/wiki/Solectron "Solectron")
  * [Sony Interactive Entertainment](https://en.wikipedia.org/wiki/Sony_Interactive_Entertainment "Sony Interactive Entertainment")
  * [SRI International](https://en.wikipedia.org/wiki/SRI_International "SRI International")
  * [Sun Microsystems](https://en.wikipedia.org/wiki/Sun_Microsystems "Sun Microsystems")
  * [Symyx](https://en.wikipedia.org/wiki/Symyx_Technologies "Symyx Technologies")
  * [Synopsys](https://en.wikipedia.org/wiki/Synopsys "Synopsys")
  * [Taligent](https://en.wikipedia.org/wiki/Taligent "Taligent")
  * [Tesla](https://en.wikipedia.org/wiki/Tesla_Inc "Tesla Inc")
  * [TiVo Corporation](https://en.wikipedia.org/wiki/TiVo_Corporation "TiVo Corporation")
  * [Uber](https://en.wikipedia.org/wiki/Uber "Uber")
  * [Verisign](https://en.wikipedia.org/wiki/Verisign "Verisign")
  * [Veritas Technologies](https://en.wikipedia.org/wiki/Veritas_Technologies "Veritas Technologies")
  * [VMware](https://en.wikipedia.org/wiki/VMware "VMware")
  * [Webex](https://en.wikipedia.org/wiki/Cisco_Webex "Cisco Webex")
  * [WhatsApp](https://en.wikipedia.org/wiki/WhatsApp "WhatsApp")
  * [Xilinx](https://en.wikipedia.org/wiki/Xilinx "Xilinx")
  * [Yahoo!](https://en.wikipedia.org/wiki/Yahoo! "Yahoo!")

  
  
  * [v](https://en.wikipedia.org/wiki/Template:PayPal_Mafia "Template:PayPal Mafia")
  * [t](https://en.wikipedia.org/wiki/Template_talk:PayPal_Mafia "Template talk:PayPal Mafia")
  * [e](https://en.wikipedia.org/wiki/Special:EditPage/Template:PayPal_Mafia "Special:EditPage/Template:PayPal Mafia")

[PayPal Mafia](https://en.wikipedia.org/wiki/PayPal_Mafia "PayPal Mafia")  
---  
Individuals| 

  * [Peter Thiel](https://en.wikipedia.org/wiki/Peter_Thiel "Peter Thiel")
  * [Reid Hoffman](https://en.wikipedia.org/wiki/Reid_Hoffman "Reid Hoffman")
  * [Max Levchin](https://en.wikipedia.org/wiki/Max_Levchin "Max Levchin")
  * [Ken Howery](https://en.wikipedia.org/wiki/Ken_Howery "Ken Howery")
  * [Luke Nosek](https://en.wikipedia.org/wiki/Luke_Nosek "Luke Nosek")
  * [Elon Musk](https://en.wikipedia.org/wiki/Elon_Musk "Elon Musk")
  * [Steve Chen](https://en.wikipedia.org/wiki/Steve_Chen "Steve Chen")
  * [Keith Rabois](https://en.wikipedia.org/wiki/Keith_Rabois "Keith Rabois")
  * [Chad Hurley](https://en.wikipedia.org/wiki/Chad_Hurley "Chad Hurley")
  * [Roelof Botha](https://en.wikipedia.org/wiki/Roelof_Botha "Roelof Botha")
  * [Jawed Karim](https://en.wikipedia.org/wiki/Jawed_Karim "Jawed Karim")
  * [Yishan Wong](https://en.wikipedia.org/wiki/Yishan_Wong "Yishan Wong")
  * [Eric M. Jackson](https://en.wikipedia.org/wiki/Eric_M._Jackson "Eric M. Jackson")
  * [David O. Sacks](https://en.wikipedia.org/wiki/David_O._Sacks "David O. Sacks")
  * [Premal Shah](https://en.wikipedia.org/wiki/Premal_Shah "Premal Shah")
  * [Russel Simmons](https://en.wikipedia.org/wiki/Russel_Simmons "Russel Simmons")
  * [Jeremy Stoppelman](https://en.wikipedia.org/wiki/Jeremy_Stoppelman "Jeremy Stoppelman")

  
Companies founded  
or co-founded| 

  * [PayPal](https://en.wikipedia.org/wiki/PayPal "PayPal")
  * [LinkedIn](https://en.wikipedia.org/wiki/LinkedIn "LinkedIn")
  * [YouTube](https://en.wikipedia.org/wiki/YouTube "YouTube")
  * [Yelp](https://en.wikipedia.org/wiki/Yelp "Yelp")
  * [Geni.com](https://en.wikipedia.org/wiki/Geni.com "Geni.com")
  * [Yammer](https://en.wikipedia.org/wiki/Yammer "Yammer")
  * [SpaceX](https://en.wikipedia.org/wiki/SpaceX "SpaceX")
  * Tesla, Inc.
  * [Palantir Technologies](https://en.wikipedia.org/wiki/Palantir_Technologies "Palantir Technologies")
  * [Kiva.org](https://en.wikipedia.org/wiki/Kiva_\(organization\) "Kiva \(organization\)")
  * [Affirm](https://en.wikipedia.org/wiki/Affirm_\(company\) "Affirm \(company\)")

  
Investments| 

  * [Friendster](https://en.wikipedia.org/wiki/Friendster "Friendster")
  * [Facebook](https://en.wikipedia.org/wiki/Facebook "Facebook")
  * [Powerset](https://en.wikipedia.org/wiki/Powerset_\(company\) "Powerset \(company\)")
  * [Six Apart](https://en.wikipedia.org/wiki/Six_Apart "Six Apart")
  * [Zynga](https://en.wikipedia.org/wiki/Zynga "Zynga")
  * [IronPort](https://en.wikipedia.org/wiki/IronPort "IronPort")
  * [Flickr](https://en.wikipedia.org/wiki/Flickr "Flickr")
  * [Digg](https://en.wikipedia.org/wiki/Digg "Digg")
  * [Grockit](https://en.wikipedia.org/wiki/Grockit "Grockit")
  * [Ooma](https://en.wikipedia.org/wiki/Ooma "Ooma")
  * [Quantcast](https://en.wikipedia.org/wiki/Quantcast "Quantcast")
  * [RapLeaf](https://en.wikipedia.org/wiki/RapLeaf "RapLeaf")
  * [SmartDrive Systems](https://en.wikipedia.org/wiki/SmartDrive_Systems "SmartDrive Systems")
  * [Wise](https://en.wikipedia.org/wiki/Wise_\(company\) "Wise \(company\)")
  * [Ping.fm](https://en.wikipedia.org/wiki/Ping.fm "Ping.fm")
  * [Nanosolar](https://en.wikipedia.org/wiki/Nanosolar "Nanosolar")
  * [Knewton](https://en.wikipedia.org/wiki/Knewton "Knewton")
  * [Kongregate](https://en.wikipedia.org/wiki/Kongregate "Kongregate")
  * [Last.fm](https://en.wikipedia.org/wiki/Last.fm "Last.fm")
  * [TokBox](https://en.wikipedia.org/wiki/TokBox "TokBox")
  * [Xoom](https://en.wikipedia.org/wiki/Xoom_\(web_hosting\) "Xoom \(web hosting\)")
  * [Joost](https://en.wikipedia.org/wiki/Joost "Joost")

  
Funds| 

  * [Founders Fund](https://en.wikipedia.org/wiki/Founders_Fund "Founders Fund")
  * [Clarium Capital](https://en.wikipedia.org/wiki/Clarium_Capital "Clarium Capital")
  * [Greylock Partners](https://en.wikipedia.org/wiki/Greylock_Partners "Greylock Partners")
  * [Sequoia Capital](https://en.wikipedia.org/wiki/Sequoia_Capital "Sequoia Capital")
  * [Valar Ventures](https://en.wikipedia.org/wiki/Valar_Ventures "Valar Ventures")

  
Other| 

  * _[The PayPal Wars](https://en.wikipedia.org/wiki/The_PayPal_Wars "The PayPal Wars")_
  * _[Thank You for Smoking](https://en.wikipedia.org/wiki/Thank_You_for_Smoking_\(film\) "Thank You for Smoking \(film\)")_
  * _[The Stanford Review](https://en.wikipedia.org/wiki/The_Stanford_Review "The Stanford Review")_

  
  
  * [v](https://en.wikipedia.org/wiki/Template:Nasdaq-100 "Template:Nasdaq-100")
  * [t](https://en.wikipedia.org/wiki/Template_talk:Nasdaq-100 "Template talk:Nasdaq-100")
  * [e](https://en.wikipedia.org/wiki/Special:EditPage/Template:Nasdaq-100 "Special:EditPage/Template:Nasdaq-100")

Companies of the [Nasdaq-100](https://en.wikipedia.org/wiki/Nasdaq-100 "Nasdaq-100") index  
---  
  
  * [Adobe](https://en.wikipedia.org/wiki/Adobe_Inc. "Adobe Inc.")
  * [ADP](https://en.wikipedia.org/wiki/ADP_\(company\) "ADP \(company\)")
  * [AMD](https://en.wikipedia.org/wiki/AMD "AMD")
  * [Airbnb](https://en.wikipedia.org/wiki/Airbnb "Airbnb")
  * [Alphabet](https://en.wikipedia.org/wiki/Alphabet_Inc. "Alphabet Inc.")
  * [Amazon](https://en.wikipedia.org/wiki/Amazon_\(company\) "Amazon \(company\)")
  * [AEP](https://en.wikipedia.org/wiki/American_Electric_Power "American Electric Power")
  * [Amgen](https://en.wikipedia.org/wiki/Amgen "Amgen")
  * [Analog Devices](https://en.wikipedia.org/wiki/Analog_Devices "Analog Devices")
  * [Apple](https://en.wikipedia.org/wiki/Apple_Inc. "Apple Inc.")
  * [Applied Materials](https://en.wikipedia.org/wiki/Applied_Materials "Applied Materials")
  * [AppLovin](https://en.wikipedia.org/wiki/AppLovin "AppLovin")
  * [Arm](https://en.wikipedia.org/wiki/Arm_Holdings "Arm Holdings")
  * [ASML](https://en.wikipedia.org/wiki/ASML_Holding "ASML Holding")
  * [AstraZeneca](https://en.wikipedia.org/wiki/AstraZeneca "AstraZeneca")
  * [Atlassian](https://en.wikipedia.org/wiki/Atlassian "Atlassian")
  * [Autodesk](https://en.wikipedia.org/wiki/Autodesk "Autodesk")
  * [Axon](https://en.wikipedia.org/wiki/Axon_Enterprise "Axon Enterprise")
  * [Baker Hughes](https://en.wikipedia.org/wiki/Baker_Hughes "Baker Hughes")
  * [Biogen](https://en.wikipedia.org/wiki/Biogen "Biogen")
  * [Booking Holdings](https://en.wikipedia.org/wiki/Booking_Holdings "Booking Holdings")
  * [Broadcom](https://en.wikipedia.org/wiki/Broadcom "Broadcom")
  * [Cadence](https://en.wikipedia.org/wiki/Cadence_Design_Systems "Cadence Design Systems")
  * [CDW](https://en.wikipedia.org/wiki/CDW "CDW")
  * [Charter Communications](https://en.wikipedia.org/wiki/Charter_Communications "Charter Communications")
  * [Cintas](https://en.wikipedia.org/wiki/Cintas "Cintas")
  * [Cisco](https://en.wikipedia.org/wiki/Cisco "Cisco")
  * [Coca-Cola Europacific Partners](https://en.wikipedia.org/wiki/Coca-Cola_Europacific_Partners "Coca-Cola Europacific Partners")
  * [Cognizant](https://en.wikipedia.org/wiki/Cognizant "Cognizant")
  * [Comcast](https://en.wikipedia.org/wiki/Comcast "Comcast")
  * [Constellation Energy](https://en.wikipedia.org/wiki/Constellation_Energy "Constellation Energy")
  * [Copart](https://en.wikipedia.org/wiki/Copart "Copart")
  * [CoStar](https://en.wikipedia.org/wiki/CoStar_Group "CoStar Group")
  * [Costco](https://en.wikipedia.org/wiki/Costco "Costco")
  * [CrowdStrike](https://en.wikipedia.org/wiki/CrowdStrike "CrowdStrike")
  * [CSX](https://en.wikipedia.org/wiki/CSX_Corporation "CSX Corporation")
  * [Datadog](https://en.wikipedia.org/wiki/Datadog "Datadog")
  * [Dexcom](https://en.wikipedia.org/wiki/Dexcom "Dexcom")
  * [Diamondback Energy](https://en.wikipedia.org/wiki/Diamondback_Energy "Diamondback Energy")
  * [DoorDash](https://en.wikipedia.org/wiki/DoorDash "DoorDash")
  * [Electronic Arts](https://en.wikipedia.org/wiki/Electronic_Arts "Electronic Arts")
  * [Exelon](https://en.wikipedia.org/wiki/Exelon "Exelon")
  * [Fastenal](https://en.wikipedia.org/wiki/Fastenal "Fastenal")
  * [Fortinet](https://en.wikipedia.org/wiki/Fortinet "Fortinet")
  * [GE HealthCare](https://en.wikipedia.org/wiki/GE_HealthCare "GE HealthCare")
  * [Gilead](https://en.wikipedia.org/wiki/Gilead_Sciences "Gilead Sciences")
  * [GlobalFoundries](https://en.wikipedia.org/wiki/GlobalFoundries "GlobalFoundries")
  * [Honeywell](https://en.wikipedia.org/wiki/Honeywell "Honeywell")
  * [Idexx Laboratories](https://en.wikipedia.org/wiki/Idexx_Laboratories "Idexx Laboratories")
  * [Intel](https://en.wikipedia.org/wiki/Intel "Intel")
  * [Intuit](https://en.wikipedia.org/wiki/Intuit "Intuit")
  * [Intuitive Surgical](https://en.wikipedia.org/wiki/Intuitive_Surgical "Intuitive Surgical")
  * [Keurig Dr Pepper](https://en.wikipedia.org/wiki/Keurig_Dr_Pepper "Keurig Dr Pepper")
  * [KLA](https://en.wikipedia.org/wiki/KLA_Corporation "KLA Corporation")
  * [Kraft Heinz](https://en.wikipedia.org/wiki/Kraft_Heinz "Kraft Heinz")
  * [Lam Research](https://en.wikipedia.org/wiki/Lam_Research "Lam Research")
  * [Linde](https://en.wikipedia.org/wiki/Linde_plc "Linde plc")
  * [Lululemon](https://en.wikipedia.org/wiki/Lululemon "Lululemon")
  * [Marriott International](https://en.wikipedia.org/wiki/Marriott_International "Marriott International")
  * [Marvell](https://en.wikipedia.org/wiki/Marvell_Technology "Marvell Technology")
  * [Mercado Libre](https://en.wikipedia.org/wiki/Mercado_Libre "Mercado Libre")
  * [Meta](https://en.wikipedia.org/wiki/Meta_Platforms "Meta Platforms")
  * [Microchip](https://en.wikipedia.org/wiki/Microchip_Technology "Microchip Technology")
  * [Micron](https://en.wikipedia.org/wiki/Micron_Technology "Micron Technology")
  * [Microsoft](https://en.wikipedia.org/wiki/Microsoft "Microsoft")
  * [MicroStrategy](https://en.wikipedia.org/wiki/MicroStrategy "MicroStrategy")
  * [Mondelez International](https://en.wikipedia.org/wiki/Mondelez_International "Mondelez International")
  * [Monster Beverage](https://en.wikipedia.org/wiki/Monster_Beverage "Monster Beverage")
  * [Netflix](https://en.wikipedia.org/wiki/Netflix,_Inc. "Netflix, Inc.")
  * [Nvidia](https://en.wikipedia.org/wiki/Nvidia "Nvidia")
  * [NXP](https://en.wikipedia.org/wiki/NXP_Semiconductors "NXP Semiconductors")
  * [O'Reilly Auto Parts](https://en.wikipedia.org/wiki/O%27Reilly_Auto_Parts "O'Reilly Auto Parts")
  * [Old Dominion](https://en.wikipedia.org/wiki/Old_Dominion_Freight_Line "Old Dominion Freight Line")
  * [onsemi](https://en.wikipedia.org/wiki/Onsemi "Onsemi")
  * [Paccar](https://en.wikipedia.org/wiki/Paccar "Paccar")
  * [Palantir](https://en.wikipedia.org/wiki/Palantir_Technologies "Palantir Technologies")
  * [Palo Alto Networks](https://en.wikipedia.org/wiki/Palo_Alto_Networks "Palo Alto Networks")
  * [Paychex](https://en.wikipedia.org/wiki/Paychex "Paychex")
  * [PayPal](https://en.wikipedia.org/wiki/PayPal "PayPal")
  * [PDD Holdings](https://en.wikipedia.org/wiki/Pinduoduo "Pinduoduo")
  * [PepsiCo](https://en.wikipedia.org/wiki/PepsiCo "PepsiCo")
  * [Qualcomm](https://en.wikipedia.org/wiki/Qualcomm "Qualcomm")
  * [Regeneron](https://en.wikipedia.org/wiki/Regeneron_Pharmaceuticals "Regeneron Pharmaceuticals")
  * [Roper Technologies](https://en.wikipedia.org/wiki/Roper_Technologies "Roper Technologies")
  * [Ross Stores](https://en.wikipedia.org/wiki/Ross_Stores "Ross Stores")
  * [Shopify](https://en.wikipedia.org/wiki/Shopify "Shopify")
  * [Starbucks](https://en.wikipedia.org/wiki/Starbucks "Starbucks")
  * [Synopsys](https://en.wikipedia.org/wiki/Synopsys "Synopsys")
  * [Take-Two Interactive](https://en.wikipedia.org/wiki/Take-Two_Interactive "Take-Two Interactive")
  * [T-Mobile US](https://en.wikipedia.org/wiki/T-Mobile_US "T-Mobile US")
  * Tesla
  * [Texas Instruments](https://en.wikipedia.org/wiki/Texas_Instruments "Texas Instruments")
  * [Thomson Reuters](https://en.wikipedia.org/wiki/Thomson_Reuters "Thomson Reuters")
  * [Trade Desk](https://en.wikipedia.org/wiki/The_Trade_Desk "The Trade Desk")
  * [Verisk](https://en.wikipedia.org/wiki/Verisk_Analytics "Verisk Analytics")
  * [Vertex](https://en.wikipedia.org/wiki/Vertex_Pharmaceuticals "Vertex Pharmaceuticals")
  * [Warner Bros. Discovery](https://en.wikipedia.org/wiki/Warner_Bros._Discovery "Warner Bros. Discovery")
  * [Workday](https://en.wikipedia.org/wiki/Workday,_Inc. "Workday, Inc.")
  * [Xcel Energy](https://en.wikipedia.org/wiki/Xcel_Energy "Xcel Energy")
  * [Zscaler](https://en.wikipedia.org/wiki/Zscaler "Zscaler")

  
  
  * [v](https://en.wikipedia.org/wiki/Template:Automotive_industry_in_the_United_States "Template:Automotive industry in the United States")
  * [t](https://en.wikipedia.org/wiki/Template_talk:Automotive_industry_in_the_United_States "Template talk:Automotive industry in the United States")
  * [e](https://en.wikipedia.org/wiki/Special:EditPage/Template:Automotive_industry_in_the_United_States "Special:EditPage/Template:Automotive industry in the United States")

[Automotive industry in the United States](https://en.wikipedia.org/wiki/Automotive_industry_in_the_United_States "Automotive industry in the United States")  
---  
  
  * [Automotive industry](https://en.wikipedia.org/wiki/Automotive_industry "Automotive industry")
  * [Economy of the United States](https://en.wikipedia.org/wiki/Economy_of_the_United_States "Economy of the United States")
  * [Transportation in the United States](https://en.wikipedia.org/wiki/Transportation_in_the_United_States "Transportation in the United States")

  
Vehicle  
manufacturers  
and brands| | Current   
([list](https://en.wikipedia.org/wiki/List_of_automobile_manufacturers_of_the_United_States "List of automobile manufacturers of the United States"))| 

  * [AGCO](https://en.wikipedia.org/wiki/AGCO "AGCO")
    * [Challenger Tractor](https://en.wikipedia.org/wiki/Challenger_Tractor "Challenger Tractor")
    * [Massey Ferguson](https://en.wikipedia.org/wiki/Massey_Ferguson "Massey Ferguson")
  * [AM General](https://en.wikipedia.org/wiki/AM_General "AM General")
  * [Anteros Coachworks](https://en.wikipedia.org/wiki/Anteros_Coachworks "Anteros Coachworks")
  * [Amp Electric Vehicles](https://en.wikipedia.org/wiki/Workhorse_Group "Workhorse Group")
  * [Arcimoto](https://en.wikipedia.org/wiki/Arcimoto "Arcimoto")
  * [Armour Group](https://en.wikipedia.org/wiki/Rhino_Runner "Rhino Runner")
  * [ATK motorcycles](https://en.wikipedia.org/wiki/ATK_motorcycles "ATK motorcycles")
  * [Autocar](https://en.wikipedia.org/wiki/Autocar_Company "Autocar Company")
  * [Blue Bird](https://en.wikipedia.org/wiki/Blue_Bird_Corporation "Blue Bird Corporation")
  * [Callaway Cars](https://en.wikipedia.org/wiki/Callaway_Cars "Callaway Cars")
  * [Caterpillar Inc.](https://en.wikipedia.org/wiki/Caterpillar_Inc. "Caterpillar Inc.")
  * [Czinger](https://en.wikipedia.org/wiki/Czinger "Czinger")
  * [Chenowth Racing Products](https://en.wikipedia.org/wiki/Desert_Patrol_Vehicle "Desert Patrol Vehicle")
  * [Environmental Performance Vehicles](https://en.wikipedia.org/wiki/Environmental_Performance_Vehicles "Environmental Performance Vehicles")
  * [Equus Automotive](https://en.wikipedia.org/wiki/Equus_Bass_770 "Equus Bass 770")
  * [Forest River](https://en.wikipedia.org/wiki/Forest_River_\(company\) "Forest River \(company\)")
    * [Champion Bus](https://en.wikipedia.org/wiki/Champion_Bus_Incorporated "Champion Bus Incorporated")
    * [Collins](https://en.wikipedia.org/wiki/Collins_Industries "Collins Industries")
    * [ElDorado National](https://en.wikipedia.org/wiki/ElDorado_\(bus_manufacturer\) "ElDorado \(bus manufacturer\)")
    * [Glaval Bus](https://en.wikipedia.org/wiki/Glaval_Bus "Glaval Bus")
    * [Starcraft Bus](https://en.wikipedia.org/wiki/Starcraft_Bus "Starcraft Bus")
  * **[Ford](https://en.wikipedia.org/wiki/Ford_Motor_Company "Ford Motor Company")**
    * [Lincoln](https://en.wikipedia.org/wiki/Lincoln_Motor_Company "Lincoln Motor Company")
    * [SVT](https://en.wikipedia.org/wiki/Special_Vehicle_Team "Special Vehicle Team")
  * [General Dynamics Land Systems](https://en.wikipedia.org/wiki/General_Dynamics_Land_Systems "General Dynamics Land Systems")
  * **[General Motors](https://en.wikipedia.org/wiki/General_Motors "General Motors")**
    * [Buick](https://en.wikipedia.org/wiki/Buick "Buick")
    * [Cadillac](https://en.wikipedia.org/wiki/Cadillac "Cadillac")
    * [Cadillac V series](https://en.wikipedia.org/wiki/Cadillac_V_series "Cadillac V series")
    * [Chevrolet](https://en.wikipedia.org/wiki/Chevrolet "Chevrolet")
    * [Chevrolet Performance](https://en.wikipedia.org/wiki/Chevrolet_Performance "Chevrolet Performance")
    * [GMC](https://en.wikipedia.org/wiki/GMC_\(marque\) "GMC \(marque\)")
  * [Gillig](https://en.wikipedia.org/wiki/Gillig "Gillig")
  * [Growler Manufacturing and Engineering](https://en.wikipedia.org/wiki/Growler_Manufacturing_and_Engineering "Growler Manufacturing and Engineering")
  * [Harley-Davidson](https://en.wikipedia.org/wiki/Harley-Davidson "Harley-Davidson")
  * [Ingersoll Rand](https://en.wikipedia.org/wiki/Ingersoll_Rand "Ingersoll Rand")
    * [Club Car](https://en.wikipedia.org/wiki/Club_Car "Club Car")
  * [HDT Global](https://en.wikipedia.org/wiki/Storm_Search_and_Rescue_Tactical_Vehicle "Storm Search and Rescue Tactical Vehicle")
  * [HME](https://en.wikipedia.org/wiki/HME,_Incorporated "HME, Incorporated")
  * [International Motors](https://en.wikipedia.org/wiki/International_Motors "International Motors")
    * [IC Bus](https://en.wikipedia.org/wiki/IC_Bus "IC Bus")
  * [John Deere](https://en.wikipedia.org/wiki/John_Deere "John Deere")
  * [Karma Automotive](https://en.wikipedia.org/wiki/Karma_Automotive "Karma Automotive")
  * [Laffite](https://en.wikipedia.org/wiki/Laffite_X-Road "Laffite X-Road")
  * [Lenco Industries](https://en.wikipedia.org/wiki/Lenco_BearCat "Lenco BearCat")
  * [Lockheed Martin](https://en.wikipedia.org/wiki/Lockheed_Martin "Lockheed Martin")
  * [Lucid Motors](https://en.wikipedia.org/wiki/Lucid_Motors "Lucid Motors")
  * [Mack Trucks](https://en.wikipedia.org/wiki/Mack_Trucks "Mack Trucks")
  * [Millennium Luxury Coaches](https://en.wikipedia.org/wiki/Millennium_Luxury_Coaches "Millennium Luxury Coaches")
  * [Morgan Olson](https://en.wikipedia.org/wiki/Morgan_Olson "Morgan Olson")
  * [Motor Coach Industries](https://en.wikipedia.org/wiki/Motor_Coach_Industries "Motor Coach Industries")
  * [Oshkosh](https://en.wikipedia.org/wiki/Oshkosh_Corporation "Oshkosh Corporation")
    * [Pierce](https://en.wikipedia.org/wiki/Pierce_Manufacturing "Pierce Manufacturing")
  * [Paccar](https://en.wikipedia.org/wiki/Paccar "Paccar")
    * [Kenworth](https://en.wikipedia.org/wiki/Kenworth "Kenworth")
    * [Peterbilt](https://en.wikipedia.org/wiki/Peterbilt "Peterbilt")
  * [Panoz](https://en.wikipedia.org/wiki/Panoz "Panoz")
  * [Polaris Industries](https://en.wikipedia.org/wiki/Polaris_Industries "Polaris Industries")
    * [Global Electric Motorcars](https://en.wikipedia.org/wiki/Global_Electric_Motorcars "Global Electric Motorcars")
    * [Indian](https://en.wikipedia.org/wiki/Indian_Motorcycle "Indian Motorcycle")
    * [Victory](https://en.wikipedia.org/wiki/Victory_Motorcycles "Victory Motorcycles")
  * [REV Group](https://en.wikipedia.org/wiki/REV_Group "REV Group")
    * [Fleetwood](https://en.wikipedia.org/wiki/Fleetwood_Enterprises "Fleetwood Enterprises")
    * [Holiday Rambler](https://en.wikipedia.org/wiki/Holiday_Rambler "Holiday Rambler")
    * [Laymor](https://en.wikipedia.org/wiki/Collins_Industries "Collins Industries")
    * [Wheeled Coach](https://en.wikipedia.org/wiki/Wheeled_Coach "Wheeled Coach")
  * [Rezvani Motors](https://en.wikipedia.org/wiki/Rezvani_Motors "Rezvani Motors")
  * [Rivian](https://en.wikipedia.org/wiki/Rivian "Rivian")
  * [Scuderia Cameron Glickenhaus](https://en.wikipedia.org/wiki/Scuderia_Cameron_Glickenhaus "Scuderia Cameron Glickenhaus")
  * [SSC North America](https://en.wikipedia.org/wiki/SSC_North_America "SSC North America")
  * [Superformance](https://en.wikipedia.org/wiki/Superformance "Superformance")
  * [Telo](https://en.wikipedia.org/wiki/Telo_Trucks "Telo Trucks")
  * Tesla
  * [Textron](https://en.wikipedia.org/wiki/Textron "Textron")
    * [Arctic Cat](https://en.wikipedia.org/wiki/Arctic_Cat "Arctic Cat")
    * [E-Z-Go](https://en.wikipedia.org/wiki/E-Z-Go "E-Z-Go")
    * [Cushman](https://en.wikipedia.org/wiki/Cushman_\(company\) "Cushman \(company\)")
  * [Trans Tech](https://en.wikipedia.org/wiki/Trans_Tech "Trans Tech")
  * [Ultimaster](https://en.wikipedia.org/wiki/Utilimaster_Corporation "Utilimaster Corporation")
  * [VIA Motors](https://en.wikipedia.org/wiki/VIA_Motors "VIA Motors")
  * [VLF Automotive](https://en.wikipedia.org/wiki/VLF_Automotive "VLF Automotive")
  * [Xos, Inc.](https://en.wikipedia.org/wiki/Xos,_Inc. "Xos, Inc.")
  * [Zero Motorcycles](https://en.wikipedia.org/wiki/Zero_Motorcycles "Zero Motorcycles")

  
---|---  
Foreign   
subsidiaries| 

  * [BMW](https://en.wikipedia.org/wiki/BMW_in_the_United_States "BMW in the United States")
  * [Daimler Truck](https://en.wikipedia.org/wiki/Daimler_Truck_North_America "Daimler Truck North America")
  * [Honda](https://en.wikipedia.org/wiki/American_Honda_Motor_Company "American Honda Motor Company")
    * [Acura](https://en.wikipedia.org/wiki/Acura "Acura")
  * [Hyundai](https://en.wikipedia.org/wiki/Hyundai_Motor_Company#Regional_operations "Hyundai Motor Company")
    * [Kia](https://en.wikipedia.org/wiki/Kia#Kia_Motors_America "Kia")
  * [Mercedes-Benz](https://en.wikipedia.org/wiki/Mercedes-Benz_USA "Mercedes-Benz USA")
  * [Mitsubishi](https://en.wikipedia.org/wiki/Mitsubishi_Motors_North_America "Mitsubishi Motors North America")
  * [Nissan](https://en.wikipedia.org/wiki/Nissan_USA "Nissan USA")
    * [Infiniti](https://en.wikipedia.org/wiki/Infiniti "Infiniti")
  * [Seres Group](https://en.wikipedia.org/wiki/Seres_Group "Seres Group")
    * [Seres](https://en.wikipedia.org/wiki/Seres_\(automobiles\) "Seres \(automobiles\)")
  * **[Stellantis](https://en.wikipedia.org/wiki/Chrysler "Chrysler")** 1
    * [Chrysler](https://en.wikipedia.org/wiki/Chrysler_\(brand\) "Chrysler \(brand\)")
    * [Dodge](https://en.wikipedia.org/wiki/Dodge "Dodge")
    * [Jeep](https://en.wikipedia.org/wiki/Jeep "Jeep")
    * [Ram](https://en.wikipedia.org/wiki/Ram_Trucks "Ram Trucks")
  * [Subaru](https://en.wikipedia.org/wiki/Subaru_of_America "Subaru of America")
  * [Toyota](https://en.wikipedia.org/wiki/Toyota_Motor_North_America "Toyota Motor North America")
  * [Toyota Motor Engineering & Manufacturing](https://en.wikipedia.org/wiki/Toyota_Motor_Engineering_%26_Manufacturing_North_America "Toyota Motor Engineering & Manufacturing North America")
  * [Volkswagen](https://en.wikipedia.org/wiki/Volkswagen_Group_of_America "Volkswagen Group of America")

  
Defunct /  
former 2| 

  * [Allis-Chalmers](https://en.wikipedia.org/wiki/Allis-Chalmers "Allis-Chalmers")
  * [American Austin](https://en.wikipedia.org/wiki/American_Austin_Car_Company "American Austin Car Company")
  * [American Electric](https://en.wikipedia.org/wiki/The_Kurrent "The Kurrent")
  * [American LaFrance](https://en.wikipedia.org/wiki/American_LaFrance "American LaFrance")
  * [American Motors](https://en.wikipedia.org/wiki/American_Motors_Corporation "American Motors Corporation")
    * [Hudson](https://en.wikipedia.org/wiki/Hudson_Motor_Car_Company "Hudson Motor Car Company")
      * [Essex](https://en.wikipedia.org/wiki/Essex_\(automobile\) "Essex \(automobile\)")
      * [Terraplane](https://en.wikipedia.org/wiki/Terraplane "Terraplane")
    * [Nash](https://en.wikipedia.org/wiki/Nash_Motors "Nash Motors")
    * [Rambler](https://en.wikipedia.org/wiki/Rambler_\(automobile\) "Rambler \(automobile\)")
  * [Armor](https://en.wikipedia.org/wiki/Armor_Holdings "Armor Holdings")
  * [Armored](https://en.wikipedia.org/wiki/Armored_Motor_Car_Company "Armored Motor Car Company")
  * [Auburn](https://en.wikipedia.org/wiki/Auburn_Automobile "Auburn Automobile")
  * [Aurica](https://en.wikipedia.org/wiki/Aurica_Motors "Aurica Motors")
  * [Autoette](https://en.wikipedia.org/wiki/Autoette "Autoette")
  * [Avanti](https://en.wikipedia.org/wiki/Avanti_\(car\) "Avanti \(car\)")
  * [Avery](https://en.wikipedia.org/wiki/Avery_Company "Avery Company")
  * [BMC](https://en.wikipedia.org/wiki/Best_Manufacturing_Company "Best Manufacturing Company")
  * [Boulder Electric Vehicle](https://en.wikipedia.org/wiki/Boulder_Electric_Vehicle "Boulder Electric Vehicle")
  * [Carbon Motors Corporation](https://en.wikipedia.org/wiki/Carbon_Motors_Corporation "Carbon Motors Corporation")
  * [Checker Motors Corporation](https://en.wikipedia.org/wiki/Checker_Motors_Corporation "Checker Motors Corporation")
  * [Clydesdale Motor Truck Company](https://en.wikipedia.org/wiki/Clydesdale_Motor_Truck_Company "Clydesdale Motor Truck Company")
  * [Coda](https://en.wikipedia.org/wiki/Coda_Automotive "Coda Automotive")2
  * [Commonwealth](https://en.wikipedia.org/wiki/Commonwealth_\(automobile_company\) "Commonwealth \(automobile company\)")
  * [Cord](https://en.wikipedia.org/wiki/Cord_\(automobile\) "Cord \(automobile\)")
  * [Case](https://en.wikipedia.org/wiki/Case_Corporation "Case Corporation")
  * [CNH Global](https://en.wikipedia.org/wiki/CNH_Global "CNH Global")
  * [Cycle-Scoot](https://en.wikipedia.org/wiki/Cycle-Scoot "Cycle-Scoot")
  * [DeLorean](https://en.wikipedia.org/wiki/DeLorean_Motor_Company "DeLorean Motor Company")
  * [Diamond-Star](https://en.wikipedia.org/wiki/Diamond-Star_Motors "Diamond-Star Motors")
  * [Duesenberg](https://en.wikipedia.org/wiki/Duesenberg "Duesenberg")
  * [Durant](https://en.wikipedia.org/wiki/Durant_Motors "Durant Motors")
    * [Flint](https://en.wikipedia.org/wiki/Flint_\(automobile\) "Flint \(automobile\)")
    * [Locomobile](https://en.wikipedia.org/wiki/Locomobile_Company_of_America "Locomobile Company of America")
    * [Mason](https://en.wikipedia.org/wiki/Mason_Truck "Mason Truck")
    * [Rugby](https://en.wikipedia.org/wiki/Rugby_\(automobile\) "Rugby \(automobile\)")
    * [Star](https://en.wikipedia.org/wiki/Star_\(automobile\) "Star \(automobile\)")
  * [Eagle Bus](https://en.wikipedia.org/wiki/Eagle_Bus "Eagle Bus")
  * [Excalibur](https://en.wikipedia.org/wiki/Excalibur_\(automobile\) "Excalibur \(automobile\)")
  * FCA US 
    * [Eagle](https://en.wikipedia.org/wiki/Eagle_\(automobile\) "Eagle \(automobile\)")
    * [Plymouth](https://en.wikipedia.org/wiki/Plymouth_\(automobile\) "Plymouth \(automobile\)")
  * [Fiberfab](https://en.wikipedia.org/wiki/Fiberfab "Fiberfab")
  * [Fitch Four Drive](https://en.wikipedia.org/wiki/Fitch_Four_Drive "Fitch Four Drive")
  * [Fisker Automotive](https://en.wikipedia.org/wiki/Fisker_Automotive "Fisker Automotive")
  * [Fisker Coachbuild](https://en.wikipedia.org/wiki/Fisker_Coachbuild "Fisker Coachbuild")
  * [Force Protection](https://en.wikipedia.org/wiki/Force_Protection_Inc "Force Protection Inc")
  * Ford 
    * [Continental](https://en.wikipedia.org/wiki/Continental_Mark_II "Continental Mark II")
    * [Edsel](https://en.wikipedia.org/wiki/Edsel "Edsel")
    * [Mercury](https://en.wikipedia.org/wiki/Mercury_\(automobile\) "Mercury \(automobile\)")
  * [FMC](https://en.wikipedia.org/wiki/FMC_Corporation "FMC Corporation")2
  * General Motors 
    * [Cartercar](https://en.wikipedia.org/wiki/Cartercar "Cartercar")
    * [Elmore](https://en.wikipedia.org/wiki/Elmore_\(automobile\) "Elmore \(automobile\)")
    * [GM Diesel](https://en.wikipedia.org/wiki/General_Motors_Diesel_Division "General Motors Diesel Division")
    * [Geo](https://en.wikipedia.org/wiki/Geo_\(automobile\) "Geo \(automobile\)")
    * [LaSalle](https://en.wikipedia.org/wiki/LaSalle_\(automobile\) "LaSalle \(automobile\)")
    * [Marquette](https://en.wikipedia.org/wiki/Marquette_\(automobile\) "Marquette \(automobile\)")
    * [McLaughlin](https://en.wikipedia.org/wiki/McLaughlin_Motor_Car_Company "McLaughlin Motor Car Company")
    * [Oakland](https://en.wikipedia.org/wiki/Oakland_Motor_Car_Company "Oakland Motor Car Company")
    * [Oldsmobile](https://en.wikipedia.org/wiki/Oldsmobile "Oldsmobile")
    * [Pontiac](https://en.wikipedia.org/wiki/Pontiac_\(automobile\) "Pontiac \(automobile\)")
    * [Saturn](https://en.wikipedia.org/wiki/Saturn_Corporation "Saturn Corporation")
    * [Scripps-Booth](https://en.wikipedia.org/wiki/Scripps-Booth "Scripps-Booth")
    * [Sheridan](https://en.wikipedia.org/wiki/Sheridan_\(automobile\) "Sheridan \(automobile\)")
    * [Viking](https://en.wikipedia.org/wiki/Viking_\(automobile\) "Viking \(automobile\)")
    * [Yellow Coach](https://en.wikipedia.org/wiki/Yellow_Coach_Manufacturing_Company "Yellow Coach Manufacturing Company")
  * [Goshen Coach](https://en.wikipedia.org/wiki/Goshen_Coach "Goshen Coach")
  * [Green](https://en.wikipedia.org/wiki/Green_Vehicles "Green Vehicles")
  * [GreenTech](https://en.wikipedia.org/wiki/GreenTech_Automotive "GreenTech Automotive")
  * [Grumman](https://en.wikipedia.org/wiki/Grumman "Grumman")
  * [Henney](https://en.wikipedia.org/wiki/Henney_Kilowatt "Henney Kilowatt")
  * [International Harvester](https://en.wikipedia.org/wiki/International_Harvester "International Harvester")
  * [Jeffery](https://en.wikipedia.org/wiki/Jeffery_\(automobile\) "Jeffery \(automobile\)")
  * [Kaiser-Frazer](https://en.wikipedia.org/wiki/Kaiser-Frazer "Kaiser-Frazer")
    * [Allstate](https://en.wikipedia.org/wiki/Allstate_\(automobile\) "Allstate \(automobile\)")
    * [Frazer](https://en.wikipedia.org/wiki/Frazer_\(automobile\) "Frazer \(automobile\)")
    * [Henry J](https://en.wikipedia.org/wiki/Henry_J "Henry J")
    * Kaiser
    * [Willys](https://en.wikipedia.org/wiki/Willys "Willys")
  * [Local](https://en.wikipedia.org/wiki/Local_Motors "Local Motors")
  * [Marathon](https://en.wikipedia.org/wiki/Marathon_Motor_Works "Marathon Motor Works")
  * [Marmon](https://en.wikipedia.org/wiki/Marmon_Motor_Car_Company "Marmon Motor Car Company")
    * [Roosevelt](https://en.wikipedia.org/wiki/Roosevelt_\(automobile\) "Roosevelt \(automobile\)")
  * [Marvel](https://en.wikipedia.org/wiki/Marvel_\(automobile\) "Marvel \(automobile\)")
  * [Matbro](https://en.wikipedia.org/wiki/Matbro "Matbro")
  * [Mercer](https://en.wikipedia.org/wiki/Mercer_\(automobile\) "Mercer \(automobile\)")
  * [Monaco Coach](https://en.wikipedia.org/wiki/Monaco_Coach_Corporation "Monaco Coach Corporation")
  * [Mosler](https://en.wikipedia.org/wiki/Mosler_Automotive "Mosler Automotive")
  * [MotoCzysz](https://en.wikipedia.org/wiki/MotoCzysz "MotoCzysz")
  * [Muntz](https://en.wikipedia.org/wiki/Muntz_Car_Company "Muntz Car Company")
  * [New United](https://en.wikipedia.org/wiki/NUMMI "NUMMI")
  * [North American Bus Industries](https://en.wikipedia.org/wiki/North_American_Bus_Industries "North American Bus Industries")
  * [Oliver Farm Equipment](https://en.wikipedia.org/wiki/Oliver_Farm_Equipment_Company "Oliver Farm Equipment Company")
  * [Packard](https://en.wikipedia.org/wiki/Packard "Packard")
  * [Peerless](https://en.wikipedia.org/wiki/Peerless_Motor_Company "Peerless Motor Company")
  * [Pierce-Arrow](https://en.wikipedia.org/wiki/Pierce-Arrow_Motor_Car_Company "Pierce-Arrow Motor Car Company")
  * [Sebring Vanguard](https://en.wikipedia.org/wiki/Sebring_Vanguard "Sebring Vanguard")
  * [Sterling](https://en.wikipedia.org/wiki/Sterling_Trucks "Sterling Trucks")
  * [Studebaker](https://en.wikipedia.org/wiki/Studebaker "Studebaker")
    * [Erskine](https://en.wikipedia.org/wiki/Erskine_\(automobile\) "Erskine \(automobile\)")
    * [Rockne](https://en.wikipedia.org/wiki/Rockne "Rockne")
  * [Stutz](https://en.wikipedia.org/wiki/Stutz_Motor_Company "Stutz Motor Company")
    * [Scion](https://en.wikipedia.org/wiki/Scion_\(automobile\) "Scion \(automobile\)")
  * [Twentieth Century](https://en.wikipedia.org/wiki/Twentieth_Century_Motor_Car_Corporation "Twentieth Century Motor Car Corporation")
  * [United Defense](https://en.wikipedia.org/wiki/United_Defense "United Defense")
  * [VPG](https://en.wikipedia.org/wiki/Vehicle_Production_Group "Vehicle Production Group")
  * [Visionary](https://en.wikipedia.org/wiki/Visionary_Vehicles "Visionary Vehicles")
  * [VL](https://en.wikipedia.org/wiki/VL_Automotive "VL Automotive")
  * [White](https://en.wikipedia.org/wiki/White_Motor_Company "White Motor Company")
  * [Wildfire](https://en.wikipedia.org/wiki/Wildfire_\(motor_company\) "Wildfire \(motor company\)")
  * [ZAP](https://en.wikipedia.org/wiki/ZAP_\(motor_company\) "ZAP \(motor company\)")
  * [Zimmer](https://en.wikipedia.org/wiki/Zimmer_\(automobile\) "Zimmer \(automobile\)")

  
  
Concept and   
pre-production| 

  * [Alpha Motor Corporation](https://en.wikipedia.org/wiki/Alpha_Motor_Corporation "Alpha Motor Corporation")
  * [Aptera Motors](https://en.wikipedia.org/wiki/Aptera_Motors "Aptera Motors")
  * [Bollinger Motors](https://en.wikipedia.org/wiki/Bollinger_Motors "Bollinger Motors")
  * [Canoo](https://en.wikipedia.org/wiki/Canoo "Canoo")
  * [Commuter Cars](https://en.wikipedia.org/wiki/Commuter_Cars "Commuter Cars")
  * [Elio Motors](https://en.wikipedia.org/wiki/Elio_Motors "Elio Motors")
  * [Faraday Future](https://en.wikipedia.org/wiki/Faraday_Future "Faraday Future")
  * [Fisker Inc](https://en.wikipedia.org/wiki/Fisker_Inc "Fisker Inc")
  * [Lordstown Motors](https://en.wikipedia.org/wiki/Lordstown_Motors "Lordstown Motors")
  * [Myers Motors](https://en.wikipedia.org/wiki/Myers_Motors "Myers Motors")
  * [Nikola](https://en.wikipedia.org/wiki/Nikola_Corporation "Nikola Corporation")
  * [Slate Auto](https://en.wikipedia.org/wiki/Slate_Auto "Slate Auto")
  * [Trion Supercars](https://en.wikipedia.org/wiki/Trion_Supercars "Trion Supercars")

  
Factories| | Active| 

  * [BMW Spartanburg](https://en.wikipedia.org/wiki/BMW_in_the_United_States "BMW in the United States")
  * [Chrysler (list)](https://en.wikipedia.org/wiki/List_of_Chrysler_factories#Current_factories "List of Chrysler factories")
  * [Ford (list)](https://en.wikipedia.org/wiki/List_of_Ford_factories#Current_production_facilities "List of Ford factories")
  * [General Motors (list)](https://en.wikipedia.org/wiki/List_of_General_Motors_factories#Current_factories "List of General Motors factories")
  * [Honda (list)](https://en.wikipedia.org/wiki/List_of_Honda_assembly_plants "List of Honda assembly plants")
  * [Hyundai (Alabama)](https://en.wikipedia.org/wiki/Hyundai_Motor_Manufacturing_Alabama "Hyundai Motor Manufacturing Alabama")
  * [Hyundai Metaplant (Georgia)](https://en.wikipedia.org/wiki/Hyundai_Motor_Group_Metaplant_America "Hyundai Motor Group Metaplant America")
  * [Kia Motors Manufacturing Georgia](https://en.wikipedia.org/wiki/List_of_Kia_design_and_manufacturing_facilities#Kia_Motors_Manufacturing_Georgia_\(KMMG\) "List of Kia design and manufacturing facilities")
  * [Mercedes-Benz (Alabama)](https://en.wikipedia.org/wiki/Mercedes-Benz_U.S._International "Mercedes-Benz U.S. International")
  * [Subaru (Indiana)](https://en.wikipedia.org/wiki/Subaru_of_Indiana_Automotive "Subaru of Indiana Automotive")
  * [Tesla (list)](https://en.wikipedia.org/wiki/List_of_Tesla_factories "List of Tesla factories")
  * [Volkswagen (Chattanooga)](https://en.wikipedia.org/wiki/Volkswagen_Chattanooga_Assembly_Plant "Volkswagen Chattanooga Assembly Plant")

  
---|---  
Defunct| 

  * [Chrysler (list)](https://en.wikipedia.org/wiki/List_of_Chrysler_factories#Current_factories "List of Chrysler factories")
  * [Ford (list)](https://en.wikipedia.org/wiki/List_of_Ford_factories#Former_production_facilities "List of Ford factories")
  * [General Motors (list)](https://en.wikipedia.org/wiki/List_of_General_Motors_factories#Closed_or_sold_Factories "List of General Motors factories")
  * [Packard](https://en.wikipedia.org/wiki/Packard_Automotive_Plant "Packard Automotive Plant")
  * [Volkswagen (Westmoreland)](https://en.wikipedia.org/wiki/Volkswagen_Westmoreland_Assembly "Volkswagen Westmoreland Assembly")

  
  
Auto component   
makers and   
[performance car](https://en.wikipedia.org/wiki/Performance_car "Performance car")   
modders| 

  * [Allison](https://en.wikipedia.org/wiki/Allison_Transmission "Allison Transmission")
  * [American Expedition Vehicles](https://en.wikipedia.org/wiki/American_Expedition_Vehicles "American Expedition Vehicles")
  * [Aptiv](https://en.wikipedia.org/wiki/Aptiv "Aptiv")
  * [BFGoodrich](https://en.wikipedia.org/wiki/BFGoodrich "BFGoodrich")
  * [BorgWarner](https://en.wikipedia.org/wiki/BorgWarner "BorgWarner")
  * [Callaway Cars](https://en.wikipedia.org/wiki/Callaway_Cars "Callaway Cars")
  * [Caterpillar](https://en.wikipedia.org/wiki/Caterpillar_Inc "Caterpillar Inc")
  * [Cummins](https://en.wikipedia.org/wiki/Cummins "Cummins")
    * [Brammo](https://en.wikipedia.org/wiki/Brammo "Brammo")
  * [Detroit Diesel](https://en.wikipedia.org/wiki/Detroit_Diesel "Detroit Diesel")
  * [Eaton](https://en.wikipedia.org/wiki/Eaton_Corporation "Eaton Corporation")
  * [Firestone](https://en.wikipedia.org/wiki/Firestone_Tire_and_Rubber_Company "Firestone Tire and Rubber Company")
  * [General Tire](https://en.wikipedia.org/wiki/General_Tire "General Tire")
  * [Goodyear](https://en.wikipedia.org/wiki/Goodyear_Tire_and_Rubber_Company "Goodyear Tire and Rubber Company")
    * [Cooper Tire & Rubber Company](https://en.wikipedia.org/wiki/Cooper_Tire_%26_Rubber_Company "Cooper Tire & Rubber Company")
  * [Hennessey](https://en.wikipedia.org/wiki/Hennessey_Performance_Engineering "Hennessey Performance Engineering")
  * [Ingersoll Rand](https://en.wikipedia.org/wiki/Ingersoll_Rand "Ingersoll Rand")
  * [Legacy](https://en.wikipedia.org/wiki/Legacy_Classic_Trucks "Legacy Classic Trucks")
  * [Lingenfelter](https://en.wikipedia.org/wiki/Lingenfelter_Performance_Engineering "Lingenfelter Performance Engineering")
  * [Nexteer](https://en.wikipedia.org/wiki/Nexteer_Automotive "Nexteer Automotive")
  * [Phoenix Motorcars](https://en.wikipedia.org/wiki/Phoenix_Motorcars "Phoenix Motorcars")
    * [Proterra (bus manufacturer)](https://en.wikipedia.org/wiki/Proterra_\(bus_manufacturer\) "Proterra \(bus manufacturer\)")
  * [Remy International](https://en.wikipedia.org/wiki/Remy_International "Remy International")
  * [Saleen](https://en.wikipedia.org/wiki/Saleen "Saleen")
  * [Shelby American](https://en.wikipedia.org/wiki/Shelby_American "Shelby American")
  * [SRT](https://en.wikipedia.org/wiki/Street_and_Racing_Technology "Street and Racing Technology")
  * [Timken](https://en.wikipedia.org/wiki/Timken_Company "Timken Company")
  * [Torrington](https://en.wikipedia.org/wiki/Torrington_Company "Torrington Company")
  * [Visteon](https://en.wikipedia.org/wiki/Visteon "Visteon")

  
Design studios| 

  * [Calty Design Research](https://en.wikipedia.org/wiki/Calty_Design_Research "Calty Design Research")
  * [Designworks](https://en.wikipedia.org/wiki/Designworks "Designworks")
  * [Rezvani Automotive Designs](https://en.wikipedia.org/wiki/Rezvani_Motors "Rezvani Motors")
  * [Wheego Electric Cars](https://en.wikipedia.org/wiki/Wheego_Electric_Cars "Wheego Electric Cars")

  
By state| 

  * [Massachusetts](https://en.wikipedia.org/wiki/Automotive_industry_in_Massachusetts "Automotive industry in Massachusetts")

  
Related topics| 

  * [AAA](https://en.wikipedia.org/wiki/American_Automobile_Association "American Automobile Association")
  * [Chicago Auto Show](https://en.wikipedia.org/wiki/Chicago_Auto_Show "Chicago Auto Show")
  * [Interstate Highway System](https://en.wikipedia.org/wiki/Interstate_Highway_System "Interstate Highway System")
  * [List of automobiles manufactured in the United States](https://en.wikipedia.org/wiki/List_of_automobiles_manufactured_in_the_United_States "List of automobiles manufactured in the United States")
  * [National Highway Traffic Safety Administration](https://en.wikipedia.org/wiki/National_Highway_Traffic_Safety_Administration "National Highway Traffic Safety Administration")
  * [New York International Auto Show](https://en.wikipedia.org/wiki/New_York_International_Auto_Show "New York International Auto Show")
  * [North American International Auto Show](https://en.wikipedia.org/wiki/North_American_International_Auto_Show "North American International Auto Show")
  * [SAE International](https://en.wikipedia.org/wiki/SAE_International "SAE International")

  
  
  * 1 Non-U.S. based parent company that owns subsidiaries headquartered in U.S.
  * 2 Company still exists but is no longer in the automotive manufacturing business
  * "[Big 3](https://en.wikipedia.org/wiki/Big_Three_\(automobile_manufacturers\) "Big Three \(automobile manufacturers\)")" in **bold**



* * *

  * [Category](https://en.wikipedia.org/wiki/Category:Automotive_industry_in_the_United_States "Category:Automotive industry in the United States")
  * [](https://en.wikipedia.org/wiki/File:Symbol_portal_class.svg "Portal") [Portal](https://en.wikipedia.org/wiki/Portal:Cars "Portal:Cars")

  
  
  * [v](https://en.wikipedia.org/wiki/Template:Trucking_industry_in_the_United_States "Template:Trucking industry in the United States")
  * [t](https://en.wikipedia.org/wiki/Template_talk:Trucking_industry_in_the_United_States "Template talk:Trucking industry in the United States")
  * [e](https://en.wikipedia.org/wiki/Special:EditPage/Template:Trucking_industry_in_the_United_States "Special:EditPage/Template:Trucking industry in the United States")

[Trucking industry in the United States](https://en.wikipedia.org/wiki/Trucking_industry_in_the_United_States "Trucking industry in the United States")  
---  
Regulated by the [Federal Motor Carrier Safety Administration](https://en.wikipedia.org/wiki/Federal_Motor_Carrier_Safety_Administration "Federal Motor Carrier Safety Administration")  
[Economy of the United States](https://en.wikipedia.org/wiki/Economy_of_the_United_States "Economy of the United States")  
Regulations| 

  * [Commercial driver's license](https://en.wikipedia.org/wiki/Commercial_driver%27s_license "Commercial driver's license")
  * [Electronic on-board recorder](https://en.wikipedia.org/wiki/Electronic_on-board_recorder "Electronic on-board recorder")
  * [Federal Bridge Gross Weight Formula](https://en.wikipedia.org/wiki/Federal_Bridge_Gross_Weight_Formula "Federal Bridge Gross Weight Formula")
  * [Hours of service](https://en.wikipedia.org/wiki/Hours_of_service "Hours of service")
  * [International Registration Plan](https://en.wikipedia.org/wiki/International_Registration_Plan "International Registration Plan")
  * [Motor Carrier Act of 1980](https://en.wikipedia.org/wiki/Motor_Carrier_Act_of_1980 "Motor Carrier Act of 1980")
  * [Motor carrier safety rating](https://en.wikipedia.org/wiki/Motor_carrier_safety_rating "Motor carrier safety rating")
  * [National Network](https://en.wikipedia.org/wiki/National_Network "National Network")

  
Manufacturers| | Truck manufacturers| 

  * [AM General](https://en.wikipedia.org/wiki/AM_General "AM General")
  * _[American LaFrance](https://en.wikipedia.org/wiki/American_LaFrance "American LaFrance")_
  * [Autocar](https://en.wikipedia.org/wiki/Autocar_Company "Autocar Company")
  * _[Bering](https://en.wikipedia.org/wiki/Bering_Truck "Bering Truck")_
  * [Bremach](https://en.wikipedia.org/wiki/Bremach "Bremach")
  * _[Brockway](https://en.wikipedia.org/wiki/Brockway_Motor_Company "Brockway Motor Company")_
  * [BYD Auto](https://en.wikipedia.org/wiki/BYD_Auto "BYD Auto")
  * [Caterpillar Inc.](https://en.wikipedia.org/wiki/Caterpillar_Inc. "Caterpillar Inc.")
  * _[Chase](https://en.wikipedia.org/wiki/Chase_Motor_Truck_Company "Chase Motor Truck Company")_
  * [Chevrolet](https://en.wikipedia.org/wiki/Chevrolet "Chevrolet")
  * [CCC](https://en.wikipedia.org/wiki/Crane_Carrier_Company "Crane Carrier Company")
  * [CNH Industrial](https://en.wikipedia.org/wiki/CNH_Industrial "CNH Industrial")
  * _[Dart](https://en.wikipedia.org/wiki/Dart_\(commercial_vehicle\) "Dart \(commercial vehicle\)")_
  * _[Diamond Reo](https://en.wikipedia.org/wiki/Diamond_Reo_Trucks "Diamond Reo Trucks")_
    * _[Diamond T](https://en.wikipedia.org/wiki/Diamond_T "Diamond T")_
    * _[Reo](https://en.wikipedia.org/wiki/Reo_Motor_Car_Company "Reo Motor Car Company")_
  * [Dina](https://en.wikipedia.org/wiki/DINA_SA "DINA SA")
  * [Dodge](https://en.wikipedia.org/wiki/Dodge "Dodge")
  * [Freightliner](https://en.wikipedia.org/wiki/Freightliner_Trucks "Freightliner Trucks")
  * [Ford](https://en.wikipedia.org/wiki/Ford_Motor_Company "Ford Motor Company")
  * [GMC](https://en.wikipedia.org/wiki/GMC_\(marque\) "GMC \(marque\)")
  * _[Hayes](https://en.wikipedia.org/wiki/Hayes_Manufacturing_Company "Hayes Manufacturing Company")_
  * [Fuso](https://en.wikipedia.org/wiki/Fuso_Trucks_America "Fuso Trucks America")
  * [Hino](https://en.wikipedia.org/wiki/Hino_Motors "Hino Motors")
  * [Hyundai Motor America](https://en.wikipedia.org/wiki/Hyundai_Motor_America "Hyundai Motor America")
  * [Hyzon](https://en.wikipedia.org/wiki/Hyzon_Motors "Hyzon Motors")
  * [International](https://en.wikipedia.org/wiki/International_Motors "International Motors")
  * [Isuzu](https://en.wikipedia.org/wiki/Isuzu "Isuzu")
  * [Kenworth](https://en.wikipedia.org/wiki/Kenworth "Kenworth")
  * _[Marmon-Herrington](https://en.wikipedia.org/wiki/Marmon-Herrington "Marmon-Herrington")_
  * [Mercedes-Benz](https://en.wikipedia.org/wiki/Mercedes-Benz "Mercedes-Benz")
  * _[Moreland](https://en.wikipedia.org/wiki/Moreland_Motor_Truck_Company "Moreland Motor Truck Company")_
  * [Mack](https://en.wikipedia.org/wiki/Mack_Trucks "Mack Trucks")
  * [Nissan](https://en.wikipedia.org/wiki/Nissan "Nissan")
  * [Oshkosh](https://en.wikipedia.org/wiki/Oshkosh_Corporation "Oshkosh Corporation")
  * [Peterbilt](https://en.wikipedia.org/wiki/Peterbilt "Peterbilt")
  * [Ram Trucks](https://en.wikipedia.org/wiki/Ram_Trucks "Ram Trucks")
  * _[Selden](https://en.wikipedia.org/wiki/Selden_Motor_Vehicle_Company "Selden Motor Vehicle Company")_
  * _[Schacht](https://en.wikipedia.org/wiki/Schacht_\(automobile\) "Schacht \(automobile\)")_
  * [Shyft Group](https://en.wikipedia.org/wiki/Shyft_Group "Shyft Group")
  * [Smith](https://en.wikipedia.org/wiki/Smith_Electric_Vehicles "Smith Electric Vehicles")
  * [Spartan](https://en.wikipedia.org/wiki/Spartan_Motors "Spartan Motors")
  * _[Sterling](https://en.wikipedia.org/wiki/Sterling_Trucks "Sterling Trucks")_
  * Tesla
  * [Tiger Truck](https://en.wikipedia.org/wiki/Tiger_Truck "Tiger Truck")
  * [Toyota](https://en.wikipedia.org/wiki/Toyota "Toyota")
  * _[Traffic](https://en.wikipedia.org/wiki/Traffic_Motor_Truck_Corporation "Traffic Motor Truck Corporation")_
  * [UD Trucks](https://en.wikipedia.org/wiki/UD_Trucks "UD Trucks")
  * [VIA](https://en.wikipedia.org/wiki/VIA_Motors "VIA Motors")
  * [Volvo](https://en.wikipedia.org/wiki/Volvo_Trucks "Volvo Trucks")
  * _[Ward LaFrance](https://en.wikipedia.org/wiki/Ward_LaFrance "Ward LaFrance")_
  * [Western Star](https://en.wikipedia.org/wiki/Western_Star_Trucks "Western Star Trucks")
  * _[White](https://en.wikipedia.org/wiki/White_Motor_Company "White Motor Company")_
  * [Workhorse Group](https://en.wikipedia.org/wiki/Workhorse_Group "Workhorse Group")

  
---|---  
Engine manufacturers| 

  * [Caterpillar Inc.](https://en.wikipedia.org/wiki/Caterpillar_Inc. "Caterpillar Inc.")
  * [Cummins](https://en.wikipedia.org/wiki/Cummins "Cummins")
  * [Detroit Diesel](https://en.wikipedia.org/wiki/Detroit_Diesel "Detroit Diesel")
  * [Mack](https://en.wikipedia.org/wiki/Mack_Trucks "Mack Trucks")
  * [MaxxForce](https://en.wikipedia.org/wiki/International_Motors "International Motors")
  * [Mercedes-Benz](https://en.wikipedia.org/wiki/Mercedes-Benz "Mercedes-Benz")
  * [Paccar](https://en.wikipedia.org/wiki/Paccar "Paccar")
  * [Volvo](https://en.wikipedia.org/wiki/Volvo_Trucks "Volvo Trucks")

  
Trailer manufacturers| 

  * [Daseke](https://en.wikipedia.org/wiki/Daseke "Daseke")
  * [Fontaine](https://en.wikipedia.org/wiki/Marmon_Group "Marmon Group")
  * [Fruehauf](https://en.wikipedia.org/wiki/Fruehauf "Fruehauf")
  * [Great Dane](https://en.wikipedia.org/wiki/Great_Dane_Trailers "Great Dane Trailers")
  * [Hyundai Translead](https://en.wikipedia.org/wiki/Hyundai_Motor_Company "Hyundai Motor Company")
  * [Lufkin Trailers](https://en.wikipedia.org/wiki/Lufkin_Industries "Lufkin Industries")
  * [Utility Trailer Manufacturing Company](https://en.wikipedia.org/wiki/Utility_Trailer_Manufacturing_Company "Utility Trailer Manufacturing Company")
  * [Vanguard](https://en.wikipedia.org/wiki/China_International_Marine_Containers "China International Marine Containers")
  * [Wabash National](https://en.wikipedia.org/wiki/Wabash_National "Wabash National")
  * [Wilson](https://en.wikipedia.org/wiki/Wilson_Trailer_Company "Wilson Trailer Company")

  
  
Motor carriers| | [Truckload](https://en.wikipedia.org/wiki/Truckload_shipping "Truckload shipping") carriers| 

  * [Averitt Express](https://en.wikipedia.org/wiki/Averitt_Express "Averitt Express")
  * [Amazon Logistics](https://en.wikipedia.org/wiki/Amazon_Logistics "Amazon Logistics")
  * _[Celadon](https://en.wikipedia.org/wiki/Celadon_Group "Celadon Group")_
  * [CFI](https://en.wikipedia.org/wiki/Contract_Freighters_Inc. "Contract Freighters Inc.")
  * [Covenant](https://en.wikipedia.org/wiki/Covenant_Transport "Covenant Transport")
  * [CRST](https://en.wikipedia.org/wiki/CRST "CRST")
  * [FFE Transportation](https://en.wikipedia.org/wiki/FFE_Transportation "FFE Transportation")
  * [J. B. Hunt](https://en.wikipedia.org/wiki/J._B._Hunt "J. B. Hunt")
  * [Knight-Swift](https://en.wikipedia.org/wiki/Knight-Swift "Knight-Swift")
  * [Landstar System](https://en.wikipedia.org/wiki/Landstar_System "Landstar System")
  * [PAM Transport](https://en.wikipedia.org/wiki/PAM_Transport "PAM Transport")
  * [Patriot Transportation](https://en.wikipedia.org/wiki/Patriot_Transportation "Patriot Transportation")
  * [Roehl Transport](https://en.wikipedia.org/wiki/Roehl_Transport "Roehl Transport")
  * [Schneider](https://en.wikipedia.org/wiki/Schneider_National "Schneider National")
  * [Swift](https://en.wikipedia.org/wiki/Swift_Transportation "Swift Transportation")
  * [Werner](https://en.wikipedia.org/wiki/Werner_Enterprises "Werner Enterprises")
  * [Western Express](https://en.wikipedia.org/wiki/Western_Express,_Inc. "Western Express, Inc.")
  * [WTI Transport](https://en.wikipedia.org/wiki/WTI_Transport "WTI Transport")

  
---|---  
[Less than truckload (LTL)](https://en.wikipedia.org/wiki/Less-than-truckload_shipping "Less-than-truckload shipping")| 

  * [ABF](https://en.wikipedia.org/wiki/ABF_Freight_System "ABF Freight System")
  * [Averitt Express](https://en.wikipedia.org/wiki/Averitt_Express "Averitt Express")
  * _[Con-way Freight](https://en.wikipedia.org/wiki/Con-way_Freight "Con-way Freight")_
  * _[Consolidated Freightways](https://en.wikipedia.org/wiki/Consolidated_Freightways "Consolidated Freightways")_
  * [Estes Express Lines](https://en.wikipedia.org/wiki/Estes_Express_Lines "Estes Express Lines")
  * [FedEx](https://en.wikipedia.org/wiki/FedEx "FedEx")
  * _[New England Motor Freight](https://en.wikipedia.org/wiki/New_England_Motor_Freight "New England Motor Freight")_
  * [Old Dominion Freight Line](https://en.wikipedia.org/wiki/Old_Dominion_Freight_Line "Old Dominion Freight Line")
  * [R+L Carriers](https://en.wikipedia.org/wiki/R%2BL_Carriers "R+L Carriers")
  * [Saia](https://en.wikipedia.org/wiki/Saia "Saia")
  * [Southeastern](https://en.wikipedia.org/wiki/Southeastern_Freight_Lines "Southeastern Freight Lines")
  * [TForce Freight](https://en.wikipedia.org/wiki/TForce_Freight "TForce Freight")
  * [XPO, Inc.](https://en.wikipedia.org/wiki/XPO,_Inc. "XPO, Inc.")
  * _[Yellow Corporation](https://en.wikipedia.org/wiki/Yellow_Corporation "Yellow Corporation")_

  
[Third-party logistics](https://en.wikipedia.org/wiki/Third-party_logistics "Third-party logistics") providers| 

  * _[Access America Transport](https://en.wikipedia.org/wiki/Access_America_Transport "Access America Transport")_
  * [American Lamprecht Transport](https://en.wikipedia.org/wiki/American_Lamprecht_Transport "American Lamprecht Transport")
  * [C.H. Robinson](https://en.wikipedia.org/wiki/C.H._Robinson "C.H. Robinson")
    * [Freightquote](https://en.wikipedia.org/wiki/Freightquote "Freightquote")
  * [CaseStack](https://en.wikipedia.org/wiki/CaseStack "CaseStack")
  * [Ryder](https://en.wikipedia.org/wiki/Ryder "Ryder")
  * [Total Quality Logistics](https://en.wikipedia.org/wiki/Total_Quality_Logistics "Total Quality Logistics")
  * [Trailer Bridge](https://en.wikipedia.org/wiki/Trailer_Bridge "Trailer Bridge")
  * [Trucker Path](https://en.wikipedia.org/wiki/Trucker_Path "Trucker Path")
  * [XPO, Inc.](https://en.wikipedia.org/wiki/XPO,_Inc. "XPO, Inc.")

  
[National parcel](https://en.wikipedia.org/wiki/Package_delivery "Package delivery") carriers| 

  * [DHL](https://en.wikipedia.org/wiki/DHL "DHL")
  * [FedEx](https://en.wikipedia.org/wiki/FedEx "FedEx")
  * [UPS](https://en.wikipedia.org/wiki/United_Parcel_Service "United Parcel Service")
  * [United States Postal Service](https://en.wikipedia.org/wiki/United_States_Postal_Service "United States Postal Service")

  
Regional parcel carriers| 

  * [GLS](https://en.wikipedia.org/wiki/GLS_Group "GLS Group")
  * [LaserShip](https://en.wikipedia.org/wiki/LaserShip "LaserShip")
  * [LSO](https://en.wikipedia.org/wiki/LSO_\(company\) "LSO \(company\)")
  * [OnTrac](https://en.wikipedia.org/wiki/OnTrac "OnTrac")

  
[Moving companies](https://en.wikipedia.org/wiki/Moving_company "Moving company")| 

  * [Allied](https://en.wikipedia.org/wiki/Allied_Van_Lines "Allied Van Lines")
  * [Atlas](https://en.wikipedia.org/wiki/Atlas_Van_Lines "Atlas Van Lines")
  * [Bekins](https://en.wikipedia.org/wiki/Bekins_Van_Lines "Bekins Van Lines")
  * [Gentle Giant Moving Company](https://en.wikipedia.org/wiki/Gentle_Giant_Moving_Company "Gentle Giant Moving Company")
  * [Global](https://en.wikipedia.org/wiki/Global_Van_Lines "Global Van Lines")
  * [Interstate](https://en.wikipedia.org/wiki/Interstate_Van_Lines "Interstate Van Lines")
  * [Mayflower](https://en.wikipedia.org/wiki/Mayflower_Transit "Mayflower Transit")
  * [National](https://en.wikipedia.org/wiki/National_Van_Lines "National Van Lines")
  * [North American](https://en.wikipedia.org/wiki/North_American_Van_Lines "North American Van Lines")
  * [PODS](https://en.wikipedia.org/wiki/PODS_\(company\) "PODS \(company\)")
  * [Two Men and a Truck](https://en.wikipedia.org/wiki/Two_Men_and_a_Truck "Two Men and a Truck")
  * [United](https://en.wikipedia.org/wiki/United_Van_Lines "United Van Lines")
  * [Wheaton](https://en.wikipedia.org/wiki/Wheaton_World_Wide_Moving "Wheaton World Wide Moving")

  
  
[Truck stops](https://en.wikipedia.org/wiki/Truck_stop "Truck stop")| 

  * [BETO Junction](https://en.wikipedia.org/wiki/BETO_Junction "BETO Junction")
  * [Bowlin Travel Centers](https://en.wikipedia.org/wiki/Bowlin_Travel_Centers "Bowlin Travel Centers")
  * [Dixie Travel Plaza](https://en.wikipedia.org/wiki/Dixie_Travel_Plaza "Dixie Travel Plaza")
  * [Iowa 80](https://en.wikipedia.org/wiki/Iowa_80 "Iowa 80")
  * [Love's](https://en.wikipedia.org/wiki/Love%27s "Love's")
  * [Pilot Flying J](https://en.wikipedia.org/wiki/Pilot_Flying_J "Pilot Flying J")
  * [Road Ranger](https://en.wikipedia.org/wiki/Road_Ranger "Road Ranger")
  * [Roady's](https://en.wikipedia.org/wiki/Roady%27s_Truck_Stops "Roady's Truck Stops")
  * [TravelCenters of America](https://en.wikipedia.org/wiki/TravelCenters_of_America "TravelCenters of America")
  * [Town Pump](https://en.wikipedia.org/wiki/Town_Pump "Town Pump")

  
People| 

  * [Frederick W. Smith](https://en.wikipedia.org/wiki/Frederick_W._Smith "Frederick W. Smith")
  * [Iyman Faris](https://en.wikipedia.org/wiki/Iyman_Faris "Iyman Faris")
  * [J. Harwood Cochrane](https://en.wikipedia.org/wiki/J._Harwood_Cochrane "J. Harwood Cochrane")
  * [Jimmy Hoffa](https://en.wikipedia.org/wiki/Jimmy_Hoffa "Jimmy Hoffa")
  * [John Hunt](https://en.wikipedia.org/wiki/Johnnie_Bryan_Hunt "Johnnie Bryan Hunt")
  * [Keith Jesperson](https://en.wikipedia.org/wiki/Keith_Hunter_Jesperson "Keith Hunter Jesperson")
  * [Kelly Reno](https://en.wikipedia.org/wiki/Kelly_Reno "Kelly Reno")
  * [Reginald Denny](https://en.wikipedia.org/wiki/Attack_on_Reginald_Denny "Attack on Reginald Denny")

  
Organizations| 

  * [American Moving & Storage Association](https://en.wikipedia.org/wiki/American_Moving_%26_Storage_Association "American Moving & Storage Association")
  * [American Trucking Associations](https://en.wikipedia.org/wiki/American_Trucking_Associations "American Trucking Associations")
  * [DAT Solutions](https://en.wikipedia.org/wiki/DAT_Solutions "DAT Solutions") (Dial-A-Truck)
  * [International Brotherhood of Teamsters](https://en.wikipedia.org/wiki/International_Brotherhood_of_Teamsters "International Brotherhood of Teamsters")
  * [National Motor Freight Classification](https://en.wikipedia.org/wiki/National_Motor_Freight_Classification "National Motor Freight Classification")
  * [National Motor Freight Traffic Association](https://en.wikipedia.org/wiki/National_Motor_Freight_Traffic_Association "National Motor Freight Traffic Association")
  * [National Private Truck Council](https://en.wikipedia.org/wiki/National_Private_Truck_Council "National Private Truck Council")
  * [SmartWay Transport Partnership](https://en.wikipedia.org/wiki/SmartWay_Transport_Partnership "SmartWay Transport Partnership")

  
  
  * [Glossary of the American trucking industry](https://en.wikipedia.org/wiki/Glossary_of_the_American_trucking_industry "Glossary of the American trucking industry")

  
[Popular culture](https://en.wikipedia.org/wiki/Trucking_industry_in_American_culture "Trucking industry in American culture")| | Film| 

  * _[Black Dog](https://en.wikipedia.org/wiki/Black_Dog_\(1998_film\) "Black Dog \(1998 film\)")_
  * _[Breakdown](https://en.wikipedia.org/wiki/Breakdown_\(1997_film\) "Breakdown \(1997 film\)")_
  * _[Breaker! Breaker!](https://en.wikipedia.org/wiki/Breaker!_Breaker! "Breaker! Breaker!")_
  * _[Convoy](https://en.wikipedia.org/wiki/Convoy_\(1978_film\) "Convoy \(1978 film\)")_
  * _[Duel](https://en.wikipedia.org/wiki/Duel_\(1971_film\) "Duel \(1971 film\)")_
  * _[F.I.S.T.](https://en.wikipedia.org/wiki/F.I.S.T._\(film\) "F.I.S.T. \(film\)")_
  * _[High-Ballin'](https://en.wikipedia.org/wiki/High-Ballin%27 "High-Ballin'")_
  * _[Joy Ride](https://en.wikipedia.org/wiki/Joy_Ride_\(2001_film\) "Joy Ride \(2001 film\)")_
  * _[Maximum Overdrive](https://en.wikipedia.org/wiki/Maximum_Overdrive "Maximum Overdrive")_
  * _[Over The Top](https://en.wikipedia.org/wiki/Over_the_Top_\(1987_film\) "Over the Top \(1987 film\)")_
  * _[Smokey& the Bandit](https://en.wikipedia.org/wiki/Smokey_%26_the_Bandit "Smokey & the Bandit")_
  * _[They Drive by Night](https://en.wikipedia.org/wiki/They_Drive_by_Night "They Drive by Night")_
  * _[The Gang's All Here](https://en.wikipedia.org/wiki/The_Gang%27s_All_Here_\(1941_film\) "The Gang's All Here \(1941 film\)")_
  * _[The Great Smokey Roadblock](https://en.wikipedia.org/wiki/The_Great_Smokey_Roadblock "The Great Smokey Roadblock")_
  * _[Trucker](https://en.wikipedia.org/wiki/Trucker_\(film\) "Trucker \(film\)")_
  * _[White Line Fever](https://en.wikipedia.org/wiki/White_Line_Fever_\(film\) "White Line Fever \(film\)")_
  * _[Hoffa](https://en.wikipedia.org/wiki/Hoffa_\(film\) "Hoffa \(film\)")_

  
---|---  
Television| 

  * _[American Loggers](https://en.wikipedia.org/wiki/American_Loggers "American Loggers")_
  * _[American Trucker](https://en.wikipedia.org/wiki/American_Trucker "American Trucker")_
  * _[B. J. and the Bear](https://en.wikipedia.org/wiki/B._J._and_the_Bear "B. J. and the Bear")_
  * _[Ice Road Truckers](https://en.wikipedia.org/wiki/Ice_Road_Truckers "Ice Road Truckers")_
  * _[Movin' On](https://en.wikipedia.org/wiki/Movin%27_On_\(TV_series\) "Movin' On \(TV series\)")_
  * _[Trick My Truck](https://en.wikipedia.org/wiki/Trick_My_Truck "Trick My Truck")_

  
Music| 

  * [A Tombstone Every Mile](https://en.wikipedia.org/wiki/A_Tombstone_Every_Mile "A Tombstone Every Mile")
  * [Big Wheels in the Moonlight](https://en.wikipedia.org/wiki/Big_Wheels_in_the_Moonlight "Big Wheels in the Moonlight")
  * [Bonnie Jean (Little Sister)](https://en.wikipedia.org/wiki/Bonnie_Jean_\(Little_Sister\) "Bonnie Jean \(Little Sister\)")
  * [Convoy](https://en.wikipedia.org/wiki/Convoy_\(song\) "Convoy \(song\)")
  * [Drivin' My Life Away](https://en.wikipedia.org/wiki/Drivin%27_My_Life_Away "Drivin' My Life Away")
  * [East Bound and Down](https://en.wikipedia.org/wiki/East_Bound_and_Down "East Bound and Down")
  * [Eighteen Wheels and a Dozen Roses](https://en.wikipedia.org/wiki/Eighteen_Wheels_and_a_Dozen_Roses "Eighteen Wheels and a Dozen Roses")
  * [Giddyup Go](https://en.wikipedia.org/wiki/Giddyup_Go "Giddyup Go")
  * [Girl on the Billboard](https://en.wikipedia.org/wiki/Girl_on_the_Billboard "Girl on the Billboard")
  * [Movin' On](https://en.wikipedia.org/wiki/Movin%27_On_\(Merle_Haggard_song\) "Movin' On \(Merle Haggard song\)")
  * [Papa Loved Mama](https://en.wikipedia.org/wiki/Papa_Loved_Mama "Papa Loved Mama")
  * [Phantom 309](https://en.wikipedia.org/wiki/Phantom_309 "Phantom 309")
  * [Roll On Big Mama](https://en.wikipedia.org/wiki/Roll_On_Big_Mama "Roll On Big Mama")
  * [Roll On (Eighteen Wheeler)](https://en.wikipedia.org/wiki/Roll_On_\(Eighteen_Wheeler\) "Roll On \(Eighteen Wheeler\)")
  * [Six Days on the Road](https://en.wikipedia.org/wiki/Six_Days_on_the_Road "Six Days on the Road")
  * [Teddy Bear](https://en.wikipedia.org/wiki/Teddy_Bear_\(Red_Sovine_song\) "Teddy Bear \(Red Sovine song\)")
  * [The White Knight](https://en.wikipedia.org/wiki/The_White_Knight_\(Cledus_Maggard_song\) "The White Knight \(Cledus Maggard song\)")

  
Radio| 

  * [Dale Sommers](https://en.wikipedia.org/wiki/Dale_Sommers "Dale Sommers")
  * [Red Eye](https://en.wikipedia.org/wiki/Red_Eye_Radio "Red Eye Radio") ([Bill Mack](https://en.wikipedia.org/wiki/Bill_Mack_\(songwriter\) "Bill Mack \(songwriter\)"))
  * [Road Dog Trucking](https://en.wikipedia.org/wiki/Road_Dog_Trucking "Road Dog Trucking") ([Dave Nemo](https://en.wikipedia.org/wiki/Dave_Nemo "Dave Nemo"))

  
Video games| 

  * _[18 Wheeler: American Pro Trucker](https://en.wikipedia.org/wiki/18_Wheeler:_American_Pro_Trucker "18 Wheeler: American Pro Trucker")_
  * _[18 Wheels of Steel](https://en.wikipedia.org/wiki/18_Wheels_of_Steel "18 Wheels of Steel")_
  * _[American Truck Simulator](https://en.wikipedia.org/wiki/American_Truck_Simulator "American Truck Simulator")_
  * _[Rig 'n' Roll](https://en.wikipedia.org/wiki/Rig_%27n%27_Roll "Rig 'n' Roll")_
  * _[Big Rigs: Over the Road Racing](https://en.wikipedia.org/wiki/Big_Rigs:_Over_the_Road_Racing "Big Rigs: Over the Road Racing")_

  
Other| 

  * [Citizens band radio](https://en.wikipedia.org/wiki/Citizens_band_radio "Citizens band radio")
  * [The Rolling Memorial](https://en.wikipedia.org/wiki/The_Rolling_Memorial "The Rolling Memorial")

  
  
  * Note: Defunct companies are shown in _italics_

  
  
[Authority control databases](https://en.wikipedia.org/wiki/Help:Authority_control "Help:Authority control") [](https://www.wikidata.org/wiki/Q478214#identifiers "Edit this at Wikidata")  
---  
International| 

  * [ISNI](https://isni.org/isni/0000000449079987)
    * [2](https://isni.org/isni/0000000480590900)
  * [VIAF](https://viaf.org/viaf/128492563)
  * [FAST](https://id.worldcat.org/fast/1917360)

  
National| 

  * [Germany](https://d-nb.info/gnd/111610105X)
  * [United States](https://id.loc.gov/authorities/n2014027514)
  * [Czech Republic](https://aleph.nkp.cz/F/?func=find-c&local_base=aut&ccl_term=ica=kv20191038193&CON_LNG=ENG)
    * [2](https://aleph.nkp.cz/F/?func=find-c&local_base=aut&ccl_term=ica=kv20191038192&CON_LNG=ENG)
  * [Norway](https://authority.bibsys.no/authority/rest/authorities/html/12062327)
  * [Poland](https://dbn.bn.org.pl/descriptor-details/9810554047605606)
  * [Israel](https://www.nli.org.il/en/authorities/987007334027305171)

  
Other| 

  * [Yale LUX](https://lux.collections.yale.edu/view/group/f173f5db-151a-482c-af4e-2b2785e9e151)

  
  
[30°13′N 97°37′W﻿ / ﻿30.22°N 97.62°W﻿ / 30.22; -97.62](https://geohack.toolforge.org/geohack.php?pagename=Tesla,_Inc.&params=30.22_N_97.62_W_type:landmark_region:US-CA)
  *[v]: View this template
  *[t]: Discuss this template
  *[e]: Edit this template
  *[c.]: circa
  *[Ref.]: Reference(s)


In [26]:
os.getenv("SEC_USER")


'druce@streeteye.com'

In [32]:
# free API for edgar filing
import sec_parser as sp
from sec_downloader import Downloader

def fn_get_10k_item_from_symbol(symbol, item="1"):
    """
    Get item 1 (or other number) of the latest 10-K annual report filing for a given symbol.

    Args:
        symbol (str): The symbol of the equity.
        item (str):   The item number to return.

    Returns:
        str: The item requested from the latest 10-K annual report filing, or None if not found.

    """

    item_text = ""
    try:
        print("Getting 10-K Item 1 for %s" % symbol)
        dl = Downloader(os.getenv("SEC_FIRM"), os.getenv("SEC_USER"))
        html = dl.get_filing_html(ticker=symbol, form="10-K")
        print("HTML length: %d characters" % len(html))
        elements = sp.Edgar10QParser().parse(html)
        tree = sp.TreeBuilder().build(elements)
        # look for e.g. "Item 1."
        # sections = [n for n in tree.nodes if re.match(r"^ITEM 1[A-Z]?\.", n.text.strip().upper())]
        sections = [n for n in tree.nodes if re.match(
            r"^ITEM\s+" + item, n.text.strip().upper())]
        print("Sections: %d" % len(sections))
        if len(sections) == 0:
            return ""
        item_node = sections[0]
        item_text = item_node.text + "\n\n" + \
            "\n".join([n.text for n in item_node.get_descendants()])
        print("Item text: %d characters" % len(item_text))
    except Exception as e:
        print("Error getting 10-K item: %s" % e)
    return item_text

edgar_10k_item1 = fn_get_10k_item_from_symbol(symbol)
with open(f'{temp_dir}/{symbol}_edgar_10k_item1.txt', 'w', encoding='utf-8') as f:
    f.write(edgar_10k_item1)
print(edgar_10k_item1)


Getting 10-K Item 1 for TSLA
HTML length: 2596459 characters
Sections: 11
Item text: 134255 characters
ITEM 1. BUSINESS

Overview
We design, develop, manufacture, sell and lease high-performance fully electric vehicles and energy generation and storage systems, and offer services related to our products. We generally sell our products directly to customers, and continue to grow our customer-facing infrastructure through a global network of vehicle showrooms and service centers, Mobile Service, body shops, Supercharger stations and Destination Chargers to accelerate the widespread adoption of our products. We emphasize performance, attractive styling and the safety of our users and workforce in the design and manufacture of our products and are continuing to develop full self-driving technology for improved safety. We also strive to lower the cost of ownership for our customers through continuous efforts to reduce manufacturing costs and by offering financial and other services tailored

/opt/anaconda3/envs/mcp/lib/python3.11/site-packages/sec_parser/processing_engine/core.py:153: UserWarning: Invalid section type for part1item1a. Defaulting to InvalidTopSectionIn10Q.
  elements = step.process(elements)
/opt/anaconda3/envs/mcp/lib/python3.11/site-packages/sec_parser/processing_engine/core.py:153: UserWarning: Invalid section type for part2item7. Defaulting to InvalidTopSectionIn10Q.
  elements = step.process(elements)
/opt/anaconda3/envs/mcp/lib/python3.11/site-packages/sec_parser/processing_engine/core.py:153: UserWarning: Invalid section type for part2item7a. Defaulting to InvalidTopSectionIn10Q.
  elements = step.process(elements)
/opt/anaconda3/envs/mcp/lib/python3.11/site-packages/sec_parser/processing_engine/core.py:153: UserWarning: Invalid section type for part2item8. Defaulting to InvalidTopSectionIn10Q.
  elements = step.process(elements)
/opt/anaconda3/envs/mcp/lib/python3.11/site-packages/sec_parser/processing_engine/core.py:153: UserWarning: Invalid sectio

In [5]:

obb.account.login(email=os.environ['OPENBB_USER'], password=os.environ['OPENBB_PW'], remember_me=True)


In [6]:
obj = obb.equity.compare.peers(symbol=symbol, provider='fmp')
peers = obj.results
peers


ImportError: cannot import name 'OBBject_EquityInfo' from 'openbb_core.app.provider_interface' (/opt/anaconda3/envs/mcp/lib/python3.11/site-packages/openbb_core/app/provider_interface.py)

In [ ]:
for peer in peers.peers_list:
    try:
        retval = []
        obj = obb.equity.profile(peer)
        results = obj.results[0]
        desc = ""
        if results.short_description:
            desc = results.short_description
        elif results.long_description:
            desc = results.long_description

        retstr = f"""
Symbol:        {peer}
Name:          {results.name}
Country:       {results.hq_country}
Industry:      {results.industry_category}
Description:   {desc}
"""
#         retval.append(results.stock_exchange)
        print(retstr)
    except Exception as e:
#         print(e)
#         print()
        continue
    print()


In [ ]:
obj.results


In [ ]:
obj = obb.equity.fundamental.filings(symbol, form_type='10-K')
r = obj.results[0]
latest_10k_url = r.report_url
latest_10k_url


In [ ]:
client = MemoryClient()

messages = [
    {"role": "user", "content": "What is Item 1 from the latest annual report from Tesla (symbol TSLA)"},
    {"role": "assistant", "content": result['content']},
]
client.add(messages, user_id="tearsheet-TSLA", output_format="v1.1")


In [ ]:
messages = [
    {"role": "user", "content": "Thinking of making a sandwich. What do you recommend?"},
    {"role": "assistant", "content": "How about adding some cheese for extra flavor?"},
    {"role": "user", "content": "Actually, I don't like cheese."},
    {"role": "assistant", "content": "I'll remember that you don't like cheese for future recommendations."}
]
client.add(messages, user_id="alex")


In [ ]:
# Example showing location and preference-aware recommendations
query = "I'm craving some pizza. Any recommendations?"
filters = {
    "AND": [
        {
            "user_id": "alex"
        }
    ]
}
client.search(query, version="v2", filters=filters)

In [13]:
import os
import pandas as pd
import numpy as np
import aiohttp
import talib
from datetime import datetime, timedelta
from typing import Dict, Any


class MarketData:
    """Handles all market data fetching operations."""

    def __init__(self):
        self.api_key = os.getenv("TIINGO_API_KEY")
        if not self.api_key:
            raise ValueError("TIINGO_API_KEY not found in environment")

        self.headers = {"Content-Type": "application/json", "Authorization": f"Token {self.api_key}"}

    async def get_historical_data(self, symbol: str, lookback_days: int = 365) -> pd.DataFrame:
        """
        Fetch historical daily data for a given symbol.

        Args:
            symbol (str): The stock symbol to fetch data for.
            lookback_days (int): Number of days to look back from today.

        Returns:
            pd.DataFrame: DataFrame containing historical market data.

        Raises:
            ValueError: If the symbol is invalid or no data is returned.
            Exception: For other unexpected issues during the fetch operation.
        """
        end_date = datetime.now()
        start_date = end_date - timedelta(days=lookback_days)

        url = (
            f"https://api.tiingo.com/tiingo/daily/{symbol}/prices?"
            f'startDate={start_date.strftime("%Y-%m-%d")}&'
            f'endDate={end_date.strftime("%Y-%m-%d")}'
        )

        try:
            async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=10)) as session:
                async with session.get(url, headers=self.headers) as response:
                    if response.status == 404:
                        raise ValueError(f"Symbol not found: {symbol}")
                    response.raise_for_status()
                    data = await response.json()

            if not data:
                raise ValueError(f"No data returned for {symbol}")

            df = pd.DataFrame(data)
            df["date"] = pd.to_datetime(df["date"])
            df.set_index("date", inplace=True)

            df[["open", "high", "low", "close"]] = df[["adjOpen", "adjHigh", "adjLow", "adjClose"]].round(2)
            df["volume"] = df["adjVolume"].astype(int)
            df["symbol"] = symbol.upper()

            return df

        except aiohttp.ClientError as e:
            raise ConnectionError(f"Network error while fetching data for {symbol}: {e}")
        except ValueError as ve:
            raise ve  # Propagate value errors (symbol issues, no data, etc.)
        except Exception as e:
            raise Exception(f"Unexpected error fetching data for {symbol}: {e}")


class TechnicalAnalysis:
    """Technical analysis toolkit using TA-Lib for improved performance."""

    @staticmethod
    def add_core_indicators(df: pd.DataFrame) -> pd.DataFrame:
        """Add a core set of technical indicators using TA-Lib."""
        try:
            # Convert to numpy arrays for TA-Lib (required format)
            high = df["high"].values
            low = df["low"].values
            close = df["close"].values
            volume = df["volume"].values

            # Adding trend indicators (Simple Moving Averages)
            df["sma_20"] = talib.SMA(close, timeperiod=20)
            df["sma_50"] = talib.SMA(close, timeperiod=50)
            df["sma_200"] = talib.SMA(close, timeperiod=200)

            # Adding volatility indicators
            df["atr"] = talib.ATR(high, low, close, timeperiod=14)

            # Calculate Average Daily Range Percentage manually
            daily_range = df["high"] - df["low"]
            adr = daily_range.rolling(window=20).mean()
            df["adrp"] = (adr / df["close"]) * 100

            # Average volume (20-day)
            df["avg_20d_vol"] = df["volume"].rolling(window=20).mean()

            # Adding momentum indicators
            df["rsi"] = talib.RSI(close, timeperiod=14)

            # MACD indicator
            macd, macd_signal, macd_hist = talib.MACD(close, fastperiod=12, slowperiod=26, signalperiod=9)
            df["macd"] = macd
            df["macd_signal"] = macd_signal
            df["macd_histogram"] = macd_hist

            return df

        except KeyError as e:
            raise KeyError(f"Missing column in input DataFrame: {str(e)}")
        except Exception as e:
            raise Exception(f"Error calculating indicators: {str(e)}")

    @staticmethod
    def check_trend_status(df: pd.DataFrame) -> Dict[str, Any]:
        """Analyze the current trend status."""
        if df.empty:
            raise ValueError("DataFrame is empty. Ensure it contains valid data.")

        latest = df.iloc[-1]

        # Handle potential NaN values
        macd_bullish = False
        if not pd.isna(latest["macd"]) and not pd.isna(latest["macd_signal"]):
            macd_bullish = latest["macd"] > latest["macd_signal"]

        return {
            "above_20sma": latest["close"] > latest["sma_20"] if not pd.isna(latest["sma_20"]) else False,
            "above_50sma": latest["close"] > latest["sma_50"] if not pd.isna(latest["sma_50"]) else False,
            "above_200sma": latest["close"] > latest["sma_200"] if not pd.isna(latest["sma_200"]) else False,
            "20_50_bullish": latest["sma_20"] > latest["sma_50"] if not pd.isna(latest["sma_20"]) and not pd.isna(latest["sma_50"]) else False,
            "50_200_bullish": latest["sma_50"] > latest["sma_200"] if not pd.isna(latest["sma_50"]) and not pd.isna(latest["sma_200"]) else False,
            "rsi": latest["rsi"] if not pd.isna(latest["rsi"]) else 0,
            "macd_bullish": macd_bullish,
        }


# Usage example
async def analyze_symbol(symbol: str):
    """Complete analysis workflow for a given symbol."""
    market_data = MarketData()
    tech_analysis = TechnicalAnalysis()

    df = await market_data.get_historical_data(symbol)
    df = tech_analysis.add_core_indicators(df)
    trend = tech_analysis.check_trend_status(df)

    analysis = f"""
Technical Analysis for {symbol}:

Trend Analysis:
- Above 20 SMA: {'✅' if trend['above_20sma'] else '❌'}
- Above 50 SMA: {'✅' if trend['above_50sma'] else '❌'}
- Above 200 SMA: {'✅' if trend['above_200sma'] else '❌'}
- 20/50 SMA Bullish Cross: {'✅' if trend['20_50_bullish'] else '❌'}
- 50/200 SMA Bullish Cross: {'✅' if trend['50_200_bullish'] else '❌'}

Momentum:
- RSI (14): {trend['rsi']:.2f}
- MACD Bullish: {'✅' if trend['macd_bullish'] else '❌'}

Latest Price: ${df['close'].iloc[-1]:.2f}
Average True Range (14): {df['atr'].iloc[-1]:.2f}
Average Daily Range Percentage: {df['adrp'].iloc[-1]:.2f}%
Average Volume (20D): {df['avg_20d_vol'].iloc[-1]:,.0f}
"""
#     print(analysis)
    return df, trend, analysis

# Example usage:
# df, trend = await analyze_symbol("AAPL")

In [21]:
temp_dir = "tmp/tx0ybr2zo" 


In [22]:
df, trend, analysis_str = await analyze_symbol(symbol)
with open(f'{temp_dir}/{symbol}_technicals.md', 'w', encoding='utf-8') as f:
    f.write(analysis_str)
print(analysis_str)


Technical Analysis for TSLA:

Trend Analysis:
- Above 20 SMA: ❌
- Above 50 SMA: ❌
- Above 200 SMA: ❌
- 20/50 SMA Bullish Cross: ❌
- 50/200 SMA Bullish Cross: ✅

Momentum:
- RSI (14): 42.92
- MACD Bullish: ❌

Latest Price: $302.63
Average True Range (14): 14.11
Average Daily Range Percentage: 3.49%
Average Volume (20D): 96,080,182



In [ ]:
from openai import OpenAI
from mem0 import Memory

openai_client = OpenAI()
memory = Memory()

def chat_with_memories(message: str, user_id: str = "default_user") -> str:
    # Retrieve relevant memories
    relevant_memories = memory.search(query=message, user_id=user_id, limit=3)
    memories_str = "\n".join(f"- {entry['memory']}" for entry in relevant_memories["results"])

    # Generate Assistant response
    system_prompt = f"You are a helpful AI. Answer the question based on query and memories.\nUser Memories:\n{memories_str}"
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": message}]
    response = openai_client.chat.completions.create(model="gpt-4o-mini", messages=messages)
    assistant_response = response.choices[0].message.content

    # Create new memories from the conversation
    messages.append({"role": "assistant", "content": assistant_response})
    memory.add(messages, user_id=user_id)

    return assistant_response

def main():
    print("Chat with AI (type 'exit' to quit)")
    while True:
        user_input = input("You: ").strip()
        if user_input.lower() == 'exit':
            print("Goodbye!")
            break
        print(f"AI: {chat_with_memories(user_input)}")

main()

In [ ]:
filters = {
   "AND": [
      {
         "user_id": "alex"
      }
   ]
}

all_memories = client.get_all(version="v2", filters=filters, page=1, page_size=50)

all_memories


In [ ]:

def extract(url):
    extractor = SECItem1Extractor(url)

    # Analyze document structure first
    print("Document Structure Analysis:")
    structure = extractor.get_document_structure()

    print(f"\nFound {len(structure['anchors'])} anchors:")
    for anchor in structure['anchors'][:10]:  # Show first 10
        print(f"  - {anchor['name']}: {anchor['text']}")

    print(f"\nFound {len(structure['headings'])} item-related headings:")
    for heading in structure['headings']:
        print(f"  - {heading['tag']}: {heading['text']}")

    print(f"\nFound {len(structure['toc_links'])} TOC links:")
    for link in structure['toc_links']:
        print(f"  - {link['text']} -> {link['href']}")

    # Extract Item 1
    print("\n" + "="*50)
    print("EXTRACTING ITEM 1")
    print("="*50)

    result = extractor.extract_item1()
    if result:
        print(f"Successfully extracted using method: {result['method']}")
        print(f"Word count: {result['word_count']}")
        print(f"First 500 characters:\n{result['content'][:500]}...")
    else:
        print("Failed to extract Item 1 content")
    return result


result = extract(latest_10k_url)
print(result['content'])


In [ ]:
print(result['content'])


In [28]:
# AlphaVantage overview
url = f"https://www.alphavantage.co/query?function=OVERVIEW&symbol={symbol}&apikey={os.environ['ALPHAVANTAGE_API_KEY']}"
r = requests.get(url)
data = r.json()
with open(f'{temp_dir}/{symbol}_fundamintals_av.json', 'w', encoding='utf-8') as f:
    f.write(json.dumps(data))
pd.DataFrame(data, index=[1]).transpose()


,1
Symbol,TSLA
AssetType,Common Stock
Name,Tesla Inc
Description,"Tesla, Inc. is an American electric vehicle an..."
CIK,1318605
Exchange,NASDAQ
Currency,USD
Country,USA
Sector,MANUFACTURING
Industry,MOTOR VEHICLES & PASSENGER CAR BODIES


'{"Symbol": "TSLA", "AssetType": "Common Stock", "Name": "Tesla Inc", "Description": "Tesla, Inc. is an American electric vehicle and clean energy company based in Palo Alto, California. Tesla\'s current products include electric cars, battery energy storage from home to grid-scale, solar panels and solar roof tiles, as well as other related products and services. In 2020, Tesla had the highest sales in the plug-in and battery electric passenger car segments, capturing 16% of the plug-in market (which includes plug-in hybrids) and 23% of the battery-electric (purely electric) market. Through its subsidiary Tesla Energy, the company develops and is a major installer of solar photovoltaic energy generation systems in the United States. Tesla Energy is also one of the largest global suppliers of battery energy storage systems, with 3 GWh of battery storage supplied in 2020.", "CIK": "1318605", "Exchange": "NASDAQ", "Currency": "USD", "Country": "USA", "Sector": "MANUFACTURING", "Industry"

In [ ]:
exchange = data['Exchange']

morningstarmap = {'NYSE': 'xnys',
                  'NASDAQ': 'xnas'
                 }


In [30]:
now = datetime.now()
end_date = now.strftime('%Y%m%dT%H%M')
sdate = now - timedelta(days=2)
# sdate = datetime(now.year, now.month-1, now.day)
start_date = sdate.strftime('%Y%m%dT%H%M')
start_date, end_date

('20250730T2005', '20250801T2005')

In [31]:
url = f'https://www.alphavantage.co/query?function=NEWS_SENTIMENT&tickers={symbol}&apikey={os.environ["ALPHAVANTAGE_API_KEY"]}&time_from={start_date}&time_to={end_date}'

r = requests.get(url)
data = r.json()

for item in data['feed']:
    markdown_str = ""
    date_object = datetime.strptime(item['time_published'], "%Y%m%dT%H%M%S")
    display_title = item['title'].replace("$", r"\$")  # so Markdown doesn't interpret as latex escape
    description = item['summary'].replace("$", r"\$")
    markdown_str += f"[{str(date_object)} {display_title}]({item['url']})\n {description}"
    display(Markdown(markdown_str))


[2025-08-01 19:37:00 SJM BREAKING INVESTIGATION: BFA Law Announces an Investigation into The J.M. Smucker Co. after Significant Impairment Charges - Contact BFA Law if You Lost Money - JM Smucker  ( NYSE:SJM ) ](https://www.benzinga.com/pressreleases/25/08/g46808389/sjm-breaking-investigation-bfa-law-announces-an-investigation-into-the-j-m-smucker-co-after-signif)
 NEW YORK, Aug. 01, 2025 ( GLOBE NEWSWIRE ) -- Leading securities law firm Bleichmar Fonti & Auld LLP announces an investigation into The J.M. Smucker Company SJM for potential violations of the federal securities laws. If you invested in J.M.

[2025-08-01 18:35:34 Tesla must pay \$329 million in damages after fatal Autopilot crash, jury says](https://www.cnbc.com/2025/08/01/tesla-must-pay-329-million-in-damages-in-fatal-autopilot-case.html)
 A jury in Florida determined that Tesla should be held partly liable for a fatal 2019 crash involving its Autopilot technology.

[2025-08-01 18:24:00 Buy, Sell or Hold Navitas Stock? Key Tips Ahead of Q2 Earnings](https://www.zacks.com/stock/news/2650562/buy-sell-or-hold-navitas-stock-key-tips-ahead-of-q2-earnings)
 NVTS' second-quarter 2025 performance is likely to suffer from muted revenue growth and margin pressure.

[2025-08-01 17:19:02 What's Going On With Tesla Stock Today? - Tesla  ( NASDAQ:TSLA ) ](https://www.benzinga.com/trading-ideas/movers/25/08/46803787/whats-going-on-with-tesla-stock-today-5)
 Tesla stock, after initially trading lower on Friday, has recovered to trade flat midday. The early session dip was in line with broader market weakness following a surprisingly soft U.S. jobs report. The market's back, and these 3 income stocks are thriving. See them here→

[2025-08-01 17:07:08 Tech Stocks Trace A Negative Pattern; Trump Weaponizes Tariffs](https://www.benzinga.com/general/market-summary/25/08/46803424/tech-stocks-trace-a-negative-pattern-figmazation-of-momo-crowd-trump-weaponizes-tariffs)
 To gain an edge, this is what you need to know today. Please click here for an enlarged chart of Invesco QQQ Trust Series 1 ( QQQ ) . President Trump has imposed sweeping tariffs on countries that have not yet reached a deal. The most notable are high tariffs of 39% on Switzerland and 35% on ...

...

[2025-07-31 12:24:15 How To Trade SPY, Top Tech Stocks Using Technical Analysis](https://www.benzinga.com/markets/equities/25/07/46754137/how-to-trade-spy-top-tech-stocks-using-technical-analysis)
 Today's economic calendar launches with the June PCE Price Index at 8:30 AM ET, a key inflation metric favored by the Fed, released alongside the Initial and Continuing Jobless Claims, which will offer fresh perspectives on labor market conditions and consumer spending power amid ongoing ...

[2025-07-31 08:52:00 Is the Vanguard Growth ETF the Simplest Way to Consistently Beat the S&P 500?](https://www.fool.com/investing/2025/07/31/buy-vanguard-growth-etf-beat-sp-500/)
 Big tech stocks continue to pole-vault the Vanguard Growth ETF to new heights.

[2025-07-31 08:02:00 2 Monster Growth Stocks to Sell Before They Fall 56% and 64% in 2025, According to Wall Street Analysts](https://www.fool.com/investing/2025/07/31/2-growth-stocks-to-sell-before-fall-64-wall-street/)
 Some Wall Street analysts anticipate substantial losses for shareholders of Circle Internet Group and Tesla.

[2025-07-31 00:27:10 Samsung second-quarter profit more than halves, missing expectations weighed by slump in chip business](https://www.cnbc.com/2025/07/31/samsung-second-quarter-profit-halves-missing-expectations.html)
 Samsung Electronics reported a second-quarter operating profit that missed expectations and more than halved from the same period last year.

[2025-07-30 21:19:03 Arm Holdings Shares Fall After Q1 Revenue Miss, Soft Q2 Guidance - ARM Holdings  ( NASDAQ:ARM ) ](https://www.benzinga.com/markets/earnings/25/07/46740862/arm-holdings-shares-move-lower-following-q1-earnings-revenue-miss-eps-in-line-soft-q2-earnings-g)
 Arm reports first-quarter revenue of \$1.053 billion, missing analyst estimates of \$1.055 billion. Arm reports first-quarter adjusted earnings of 35 cents per share, in line with analyst estimates. The market's back, and these 3 income stocks are thriving. See them here→

In [32]:
NEWSAPI_API_KEY = os.environ['NEWSAPI_API_KEY']

page_size = 100
q = company
date_24h_ago = datetime.now() - timedelta(hours=24)
formatted_date = date_24h_ago.strftime("%Y-%m-%dT%H:%M:%S")
print(f"Fetching top {page_size} stories matching {q} since {formatted_date} from NewsAPI")

api = NewsApiClient(api_key=NEWSAPI_API_KEY)
# sources = api.get_sources()
# pd.DataFrame(sources['sources'])

baseurl = 'https://newsapi.org/v2/everything'

# Define search parameters
params = {
    'q': q,
    'from': formatted_date,
    'language': 'en',
    'sortBy': 'relevancy',
    'apiKey': NEWSAPI_API_KEY,
    'pageSize': 100
}

# Make API call with headers and params
response = requests.get(baseurl, params=params, timeout=60)
if response.status_code != 200:
    print('ERROR: API call failed.')
    print(response.text)
else:
    data = response.json()
    newsapi_df = pd.DataFrame(data['articles'])

    # Print the articles
    markdown_str = ""
    for row in newsapi_df.itertuples():
#         date_object = datetime.strptime(row.publishedAt, "%Y%m%dT%H%M%S")
        display_title = row.title.replace("$", r"\$")  # so Markdown doesn't interpret as latex escape
        markdown_str += f"[{row.publishedAt} {display_title}]({row.url}) \n\n"

    display(Markdown(markdown_str))


Fetching top 100 stories matching Tesla since 2025-07-31T20:05:26 from NewsAPI


[2025-07-31T20:35:10Z Tesla starts sort-of Robotaxi service in San Francisco by invite only](https://www.theregister.com/2025/07/31/tesla_robotaxi_san_francisco/) 

[2025-07-31T20:06:48Z CSX Is Said to Work With Goldman Sachs to Explore Options](https://biztoc.com/x/d3abff762c854ecc) 

[2025-07-31T20:06:59Z Pakistan to get first U.S. oil shipment as Cnergyico seals import deal](https://biztoc.com/x/759b1c3b7dd479b6) 

[2025-07-31T20:06:53Z For Trump’s Harvard Deal, \$500 Million Is Only a Starting Point](https://biztoc.com/x/79c0b1e3362e6a5a) 

[2025-07-31T20:06:33Z Stocks end mostly lower but book strong July gains](https://biztoc.com/x/5ded98817b331b98) 

[2025-07-31T20:06:36Z NYC Subway Havoc Returns as Heat Wave, Heavy Rains Beset City](https://biztoc.com/x/091bb8f825aaba6c) 

[2025-07-31T22:05:00Z Tesla vs. Bezos, Slate, Windrose, Lucid, and Paul ‘Muad’Dib’ Atreides](http://electrek.co/2025/07/31/tesla-vs-bezos-slate-windrose-lucid-and-paul-muaddib-atreides/) 

[2025-07-31T23:03:00Z LGES inks major LFP battery supply deal, Tesla tipped as mystery client](https://www.digitimes.com/news/a20250731PD226/lges-lfp-battery-energy-storage-tesla-2025.html) 

[2025-07-31T21:47:01Z Tesla Robotaxi pulls ahead of Waymo in San Francisco](https://biztoc.com/x/c8243e15ab78ad3a) 

[2025-07-31T20:33:24Z An electric Subaru BRZ? Don’t rule out an EV version just yet](http://electrek.co/2025/07/31/electric-subaru-brz-dont-rule-out-an-ev-version-yet/) 

[2025-08-01T00:01:48Z France new car registrations down 7.66% in July, Tesla sales drop 26.57%](https://biztoc.com/x/19d55a0d6c267b77) 

[2025-07-31T20:39:54Z Tesla starts sort-of Robotaxi service in San Francisco by invite only](https://biztoc.com/x/a497d53287bd45f5) 

[2025-07-31T23:39:12Z China EV war: Tesla, Nio and Li Auto target mainland families with premium SUVs](https://biztoc.com/x/119694eeebc8cbe6) 

[2025-07-31T20:40:15Z Exxon Mobil Reports Earnings Friday. What’s Holding the Stock Back](https://biztoc.com/x/7775ba676ab1beb6) 

[2025-07-31T21:13:46Z Apple Earnings Call Begins. Listen for Updates on Tariffs and Guidance](https://biztoc.com/x/64068e57bc9a5868) 

[2025-07-31T20:51:20Z Eversource's quarterly profit climbs on transmission, distribution strength](https://biztoc.com/x/0410fd71d1375c00) 

[2025-07-31T20:51:17Z Amazon cloud computing results fail to impress, shares drop after hours](https://biztoc.com/x/604a111646cabe6e) 

[2025-07-31T21:02:17Z First Solar Rises After Earnings. Management Sees Hope in Darkening Solar Industry](https://biztoc.com/x/7f7e2aebdd1dcafa) 

[2025-07-31T20:39:59Z Stocks Erase Early Gains as Chip Makers and Big Pharma Retreat](https://biztoc.com/x/a830ebd2163d862f) 

[2025-07-31T20:39:49Z Amazon cloud computing results fail to impress, shares drop after hour](https://biztoc.com/x/7744f7d434466ba4) 

[2025-07-31T21:02:14Z ‘Chamber of horrors’ being exhumed at Ireland mass baby grave](https://biztoc.com/x/92b09049d84fcba5) 

[2025-07-31T21:02:36Z Resmed beats quarterly profit estimates on strong demand for sleep devices](https://biztoc.com/x/aa1a206f948a48f5) 

[2025-07-31T21:02:14Z Coinbase Falls After Revenue Misses Estimates on Lower Volumes](https://biztoc.com/x/411371aaa76057a1) 

[2025-07-31T21:02:21Z First Solar raises annual sales outlook, expects higher prices due to tariffs](https://biztoc.com/x/fc22dcb23231fd0b) 

[2025-07-31T21:24:41Z The Secret Rise of China’s AI Desert Empire](https://biztoc.com/x/db961df4c05876be) 

[2025-07-31T21:24:23Z American Eagle's 'good jeans' ads with Sydney Sweeney spark a debate on race and beauty standards](https://biztoc.com/x/162fd42d0f682c7e) 

[2025-07-31T21:24:47Z Figma’s Blockbuster IPO Gives CEO Field a \$6 Billion Fortune](https://biztoc.com/x/582744cca7339be6) 

[2025-07-31T21:35:57Z UnitedHealth appoints Wayne DeVeydt as CFO in another management shake-up](https://biztoc.com/x/45a4a1a352980f70) 

[2025-07-31T21:35:47Z Canada Called Today, But the White House Hasn't Spoken With Them](https://biztoc.com/x/729ea362ce13992e) 

[2025-07-31T21:24:49Z Apple got a much-needed iPhone and China boost](https://biztoc.com/x/f830790e405f72ed) 

[2025-07-31T21:24:48Z The World's Biggest Iron Ore Windfall Is Fading for Australia](https://biztoc.com/x/e097baaf5c6216a0) 

[2025-07-31T21:24:23Z Amazon cloud computing results fail to impress, shares slide after hours](https://biztoc.com/x/450e83026f85c289) 

[2025-07-31T20:51:38Z Moderna to Report Earnings Amid Shrinking Cash Balance. Investors Want to See If It Can Stop the Bleeding](https://biztoc.com/x/cf65dd4babaedcf7) 

[2025-07-31T20:18:01Z Roku beats second-quarter revenue estimates](https://biztoc.com/x/8f11947922a68aa0) 

[2025-07-31T20:18:10Z U.S. Indexes Decreased Thursday; Align Technology Posted Biggest Loss](https://biztoc.com/x/f408f60dc35a56b7) 

[2025-07-31T20:07:11Z Paramount Profit Beats Estimates Ahead of Skydance Merger](https://biztoc.com/x/30c247c700da0557) 

[2025-07-31T20:07:10Z Anthropic Follows OpenAI, xAI in Seeking Funds From Middle East](https://biztoc.com/x/b6dc797d89de20a6) 

[2025-07-31T20:06:56Z What’s Productive About Bunnies Jumping on a Trampoline?](https://biztoc.com/x/8c61d158cc3828ac) 

[2025-07-31T20:17:32Z Nantucket officials accuse offshore wind developer of going into hiding since Trump's election](https://biztoc.com/x/d7d2f9372d57165a) 

[2025-07-31T20:17:57Z Reddit forecasts strong revenue on AI-driven ad strength, shares surge](https://biztoc.com/x/9ca07c60b0e83330) 

[2025-07-31T20:18:13Z When Crisis Strikes, Does Your Firm Have a Continuity Plan?](https://biztoc.com/x/5087f761e712bcf8) 

[2025-07-31T20:18:00Z Paramount Global beats earnings target ahead of sale to Skydance](https://biztoc.com/x/8c025d6aa21ecb1e) 

[2025-07-31T20:17:50Z Point72 Preps First Venture Fund for Clients, Focuses on Defense](https://biztoc.com/x/16770cac6a546764) 

[2025-07-31T20:29:04Z Coinbase quarterly profit jumps on subscription revenue boost](https://biztoc.com/x/0d1375cee87b9283) 

[2025-07-31T20:29:08Z Reddit Stock Surges Higher on Strong Advertising Projections](https://biztoc.com/x/17b83cf55c216ffd) 

[2025-07-31T21:13:12Z What stops the tech juggernaut?](https://biztoc.com/x/0dda7a98ee595695) 

[2025-07-31T21:47:12Z MicroStrategy just posted a huge profit, thanks to this accounting change](https://biztoc.com/x/9dbe56f2714cdbab) 

[2025-07-31T20:07:10Z Texas Local Officials Grilled Over Response to Deadly Floods](https://biztoc.com/x/6ad0393c60955c9e) 

[2025-07-31T20:07:09Z China's Humanoids Will Soon Be Playing the Hunger Games](https://biztoc.com/x/5428e5e0840daa8f) 

[2025-07-31T20:06:59Z Pakistan to get first U.S. oil shipment as Cnergyico seals import deal](https://biztoc.com/x/759b1c3b7dd479b6) 

[2025-07-31T20:06:40Z Brazilian companies react to Trump's tariffs with relief and doubt](https://biztoc.com/x/e53acbb4e68c8c5c) 

[2025-07-31T20:18:03Z Why It’s So Hard to Buy or Sell a House Right Now](https://biztoc.com/x/763e560dff79a3ad) 

[2025-07-31T20:18:10Z Clorox’s Outlook Hit as Software Switch Disrupts Retail Orders](https://biztoc.com/x/d4ca834b74c97d6b) 

[2025-07-31T20:17:59Z Lumen narrows loss, lifts free cash flow forecast on tax savings](https://biztoc.com/x/1c4f2fffebaff770) 

[2025-07-31T20:17:48Z NuScale vs. Oklo Stock: A Tale of Two Nuclear Futures and Why Investors May Be Getting Way Ahead of Themselves](https://biztoc.com/x/59b923b5e6c62c8d) 

[2025-07-31T20:29:09Z Amazon Projects Profit That Underwhelms on Expense of AI Race](https://biztoc.com/x/230713fbaf74fa98) 

[2025-07-31T20:29:04Z Trump again slams Fed chair Powell after rates hold](https://biztoc.com/x/7730461e15fe2be3) 

[2025-07-31T20:28:44Z Childhood Vaccination Rates Fall as School Exemptions Hit Record](https://biztoc.com/x/75a66c140269dd47) 

[2025-07-31T20:40:06Z CSX working with Goldman Sachs to explore strategic options, Bloomberg News reports](https://biztoc.com/x/185303c910921621) 

[2025-07-31T20:51:33Z US gasoline demand in May hits lowest seasonal level since 2020, EIA says](https://biztoc.com/x/9d01a08d97202da2) 

[2025-07-31T21:36:06Z Moderna to cut more than 800 jobs this year as it burns through cash](https://biztoc.com/x/06921c0b84ed9e85) 

[2025-07-31T21:35:45Z Trump revives the Presidential Fitness Test for American schoolchildren](https://biztoc.com/x/7f4bdd5c62050c87) 

[2025-07-31T21:35:37Z How Embraer came away unscathed from Trump's tariff blitz](https://biztoc.com/x/1041b00f25dadf73) 

[2025-07-31T21:13:50Z Trading Day: What stops the tech juggernaut?](https://biztoc.com/x/bbb89519c4f9daa4) 

[2025-07-31T21:13:37Z How's Apple managing the tariff situation? This number holds a clue](https://biztoc.com/x/043a6236073d25b8) 

[2025-07-31T21:47:11Z Southwest Airlines names Doug Brooks as board chairperson](https://biztoc.com/x/9755f850a759d88c) 

[2025-07-31T21:24:33Z American Leadership in the Digital Finance Revolution](https://biztoc.com/x/bfbac264f01806e1) 

[2025-07-31T21:24:46Z UnitedHealth names Wayne DeVeydt as CFO](https://biztoc.com/x/f9a71cad1792e584) 

[2025-07-31T21:24:29Z Trump Says Witkoff to Visit Russia, Expects to Impose Sanctions](https://biztoc.com/x/bb0b51cedcd1f46d) 

[2025-07-31T21:02:29Z How Trump Let \$1 Trillion Worth of Imports Escape His Tariff Hammer](https://biztoc.com/x/bc5cf5f77b3c47da) 

[2025-07-31T21:02:22Z Singapore beefs up defense clout while targeting diplomatic anchor role](https://biztoc.com/x/23394d2834748063) 

[2025-07-31T21:02:32Z Where Wine Lovers Should Travel Next](https://biztoc.com/x/3c4f9900b31566b3) 

[2025-07-31T21:13:21Z Trump Says He Made a Mistake Naming Powell Fed Chair](https://biztoc.com/x/50a982c7b2efb950) 

[2025-07-31T20:18:17Z Coinbase reports rise on quarterly profit subscription strength](https://biztoc.com/x/620ab96e9afcc1d4) 

[2025-07-31T20:17:54Z Amazon projects quarterly revenue above market estimates](https://biztoc.com/x/d33512534acbb1bc) 

[2025-07-31T20:18:02Z An Army Contract Fueled Red Cat Stock’s 911% Surge in 13 Months: Is RCAT Still a Buy?](https://biztoc.com/x/763471fc27ca0c89) 

[2025-07-31T20:18:02Z Reddit Earnings Beat Estimates As Revenue Soars](https://biztoc.com/x/2164ed5a6e5c6e21) 

[2025-07-31T20:17:59Z Trump Asks Bank CEOs to Pitch on Fannie, Freddie Stock Offerings](https://biztoc.com/x/7eab4bc7f82f92a4) 

[2025-07-31T21:02:36Z Crypto-Treasury Financings Are Building Up in a Fragile Market](https://biztoc.com/x/20d28238bd6ce5d3) 

[2025-07-31T21:02:27Z Coinbase quarterly profit falls on trading dip, shares fall](https://biztoc.com/x/75603ce008bacc43) 

[2025-07-31T21:02:46Z Stryker raises annual profit forecast on steady demand for surgical devices](https://biztoc.com/x/9913511e1a64eff9) 

[2025-07-31T20:40:06Z How This Morgan Stanley Advisor Helps the Ultrarich Plan Their Estates](https://biztoc.com/x/a7a58f821d03ef9f) 

[2025-07-31T20:39:54Z Importers Left Waiting for Details on Trump’s Tariff Rollout](https://biztoc.com/x/a0371a09b3a8b064) 

[2025-07-31T20:51:21Z Fukushima plant operator takes another big loss on fuel removal costs](https://biztoc.com/x/2a67223a0d1afce5) 

[2025-07-31T20:51:29Z Flash flood threat prompts emergency declarations in New York, New Jersey](https://biztoc.com/x/b3e7936488b3ddbd) 

[2025-07-31T20:39:54Z Apple's stock is rising after a huge revenue beat. Did tariffs help?](https://biztoc.com/x/4602cf81d402377d) 

[2025-07-31T20:39:56Z Apple Revenue Tops Estimates on Strength of iPhone, China Market](https://biztoc.com/x/a234338687b08ad7) 

[2025-07-31T20:39:54Z LPL Financial Misses Wall Street Estimates as Company Prepares to Close Commonwealth Acquisition](https://biztoc.com/x/349cbc81df1b7500) 

[2025-07-31T20:28:57Z Saylor’s Strategy Posts Massive Unrealized Gain During Quarter](https://biztoc.com/x/778a7d8969e12d13) 

[2025-07-31T20:28:49Z Roku beats second-quarter revenue estimates, shares rise](https://biztoc.com/x/54c5eaa6c50d49f8) 

[2025-07-31T20:28:57Z Live blog: Amazon extends streak of profit beats, but the stock pulls back](https://biztoc.com/x/3f4afca631bf2e0b) 

[2025-07-31T20:17:49Z Japan’s Dai-ichi Life Eyes M&A in Southeast Asia to Grow Abroad](https://biztoc.com/x/4834d429e04ee60b) 

[2025-07-31T21:13:25Z Ericsson in Talks to Invest in Intel Standalone Network Business](https://biztoc.com/x/8ae13048aa4a59f2) 

[2025-07-31T21:13:36Z A trophy hunter killed a lion in Zimbabwe that was part of a research project, sparking anger](https://biztoc.com/x/66ae64ab6f792c3b) 

[2025-07-31T21:13:52Z Trump Team Outlines Push for Rare Earths in Meeting With Executives](https://biztoc.com/x/43f0be9338f24a31) 

[2025-07-31T21:13:16Z Mettler-Toledo raises annual profit forecast on strong demand for lab instruments](https://biztoc.com/x/525206b32801eef0) 

[2025-07-31T21:24:52Z Saylor's Strategy swings to quarterly profit amid crypto goldrush](https://biztoc.com/x/3e2180898cfdf0f4) 

[2025-07-31T21:46:55Z Apple Talks About Search As AI Changes Internet Landscape](https://biztoc.com/x/df1f15206eec20ff) 



In [33]:
now = datetime.now()
end_date = now.strftime('%Y-%m-%d')
# start_date = datetime.now() - timedelta(days=30)
sdate = datetime(now.year, now.month-1, now.day)
start_date = sdate.strftime('%Y-%m-%d')
start_date, end_date


('2025-07-01', '2025-08-01')

In [34]:
API_KEY = os.environ['NEWSFILTER_API_KEY']
API_ENDPOINT = "https://api.newsfilter.io/search?token={}".format(API_KEY)

# Define the news search parameters
queryString = f"symbols:{symbol} AND publishedAt:[{start_date} TO {end_date}]"

payload = {
    "queryString": queryString,
    "from": 0,
    "size": 10
}

# Send the search query to the Search API
response = requests.post(API_ENDPOINT, json=payload)

# Read the response
articles = response.json()


# Read the response
articles = response.json()

for item in articles['articles']:
    markdown_str = ""
    date_array = item['publishedAt'].split("T")
    datestr = date_array[0] + "T" + date_array[1][:8]
#     print(item['publishedAt'])
#     print(date_array)
#     print(datestr)
    date_object = datetime.strptime(datestr, "%Y-%m-%dT%H:%M:%S")
    display_title = item['title'].replace("$", r"\$")  # so Markdown doesn't interpret as latex escape
    description = item['description'].replace("$", r"\$")

    markdown_str += f"[{str(date_object)} {display_title}]({item['url']})\n {description}"
    display(Markdown(markdown_str))

[2025-08-01 21:45:00 E-Cite Motors (VAPR) Outpaces EV Industry in Stock Performance This Week and YTD](https://newsfilter.io/articles/e-cite-motors-vapr-outpaces-ev-industry-in-stock-performance-this-week-and-ytd-007eaa386049fa115078463e01caa5d7)
 E-Cite Motors Outperforms Tesla , Rivian , and Lucid Motors in Weekly and Year-to-Date Stock Gains Amid Breakthrough Technologies and Strategic U.S. Expansion BOTHELL, WA / ACCESS Newswire /

[2025-08-01 21:03:15 Tesla ordered by Florida jury to pay \$243 million in fatal Autopilot crash](https://newsfilter.io/articles/tesla-ordered-by-florida-jury-to-pay-243-million-in-fatal-autopilot-crash-ce27f60a1b1af713c254fc66c507d48f)
 A Florida jury on Friday found Tesla liable in the 2019 fatal crash of an Autopilot-equipped Model S and ordered Elon Musk's automaker to pay \$243 million to the family of a deceased woman and an injured survivor.

[2025-08-01 19:29:10 Tesla ordered by Florida jury to pay \$329 million in Autopilot crash](https://newsfilter.io/articles/tesla-ordered-by-florida-jury-to-pay-329-million-in-autopilot-crash-952f7139257075ac297ddc84c597b0b6)
 A Florida jury on Friday found Tesla liable in the 2019 fatal crash of an Autopilot-equipped Model S, and ordered Elon Musk's automaker to pay \$329 million to the family of a deceased woman and an injured survivor.

[2025-08-01 18:29:51 Tesla found partially liable for fatal 2019 crash, Washington Post reports](https://newsfilter.io/articles/tesla-found-partially-liable-for-fatal-2019-crash-washington-post-reports-c6f6c76b6ce69cef393b90ef8c0fa42a)
 A jury found Tesla partially liable for a fatal 2019 crash involving driver-assistance technology in Key Largo, Florida, and slapped the company with \$200 million in punitive damages, the Washington Post reported on Friday.

[2025-08-01 18:20:53 Tesla must pay \$329 million in damages after fatal Autopilot crash, jury says](https://newsfilter.io/articles/tesla-must-pay-329-million-in-damages-after-fatal-autopilot-crash-jury-says-23fff4df2ffde47eb06b8b33361854c2)
 A jury in Florida determined that Tesla should be held partly liable for a fatal 2019 crash involving its Autopilot technology. 

[2025-08-01 16:42:37 Tesla sales drop again around Europe despite Model Y revamp](https://newsfilter.io/articles/tesla-sales-drop-again-around-europe-despite-model-y-revamp-9eef9611bfc8016d0d96033e0130c051)
 Registrations of new Tesla cars in several key European markets fell in July, despite a revamp to its signature Model Y, as the EV maker struggles with a backlash against CEO Elon Musk's political views, regulatory challenges and rising competition.

[2025-08-01 16:21:58 Wall St selloff sparked by tariff onslaught, weak payrolls](https://newsfilter.io/articles/wall-st-selloff-sparked-by-tariff-onslaught-weak-payrolls-21544ae8098429bb69a3de785c956fcb)
 Wall Street's main indexes led a global selloff on Friday as new U.S. tariffs on dozens of trading partners weighed on sentiment, while a weaker-than-expected payrolls report added to risk aversion.

[2025-08-01 14:08:23 Apple's tariff-fueled iPhone sales surge raises doubts about sustainability](https://newsfilter.io/articles/apples-tariff-fueled-iphone-sales-surge-raises-doubts-about-sustainability-edd8b1952548298a4edf1d83c596928d)
 Apple's best revenue growth in three years failed to impress, with shares down about 1% in volatile trading on Friday, as investors questioned whether a tariff-driven surge in iPhone sales would last.

[2025-08-01 13:45:52 BYD's July production falls for first time in 17 months as expansion spree slows](https://newsfilter.io/articles/byds-july-production-falls-for-first-time-in-17-months-as-expansion-spree-slows-579a2a172f22a3bf8e5403135a81caf2)
 BYD's vehicle production fell 0.9% in July from a year earlier, ending a 16-month growth streak that has catapulted the Chinese automaker into the world's largest electric vehicle maker.

[2025-08-01 11:43:12 BYD's July production falls 0.9% y/y as expansion spree slows](https://newsfilter.io/articles/byds-july-production-falls-09-yy-as-expansion-spree-slows-ed82e8fcc311b7d04d7b1a15219ce476)
 BYD's vehicle production fell 0.9% in July from a year earlier, ending a 16-month growth streak that has catapulted the Chinese automaker into the world's largest electric vehicle maker.

In [ ]:
# https://sethhobson.com/2025/01/building-a-stock-analysis-server-with-mcp-part-1/

class MarketData:
    """Handles all market data fetching operations."""

    def __init__(self):
        self.api_key = os.getenv("TIINGO_API_KEY")
        if not self.api_key:
            raise ValueError("TIINGO_API_KEY not found in environment")

        self.headers = {"Content-Type": "application/json", "Authorization": f"Token {self.api_key}"}

    async def get_historical_data(self, symbol: str, lookback_days: int = 365) -> pd.DataFrame:
        """
        Fetch historical daily data for a given symbol.

        Args:
            symbol (str): The stock symbol to fetch data for.
            lookback_days (int): Number of days to look back from today.

        Returns:
            pd.DataFrame: DataFrame containing historical market data.

        Raises:
            ValueError: If the symbol is invalid or no data is returned.
            Exception: For other unexpected issues during the fetch operation.
        """
        end_date = datetime.now()
        start_date = end_date - timedelta(days=lookback_days)

        url = (
            f"https://api.tiingo.com/tiingo/daily/{symbol}/prices?"
            f'startDate={start_date.strftime("%Y-%m-%d")}&'
            f'endDate={end_date.strftime("%Y-%m-%d")}'
        )

        try:
            async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=10)) as session:
                async with session.get(url, headers=self.headers) as response:
                    if response.status == 404:
                        raise ValueError(f"Symbol not found: {symbol}")
                    response.raise_for_status()
                    data = await response.json()

            if not data:
                raise ValueError(f"No data returned for {symbol}")

            df = pd.DataFrame(data)
            df["date"] = pd.to_datetime(df["date"])
            df.set_index("date", inplace=True)

            df[["open", "high", "low", "close"]] = df[["adjOpen", "adjHigh", "adjLow", "adjClose"]].round(2)
            df["volume"] = df["adjVolume"].astype(int)
            df["symbol"] = symbol.upper()

            return df

        except aiohttp.ClientError as e:
            raise ConnectionError(f"Network error while fetching data for {symbol}: {e}")
        except ValueError as ve:
            raise ve  # Propagate value errors (symbol issues, no data, etc.)
        except Exception as e:
            raise Exception(f"Unexpected error fetching data for {symbol}: {e}")


In [ ]:
class TechnicalAnalysis:
    """Technical analysis toolkit with improved performance and readability."""

    @staticmethod
    def add_core_indicators(df: pd.DataFrame) -> pd.DataFrame:
        """Add a core set of technical indicators."""
        try:
            # Adding trend indicators
            df["sma_20"] = ta.sma(df["close"], length=20)
            df["sma_50"] = ta.sma(df["close"], length=50)
            df["sma_200"] = ta.sma(df["close"], length=200)

            # Adding volatility indicators and volume
            daily_range = df["high"].sub(df["low"])
            adr = daily_range.rolling(window=20).mean()
            df["adrp"] = adr.div(df["close"]).mul(100)
            df["avg_20d_vol"] = df["volume"].rolling(window=20).mean()

            # Adding momentum indicators
            df["atr"] = ta.atr(df["high"], df["low"], df["close"], length=14)
            df["rsi"] = ta.rsi(df["close"], length=14)
            macd = ta.macd(df["close"], fast=12, slow=26, signal=9)
            if macd is not None:
                df = pd.concat([df, macd], axis=1)

            return df

        except KeyError as e:
            raise KeyError(f"Missing column in input DataFrame: {str(e)}")
        except Exception as e:
            raise Exception(f"Error calculating indicators: {str(e)}")

    @staticmethod
    def check_trend_status(df: pd.DataFrame) -> Dict[str, Any]:
        """Analyze the current trend status."""
        if df.empty:
            raise ValueError("DataFrame is empty. Ensure it contains valid data.")

        latest = df.iloc[-1]
        return {
            "above_20sma": latest["close"] > latest["sma_20"],
            "above_50sma": latest["close"] > latest["sma_50"],
            "above_200sma": latest["close"] > latest["sma_200"],
            "20_50_bullish": latest["sma_20"] > latest["sma_50"],
            "50_200_bullish": latest["sma_50"] > latest["sma_200"],
            "rsi": latest["rsi"],
            "macd_bullish": latest.get("MACD_12_26_9", 0) > latest.get("MACDs_12_26_9", 0),
        }

In [ ]:
async def fetch_technical_analysis(symbol):

    market_data = MarketData()
    tech_analysis = TechnicalAnalysis()
    tadf = await market_data.get_historical_data(symbol)
    tadf = tech_analysis.add_core_indicators(tadf)
    trend = tech_analysis.check_trend_status(df)
    analysis = f"""
    Technical Analysis for {symbol}:

    Trend Analysis:
    - Above 20 SMA: {'✅' if trend['above_20sma'] else '❌'}
    - Above 50 SMA: {'✅' if trend['above_50sma'] else '❌'}
    - Above 200 SMA: {'✅' if trend['above_200sma'] else '❌'}
    - 20/50 SMA Bullish Cross: {'✅' if trend['20_50_bullish'] else '❌'}
    - 50/200 SMA Bullish Cross: {'✅' if trend['50_200_bullish'] else '❌'}

    Momentum:
    - RSI (14): {trend['rsi']:.2f}
    - MACD Bullish: {'✅' if trend['macd_bullish'] else '❌'}

    Latest Price: ${df['close'].iloc[-1]:.2f}
    Average True Range (14): {df['atr'].iloc[-1]:.2f}
    Average Daily Range Percentage: {df['adrp'].iloc[-1]:.2f}%
    Average Volume (20D): {df['avg_20d_vol'].iloc[-1]}
    """
    return analysis

print(await fetch_technical_analysis('TSLA'))



In [ ]:

print(analysis)


In [35]:
# open pages via selenium and firefox
outputdir = "htmldata"

# Print the formatted time
print(datetime.now().strftime('%H:%M:%S'), "Starting", flush=True)

firefox_app_path = '/Applications/Firefox.app'
# Path to your geckodriver
geckodriver_path = '/Users/drucev/webdrivers/geckodriver'
# Set up Firefox options to use your existing profile
# important for some sites that need a login, also a generic profile fingerprint that looks like a bot might get blocked
firefox_profile_path = '/Users/drucev/Library/Application Support/Firefox/Profiles/x6dmpnoo.default-release-1'
options = Options()
options.profile = firefox_profile_path

print(datetime.now().strftime('%H:%M:%S'), "Initialized profile", flush=True)

# Create a Service object with the path
service = Service(geckodriver_path)

print(datetime.now().strftime('%H:%M:%S'), "Initialized service", flush=True)
# Set up the Firefox driver
driver = webdriver.Firefox(service=service, options=options)

print(datetime.now().strftime('%H:%M:%S'), "Initialized webdriver", flush=True)
sleeptime = 10


20:07:03 Starting
20:07:04 Initialized profile
20:07:04 Initialized service
20:07:07 Initialized webdriver


In [36]:
# Open a new tab
def open_tab(driver, url):
    driver.execute_script(f"window.open('{url}');")
    # Switch to last one opened - sometimes hangs?
    # driver.switch_to.window(driver.window_handles[-1])
    # print(url)
    # time.sleep(sleeptime)


In [37]:
url = f"https://stockcharts.com/h-sc/ui?s={symbol}&id=p33407302522&def=Y&listNum=1#"
source = "Stockcharts"
# Open the page
open_tab(driver, url)

# Wait for the page to load
# time.sleep(sleeptime)
display(Markdown(f"[{source}]({url})"))

[Stockcharts](https://stockcharts.com/h-sc/ui?s=TSLA&id=p33407302522&def=Y&listNum=1#)

In [38]:
url = f'https://www.bloomberg.com/search?query={company}'
source = "Bloomberg"
open_tab(driver, url)
display(Markdown(f"[{source}]({url})"))

[Bloomberg](https://www.bloomberg.com/search?query=Tesla)

In [39]:
url = f'https://www.reuters.com/site-search/?query={company}&offset=0'
source = "Reuters"
open_tab(driver, url)
display(Markdown(f"[{source}]({url})"))

[Reuters](https://www.reuters.com/site-search/?query=Tesla&offset=0)

In [40]:
url = f'https://finance.yahoo.com/quote/{symbol}/news'
source = "Yahoo quote"
open_tab(driver, url)
display(Markdown(f"[{source}]({url})"))

[Yahoo quote](https://finance.yahoo.com/quote/TSLA/news)

In [ ]:
url = f'https://finance.yahoo.com/quote/{symbol}/key-statistics?ltr=1'
source = "Yahoo stats"
open_tab(driver, url)
display(Markdown(f"[{source}]({url})"))

In [ ]:
url = f'https://www.ft.com/search?q={company}'
source = "FT"
open_tab(driver, url)
display(Markdown(f"[{source}]({url})"))

In [ ]:
url = f'https://www.marketwatch.com/investing/stock/{symbol}?mod=search_symbol'
source = "Marketwatch"
open_tab(driver, url)
display(Markdown(f"[{source}]({url})"))

In [ ]:
url = f'https://www.barrons.com/market-data/stocks/{symbol}?mod=searchresults_companyquotes&mod=searchbar&search_keywords={company}&search_statement_type=suggested'
source = "Barrons"
open_tab(driver, url)
display(Markdown(f"[{source}]({url})"))

In [ ]:
url = f'https://www.businessinsider.com/answers#{company}'

source = "Insider"
open_tab(driver, url)
display(Markdown(f"[{source}]({url})"))

In [ ]:
url = f'https://www.google.com/finance/quote/{symbol}:{exchange}'
source = "Google Finance"
open_tab(driver, url)
display(Markdown(f"[{source}]({url})"))

In [ ]:
url = f'https://finviz.com/quote.ashx?t={symbol}&p=d'
source = "FinViz"
open_tab(driver, url)
display(Markdown(f"[{source}]({url})"))

In [ ]:
url = f'https://www.reddit.com/search?q={company}&include_over_18=on&sort=relevance&t=all'
source = "Reddit"
open_tab(driver, url)
display(Markdown(f"[{source}]({url})"))

In [ ]:
morn_exch = morningstarmap[exchange]
url = f'https://www.morningstar.com/stocks/{morn_exch}/{symbol}/quote'
source = "Morningstar"
open_tab(driver, url)
display(Markdown(f"[{source}]({url})"))


In [41]:
url = f'https://whalewisdom.com/stock/{symbol}'
source = "WhaleWisdom"
open_tab(driver, url)
display(Markdown(f"[{source}]({url})"))


[WhaleWisdom](https://whalewisdom.com/stock/TSLA)

In [42]:
url = f'https://www.gurufocus.com/stock/{symbol}/guru-trades'
source = "GuruFocus"
open_tab(driver, url)
display(Markdown(f"[{source}]({url})"))


[GuruFocus](https://www.gurufocus.com/stock/TSLA/guru-trades)

In [ ]:
# https://www.barchart.com/stocks/highs-lows/lows?orderBy=highPercent1y&orderDir=asc
#     https://stockcharts.com/h-sc/ui?s=%24DJUSEN&id=p33407302522&def=Y&listNum=1
#         https://stockcharts.com/h-sc/ui?s=%24DJUSDN&id=p33407302522&def=Y&listNum=1

In [ ]:
# driver.close()


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta

def create_advanced_stock_chart(symbol, period='2y'):
    """
    Create an advanced stock chart with candlesticks, moving averages, volume, and relative performance

    Parameters:
    symbol (str): Stock symbol (e.g., 'TSLA')
    period (str): Time period ('1y', '2y', '5y', etc.)
    """

    # Fetch stock data
    stock = yf.Ticker(symbol)
    df = stock.history(period=period)

    # Fetch S&P 500 data for relative performance
    spy = yf.Ticker('SPY')
    spy_df = spy.history(period=period)

    # Calculate moving averages
    df['MA_13'] = df['Close'].rolling(window=13).mean()  # ~13 weeks (65 trading days)
    df['MA_52'] = df['Close'].rolling(window=52).mean()  # ~52 weeks (260 trading days)

    # Calculate relative performance vs S&P 500
    # Normalize both to starting point and calculate ratio
    stock_normalized = df['Close'] / df['Close'].iloc[0]
    spy_normalized = spy_df['Close'] / spy_df['Close'].iloc[0]
    relative_performance = stock_normalized / spy_normalized

    # Create subplots
    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.02,
        row_heights=[0.6, 0.2, 0.2],
        subplot_titles=(f'{symbol} Stock Chart', 'Volume', f'{symbol} vs S&P 500 Relative Performance')
    )

    # Add candlestick chart
    fig.add_trace(
        go.Candlestick(
            x=df.index,
            open=df['Open'],
            high=df['High'],
            low=df['Low'],
            close=df['Close'],
            name=symbol,
            showlegend=False
        ),
        row=1, col=1
    )

    # Add 13-week moving average
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df['MA_13'],
            mode='lines',
            name='13-Week MA',
            line=dict(color='blue', width=1.5),
            opacity=0.8
        ),
        row=1, col=1
    )

    # Add 52-week moving average
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df['MA_52'],
            mode='lines',
            name='52-Week MA',
            line=dict(color='red', width=1.5),
            opacity=0.8
        ),
        row=1, col=1
    )

    # Add volume bars
    colors = ['red' if close < open else 'green' for close, open in zip(df['Close'], df['Open'])]
    fig.add_trace(
        go.Bar(
            x=df.index,
            y=df['Volume'],
            name='Volume',
            marker_color=colors,
            opacity=0.6,
            showlegend=False
        ),
        row=2, col=1
    )

    # Add relative performance
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=relative_performance,
            mode='lines',
            name=f'{symbol} vs SPY',
            line=dict(color='black', width=1.5),
            fill='tonexty' if relative_performance.iloc[-1] > 1 else 'tozeroy',
            fillcolor='rgba(0,255,0,0.1)' if relative_performance.iloc[-1] > 1 else 'rgba(255,0,0,0.1)',
            showlegend=False
        ),
        row=3, col=1
    )

    # Add horizontal line at 1.0 for relative performance
    fig.add_hline(y=1.0, line_dash="dash", line_color="gray", opacity=0.5, row=3, col=1)

    # Update layout
    fig.update_layout(
        title=f'{symbol} - Advanced Stock Analysis',
        height=800,
        showlegend=True,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        ),
        margin=dict(l=50, r=50, t=80, b=50),
        plot_bgcolor='rgba(240,240,240,0.5)',
        paper_bgcolor='white'
    )

    # Update x-axes
    fig.update_xaxes(
        rangeslider_visible=False,
        showgrid=True,
        gridwidth=1,
        gridcolor='rgba(128,128,128,0.2)'
    )

    # Update y-axes
    fig.update_yaxes(
        showgrid=True,
        gridwidth=1,
        gridcolor='rgba(128,128,128,0.2)',
        title_text="Price ($)",
        row=1, col=1
    )

    fig.update_yaxes(
        title_text="Volume",
        row=2, col=1
    )

    fig.update_yaxes(
        title_text="Relative Performance",
        row=3, col=1
    )

    # Remove range slider from candlestick
    fig.layout.xaxis.rangeslider.visible = False

    return fig

# Example usage
if __name__ == "__main__":
    # Create chart for Tesla
    fig = create_advanced_stock_chart('TSLA', period='2y')
    fig.show()

    # You can also create charts for other stocks
    # fig = create_advanced_stock_chart('AAPL', period='1y')
    # fig.show()

# Additional customization options:

def add_technical_indicators(fig, df, row=1):
    """Add additional technical indicators to the chart"""

    # Bollinger Bands
    window = 20
    df['BB_Middle'] = df['Close'].rolling(window=window).mean()
    df['BB_Std'] = df['Close'].rolling(window=window).std()
    df['BB_Upper'] = df['BB_Middle'] + (df['BB_Std'] * 2)
    df['BB_Lower'] = df['BB_Middle'] - (df['BB_Std'] * 2)

    # Add Bollinger Bands
    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df['BB_Upper'],
            mode='lines',
            name='BB Upper',
            line=dict(color='purple', width=1, dash='dot'),
            opacity=0.6
        ),
        row=row, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=df.index,
            y=df['BB_Lower'],
            mode='lines',
            name='BB Lower',
            line=dict(color='purple', width=1, dash='dot'),
            fill='tonexty',
            fillcolor='rgba(128,0,128,0.1)',
            opacity=0.6
        ),
        row=row, col=1
    )

    return fig

def customize_chart_appearance(fig):
    """Apply custom styling to match the original chart aesthetic"""

    fig.update_layout(
        plot_bgcolor='rgba(245,245,220,0.8)',  # Beige background
        paper_bgcolor='white',
        font=dict(size=10, color='black'),
        title_font_size=14
    )

    # Update candlestick colors
    fig.data[0].increasing.fillcolor = 'rgba(0,150,0,0.8)'
    fig.data[0].increasing.line.color = 'rgba(0,100,0,1)'
    fig.data[0].decreasing.fillcolor = 'rgba(200,0,0,0.8)'
    fig.data[0].decreasing.line.color = 'rgba(150,0,0,1)'

    return fig

# Enhanced example with all features
def create_complete_chart(symbol='TSLA', period='2y'):
    """Create a complete chart with all technical indicators and styling"""

    fig = create_advanced_stock_chart(symbol, period)

    # Get the data again for technical indicators
    stock = yf.Ticker(symbol)
    df = stock.history(period=period)

    # Add technical indicators
    fig = add_technical_indicators(fig, df)

    # Apply custom styling
    fig = customize_chart_appearance(fig)

    return fig

# Usage:
# fig = create_complete_chart('TSLA', '2y')
# fig.show()

In [ ]:
import yfinance as yf
import pandas as pd
import plotly.graph_objs as go
from plotly.subplots import make_subplots

def make_stock_chart(symbol):

    # Download weekly data
    tsla = yf.download(symbol, interval="1wk", period="4y")
    tsla.columns = [col[0] if col[1] == symbol else col[0] for col in tsla.columns]
    spx = yf.download("^GSPC", interval="1wk", period="4y")
    spx.columns = [col[0] if col[1] == '^GSPC' else col[0] for col in spx.columns]

    # Compute moving averages
    tsla['MA13'] = tsla['Close'].rolling(window=13).mean()
    tsla['MA52'] = tsla['Close'].rolling(window=52).mean()

    # Compute relative strength vs SPX
    relative = tsla['Close'] / spx['Close']
    tsla['Rel_SPX'] = relative

    # Create figure with secondary y-axis in the first row
    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        row_heights=[0.7, 0.3],
        vertical_spacing=0.05,
        specs=[[{"secondary_y": True}], [{}]],  # row 1 has secondary y-axis
        subplot_titles=[f"{symbol} Price with Moving Averages & Volume", f"{symbol} Relative to S&P 500"]
    )

    # --- Row 1: Price Candlesticks & MAs (primary y-axis) ---
    fig.add_trace(go.Candlestick(
        x=tsla.index,
        open=tsla['Open'],
        high=tsla['High'],
        low=tsla['Low'],
        close=tsla['Close'],
        name=symbol,
        increasing_line_color='black',
        decreasing_line_color='red'
    ), row=1, col=1, secondary_y=False)

    fig.add_trace(go.Scatter(
        x=tsla.index,
        y=tsla['MA13'],
        mode='lines',
        name='13-week MA',
        line=dict(color='blue')
    ), row=1, col=1, secondary_y=False)

    fig.add_trace(go.Scatter(
        x=tsla.index,
        y=tsla['MA52'],
        mode='lines',
        name='52-week MA',
        line=dict(color='orange')
    ), row=1, col=1, secondary_y=False)

    # --- Row 1: Volume on right axis (secondary y-axis) ---
    fig.add_trace(go.Bar(
        x=tsla.index,
        y=tsla['Volume'],
        name='Volume',
        marker_color='rgba(0, 128, 0, 0.4)',
        showlegend=False
    ), row=1, col=1, secondary_y=True)

    # --- Row 2: Relative to SPX ---
    fig.add_trace(go.Scatter(
        x=tsla.index,
        y=tsla['Rel_SPX'],
        name=symbol + ' / SPX',
        mode='lines',
        line=dict(color='black')
    ), row=2, col=1)

    # Layout adjustments
    fig.update_layout(
        title=symbol + ' Weekly Chart with MAs, Volume (Right Axis), and Relative Strength',
        height=800,
        xaxis=dict(rangeslider_visible=False),
        showlegend=True
    )

    fig.update_yaxes(title_text="Price", row=1, col=1, secondary_y=False)
    fig.update_yaxes(title_text="Volume", row=1, col=1, secondary_y=True)
    fig.update_yaxes(title_text=f"{symbol} / SPX", row=2, col=1)

    fig.show()

make_stock_chart('TSLA')

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Create figure with secondary y-axis in the first row
fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    row_heights=[0.7, 0.3],
    vertical_spacing=0.05,
    specs=[[{"secondary_y": True}], [{}]],  # row 1 has secondary y-axis
    subplot_titles=["TSLA Price with Moving Averages & Volume", "TSLA Relative to S&P 500"]
)

# --- Row 1: Price Candlesticks & MAs (primary y-axis) ---
fig.add_trace(go.Candlestick(
    x=tsla.index,
    open=tsla['Open'],
    high=tsla['High'],
    low=tsla['Low'],
    close=tsla['Close'],
    name='TSLA',
    increasing_line_color='black',
    decreasing_line_color='red'
), row=1, col=1, secondary_y=False)

fig.add_trace(go.Scatter(
    x=tsla.index,
    y=tsla['MA13'],
    mode='lines',
    name='13-week MA',
    line=dict(color='blue')
), row=1, col=1, secondary_y=False)

fig.add_trace(go.Scatter(
    x=tsla.index,
    y=tsla['MA52'],
    mode='lines',
    name='52-week MA',
    line=dict(color='orange')
), row=1, col=1, secondary_y=False)

# --- Row 1: Volume on right axis (secondary y-axis) ---
fig.add_trace(go.Bar(
    x=tsla.index,
    y=tsla['Volume'],
    name='Volume',
    marker_color='rgba(0, 128, 0, 0.4)',
    showlegend=False
), row=1, col=1, secondary_y=True)

# --- Row 2: Relative to SPX ---
fig.add_trace(go.Scatter(
    x=tsla.index,
    y=tsla['Rel_SPX'],
    name='TSLA / SPX',
    mode='lines',
    line=dict(color='black')
), row=2, col=1)

# Layout adjustments
fig.update_layout(
    title='TSLA Weekly Chart with MAs, Volume (Right Axis), and Relative Strength',
    height=800,
    xaxis=dict(rangeslider_visible=False),
    showlegend=True
)

fig.update_yaxes(title_text="Price", row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text="Volume", row=1, col=1, secondary_y=True)
fig.update_yaxes(title_text="TSLA / SPX", row=2, col=1)

fig.show()

In [ ]:
tsla.columns = ['_'.join(filter(None, col)).strip() for col in tsla.columns.values]
tsla


In [ ]:
tsla = yf.download("TSLA", interval="1wk", period="6y")
tsla.columns = [col[0] if col[1] == 'TSLA' else col[0] for col in tsla.columns]
tsla


In [ ]:
tod

In [ ]:
xxx

In [46]:
system_prompt = f""" 
You are an expert investment research analyst tasked with creating comprehensive up-to-date company profiles from an institutional investment perspective. Your role is to conduct thorough, objective research and analysis to inform investment decision-making.

Core Principles

Analytical Rigor: Apply systematic, data-driven analysis with appropriate skepticism. Validate information across multiple sources and flag any inconsistencies or data quality concerns.
Investment Focus: Frame all analysis through the lens of investment implications. Connect operational details to financial performance, competitive positioning, and risk/return profiles.
Objectivity: Present balanced analysis that acknowledges both positive and negative factors. Avoid promotional language and maintain professional neutrality throughout.
Timely Information: Attempt to present the latest information as of today, {date_today}. Note dates of sources referenced. Do not treat possibly outdated information as current.
Forward-looking perspective: Include historical information for context to understand the current situation and future prospects. Information that impacts future prospects and drives investment returns going forward is of most interest.
Research Methodology
Multi-Source Verification: Cross-reference information from company filings, financial databases, industry reports, and credible news sources. When sources conflict, note discrepancies and assess reliability.
Systematic Information Gathering: Use available tools methodically:

Financial data APIs for quantitative metrics and peer comparisons
Web search for recent developments, analyst coverage, and industry context
Company filings and investor materials for official positions
News aggregation for market sentiment and risk factor identification

Quality Standards: Prioritize authoritative sources (SEC filings, earnings calls, established financial media, recognized research firms) over speculative or promotional content. Flag when analysis relies on limited or potentially biased sources.
Analytical Framework
Strategic Context: Position the company within its industry ecosystem, considering competitive dynamics, market maturity, regulatory environment, and secular trends.
Financial Analysis: Go beyond basic metrics to understand business quality, cash generation, capital efficiency, and financial flexibility. Connect financial trends to underlying business drivers.
Risk Assessment: Identify and categorize risks systematically (operational, financial, competitive, regulatory, ESG). Assess probability and potential impact of key risk scenarios.
Valuation Perspective: While not providing specific price targets, discuss valuation approaches relevant to the business model and highlight key variables that drive investment returns.
Output Expectations
Professional Tone: Write in the clear, authoritative style of institutional research. Use precise financial terminology while remaining accessible to sophisticated investors.
Evidence-Based Conclusions: Support all analytical points with specific data, examples, or credible sources. Distinguish between factual information and analytical interpretation.
Actionable Insights: Focus on information that would influence investment decisions. Highlight critical factors for ongoing monitoring and key questions for further due diligence.
Structured Presentation: Follow the specific organizational structure provided in the user prompt, ensuring logical flow and appropriate depth for each section.
Key Reminders

Maintain objectivity and avoid advocacy for any investment position
Clearly distinguish between company-provided information and independent analysis
Note when information is incomplete, uncertain, or requires further investigation
Focus on material factors that could significantly impact investment outcomes
Present analysis that would meet institutional investment committee standards

Your goal is to produce a comprehensive, professional-grade company profile that enables informed investment decision-making through rigorous research and balanced analysis."""

In [47]:
user_prompt = f"""

Using everything found so far and everything you can find by using tools and doing deep research, write a report on Tesla (symbol TSLA) in the straightforward factual style of a Wall Street equity research analyst, in 8 sections:

1. Profile
• History with origin story and key historical milestones
• Core business and competitors
• Major news events since {last_year}

2. Business Model:
• Describe their core businesses, products and services.
• Outline their key revenue streams, customer segments, and monetization strategies.
• Analyze key characteristics of markets it operates in:
    - customer acquisition costs
    - retention metrics
    - sales cycles
    - seasonal or cyclical business patterns
    - margins, market size, growth trajectory and factors affecting them
• Explain sources of competitive advantage such as network effects, switching costs, brands, intellectual property, regulatory moats, and other barriers to entry.

3. Competitive Landscape:
• Identify their main competitors, including direct, adjacent, and emerging competitors.
• Compare key metrics such as market share, product differentiation, pricing power, and growth trajectories.

4. Supply Chain Positioning:
• Describe their role in the upstream (supplier-side) and downstream (customer/distribution) parts of the supply chain.
• Identify key suppliers, partners, distributors, and any major dependencies or concentrations.

5. Financial and Operating Leverage:
• Analyze the company’s use of financial leverage (debt levels, interest obligations, credit ratings).
• Analyze operating leverage (fixed vs. variable cost structure, scalability, margin sensitivity to revenue changes).
• Analyze cash flow generation and working capital dynamics.
• Analyze capital allocation strategy (dividends, buybacks, reinvestment).

6. Valuation:
• Identify appropriate valuation methodologies, including income-based (e.g., DCF), asset-based (eg book value and sum of parts), market-based (e.g. peer multiples and comparisons), and LBO analysis
• Highlight important valuation inputs and metrics (growth rates, margins, discount rates, terminal value assumptions).
• Summarize current ratings and analyst opinions, including recent changes.
• Note the stock's volatility, liquidity, if it is widely covered and owned, if it is a hedge fund story stock or meme stock, what macro factors it is sensitive to

7. Recent developments, News Search and Risk Factors:
• Conduct a deep news search for significant positive and negative news items since {last_year}, including:
    • Revenue and earnings trends
    • Management changes
    • New product launches
    • Restructurings, mergers, acquisitions, divestitures, strategic partnerships
    • Short-seller reports or allegations.
    • Regulatory investigations or lawsuits.
    • Product failures, operational issues, or supply chain disruptions.
    • Major wins (e.g., partnerships, large customer wins, successful product launches).
    • Insider trading activity and institutional ownership changes
• Summarize key themes from media coverage, analyst reports, and public filings since {last_year}.
• Note controversies, reputational risks, and governance concerns if any.
• Discuss what companies might be potential acquisition targets or acquirers of the company, based on overlapping or complementary customer bases, product offerings, and technical capabilities.
• Note any other key themes or trends that you think are important.

8. Overall Assessment:
• Summarize the company’s strategic position and the stock's investment risk/reward profile
    • Strengths, weaknesses, opportunities, threats
    • Bear case and bull case
    • The level of risk
    • Any critical “watch points” for further due diligence and ongoing monitoring.

"""

In [48]:
!ls tmp/tx0ybr2zo/ 


TSLA_edgar_10k_item1.txt   TSLA_perplexity_profile.md
TSLA_fundamintals_av.json  TSLA_perplexity_ratings.md
TSLA_peers.txt             TSLA_technicals.md
TSLA_perplexity_news.md    TSLA_wikipedia.md


In [ ]:
balance sheet
income statement


In [89]:
upload_files = {
    'edgar_10k_item1.txt': f'SEC 10K Item 1',
    'perplexity_profile.md': f'Company profile from Perplexity',
    'peers.txt': f'List of Peers',
    'fundamentals_av.json': f'Fundamentals for company and peers',
    'perplexity_ratings.md': f'Analyst ratings from Perplexity',
    'technicals.md': f'Current technicals',
    'perplexity_news.md': f'Company news from Perplexity',
    'wikipedia.md': f'Wikipedia profile'
}

In [77]:
# current_files = [f"{temp_dir}/{symbol}_{f}" for f in upload_files]
current_files = [str(f) for f in Path(temp_dir).iterdir() if f.is_file()]

# Sort files by size (smallest to largest)
current_files_by_size = sorted(current_files, key=lambda f: Path(f).stat().st_size)
current_files_by_size


['tmp/tx0ybr2zo/TSLA_technicals.md',
 'tmp/tx0ybr2zo/TSLA_fundamentals_av.json',
 'tmp/tx0ybr2zo/TSLA_perplexity_news.md',
 'tmp/tx0ybr2zo/TSLA_wikipedia.md',
 'tmp/tx0ybr2zo/TSLA_perplexity_ratings.md',
 'tmp/tx0ybr2zo/.DS_Store',
 'tmp/tx0ybr2zo/TSLA_perplexity_profile.md',
 'tmp/tx0ybr2zo/TSLA_peers.txt',
 'tmp/tx0ybr2zo/TSLA_edgar_10k_item1.txt']

In [92]:
for file_path in current_files_by_size:
    parts = file_path.split('_', 1)
    result = file_path[len(parts[0])+1:]
    if result in upload_files:
        print(result, ": ", upload_files[result])
    else:
        print("notfound") 

technicals.md :  Current technicals
fundamentals_av.json :  Fundamentals for company and peers
perplexity_news.md :  Company news from Perplexity
wikipedia.md :  Wikipedia profile
perplexity_ratings.md :  Analyst ratings from Perplexity
notfound
perplexity_profile.md :  Company profile from Perplexity
peers.txt :  List of Peers
edgar_10k_item1.txt :  SEC 10K Item 1


In [96]:
import tiktoken

# Initialize tiktoken encoder (using GPT-4 encoding)
encoding = tiktoken.get_encoding("cl100k_base")  # or "gpt-4" 

additional_context = ""
total_tokens = 0
files_processed = []

for file_path in current_files_by_size:
    try:
        # Read file content
        with open(file_path, 'r', encoding='utf-8') as file:
            file_content = file.read()
        
        # Count tokens in this file
        file_tokens = len(encoding.encode(file_content))
        
        # Check if adding this file would exceed the limit
        if total_tokens + file_tokens > 90000:
            print(f"Stopping before {file_path} - would exceed 90,000 tokens")
            break
        
        # Add to context and update counters
        parts = file_path.split('_', 1)
        section_title = file_path[len(parts[0])+1:]
        additional_context += f"# {symbol} {section_title}\n"
        additional_context += file_content + "\n\n"  # Add some separation between files
        total_tokens += file_tokens
        files_processed.append(file_path)
        
        print(f"Added {file_path}: {file_tokens} tokens (Total: {total_tokens})")
        
    except (UnicodeDecodeError, OSError, FileNotFoundError) as e:
        print(f"Error reading {file_path}: {e}")
        continue

print(f"\nProcessed {len(files_processed)} files")
print(f"Total tokens: {total_tokens}")
print(f"Total characters in additional_context: {len(additional_context)}")


Added tmp/tx0ybr2zo/TSLA_technicals.md: 130 tokens (Total: 130)
Added tmp/tx0ybr2zo/TSLA_fundamentals_av.json: 731 tokens (Total: 861)
Added tmp/tx0ybr2zo/TSLA_perplexity_news.md: 1117 tokens (Total: 1978)
Added tmp/tx0ybr2zo/TSLA_wikipedia.md: 1117 tokens (Total: 3095)
Added tmp/tx0ybr2zo/TSLA_perplexity_ratings.md: 1497 tokens (Total: 4592)
Error reading tmp/tx0ybr2zo/.DS_Store: 'utf-8' codec can't decode byte 0x80 in position 3131: invalid start byte
Added tmp/tx0ybr2zo/TSLA_perplexity_profile.md: 1929 tokens (Total: 6521)
Added tmp/tx0ybr2zo/TSLA_peers.txt: 2246 tokens (Total: 8767)
Added tmp/tx0ybr2zo/TSLA_edgar_10k_item1.txt: 23395 tokens (Total: 32162)

Processed 8 files
Total tokens: 32162
Total characters in additional_context: 171305


In [97]:
print(additional_context)

# TSLA technicals.md

Technical Analysis for TSLA:

Trend Analysis:
- Above 20 SMA: ❌
- Above 50 SMA: ❌
- Above 200 SMA: ❌
- 20/50 SMA Bullish Cross: ❌
- 50/200 SMA Bullish Cross: ✅

Momentum:
- RSI (14): 42.92
- MACD Bullish: ❌

Latest Price: $302.63
Average True Range (14): 14.11
Average Daily Range Percentage: 3.49%
Average Volume (20D): 96,080,182


# TSLA fundamentals_av.json
{"Symbol": "TSLA", "AssetType": "Common Stock", "Name": "Tesla Inc", "Description": "Tesla, Inc. is an American electric vehicle and clean energy company based in Palo Alto, California. Tesla's current products include electric cars, battery energy storage from home to grid-scale, solar panels and solar roof tiles, as well as other related products and services. In 2020, Tesla had the highest sales in the plug-in and battery electric passenger car segments, capturing 16% of the plug-in market (which includes plug-in hybrids) and 23% of the battery-electric (purely electric) market. Through its subsidiary Tesl

In [100]:
keylist = list(upload_files.keys())
keylist = keylist[len(files_processed):]
file_list=[client.files.create(
    file=open(f"{temp_dir}/{symbol}_{f}", "rb"), purpose="assistants") 
           for f in keylist]


In [101]:
file_list

[]

In [ ]:
    • company +  "analyst report" OR "research note" downgrade OR upgrade
    • use perplexity to search for company + "profile" or "executive profile" and ask: "What are the most significant investigative reports and executive profiles about company published in 2023-2024?"


In [58]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

response = client.responses.create(
  model="o3-deep-research",
  input=[
    {
      "role": "developer",
      "content": [
        {
          "type": "input_text",
          "text": system_prompt,
        }
      ]
    },
    {
      "role": "user",
      "content": [
        {
          "type": "input_text",
          "text": user_prompt,
        }
      ]
    }
  ],
  reasoning={
    "summary": "auto"
  },
  tools=[
    {
      "type": "web_search_preview"
    },
    {
      "type": "code_interpreter",
      "container": {
        "type": "auto",
        "file_ids": [f.id for f in file_list]
      }
    }
  ]
)

In [ ]:
since current year -1

In [71]:
response_text = response.output[-1].content[0].dict()['text']
response_text = response_text.replace(r'$',r'\$')
display(Markdown(response_text))


/var/folders/6d/3xz907yn5ylg43s2vlnnzptr0000gn/T/ipykernel_70763/697380116.py:1: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  response_text = response.output[-1].content[0].dict()['text']


# Tesla, Inc. (TSLA) – Equity Research Analyst Report

## 1. Profile  
**History & Milestones:** Tesla, Inc. was founded in July 2003 by engineers Martin Eberhard and Marc Tarpenning in San Carlos, California ([en.wikipedia.org](https://en.wikipedia.org/wiki/Tesla%2C_Inc.#:~:text=Tesla%20was%20incorporated%20in%20July,2022%20and%20the%20%2019)). The company was named in honor of inventor Nikola Tesla. Elon Musk led the first major funding round in 2004 and became chairman, ultimately taking over as CEO in 2008 ([en.wikipedia.org](https://en.wikipedia.org/wiki/Tesla%2C_Inc.#:~:text=Tesla%20was%20incorporated%20in%20July,2022%20and%20the%20%2019)). In its early years, Tesla focused on proving that electric vehicles (EVs) could be high-performance and desirable. The first model, the Tesla **Roadster**, launched in 2008, was a sports car that used lithium-ion batteries and showcased Tesla’s cutting-edge EV technology ([en.wikipedia.org](https://en.wikipedia.org/wiki/Tesla%2C_Inc.#:~:text=funding%20round%20and%20became%20the,2022%20and%20the%20%2019)). This was followed by pivotal model launches: the **Model S** luxury sedan in 2012, the **Model X** SUV in 2015, the mass-market **Model 3** sedan in 2017, and the **Model Y** crossover in 2020 ([en.wikipedia.org](https://en.wikipedia.org/wiki/Tesla%2C_Inc.#:~:text=funding%20round%20and%20became%20the,2022%20and%20the%20%2019)). Each new model significantly expanded Tesla’s customer base. Tesla went public in 2010 and, over the 2010s, it grew from a niche startup into one of the world’s most valuable automakers. Key milestones include the development of Gigafactory battery plants (starting with Nevada in partnership with Panasonic in 2014), achieving its first full year of profitability in 2020, and being added to the S&P 500 index in late 2020. By 2021, Tesla briefly reached a trillion-dollar market capitalization – a first for any automaker ([en.wikipedia.org](https://en.wikipedia.org/wiki/Tesla%2C_Inc.#:~:text=Tesla%20is%20one%20of%20the,in%20the%20Forbes%20Global%202000)). The company’s global expansion also accelerated in recent years, with new production Gigafactories opened in Shanghai (2019), Berlin (2022), and Austin, Texas (2022). These facilities have enabled rapid growth in vehicle output. In late 2022 Tesla delivered its first **Tesla Semi** (electric freight truck), and in late 2023 it began initial production of the long-awaited **Cybertruck** pickup ([en.wikipedia.org](https://en.wikipedia.org/wiki/Tesla%2C_Inc.#:~:text=funding%20round%20and%20became%20the,2022%20and%20the%20%2019)). 

**Core Business & Products:** Tesla’s core business is designing, manufacturing, and selling **battery electric vehicles** (BEVs) and related clean energy products ([en.wikipedia.org](https://en.wikipedia.org/wiki/Tesla%2C_Inc.#:~:text=correct%2C%20though%20Nikola%20Tesla%20%27s,and%20related%20products%20and%20services)). The automotive segment, which includes the sale of electric cars and related services, accounts for the majority of revenue. Tesla currently offers six vehicle models – the Model S, Model X, Model 3, Model Y, the Tesla Semi, and the new Cybertruck – covering segments from luxury sedans and SUVs to pickup trucks and commercial vehicles ([en.wikipedia.org](https://en.wikipedia.org/wiki/Tesla%2C_Inc.#:~:text=funding%20round%20and%20became%20the,2022%20and%20the%20%2019)) ([en.wikipedia.org](https://en.wikipedia.org/wiki/Tesla%2C_Inc.#:~:text=Automotive%20products%20and%20services)). In addition, Tesla provides software features such as its **Full Self-Driving (FSD)** driver assistance package (sold as an upgrade), and it operates a global network of **Supercharger** stations for EV fast-charging as a service to its customers. Beyond vehicles, Tesla’s second business segment is **clean energy & storage**, which includes solar panels and solar roofs, home energy storage units (Powerwall), and large-scale energy storage solutions (Megapack batteries) ([en.wikipedia.org](https://en.wikipedia.org/wiki/Tesla%2C_Inc.#:~:text=correct%2C%20though%20Nikola%20Tesla%20%27s,and%20related%20products%20and%20services)). These products leverage Tesla’s battery technology for residential and utility-scale energy management. While smaller in revenue than the automotive segment, the energy division is growing as demand for battery storage rises with renewable energy adoption. 

**Competitors:** In electric vehicles, Tesla’s primary competitors include both traditional automakers entering the EV market and pure-play EV startups. In the U.S. market, major rivals are starting to roll out competing electric models – for example, **Ford** (Mustang Mach-E SUV, F-150 Lightning pickup) and **General Motors** (Chevy Bolt, Cadillac Lyriq, etc.), and these legacy firms plan dozens of new EV offerings. In the luxury and mid-market segments, Tesla’s Model S and X face competition from brands like **Mercedes-Benz EQS/EQE**, **Audi e-tron**, **Porsche Taycan**, and **Lucid** (an EV startup focusing on premium sedans). Tesla’s high-volume Model 3 and Y are up against emerging rivals like **Volkswagen’s ID series** in Europe, **Hyundai/Kia’s EV lineup**, as well as lower-cost Chinese EVs. Notably, in China – the world’s largest EV market – Tesla competes with aggressive local manufacturers such as **BYD, NIO, Xpeng,** and **Li Auto**. BYD in particular has become a formidable global competitor, leveraging vertical integration and scale in China to sell *nearly* as many pure EVs as Tesla, and even more if counting plug-in hybrids ([www.statista.com](https://www.statista.com/chart/27733/battery-electric-vehicles-manufacturers/#:~:text=As%20of%20September%202024%2C%20Tesla,EV%20Volumes%20published%20on%20CleanTechnica)) ([www.reuters.com](https://www.reuters.com/business/autos-transportation/byd-may-hand-back-top-ev-seller-title-tesla-after-q1-sales-decline-2024-04-02/#:~:text=BYD%2C%20China%27s%20leading%20EV%20manufacturer%2C,competition%20in%20the%20Chinese%20market)). Tesla’s clean energy business competes with other solar installation and battery providers (like Sunrun in solar, or LG and Samsung in battery storage), though no single company mirrors Tesla’s integrated energy storage model directly. Overall, Tesla’s broad competition ranges from automotive giants transitioning to EVs, to specialized EV startups and tech companies (in autonomous driving), making its competitive landscape one of the most dynamic in the industry.

**Recent Major News:** In the past year, Tesla’s growth and strategic moves have been closely watched by the market. The company achieved **record vehicle production and deliveries** in 2022 (around 1.31 million vehicles delivered, a ~40% growth YoY) and continued to increase volumes into 2023. However, to support demand, Tesla implemented significant **price cuts** across its vehicle lineup in late 2022 and early 2023 ([apnews.com](https://apnews.com/article/fac648952687ea903477422f5001ef0a#:~:text=2023,a%20%247%2C500%20federal%20tax%20credit)). These price reductions (up to ~20% on some models) aimed to widen the customer base and also ensured more Tesla models qualified for new EV tax credits in the U.S. ([apnews.com](https://apnews.com/article/fac648952687ea903477422f5001ef0a#:~:text=2023,a%20%247%2C500%20federal%20tax%20credit)). The strategy boosted sales but began to **compress profit margins**. By mid-2023, Tesla’s automotive gross margin had fallen to its lowest level in years (around 14–18%, ex-credit, vs. ~25–30% a year prior) due to the price cuts ([www.reuters.com](https://www.reuters.com/business/autos-transportation/tesla-revenue-sees-surprise-rise-second-quarter-revenue-2024-07-23/#:~:text=Automotive%20gross%20margin%20excluding%20regulatory,unveiling%20delayed%20to%20October%2010)). In other major developments, Tesla made headlines by opening up its previously proprietary Supercharger network to competitors: agreements in 2023 will allow EVs from Ford, GM, and other makers to use Tesla’s fast chargers, positioning Tesla’s charging plug as a de facto standard in North America. This not only potentially adds a new revenue stream (via charging access fees) but also expands Tesla’s influence in the EV ecosystem. On the product front, Tesla finally commenced limited deliveries of the **Cybertruck** (its futuristic pickup) in late 2023 after delays, and it’s gearing up for a larger production ramp in 2024. The company also delivered its first **Tesla Semi** trucks to customers like PepsiCo, marking its entry into heavy-duty EVs. In management news, Tesla’s longtime Chief Financial Officer, Zach Kirkhorn, unexpectedly stepped down in August 2023 (after 13 years with the company), raising some investor questions about succession ([www.cnbc.com](https://www.cnbc.com/2023/08/07/tesla-cfo-zach-kirkhorn-steps-down.html#:~:text=Skip%20Navigation%20Key%20Points%20,Kirkhorn%20will%20stay%20on%20at)). Elon Musk’s continuing role as CEO has also drawn attention as he took on additional commitments (notably his late-2022 acquisition of Twitter, now X), though Musk has since appointed a separate CEO at his social media venture to refocus on Tesla. Overall, recent news highlights Tesla’s push for volume growth (even at the expense of short-term margins), its strategic leveraging of infrastructure (charging network), and the ongoing balancing act of managing executive leadership amid rapid expansion. 

## 2. Business Model  
**Core Businesses & Products:** Tesla operates **two primary business segments**: (1) **Automotive** and (2) **Energy Generation & Storage**. The **Automotive** segment comprises the design, manufacture, and sale/lease of electric vehicles, which is by far the largest revenue contributor. This includes multiple revenue streams: new vehicle sales, leasing revenue, sales of regulatory credits (Tesla earns zero-emission vehicle credits which it can sell to other automakers), and after-sales services (maintenance, repairs, insurance, and software upgrades like the FSD package). Tesla’s vehicles are sold directly to consumers, bypassing traditional dealer networks – customers order online or at Tesla stores, and the company handles its own distribution and service. Tesla also recognizes revenue from its Autopilot and Full Self-Driving software either at point of sale or over time (for features delivered via updates). The (2) **Energy** segment includes sales of solar photovoltaic systems (solar panels and the **Solar Roof** tiles) and battery storage products. Key products are the **Powerwall** (a home battery for backup power and energy management) and the **Megapack** (a large-scale battery system for utilities and commercial projects). This segment generates revenue through product sales and also through services like solar energy subscriptions. While currently smaller than auto, Tesla’s energy business is growing and management sees it as a pivotal part of a sustainable energy future (with some analysts predicting it could rival the auto business in the long run) ([www.reuters.com](https://www.reuters.com/business/autos-transportation/tesla-jumps-replacing-ford-morgan-stanleys-top-pick-us-auto-sector-2024-07-29/#:~:text=Tesla%27s%20shares%20surged%20by%206.3,and%20the%20upcoming%20robotaxi%20launch)). Tesla’s **monetization strategy** thus spans one-time hardware sales (vehicles, solar installations, batteries) and recurring or software income (charging fees, software licenses/upgrades, energy services), providing a mix of upfront and ongoing revenue streams.

**Customer Segments & Sales Approach:** Tesla’s customer base ranges from individual consumers to commercial clients and utilities. The *typical automotive customer* includes both retail consumers and corporate fleets that value EV performance, technology, and environmental benefits. Early on, Tesla’s market skewed toward luxury buyers (with the high-end Roadster, Model S and X), but with the Model 3/Y, it now serves the premium **mass-market segment**, attracting buyers who might otherwise consider mid-range luxury combustion cars (BMW 3-Series, Audi Q5, etc.). Tesla’s vehicles appeal through a combination of high performance, advanced tech features, and brand prestige. The company has notably **minimal spending on marketing/advertising** – instead, it relies on strong brand recognition, word-of-mouth, and media coverage for customer acquisition. This results in unusually low **customer acquisition costs** for an automaker (Tesla historically spent \$0 on conventional ads while competitors spend hundreds of dollars per car on marketing). Tesla also uses a direct-to-consumer sales model, selling via its website and company-owned stores, which shortens the sales cycle and keeps distribution costs down (though it had to navigate or lobby changes to dealer franchise laws in some U.S. states). **Sales cycle** in auto typically can span weeks from ordering to delivery (especially if a vehicle is built-to-order), and Tesla’s made-to-order approach means customers often place a deposit and wait for delivery. This model gives Tesla some cash-flow benefit and demand visibility, but it also means Tesla carries the risk of inventory if production outpaces orders (as seen occasionally when Tesla produces vehicles in batches). In the energy segment, customers include homeowners (for solar panels/Powerwalls) as well as businesses and utilities for Megapacks; sales here often involve longer project cycles and partnership with installers or developers.

**Market Characteristics:** Tesla participates in the **global automotive industry** (a vast market historically dominated by internal combustion vehicles) and the rapidly growing **EV sub-market**. The EV market is expanding quickly, roughly 50%+ annual growth in recent years, but it is also **highly competitive and capital-intensive**. Automotive markets are generally cyclical – influenced by economic conditions, interest rates, and consumer confidence – and somewhat seasonal, with typically higher sales in second-half of the year. Tesla often has a delivery push each quarter’s end, and Q4 has historically seen peak deliveries due to seasonal consumer buying and company sales incentives. The **sales cycle** for vehicles can be impacted by model refresh timelines and expiration/introduction of government incentives (for example, Tesla saw demand surges before EV tax credit phase-outs in the past, and again when new credits were introduced in 2023). **Customer retention** for Tesla appears strong – owners often stay within the brand for repeat purchases, helped by Tesla’s continuous improvement via over-the-air software updates and the ecosystem (Supercharger network making long-term ownership convenient). Tesla also enjoys one of the highest **net promoter scores** in the auto industry, indicating positive word-of-mouth. However, as more EV options enter the market, Tesla’s customer loyalty will be tested by new offerings.

**Revenue Streams:** In 2022, about ~87% of Tesla’s revenue came from the automotive segment (including regulatory credit sales) and ~13% from energy generation/storage and services. Within automotive, the bulk is from new vehicle sales. Tesla also generates noteworthy income from selling **regulatory credits** to other automakers who need them to meet emissions requirements – this has been a 100% margin revenue source that added hundreds of millions of dollars per year (though as others produce more EVs, this may dwindle). Another growing revenue stream is software: Tesla’s FSD feature is sold for a high upfront fee (as much as \$15,000 per vehicle) or a monthly subscription, and because the software is developed in-house, this is a high-margin stream that can expand as more customers opt in. On the energy side, revenue comes from solar panel sales/leases and storage unit sales; Tesla often packages solar + Powerwall to homeowners. Energy revenues are also supported by services agreements (maintenance or energy trading). 

**Cost Structure & Margins:** As an automaker, Tesla has a heavy manufacturing cost structure, with significant **fixed costs** (factories, equipment, tooling, and R&D) and also large **variable costs** for components (especially batteries, which are one of the most expensive parts of an EV). Tesla’s early investment in highly automated manufacturing (like the **Gigafactory** concept for battery production and the use of mega-casting machines for car bodies) is intended to achieve economies of scale and lower unit costs as volume grows. The company’s **gross margins** have historically been higher than legacy auto peers. Prior to recent price cuts, Tesla’s automotive gross margin (GAAP) was over 25% – well above industry averages in the teens – thanks to its premium pricing, direct sales (no dealer margin), and manufacturing efficiencies. Even after price cuts in 2023, gross margins around mid-teens percent remain healthy relative to competitors ([www.reuters.com](https://www.reuters.com/business/autos-transportation/tesla-revenue-sees-surprise-rise-second-quarter-revenue-2024-07-23/#:~:text=Automotive%20gross%20margin%20excluding%20regulatory,unveiling%20delayed%20to%20October%2010)). Operating margins have also been positive in recent years – Tesla exceeded 16% operating margin in 2022, reflecting the scalability of its cost base at high volumes. However, margins are sensitive to **utilization and scale**: building new factories or launching new models can pressure margins in the short run (due to under-utilized capacity or initial inefficiencies), whereas running factories near full capacity (as seen with the Model 3 line by 2019) drives unit costs down significantly. The EV industry also faces **commodity cost pressures** (lithium, nickel, cobalt prices can affect battery costs). Tesla has navigated this by securing supply contracts and even considering direct raw material processing, but spikes in raw material prices can squeeze margins unless offset by pricing.

**Growth Trajectory & Market Factors:** Tesla is positioned in markets with strong secular growth. The **global EV market** is expected to grow at a high CAGR for the next decade as consumers shift from gasoline to electric and governments implement stricter emissions regulations. Tesla’s unit sales have grown ~50% per year on average over the last few years, and the company has publicly aimed for around 50% annual growth over a multi-year horizon (though actual growth may fluctuate year to year). Key factors affecting Tesla’s growth include: (1) **EV adoption rates** and government incentives (such as tax credits, which lower effective prices and encourage buyers); (2) **competition intensity** – more competitors launching EVs may pressure Tesla’s market share or force further price adjustments; (3) **production capacity expansion** – Tesla’s ability to ramp new Gigafactories (e.g., a potential future plant or expansion in regions like India or another U.S. factory) will determine how quickly it can increase supply; and (4) technology improvements like battery cost reduction (which allow lower-priced models and tap into new customer segments). The company is also subject to typical auto industry cyclicality: in economic downturns, consumers may delay car purchases (especially higher-end models), which could impact Tesla’s growth or force promotional pricing. There may also be **seasonal patterns** – for instance, in some markets EV sales spike in the fourth quarter due to regulatory credit deadlines or consumers aiming for year-end incentive cut-offs. Tesla’s **customer acquisition** benefits from its strong brand (many customers actively seek out Tesla, requiring less traditional marketing), and **customer retention** is aided by a high satisfaction rate and continuous software updates that refresh the ownership experience. Nonetheless, as competition offers alternative EVs, Tesla will need to maintain a technology and cost edge to keep customers within its ecosystem.

**Competitive Advantages:** Tesla has established several competitive advantages that support its business model:
- **Brand and First-Mover Advantage:** Tesla is arguably the first automaker to prove that EVs can be desirable, high-performance, and mass-produced. This first-mover status in modern EVs has given it a strong brand associated with innovation and premium quality. Tesla consistently ranks atop consumer surveys for EV consideration and has a loyal fan base, giving it marketing advantages over both legacy automakers and newer entrants.
- **Technology Leadership:** Tesla has a lead in critical EV technologies, particularly in battery and powertrain efficiency and in software. The company’s vehicles often have superior range and performance metrics for their class, reflecting Tesla’s prowess in battery management and drivetrain design. Moreover, Tesla’s cars operate on a unified software platform that allows **over-the-air updates** – this keeps vehicles improving over time (a compelling ownership proposition traditional OEMs are now trying to emulate). Tesla’s development of autonomous driving algorithms (FSD) and custom AI chips for its vehicles also highlight a tech-driven approach atypical for car companies.
- **Vertical Integration:** Unlike many automakers that outsource large portions of component manufacturing, Tesla is highly vertically integrated. It designs its own electric motors, battery packs, and software in-house, and even its own infotainment software and some semiconductor components. Tesla also manufactures batteries jointly with partners (and is developing its **own 4680 battery cells**). This integration can lead to cost advantages and faster innovation cycles – e.g., Tesla introduced large casting techniques to simplify car assembly, reducing parts count and cost. Vertical integration extends to distribution and charging infrastructure: Tesla owns its sales and service centers and built the **Supercharger network** exclusively for its customers (now being partially opened to others). The Supercharger network presence has been a selling point and creates a **network effect** – the more Teslas on the road, the more justification to expand chargers, which in turn attracts more buyers for the convenience. With recent agreements to share this network with other brands, Tesla may even turn this asset into a broader industry standard ([www.reuters.com](https://www.reuters.com/business/autos-transportation/tesla-jumps-replacing-ford-morgan-stanleys-top-pick-us-auto-sector-2024-07-29/#:~:text=Tesla%27s%20shares%20surged%20by%206.3,and%20the%20upcoming%20robotaxi%20launch)).
- **Scale and Manufacturing Efficiency:** Tesla has achieved a scale in EV production (1.3+ million cars/year in 2022) far above most pure EV rivals. This scale brings procurement benefits, manufacturing experience, and data collection advantages. For example, Tesla has accumulated billions of miles of real-world driving data from its cars’ sensors – invaluable for refining autonomous driving features. Scale also spreads R&D and fixed costs over more units, which helped Tesla reach industry-leading margins. The gigafactory concept (large, highly automated plants focused on one or two models) has allowed rapid output expansion once operational. Tesla’s relentless focus on cost reduction (they often talk of simplifying design, removing parts, using cheaper materials without compromising function) supports a trend of improving margins over the long term (barring temporary setbacks from pricing actions or new factory costs).
- **Switching Costs / Ecosystem:** Tesla owners to some extent are “locked in” to an ecosystem – they rely on Tesla software and services (navigation, music streaming, etc. integrated into the car), and many invest in home charging and Tesla’s mobile app for monitoring the vehicle. The Supercharger network has been a proprietary advantage for long road trips. While another EV could use third-party public chargers, many Tesla owners value the integrated charging experience. Additionally, any purchase of expensive software like FSD is tied to the car (or account), making an owner less inclined to switch brands and lose that investment. These factors create some **switching cost** or at least an inconvenience barrier for customers to leave the Tesla fold and go to a competitor.
- **Intellectual Property and Innovation:** Tesla holds numerous patents in battery design, power electronics, and software. But beyond formal IP, much of Tesla’s advantage comes from *know-how* – years of engineering solving EV-specific challenges (battery thermal management, vehicle software integration, etc.). This know-how and Tesla’s culture of rapid innovation (for instance, designing custom machinery when off-the-shelf equipment proved inadequate in the Model 3 ramp) is hard for competitors to replicate quickly. Tesla also benefits from a high-profile CEO and vision that helps attract top engineering talent, contributing to its innovation capacity.
- **Regulatory & Policy Edge:** As an established EV leader, Tesla has benefited from pro-EV regulations (such as ZEV credits, and subsidies that favor EV purchases). It has navigated regulatory environments globally and even earned revenue from credits. Also, being purely focused on EVs means Tesla doesn’t bear the burden of transitioning from legacy combustion technology – a challenge incumbent automakers face both financially and strategically. This singular focus can be considered a competitive edge as the industry transforms.

In summary, Tesla’s business model is built around high-value technology products (vehicles and energy systems) sold directly to customers, leveraging a strong brand, integrated technology, and scale advantages. The company’s ability to generate multiple revenue streams (hardware, software, services) and control much of the value chain underpins its strong margins and growth, while its continuous innovation and infrastructure investments reinforce competitive moats such as brand loyalty, network effects (charging, data), and cost leadership in the EV space.

## 3. Competitive Landscape  
Tesla faces intensifying competition as the automotive industry shifts towards electric mobility. **Direct competitors** in the electric vehicle market include both established automakers with EV programs and pure-play EV companies:

- **Traditional Automakers (Legacy OEMs):** Companies like **Volkswagen Group**, **Toyota**, **Ford**, **General Motors**, **Hyundai-Kia**, and **Stellantis** are investing tens of billions into EV development and ramping up a slate of battery-electric models. Volkswagen, for instance, has the ID.3/ID.4 in Europe and a goal to become a global EV leader, achieving about 7-8% global BEV market share as of 2024 ([www.statista.com](https://www.statista.com/chart/27733/battery-electric-vehicles-manufacturers/#:~:text=not%20increase%20that%20much%20in,vehicles%20in%20the%20upcoming%20year)). Legacy competitors enjoy well-known brands, existing manufacturing scale, and extensive dealership/service networks, but many are still catching up to Tesla in EV technology and software. In the U.S., Ford and GM have launched popular EVs (the Ford F-150 Lightning electric pickup and Mustang Mach-E, GM’s Cadillac Lyriq and forthcoming Chevrolet EVs), but they have encountered challenges scaling production profitably. Ford’s EV division, for example, has been incurring large losses as it scales up, highlighting the difficulty of matching Tesla’s cost structure and battery supply chain efficiency. These legacy automakers generally have lower pricing power in EVs due to lack of scale and are often pricing aggressively (sometimes below cost) to gain market share – a dynamic that Tesla has been responding to with its own price cuts to stay competitive.

- **Pure-Play and New EV Entrants:** This category includes U.S. startups like **Rivian** (focused on electric pickups and SUVs) and **Lucid Motors** (luxury electric sedans), as well as Chinese EV specialists like **BYD, NIO, Xpeng, Li Auto**, and others. **BYD** (Build Your Dreams) is the most noteworthy – it has rapidly grown to challenge Tesla’s global EV volume. In 2023, BYD (a Chinese company with backing in battery tech and manufacturing) sold roughly 1.8 million new energy vehicles, including about 0.9–1.0 million pure electric vehicles, putting it nearly on par with Tesla’s 1.31 million that year ([www.ft.com](https://www.ft.com/content/81f97fd3-c889-46af-990c-051d32821102#:~:text=BYD%20has%20overtaken%20Tesla%20as,dominance%20has%20eroded%20due%20to)) ([www.ft.com](https://www.ft.com/content/b9797e5a-1346-4779-9125-686932c9d8ec#:~:text=In%202024%2C%20BYD%2C%20China%27s%20leading,79%20million)). BYD combines its own battery production and a range of models (from budget to luxury) mostly in China, and it has started expanding overseas. BYD’s market share in global BEV rose significantly between 2021 and 2024 (gaining ~9 percentage points) while Tesla’s share, though still the largest, edged down to ~18–19% ([www.statista.com](https://www.statista.com/chart/27733/battery-electric-vehicles-manufacturers/#:~:text=As%20of%20September%202024%2C%20Tesla,EV%20Volumes%20published%20on%20CleanTechnica)). Other Chinese automakers like **SAIC** (which produces EVs including under the MG brand and joint ventures) also hold significant share, especially in China’s mass-market EV segment. U.S. startups Rivian and Lucid have high-end products but much smaller scale (tens of thousands of units or less) and face their own production ramp and financial challenges, so they are not yet threatening Tesla’s broad market leadership, though they do compete in niche segments (e.g., Rivian in pickup trucks before Tesla’s Cybertruck arrives). **Tesla’s product differentiation** remains significant: its vehicles are known for superior range, a cohesive software experience, and the expansive charging network. Many competitors have yet to match Tesla’s combination of range and performance at a given price point – for instance, Tesla’s Model Y dominates global EV sales in the crossover category due to its balance of range, tech features, and price after recent cuts. However, competitors are narrowing the gap; some offer features Tesla currently doesn’t (like varied body styles – e.g., compact EVs, which Tesla won’t have until its next-generation platform). **Pricing power** has been a Tesla strength – it historically charged premium prices and earned high margins, but with 2023’s price cuts, Tesla demonstrated willingness to reduce prices to defend its volumes. This indicates that as competition increases, Tesla’s pricing power is challenged; it can no longer price significantly above rivals without affecting demand. Still, Tesla’s cost leadership often means it can cut prices and remain profitable, whereas some competitors selling at similar prices may be taking losses (especially true for newer entrants).

- **Adjacent and Emerging Competitors:** Outside of direct car-to-car competition, Tesla faces potential competition from tech companies and mobility service providers in areas like autonomous driving and transportation services. For example, **Alphabet’s Waymo** and **GM’s Cruise** are leaders in self-driving car technology and are operating robo-taxi services in select cities. If autonomous ride-hailing becomes a major segment, Tesla’s approach of developing full self-driving in-house (with plans for its own robotaxi network) pits it against these players. Similarly, any hint of **Apple** or other tech giants developing an electric car could create a formidable competitor given their resources and brand (Apple has long been rumored to work on an “Apple Car”, though no product has materialized). In the energy sector, Tesla’s solar and storage business competes with solar installers and battery providers; for instance, **Sunrun** and **SunPower** are big residential solar rivals, and **LG Chem**, **Panasonic**, **Samsung SDI**, and others provide battery storage solutions that compete with Powerwall and Megapack in certain projects. These adjacent competitors don’t challenge Tesla’s automotive sales directly but vie for share in the broader clean energy and future mobility ecosystem.

**Comparison & Market Share:** Tesla remains the **global BEV market leader** by volume for pure electric vehicles, with about 18–20% worldwide BEV market share in recent years ([www.statista.com](https://www.statista.com/chart/27733/battery-electric-vehicles-manufacturers/#:~:text=As%20of%20September%202024%2C%20Tesla,EV%20Volumes%20published%20on%20CleanTechnica)). Its nearest global competitor, BYD, has roughly a similar share when excluding plug-in hybrids, and both companies together account for roughly one-third of global BEV sales ([www.statista.com](https://www.statista.com/chart/27733/battery-electric-vehicles-manufacturers/#:~:text=As%20of%20September%202024%2C%20Tesla,EV%20Volumes%20published%20on%20CleanTechnica)). In the United States, Tesla’s dominance is even greater – it commanded an estimated ~60–65% of the U.S. EV market in 2022-2023, though this is slowly declining as more models from competitors launch. In China, Tesla is among the top sellers of premium EVs (like Model Y is often the best-selling SUV in its category), but local brands (BYD, Wuling, etc.) lead the lower-priced segments. Europe has been a more competitive region for Tesla; it faces strong competition from Volkswagen Group, Stellantis (Peugeot, Fiat EVs), and Hyundai/Kia, and has seen its market share there come down as local brands offer popular EVs. For instance, Tesla held a very high share in Europe around 2019–2020 with the Model 3, but by 2023 its share of Europe’s EV market had slipped as new entrants arrived ([insideevs.com](https://insideevs.com/news/727744/tesla-market-share-2024-q2/#:~:text=If%20we%20take%20a%20look,%E2%80%94probably%20around%203.8)) ([insideevs.com](https://insideevs.com/news/727744/tesla-market-share-2024-q2/#:~:text=In%20Europe%2C%20the%20weakening%20is,later%20this%20year)). Still, Tesla’s **Model Y became one of Europe’s top-selling vehicles overall (EV or not) in 2023**, reflecting that with sufficient supply it can outsell many competitors’ offerings.

In terms of **growth trajectory**, Tesla and its competitors are on different curves: Tesla, at a larger base, grew vehicle deliveries ~40% in 2022, while many startups grew faster percentage-wise from a small base. Chinese EV makers, collectively, are growing very rapidly domestically and beginning export pushes (e.g., BYD expanding to Europe, NIO also exploring overseas) which could challenge Tesla internationally. Legacy automakers are aiming for high EV growth rates (Ford, GM target hundreds of thousands of EVs annually within a couple of years), but they often still sell far more combustion vehicles than EVs, whereas Tesla is an *all-EV* company – 100% of its sales are electric, which continues to differentiate it as consumers looking specifically for an electric specialist often gravitate to Tesla.

**Key Competitive Metrics:** Beyond market share, other metrics to compare:
- **Product Range & Lineup:** Tesla’s lineup is relatively limited (few models, all larger than a compact, and all priced above ~\$40k after recent cuts), whereas competitors collectively offer a wider variety of body styles and price points. For example, GM and Nissan offer some cheaper EVs (Chevy Bolt, Nissan Leaf around \$30k) which Tesla doesn’t yet play in. Tesla’s product cadence (next generation platform for a lower-cost model) will be crucial to fend off competitors at the low end. On the high end, Lucid’s Air sedan beats Tesla in some specs like range and luxury features, but at much higher price and low volume. Mercedes, BMW, Audi EVs bring brand cachet in luxury, but often inferior range or higher price for similar performance compared to Tesla.
- **Technology & Differentiation:** Tesla’s strength is in software and integration (its cars feel like “tech gadgets” to consumers, with sleek user interfaces and frequent updates). Some competitors like Mercedes have advanced driver assists and interior tech too, but Tesla’s Autopilot/FSD is a unique selling point (despite controversies, many customers value the semi-autonomous capabilities). Traditional OEMs are still developing OTA update capabilities and unified software – an area Tesla leads. Battery range and efficiency is another metric; Tesla vehicles typically top range comparisons in their segments thanks to efficient powertrain and aerodynamics. Competitors like Lucid have one model with higher range, but generally Tesla leads in efficiency (miles per kWh).
- **Cost and Pricing:** Tesla’s recent pricing strategy shows it can undercut many rivals on price while still earning a profit. For instance, after price cuts, the base Model Y in the U.S. became cheaper than some competing electric SUVs from Ford and Volkswagen, putting pressure on those competitors who have higher production costs. Tesla’s direct sales and lack of dealer markups also mean customers pay essentially the sticker price, whereas some high-demand competitor EVs saw dealer markups. **Gross margin** is a telling metric: even after cutting prices, Tesla’s automotive gross margin (~15-20% in 2023) remains above the gross margins of many rivals’ EV businesses, which are often negative or low-single-digit until they scale. This suggests Tesla retains a cost advantage.
- **Growth & Investment:** Tesla’s growth (40-50% annually recently) outpaced the overall auto industry (flat-to-single-digit) and even the overall EV market growth in some years. However, as Tesla’s base grows, maintaining high growth is challenging, and competitors growing from a smaller base may grow faster in percentage terms. Many competitors are investing heavily (e.g., GM committed ~\$35B to EVs through 2025, VW similarly massive budgets), potentially closing the gap in technology and manufacturing over time. Tesla’s R&D spend, while large in absolute terms (~\$3.0B in 2022), is smaller than some legacy OEMs’ total R&D – but it’s highly focused on EV/tech, whereas legacy have to split budgets with engine development etc. 

Overall, Tesla currently stands as the **benchmark** in the EV industry in terms of volume, technology integration, and profitability, but the landscape is quickly evolving. Major automakers bring scale and manufacturing expertise, and Chinese players bring aggressive innovation and cost competition. This competitive backdrop has already led to a **price war** in key markets – for example, Tesla’s price cuts in China prompted BYD and others to also cut prices ([www.reuters.com](https://www.reuters.com/business/autos-transportation/byd-may-hand-back-top-ev-seller-title-tesla-after-q1-sales-decline-2024-04-02/#:~:text=33.7,year)), which benefits consumers but could squeeze everyone’s margins. Tesla’s challenge will be to maintain its lead through continual innovation (e.g., introducing a next-gen low-cost model, advancing its autonomous tech) and leveraging its infrastructure (charging, brand community) to keep customers within its ecosystem, as competitors target every segment Tesla operates in.

## 4. Supply Chain Positioning  
Tesla’s supply chain strategy is a mix of **vertical integration** in key areas and strategic partnerships in others, positioning it uniquely upstream and downstream compared to traditional automakers.

**Upstream (Suppliers & Manufacturing Inputs):** One of the most critical areas for Tesla is its battery supply. Tesla is **highly dependent on suppliers for battery cells**, relying primarily on a few key partners. The main suppliers include **Panasonic** (its long-time partner producing 18650 and 2170 cells, especially at Tesla’s Nevada Gigafactory) and **Contemporary Amperex Technology Co. Limited (CATL)**, the Chinese battery giant that supplies Tesla’s Shanghai factory with lithium iron phosphate cells for standard range models ([www.sec.gov](https://www.sec.gov/Archives/edgar/data/1318605/000095017023001409/tsla-20221231.htm#:~:text=We%20are%20dependent%20on%20the,In%20the%20long%20term%2C%20we)). Tesla has also sourced cells from **LG Energy Solution** for certain vehicles. Notably, these relationships often involve Tesla co-investing or tying up capacity; for instance, at Gigafactory Nevada, Panasonic produces cells on site exclusively for Tesla under a partnership agreement ([www.sec.gov](https://www.sec.gov/Archives/edgar/data/1318605/000095017023001409/tsla-20221231.htm#:~:text=Panasonic%20has%20partnered%20with%20us,As%20the%20terms%20of%20the)). Because of this dependency, Tesla has *to date qualified only a limited number of suppliers for battery cells*, meaning any disruption at a key supplier could impact Tesla’s production ([www.sec.gov](https://www.sec.gov/Archives/edgar/data/1318605/000095017023001409/tsla-20221231.htm#:~:text=We%20are%20dependent%20on%20the,In%20the%20long%20term%2C%20we)). To mitigate this, Tesla is pursuing **in-house battery manufacturing**: it has developed its own larger-format 4680 battery cell and is ramping production of these at facilities in Fremont and Texas. If Tesla succeeds at scale with its 4680 cell, it could reduce reliance on external suppliers and secure more control over one of the costliest components in its cars. 

Beyond batteries, Tesla manufactures many critical components internally: electric motors, inverters, vehicle control units, and even seats are made by Tesla. However, it still sources standard automotive parts (tires, glass, semiconductors, etc.) from a broad supply chain. During the global semiconductor shortage in 2021, Tesla’s flexibility in rewriting software to use alternative chips was credited for keeping production running, but it still faced challenges securing enough automotive chips. Tesla sometimes relies on **single-source suppliers** for specialized parts (for example, certain chips or the large aluminum castings used in its Model Y chassis, which come from unique suppliers of giga-press machines). The company’s 10-K filings acknowledge that inability of any single-source supplier to deliver on time could materially affect production ([www.sec.gov](https://www.sec.gov/Archives/edgar/data/1318605/000095017023001409/tsla-20221231.htm#:~:text=We%20are%20dependent%20on%20our,financial%20condition%20and%20operating%20results)). To manage risk, Tesla has engaged in strategic commodity sourcing: it has multi-year contracts with producers of lithium, nickel, and other battery minerals, sometimes including upfront commitments. Elon Musk has even mentioned Tesla might vertically integrate into lithium refining to secure that part of the supply chain. 

Geographically, Tesla’s supply chain spans the globe. For the Fremont (California) and Texas car plants, many parts (besides batteries) come from North American or global suppliers; for the Shanghai Gigafactory, Tesla sources extensively in China (benefiting from China’s robust EV supply base); and in Europe (Berlin Gigafactory), it’s building a localized supply chain within Europe. Localizing supply helps reduce logistics costs and currency risk. Tesla’s raw material sourcing is also diversified geographically – e.g., lithium from Australia or South America through partners, cobalt mostly from responsible mining in countries like Australia (Tesla has moved to reduce cobalt use significantly in its batteries). Tesla’s vertical integration in manufacturing extends upstream in another way: it acquired companies like **Grohmann Engineering** in Germany (automation engineering) and **Maxwell Technologies** (supercapacitor and battery tech) to absorb their technology and talent into Tesla’s production process. This indicates Tesla’s strategy of owning crucial production know-how and not being overly reliant on external machine suppliers or processes for breakthroughs. 

**Downstream (Distribution & Customers):** Tesla’s position in the downstream supply chain is distinctly **direct-to-consumer**. Unlike virtually all legacy automakers, Tesla **owns its distribution network**: there are no franchised Tesla dealerships. Instead, Tesla operates showrooms (often in malls or high-traffic areas) and “galleries,” but purchases are done through Tesla’s website with fixed pricing. Vehicles are delivered through Tesla’s own delivery centers or shipped directly to customers. This model gives Tesla more control over the customer experience and pricing, but it also means Tesla carries inventory and logistics responsibilities that dealers normally would (Tesla must handle vehicle delivery, registration, and sometimes trade-ins). It also faced legal challenges in some U.S. states that have dealership franchise laws; Tesla has lobbied and negotiated case by case to allow its own stores in many states, though some still require workaround methods (like selling online and delivering from out-of-state). 

In terms of **customer service and support**, Tesla runs its own service centers for maintenance and repairs and mobile service fleets that can do repairs at a customer’s location. This is an ongoing challenge, as the rapid increase in Tesla’s cars on the road has sometimes outpaced the expansion of service infrastructure, leading to customer complaints about service delays. However, Tesla’s strategy is to design cars that need minimal regular maintenance (e.g., no regular oil changes, fewer moving parts than gas cars) and to use over-the-air diagnostics and fixes when possible (software updates to fix certain issues remotely).

Tesla also directly manages the **charging infrastructure** for its customers via the **Supercharger network**. This network of DC fast-charging stations, which Tesla began building in 2012, has grown to over 45,000 connectors globally. It’s a vertically integrated ecosystem: Tesla makes its own charging stations and installs/operates them (often on its own or leased property near highways). For years, this network was a Tesla-exclusive asset – only Tesla vehicles could charge there – enhancing the value proposition of owning a Tesla for customers who road-trip frequently. In 2023, Tesla announced deals to open portions of the network to other EVs (using a standardized adapter or Tesla’s North American Charging Standard plug) in exchange for government funding and presumably usage fees. Downstream, this positions Tesla almost as a *utility or service provider* to the broader EV customer base, not just its own customers, which could turn charging into a notable revenue stream and strengthen Tesla’s influence on charging standards across the industry.

**Key Partners and Dependencies:** Tesla’s major upstream partners include:
- **Battery Partners:** As noted, Panasonic (production partner at Nevada Gigafactory), CATL (lithium iron phosphate cells in China), and LG Energy Solution. These relationships are crucial; for example, at one point Panasonic’s production issues constrained Model 3 ramp until resolved. Any disruption (e.g., if a partner’s factory has technical issues or if a contract lapses without replacement) is a significant risk ([www.sec.gov](https://www.sec.gov/Archives/edgar/data/1318605/000095017023001409/tsla-20221231.htm#:~:text=We%20are%20dependent%20on%20the,In%20the%20long%20term%2C%20we)).
- **Suppliers of Motors/Electronics:** While Tesla designs its own motors and electronics, it still sources materials like high-grade silicon carbide for inverters (from suppliers like STMicro) or aluminum for castings. Specific unique suppliers (like Italy’s IDRA Group, which provides the giant casting presses Tesla uses) are important partners for Tesla’s manufacturing innovation, though not in terms of recurring supply of parts, rather supply of manufacturing equipment.
- **Manufacturing Partners:** In its early years, Tesla partnered with Lotus (for the Roadster’s glider chassis) and Toyota (the NUMMI plant which became Tesla’s Fremont Factory was acquired from Toyota). These legacy partnerships are no longer central, as Tesla now largely builds in-house, but it shows how Tesla was once dependent on others to get started. Today, a kind of manufacturing partner is **Tesla’s suppliers of factory automation** – companies that provide robotics, assembly lines, etc. Tesla’s acquisition of Grohmann Engineering was to bring a lot of that expertise internally so it’s less dependent on third-party integrators.

Downstream, Tesla’s “partners” are fewer since it eschews dealers. However:
- It partners with some **external installers** for solar products (Tesla has its internal solar installers, but also certified third-party installers).
- For energy projects (like installing Megapacks at a utility site), Tesla works closely with construction and engineering firms that act as partners/contractors.
- In charging, Tesla has started partnering with restaurants, hotels, and other businesses to host Superchargers (site hosts), and with governments (to access funding for expanding charging networks).
- Tesla also has a customer referral program which effectively turns loyal customers into marketing partners (referrals for Tesla purchases can earn credits or perks).

**Concentration and Dependencies:** Tesla’s supply chain has some notable concentrations:
   - **Battery Cell Supply:** Concentrated among a few suppliers. This is a dependency that Tesla is actively addressing by developing its own cell production. Nonetheless, in the near term, if Panasonic or CATL had a major disruption, Tesla’s production would be directly impacted ([www.sec.gov](https://www.sec.gov/Archives/edgar/data/1318605/000095017023001409/tsla-20221231.htm#:~:text=We%20are%20dependent%20on%20the,In%20the%20long%20term%2C%20we)).
   - **Geographical Concentration:** Tesla’s manufacturing is concentrated in a few plants (California, two in the U.S. including Texas, one in China, one in Germany). Particularly, the Shanghai Gigafactory has become one of Tesla’s highest output plants (~50% of output in 2022). This makes Tesla somewhat exposed to China-specific risks (e.g., COVID lockdowns affected Shanghai production in 2022, as did temporary power rationing). Similarly, supply base in China is local – any trade restrictions or geopolitical issues could impact parts supply or export ability from that plant. Tesla is partially mitigating this by building plants on three continents to localize production.
   - **Key Personnel/Talent:** While not a supplier “dependency” per se, Tesla has dependence on certain key talent and leadership (notably Elon Musk and a relatively small executive team). This isn’t about physical supply chain, but it is a single point of potential disruption if leadership were to change suddenly.
   - **Customers/Market Concentration:** On the downstream side, Tesla’s sales are broadly distributed (no single customer makes up a material percentage – cars are sold to individual consumers). However, it does have regional concentration: China accounted for about 20-25% of sales in 2022, and the U.S. around 40-50%. So demand or policy changes in China (like EV subsidy changes or rising local competition) are an external factor that could significantly affect Tesla’s volume if not balanced by growth elsewhere.

In summary, Tesla’s supply chain positioning is characterized by a **high degree of control** over important inputs (doing more in-house than traditional carmakers) while still relying on specialized partners for critical components like battery cells. Its direct-to-consumer model cuts out intermediaries, giving it more downstream control and customer data, but also means Tesla bears the costs and responsibilities of distribution, sales, and service infrastructure. The company’s success has, in part, come from deftly managing this unique supply chain – securing materials and ramping production in a new segment (EVs) faster than others – and this will remain a focus area as global demand and competition in EVs rise.

## 5. Financial and Operating Leverage  
**Financial Leverage (Debt & Balance Sheet):** Tesla has been reducing its financial leverage in recent years to the point of having a very strong balance sheet for a manufacturing company. Historically, Tesla used a mix of equity raises, convertible debt, and corporate bonds to fund its growth (especially during heavy investment periods like Model 3 ramp around 2017-2019). However, after achieving consistent profitability and large positive cash flows, Tesla paid down much of its debt. As of year-end 2022, Tesla’s total debt was around \$1.0 billion (excluding vehicle and energy product financing), which is very low relative to its equity base ([www.sec.gov](https://www.sec.gov/Archives/edgar/data/1318605/000095017023001409/tsla-20221231.htm#:~:text=Current%20portion%20of%20long,991)). The company held about \$22 billion in cash and equivalents at that time, meaning it has **net cash** rather than net debt on its balance sheet. This conservative debt position significantly reduces financial risk; Tesla’s debt-to-equity ratio is close to zero and interest obligations are minimal (interest expense in 2022 was small, and Tesla actually earns interest income given its cash pile in a higher rate environment).

Recognizing this improvement, credit rating agencies have upgraded Tesla. In late 2022, S&P Global Ratings raised Tesla’s credit rating to **investment grade (BBB)** ([electrek.co](https://electrek.co/2022/10/06/tesla-tsla-upgraded-investment-grade/#:~:text=upgraded%20to%20investment%20grade%20by,TSLA%29%20has)), citing its solid cash flow and operating performance. Moody’s followed in early 2023, also upgrading Tesla to Baa3 (investment grade). These upgrades reflect that Tesla is now seen as financially stable, with manageable debt levels and strong interest coverage. Tesla’s current **credit profile** is supported by its robust market capitalization (which provides financial flexibility) and consistent free cash flow generation. If needed, Tesla can access capital markets relatively easily – it took advantage of high share prices in 2020-21 to issue new equity and strengthen its balance sheet. Thus, Tesla’s **financial leverage** is low: it does not rely heavily on borrowed money to finance operations or growth at this stage, which is somewhat unique for an automaker (most peers carry substantial debt). The company also doesn’t have defined-benefit pension liabilities that burden some legacy auto manufacturers. 

The flip side is Tesla’s growth is largely equity-financed or internally funded. It pays no dividend and, to date, has not engaged in significant share buybacks (there have been discussions and shareholder interest in buybacks given cash accumulation, but Tesla has prioritized retaining cash for expansion). Capital allocation at Tesla has been focused on **reinvestment** – building new factories (capex) and R&D in new technologies (like autonomy, AI chips, etc.), rather than returning cash to shareholders. This aligns with management’s growth-oriented strategy.

**Operating Leverage (Cost Structure & Margins):** Tesla’s cost structure includes a high proportion of fixed costs – these are expenses that do not vary directly with short-term production volume, such as depreciation on factories and equipment, R&D spending, corporate overhead, and certain labor costs. Building and running large factories involves significant fixed costs (equipment, automation systems, facility maintenance). Tesla’s strategy of high automation (e.g., robotic manufacturing lines) increases upfront fixed costs but can reduce variable labor costs per unit. The **operating leverage** refers to how changes in revenue translate to changes in operating profit. In Tesla’s case, once the factories are built and the R&D done, producing more cars spreads those fixed costs over more units, which historically led to widening gross and operating margins as volume grew (this was evident when Model 3 production scaled up: unit costs fell and margins improved dramatically from 2018 to 2019). However, operating leverage also means that if volumes fall short, Tesla’s margins can contract quickly since those fixed costs still must be paid. The company experienced such pressure in early years when deliveries were below capacity; it needed high volume to turn a profit after investing in capacity.

By 2022, Tesla achieved an operating margin of about 16–17%, which is extremely high for an automaker, thanks to high volume and pricing. In 2023, with price cuts, the **margin sensitivity** became apparent: gross margin fell ~10 percentage points from 2022 to mid-2023 with the lower pricing, as fixed costs were spread over increasing volume but not enough to fully offset the revenue per unit drop ([www.reuters.com](https://www.reuters.com/business/autos-transportation/tesla-revenue-sees-surprise-rise-second-quarter-revenue-2024-07-23/#:~:text=Automotive%20gross%20margin%20excluding%20regulatory,unveiling%20delayed%20to%20October%2010)). Still, Tesla remained solidly profitable. The company has some flexibility in adjusting costs – for example, it can slow hiring or scale back certain expenses if needed. In mid-2023, Elon Musk indicated Tesla would push for higher volume even if it means lower margins in the short term, implying Tesla is willing to use its cost advantage and absorb lower per-unit profit to grow (a luxury many competitors without profitability cannot afford). This strategy banks on **scalability**: those larger volumes could later be monetized via software (like FSD subscriptions) or simply lead to greater economies of scale in production and procurement, restoring margins in the long run.

Tesla’s **gross margin** structure also benefits from revenue streams with little variable cost, notably regulatory credit sales (pure profit) and software sales like FSD (development cost is fixed, but selling an upgrade is nearly pure margin). These contribute to operating leverage: after covering the big fixed costs of developing software, each extra sale is high margin.

**Cash Flow Generation:** Tesla’s business turned into a strong cash generator in recent years. **Operating cash flow** has grown with profits – in 2022, Tesla generated around \$14.7 billion of operating cash flow. Cash flow is bolstered by Tesla’s working capital dynamics: Tesla often has a favorable cash conversion cycle. Customers typically pay (or arrange financing) at or before delivery, and Tesla in many cases receives cash quickly upon sale. Meanwhile, Tesla enjoys some credit from suppliers (accounts payable). In fact, Tesla often had a **negative cash conversion cycle**, meaning it received customer cash faster than it paid its suppliers. This dynamic, along with customer deposits on orders (e.g., reservations for Cybertruck or Semi provide interest-free cash upfront), has helped Tesla’s cash flow. **Free cash flow** (operating cash flow minus capital expenditures) has been positive and healthy since 2019. In 2022, Tesla’s free cash flow was about \$7.6 billion, even after heavy investments in new factories in Texas and Germany. Tesla’s capital expenditures have been increasing but at a controlled pace (around \$7–9 billion annually recently) as it builds production and battery capacity. The ability to fund capex out of operating cash flow is a positive sign of financial strength.

**Working Capital:** Tesla’s working capital management is efficient by design. The direct sales model means Tesla does not have large receivables from dealers (which traditional OEMs often do). Inventory is managed tightly – vehicles are usually made to order or quickly assigned to customers, though Tesla’s inventory days have inched up as volume grew and it moves to batch production. Nonetheless, compared to the industry, Tesla’s inventory levels relative to sales are low. Accounts payable (to suppliers) provide a source of short-term financing. Overall, Tesla’s working capital is often near **breakeven or slightly negative**, meaning the business isn’t tying up much cash in day-to-day operations – a rarity in auto. This helps with cash flow and reduces the need for working capital financing.

**Capital Allocation:** Tesla has thus far **reinvested the majority of its earnings** back into the business. The company does not pay dividends (common for growth companies). In late 2022, management discussed the possibility of share buybacks, and the board authorized a potential buyback program, but as of mid-2023 no significant buybacks had been executed. The hesitation likely stems from the priority to keep cash for expansion (new models, new factories, vertical integration in battery supply, etc.) and also to maintain a war chest given economic uncertainty. Tesla did execute stock splits (to improve liquidity and accessibility of the stock price) – a 5-for-1 split in 2020 and a 3-for-1 split in 2022 – but those don’t change capital structure fundamentally. Tesla’s **capital allocation strategy** can be summarized as: invest in growth (capex, R&D), maintain a strong cash buffer, and avoid unnecessary debt. In fact, Tesla’s positive free cash flow allowed it to start **paying down debt early** – many of its convertible bonds were converted to equity or paid off, and other loans (including some from Chinese banks used for the Shanghai factory) were largely repaid by 2021. This leaves Tesla with a very clean balance sheet. If growth opportunities arose (e.g., building an additional Gigafactory or acquiring a strategic resource company), Tesla has the financial capacity to deploy cash or raise funds as needed. Another aspect of capital allocation is acquisitions: Tesla has been generally *selective and modest* in M&A – opting for small strategic acquisitions (as mentioned, Grohmann Engineering, Maxwell, etc.) rather than large mergers. There’s no sign Tesla plans any large acquisition; it has favored organic growth.

In terms of **financial risk**, Tesla’s low debt and strong cash flow mean it has low financial leverage risk. It also means the company could weather a downturn more easily than if it had high interest burdens. The main financial sensitivity for Tesla is operating performance – because if a severe recession or a big drop in demand occurred, Tesla’s profits and operating cash might drop (due to high operating leverage), but the company has the cash reserves and lack of debt to survive periods of weaker earnings without insolvency concerns that more leveraged companies might face.

**Summary of Leverage:** Tesla’s combination of *low financial leverage* and *high operating leverage* creates a potentially volatile but powerful financial profile. High operating leverage contributed to big profit increases as volume grew, and it could lead to profit volatility if sales stagnate or decline. However, with essentially no net debt, Tesla’s equity holders are not heavily exposed to creditor claims, and management has flexibility to adjust spending if needed. The strong free cash flow generation in recent years indicates that Tesla’s growth is increasingly self-funded – an important milestone for a once capital-hungry startup now turned into a financially sturdy industry leader.

## 6. Valuation  
Valuing Tesla has been a subject of much debate, as the company straddles the line between a high-growth tech profile and a capital-intensive automaker profile. Several valuation methodologies can be considered:

- **Income-Based Valuation (DCF):** A **Discounted Cash Flow (DCF)** analysis is a fundamental approach that would project Tesla’s future free cash flows and discount them to present value. Key inputs for a DCF on Tesla include the assumed revenue growth rate (Tesla has targeted ~50% annual vehicle delivery growth in the near term, but consensus expects that to moderate over time), profit margins (gross and operating margins which could improve with scale but are currently pressured by price cuts), capital expenditures (ongoing factory investments), and the discount rate (which would reflect Tesla’s cost of capital – largely equity, as debt is minimal). DCF valuations for Tesla are *very sensitive* to long-term assumptions such as the terminal growth rate and margins, given that much of Tesla’s presumed value comes from its earnings far in the future (a characteristic of growth companies). For example, a DCF might assume Tesla eventually sells tens of millions of cars per year decades ahead or generates significant cash from robotaxi services – small changes in those endpoints can swing the valuation significantly. Analysts often use scenario-based DCFs for Tesla: a **bull case** might assume successful expansion to a 10+ million vehicles/year company with software-like margins on autonomous services, whereas a **bear case** might assume Tesla grows to only a few million cars/year with automotive-average margins. The discount rate (WACC) for Tesla is usually set relatively high (to account for execution risk and stock volatility), and with rising interest rates in the broader economy, the discounting has a larger impact on such long-dated cash flows.

- **Market-Based Valuation (Multiples):** Many analysts look at **relative valuation multiples**. Traditional auto metrics like price-to-earnings (P/E) or enterprise value-to-EBITDA for Tesla have typically been far higher than legacy peers. As of mid-2023, Tesla traded at a forward P/E in the range of **50-70x earnings**, vastly above legacy automakers that often trade in single-digit P/Es. The rationale for such a premium is Tesla’s superior growth and margin profile. However, there’s been some convergence: Tesla’s stock price corrected from all-time highs and its earnings grew, so the P/E has moderated from extreme levels seen in 2020-21. Other multiples include price-to-sales (P/S); Tesla’s P/S might be around 8–10x, again much higher than traditional auto companies often below 1x. Some analysts compare Tesla to high-growth tech or consumer companies (like Apple or semiconductor firms) rather than autos, which could justify higher multiples. A **PEG ratio** (P/E to growth rate) is sometimes cited – if Tesla is growing earnings 40-50%, a high P/E can be rationalized. Peers for multiple comparison are tricky: one might compare to other EV pure-plays (like NIO, Lucid, Rivian) which also have high multiples or no earnings, or compare to general autos (GM, Ford, Toyota) which makes Tesla look expensive but those peers lack Tesla’s growth. Many equity research reports use a blend: for example, applying a multiple to Tesla’s automotive business based on peers plus a higher multiple or DCF for Tesla’s potential software and energy businesses (a sum-of-parts approach).

- **Asset-Based & Sum-of-Parts:** An asset-based valuation (e.g., book value) is not very meaningful for Tesla as a going concern because its market value far exceeds book value (like many tech companies). However, a **sum-of-parts** analysis can be informative: it would attempt to separate Tesla’s businesses (e.g., the core vehicle manufacturing, the high-margin software/autonomy component, the energy segment, perhaps the charging network) and value each somewhat independently. For instance, one might value the auto manufacturing portion at an EBITDA multiple closer to autos, the software/FSD portion at a tech-like multiple, and the energy storage business at a renewable or utility-like multiple. This is complex, but it recognizes that Tesla isn’t a monolithic automaker – it has unique pieces that could warrant different valuations. As of now, the market largely prices Tesla in one lump sum, but bulls argue that embedded in Tesla’s valuation is not just car making, but potential future businesses (like a robotaxi network, Tesla Energy becoming a utility-scale player, etc.). Additionally, an **LBO (Leveraged Buyout) valuation** is generally not applicable for Tesla – the company’s market cap (hundreds of billions) and growth focus make it far too large and not suitable for an LBO (which typically targets mature companies with stable cash flows that can support heavy debt). Any theoretical LBO model would find Tesla’s cash flows insufficient for the level of debt needed to take it private at current valuations (indeed, Elon Musk’s own 2018 attempt to consider taking Tesla private at \$420/share did not materialize, illustrating the difficulty and enormous capital that would be required).

**Important Valuation Inputs & Metrics:** For Tesla, several specific metrics are closely watched:
  - **Delivery Numbers and Growth Rate:** The pace of vehicle delivery growth (quarterly and annual) is one of the most immediate drivers of valuation sentiment. Tesla’s stock often moves on whether it meets or misses delivery expectations. Sustained 40-50% growth supports higher valuation assumptions, whereas any sign of plateau or slowing to, say, 10-20% could compress multiples quickly as the narrative shifts from “high growth” to “maturing”.
  - **Profit Margins:** Gross margin, operating margin, and EBITDA margins are crucial. Tesla’s strong margins have underpinned much of the optimism that it can be both high-growth and profitable. If margins erode (as happened with the 2023 price cuts), analysts revisit their earnings forecasts and valuation – indeed, Tesla’s stock saw downward pressure when its Q2 2023 margins came in at a five-year low ([www.reuters.com](https://www.reuters.com/business/autos-transportation/tesla-revenue-sees-surprise-rise-second-quarter-revenue-2024-07-23/#:~:text=Automotive%20gross%20margin%20excluding%20regulatory,unveiling%20delayed%20to%20October%2010)). Conversely, improvements or stabilization in margin are taken positively.
  - **Capital Expenditure and Efficiency:** How much Tesla must spend to grow is a factor – a lower capex per additional unit of capacity improves DCF values. Tesla has been able to drop its capex per new vehicle produced thanks to manufacturing efficiencies (e.g., the Gigafactory Shanghai was built quickly and at lower cost). Analysts model Tesla’s capex needs to forecast free cash flow; if Tesla needed to, say, double capex to fund new battery plants, that could reduce free cash flows near-term and affect valuation.
  - **Discount Rate and Risk Premium:** Given Tesla’s higher risk profile, many models use a higher discount rate (e.g., 10-12% for cost of equity) which in a DCF significantly reduces the present value of far-future cash flows. Changes in macro interest rates also impact this – as risk-free rates rise, the bar for Tesla’s future earnings is raised (this was one reason high-growth stocks like Tesla saw volatility when interest rates jumped in 2022).
  - **Terminal Value Assumptions:** In DCFs, what happens in the very long term (say 10+ years out) is pivotal. Does one assume Tesla becomes a stable 2% growth firm with, say, 15% operating margin (like a typical large company)? Or does one assume it continues growing rapidly or dominating multiple industries (energy, autos, AI) and thus can sustain a higher terminal growth or margin? These varied assumptions explain why valuation estimates for Tesla can differ by orders of magnitude among analysts.

**Market Sentiment and Analyst Opinions:** Tesla is a widely covered stock, with dozens of Wall Street analysts issuing reports. Analyst opinions range across the spectrum:
  - **Bullish View:** Bulls argue Tesla is not just an automaker; it’s a transformative technology company with potential high-margin software revenue (from FSD/robotaxis) and a large share of the global automotive and energy markets. They point to Tesla’s industry-leading margins, growth, and technology as justification for premium valuation. Extremely bullish analysts (e.g., ARK Invest’s Cathie Wood) have even higher price targets based on scenarios of robo-taxi networks and Tesla entering new businesses like AI training or HVAC systems. Some have price targets factoring in Tesla selling 10-20 million cars a year by 2030 with significant autonomous revenue per car.
  - **Bearish View:** Bears and skeptics focus on the risk that Tesla’s growth will slow and margins will fall as competition catches up, essentially seeing Tesla’s valuation as reflecting *best-case scenarios*. They might use comparables like “if Tesla were valued like GM or Toyota on a per-car or earnings basis, the stock would be a fraction of its current price”. Some have very low price targets (there have been outlier low targets under \$50, implying a belief that Tesla’s stock is dramatically overvalued). Concerns include Elon Musk’s distractions (e.g., spending time on Twitter), inconsistent profitability in the face of price wars, and governance issues.
  - **Consensus:** As of recent data, the **consensus analyst rating** tends to be in the middle – often a “Hold” or modest **Buy**, with a wide range of price targets. For instance, the average target price might be around the low \$300s (when the stock is say \$250), with highs of \$500+ and lows around \$100 or less ([www.tipranks.com](https://www.tipranks.com/stocks/tsla/forecast#:~:text=TipRanks,List%20of%20Analyst%20Forecasts%E2%80%8B%20Detailed)). The breadth of those targets is much wider than for most large-cap stocks, reflecting differing fundamental narratives. Notably, there have been frequent **target price changes**; after Tesla’s big rally in 2020-21, some analysts cut ratings due to valuation stretch, then when the stock fell in late 2022, some upgraded from Sell to Hold or Hold to Buy thinking the price now better reflects fundamentals. Tesla’s volatile share price (it can swing 5-10% on an earnings report or Elon Musk tweet) also means analysts often emphasize the risk factors in either direction.

**Stock Volatility and Ownership:** Tesla’s stock is known for its **high volatility**. Its **beta** (a measure of volatility relative to the market) has historically been above 2.0, meaning it moves on average twice as volatile as the overall market. The stock has seen huge swings: for example, it skyrocketed ~700% in 2020, then in late 2022 it fell about 65% from its peak amid market downturn and Musk’s stock sales. Part of this volatility comes from the company’s results and news flow (which can be dramatic, e.g., hitting production records or facing recalls) and part from its substantial **retail investor following**. Tesla is often considered a “**cult stock**” or even a meme stock in the sense that it’s heavily discussed on social media and online forums, and it has passionate bull communities (like Tesla fans) and bear communities. It’s one of the most traded stocks by retail investors, which can lead to momentum-driven moves. It’s also heavily owned by institutional investors given its size – large index funds (e.g., Vanguard, BlackRock) hold it because Tesla is a top-10 company by market cap. **Liquidity** is very high (tens of millions of shares trade per day), so large positions can be entered/exited relatively easily compared to smaller stocks. Hedge funds have at times taken big swings on Tesla: some prominent ones were short (betting against it) historically and got burned as the stock rose, while others have ridden the momentum on the long side.

**Coverage and Story:** Almost all major Wall Street firms cover Tesla now. It’s a staple of discussion not just in auto-sector research but also among macro and tech investors. Some consider Tesla an indicator of risk sentiment in the market given its high beta. It’s also been grouped with “big tech” in some contexts because of Musk’s tech persona and the company’s Silicon Valley DNA, though fundamentally it’s manufacturing-based. Tesla is sensitive to **macro factors** as well: for instance, interest rates – higher rates can hurt valuations of growth stocks (Tesla included) and can also increase consumers’ auto loan rates which might weaken car demand. Broad economic conditions (like a recession) could hit discretionary purchases like cars. Also, commodity prices (like lithium, nickel) matter for margins. Another macro factor is fuel prices – historically, higher gasoline prices spur more interest in EVs, benefiting Tesla, whereas if fuel is cheap, some consumers feel less urgency to switch to an EV.

In terms of **current market sentiment** (as of mid-2023 to 2024), Tesla’s stock had rebounded strongly from late-2022 lows after investors saw improvements like stabilizing demand (helped by price cuts) and positive developments like the Supercharger network opening deals and optimism around AI. However, the stock’s valuation remains a point of contention, reflecting its tricky position: it’s **expensive relative to current earnings** but potentially cheap if one believes in a future of ubiquitous Tesla robotaxis and energy dominance. Thus, analysts often advise monitoring key metrics (deliveries, margins, and any progress on self-driving) closely, as these will justify or call into question the growth assumptions in the valuation.

## 7. Recent Developments, News & Risk Factors  
In the last 12 months, Tesla has experienced a multitude of significant developments – both positive catalysts and challenges:

**Financial Performance & Demand:** Tesla’s **revenue and earnings trends** have shown strong growth in absolute terms but with some margin compression. For 2022, Tesla posted record revenue of \$81.5 billion (up ~51% YoY) and record net income of \$12.6 billion (up 128% YoY), capping a year of robust expansion ([www.axios.com](https://www.axios.com/2023/01/25/tesla-earnings-elon-musk-price-cuts#:~:text=Tesla%20reported%20record%20profit%20and,Despite%20these%20gains%2C%20Tesla%20acknowledged)). Entering 2023, Tesla faced a more complex picture: concerns about demand elasticity led to a series of **price cuts** on its models globally. In January 2023, Tesla cut prices in the U.S. and Europe by as much as 20% on certain trims ([apnews.com](https://apnews.com/article/fac648952687ea903477422f5001ef0a#:~:text=2023,a%20%247%2C500%20federal%20tax%20credit)), and further adjustments continued through the year. CEO Elon Musk defended these cuts by saying driving higher volume and fleet size is the priority, even if it means lower margins ([www.axios.com](https://www.axios.com/2023/01/25/tesla-earnings-elon-musk-price-cuts#:~:text=2023,Despite%20these%20gains%2C%20Tesla%20acknowledged)). The impact was evident in financials – by the second quarter of 2023, Tesla’s gross profit margin had fallen to around 18.2% (vs ~27% a year earlier) and automotive gross margin (ex-credits) hit a five-year low of 14.6% ([www.reuters.com](https://www.reuters.com/business/autos-transportation/tesla-revenue-sees-surprise-rise-second-quarter-revenue-2024-07-23/#:~:text=Automotive%20gross%20margin%20excluding%20regulatory,unveiling%20delayed%20to%20October%2010)). Nevertheless, **vehicle deliveries** kept climbing: Tesla delivered 422,875 vehicles in Q1 2023 and 466,140 in Q2 2023, both quarterly records. Full-year 2023 deliveries were around 1.31 million, up ~40% from 2022, though slightly below Tesla’s initial target (1.8 million). The company cited high interest rates and a tougher economic environment as headwinds to near-term demand, in addition to increased competition.

**Production & New Models:** On the operational side, Tesla has been ramping new factories and preparing new models:
  - The **Berlin Gigafactory (Germany)** and **Austin Gigafactory (Texas)**, both opened in 2022, have been increasing production of Model Y. By mid-2023, each was targeting 5,000 cars per week output. These ramps are crucial for supplying Europe and the central U.S. respectively without import constraints.
  - Tesla’s first **Tesla Semi** trucks were delivered in December 2022 to PepsiCo, marking Tesla’s entry into heavy trucks after years of prototype development. Production is still limited, but Tesla is planning to scale Semi manufacturing in Nevada.
  - The **Cybertruck**, unveiled in 2019, finally saw its first production unit built in mid-2023, with an official handover event taking place in Q4 2023. Initial deliveries were symbolic (a batch of trucks delivered to employees/customers in late November 2023), and volume production is slated for 2024. The Cybertruck is a key product – it targets the lucrative pickup truck segment (huge in North America) and has over 1 million reservations reportedly. However, its radical design and new manufacturing techniques (e.g., stainless steel exoskeleton) pose execution risks.
  - Tesla has teased a “next generation” platform for a future compact model (often dubbed **Model 2** by media, expected to be a smaller, more affordable EV). In 2023’s Investor Day, Tesla discussed how its next-gen vehicle would be produced with 50% lower cost, but details on launch timing remain undisclosed. The market sees this future model as critical for expanding Tesla’s addressable market and maintaining growth in the later 2020s.

**Leadership and Management Changes:** A notable management change was the **resignation of CFO Zachary Kirkhorn** in August 2023 ([www.cnbc.com](https://www.cnbc.com/2023/08/07/tesla-cfo-zach-kirkhorn-steps-down.html#:~:text=Skip%20Navigation%20Key%20Points%20,Kirkhorn%20will%20stay%20on%20at)). Kirkhorn was respected on Wall Street and seen as a key figure in Tesla’s turnaround to profitability over the past years. His departure (he stayed on through end-2023 as an advisor) caused some stir; it coincided with Musk’s increasingly hands-on involvement at Twitter, leading to speculation about Tesla’s management depth and Musk potentially overextended. Tesla quickly appointed its Chief Accounting Officer, Vaibhav Taneja, as the new CFO. No explicit reason was given for Kirkhorn’s exit, but it came as a surprise, making investors watchful of any further high-level turnover. On the governance front, Tesla’s board had faced criticism for being too closely tied to Musk. In 2023, Tesla added **JB Straubel** (Tesla’s former CTO and co-founder) to the board – a move that was seen positively, adding an EV industry veteran and someone familiar with Tesla’s operations. Musk himself remains the largest shareholder and firmly in control; there is no clear succession plan disclosed, which is a noted governance risk if Musk were to step aside unexpectedly.

**Elon Musk’s Outside Activities:** Elon Musk’s acquisition of Twitter (renamed **X**) in October 2022 had indirect effects on Tesla. Musk sold roughly \$20+ billion worth of Tesla stock in late 2022 to help finance the Twitter deal, which put downward pressure on Tesla’s share price and concerned investors about Musk’s focus. Through late 2022 and early 2023, there were worries that the Twitter saga and Musk’s controversial decisions there (mass layoffs, policy changes) were tarnishing his personal brand and by extension Tesla’s brand, possibly alienating some customers. In 2023, Musk hired a new CEO for X (Linda Yaccarino) and indicated he would reduce his time commitment there, which somewhat allayed concerns. Still, Musk’s persona – outspoken, sometimes erratic on social media – remains a double-edged sword for Tesla: it provides immense free marketing and a loyal fan base, but also courts controversy (e.g., his tweets about politics, or an ongoing SEC consent decree from past tweeting about Tesla’s stock).

**Autopilot and Self-Driving Developments:** Tesla’s Autopilot and Full Self-Driving (FSD) software continue to be in the news. On one hand, Tesla has expanded its FSD Beta program to more customers (it allows Tesla cars to attempt to drive autonomously on city streets under driver supervision). On the other hand, regulators are scrutinizing the technology:
  - In February 2023, Tesla had to **recall** over 360,000 vehicles equipped with FSD Beta after NHTSA found that the software could allow the cars to behave unsafely (rolling past stop signs, speeding through intersections) in certain scenarios. Tesla issued an over-the-air update to address these issues ([en.wikipedia.org](https://en.wikipedia.org/wiki/Tesla%2C_Inc.#:~:text=Tesla%20has%20been%20the%20subject,in%20the%20second%20Trump%20presidency)). This was a high-profile recall because it directly tackled Tesla’s touted FSD capability.
  - NHTSA (the U.S. traffic safety regulator) and the Department of Justice are investigating Tesla’s Autopilot system and the company’s claims. Specifically, they are probing whether Tesla’s marketing of “Full Self-Driving” is misleading (since the system is not fully autonomous and requires driver attention, contrary to what some consumers might infer from the name). There have been several accidents, some fatal, involving Tesla vehicles on Autopilot, which have drawn regulatory scrutiny. As of mid-2023, NHTSA’s investigation into Tesla’s Autopilot (opened in 2021 regarding crashes into emergency vehicles) remained open, and the DOJ reportedly requested documents on FSD descriptions. These investigations pose a risk of potential forced changes to the technology or even legal penalties.
  - Despite these issues, Musk has continued to say Tesla is making progress on autonomy and even suggested that by mid-decade Tesla might achieve true robotaxis. Many observers remain skeptical given prior over-optimistic timelines. Autonomy is a significant potential value driver (or risk factor if it underdelivers) for Tesla, and it remains an area of intense focus.

**Legal and Regulatory Issues:** Tesla faces various legal and regulatory challenges:
  - **Product-related lawsuits:** e.g., families of victims in Autopilot-related crashes have filed suits. One notable case concluded in April 2023 where a jury found Tesla’s Autopilot was *not* at fault in a 2019 crash, a win for Tesla. However, another case in 2024 about a Model 3 that veered and killed passengers resulted in a jury finding Tesla partially liable due to a defective Autopilot system. Outcomes are mixed, and such cases will continue to test Tesla’s liability exposure.
  - **Workplace and HR issues:** Tesla has faced suits alleging racial discrimination at its Fremont factory (a major case by California’s civil rights agency is pending, and Tesla has faced previous verdicts against it in such matters). Claims of sexual harassment and poor working conditions have also been raised by some employees. Tesla, notably, has disbanded its PR department and often does not comment to media, which some say exacerbates reputation issues because the company doesn’t formally address criticisms.
  - **Union and labor relations:** Tesla’s workforce is not unionized (unlike Ford/GM). This has been a friction point – the U.S. President in 2021 pointedly invited GM and Ford to an EV summit and snubbed Tesla, largely due to union considerations. The risk of unionization at Tesla or labor disputes is something to watch; any organizing could increase labor costs or lead to conflicts (Tesla was found by a labor judge to have violated some labor laws in the past over an employee firing and Musk’s tweets discouraging unions).

**Competitive Moves and Industry Partnerships:** There were some **notable partnerships** and moves by other companies involving Tesla:
  - In 2023, as mentioned, **Ford, General Motors, and other automakers (Rivian, Volvo, etc.) struck agreements with Tesla to adopt Tesla’s charging plug standard (NACS) and gain access to Tesla’s Supercharger network for their EV owners starting 2024**. This was a major industry development. It positioned Tesla’s charging design as likely the North American standard, and by opening its network, Tesla stands to earn additional revenue (through charging fees) and government subsidies. It also alleviates one concern for non-Tesla EV buyers (charging availability), indirectly helping EV adoption. For Tesla, while it means sharing a prior exclusive asset, the upside is revenue and perhaps even converting some non-Tesla EV owners into Tesla customers in the future when they see the Tesla ecosystem benefits.
  - In the battery space, Tesla continues to secure deals. It signed contracts for U.S. domestic lithium supply (e.g., agreements with Piedmont Lithium and others) to support its supply chain and align with the Inflation Reduction Act (IRA) requirements for battery sourcing to qualify for consumer tax credits.
  - **Competitive launches:** In the last year, many competitors launched notable EVs: GM’s Chevrolet Blazer and Equinox EV (aiming below Tesla’s price point), Ford’s electric pickup, new Chinese models with advanced features (BYD’s luxury Seal sedan, Xpeng’s G9 SUV with fast charging, etc.). Tesla hasn’t launched a brand-new mass-market model since the Model Y in 2020, so these new entrants started nibbling at various market segments. For example, in China, Tesla had to contend with BYD’s Song Plus and Seal which undercut Tesla’s prices. Tesla responded with localized measures like offering insurance subsidies and other promotions in China to spur sales in late 2022. As a result of all this, Tesla’s market share in China and Europe has seen slight declines ([insideevs.com](https://insideevs.com/news/727744/tesla-market-share-2024-q2/#:~:text=During%20the%20first%20half%20of,the%20EV%20segment%20is%20expanding)). This underscores that competition is a real force: Tesla no longer operates in a near-vacuum as it did 5 years ago.
  
**Mergers & Acquisitions:** Tesla itself has not made any large acquisitions in the past year. Its growth has been organic. However, industry M&A is an area to watch. Some struggling EV startups (like Lordstown Motors or Electric Last Mile Solutions) went bankrupt in 2022-2023, though Tesla didn’t partake in any acquisitions there. Potential **acquisition targets for Tesla** could conceptually be in areas of battery materials (to secure supply) or autonomous software (to accelerate FSD development), but Tesla tends to prefer building in-house. On the flip side, **Tesla as an acquisition target** is highly unlikely given its size (>\$700B market cap) and Musk’s controlling stake – no acquirer could realistically buy Tesla outright. Instead, partnerships are more likely than acquisitions in Tesla’s strategy (like the charging network partnerships). 

**Short Sellers and Market Sentiment:** Tesla has historically been one of the most shorted stocks – critics on Wall Street bet against it, citing high valuation and various controversies. At times, over 20% of Tesla’s float was sold short (several years ago). Short-sellers have published reports and incessant social media commentary attacking Tesla’s demand, accounting (some claimed Tesla was not as profitable without credits, etc.), and leadership issues. In late 2022, as Tesla’s stock fell, short sellers made some profits, but in the first half of 2023 Tesla’s 100%+ rally inflicted heavy losses on those betting against it. There hasn’t been a single defining “short-seller report” in the mold of Hindenburg (which targets companies like Nikola or Adani in other cases) specifically in the last 12 months, but Tesla remains a battleground between bulls and bears. Notably, one short-seller theme that emerged is *competition from China* – some bearish analysts point out that Tesla’s growth could be eaten by Chinese automakers expanding globally (perhaps akin to how Japanese and Korean automakers took share in the US historically). Another theme bears highlight is Tesla’s reliance on Musk – any negative event around him could impact Tesla strongly, as seen when he sold stock or when there was a jury trial over his 2018 tweet (“funding secured” case, which Musk won in early 2023 avoiding investor damages).

**Institutional Ownership Changes:** Given Tesla’s inclusion in the S&P 500 (since Dec 2020), index funds are major holders. Over the past year, some active funds adjusted positions: for instance, there were reports in early 2023 regulatory filings that some large investors like **George Soros’s fund reduced or exited Tesla**, while others like **ARK Invest (Cathie Wood)** used dips to buy more. Institutional ownership is broad, with no single institution (besides Musk himself ~13%) owning an overwhelming share. Tesla did see some unique investor moves: the Saudi Arabia PIF (sovereign fund) once a big backer (they had been involved around the time of the “going private” attempt) exited around 2020; their stake has not been a factor since. In 2023, Norways’s sovereign wealth fund (a major global investor) publicly called for better Tesla corporate governance relating to handling of racism claims and such, reflecting that ESG-minded investors are scrutinizing Tesla’s behavior beyond just its clean tech credentials.

**Controversies and Reputational Factors:** Tesla’s overall brand among customers remains strong (surveys show high satisfaction with the cars themselves), but the company and Musk attract controversy:
  - **Quality and Service:** Tesla has had well-publicized quality issues (panel gaps, paint issues, etc., especially in early production of new models). While these have improved, some surveys (like J.D. Power initial quality) still rank Tesla lower than average on defects. Tesla also had to recall a significant number of vehicles for issues ranging from power steering bolt failures to window automatic reversal problems; most were minor and fixed via software, but it feeds a narrative by critics that Tesla rushes products.
  - **No PR Policy:** Tesla’s approach of not engaging with media often leaves negative press unanswered formally. This is unconventional for a large company. It relies on Musk’s own communications (Twitter posts, etc.) and fans to counter what Musk often calls “FUD” (fear, uncertainty, doubt) spread by detractors.
  - **Environmental and Supply Chain:** As a clean energy company, Tesla’s core mission is environmentally positive. However, the sourcing of materials like lithium and cobalt can raise concerns about mining practices (child labor in cobalt mining, for example). Tesla has moved to cobalt-free batteries in many models and publishes impact reports on responsible sourcing to address this.
  - **Tesla’s Bitcoin experiment:** In 2021 Tesla bought \$1.5B in Bitcoin and started accepting it briefly, then stopped due to environmental concerns of mining, and later sold most of its Bitcoin holdings in 2022. This was a non-core saga but brought some reputational questions regarding corporate focus and treasury management. It’s largely in the rearview now, with minimal remaining crypto on Tesla’s balance sheet.
  
**Potential M&A in Industry (Other Players):** While Tesla itself likely won’t be acquired, the broader industry might see tie-ups in reaction to Tesla’s dominance. For example, some speculate legacy automakers might acquire EV startups to catch up (like GM buying a stake in Nikola earlier, or more plausibly, a major OEM buying a battery supplier). Another angle is tech companies partnering or merging with car companies (e.g., Honda’s partnership with Sony on EVs, or rumors of Apple partnering with a manufacturer for its car). Tesla’s position could also drive **consolidation among suppliers** (like battery makers merging to compete with Tesla’s scale). These industry moves can indirectly affect Tesla by altering the competitive landscape or supply chain costs.

**Risk Factors:** Summarizing the key risks highlighted by recent developments:
  - **Competition and Market Share Erosion:** As noted, Tesla’s share is slipping slightly in key markets ([insideevs.com](https://insideevs.com/news/727744/tesla-market-share-2024-q2/#:~:text=During%20the%20first%20half%20of,the%20EV%20segment%20is%20expanding)). If competitors like BYD continue to undercut on price or flood markets with new models, Tesla may face an increasingly crowded field requiring more spending or further price cuts.
  - **Macro-Economic Risks:** High inflation and interest rates increase the cost of auto loans, potentially dampening consumer demand for big-ticket items like cars. Tesla’s own financing rates became less attractive as rates rose, which can slow orders. A global recession could particularly impact the luxury and premium segments where Tesla still plays heavily.
  - **Commodity and Supply Risks:** While Tesla navigated the chip shortage relatively well, any renewed supply chain disruption (e.g., a critical chip supplier issue or material shortage) could constrain production or increase costs. Battery materials remain a risk; although lithium prices have fluctuated, any sustained spike could hurt Tesla or force vehicle price increases.
  - **Regulatory/Policy Risks:** Changes in EV incentives (for example, if a new government reduced or eliminated credits) could affect Tesla’s pricing advantage. In the U.S., the 2023 IRA grants tax credits but with conditions – Tesla mostly qualifies due to local assembly, but if those rules tighten or if other countries impose tariffs or rules (like the EU considering tariffs on Chinese EVs which could affect Tesla’s China-made exports), Tesla would have to adapt. There’s also the risk of safety regulations – if, say, regulators mandated that automakers include lidar or other sensors on autonomous cars, Tesla would need to re-engineer vehicles (Tesla currently avoids expensive lidar).
  - **Elon Musk / Key Person Risk:** Musk’s importance to Tesla cannot be overstated – he is deeply involved in product decision-making and is the public face of the company. If for any reason he were no longer able to serve (health, controversy, etc.), it could shock investor confidence. Additionally, if Musk continues to sell shares for external ventures or becomes embroiled in political controversies (as he’s taken to sharing polarizing opinions on X), Tesla could face consumer backlash from some demographics.
  - **Execution of New Ventures:** Tesla’s future growth partially rests on projects that are not yet proven commercial successes: the Cybertruck’s acceptance and profitability (its manufacturing is very different), scaling the Semi for meaningful revenue, the success of 4680 in-house cell production (which has been slower than hoped), and eventually a robotaxi network. Each of these carries execution risk. For example, if the Cybertruck has delays or quality issues, it could cede the electric pickup space to Rivian or Ford longer. Or if FSD continues to be unfinished, it could mean Tesla doesn’t realize the high-margin software revenue envisioned.
  - **Valuation & Market Risk:** Tesla’s stock, being relatively richly valued, could be volatile or decline if the company fails to meet growth expectations. This could also impact employee morale and retention, since Tesla uses stock compensation heavily – a falling stock price makes it harder to attract/retain talent in a competitive tech labor market.

**Potential Acquisition Targets or Acquirers:** While Tesla is unlikely to be acquired, it might look at **strategic acquisitions** to support its growth. Based on overlapping or complementary needs:
  - A battery materials company (lithium mining or processing firm) could make sense to secure raw materials. Tesla already has supply deals; if they wanted more control, they might acquire a lithium miner or recycler.
  - Suppliers of autonomous driving tech or AI that could enhance Tesla’s FSD. Tesla thus far has developed its own tech (even building its own chips and Dojo supercomputer), but it might acquire a small AI startup if it had talent or tech Tesla wants.
  - A carmaker acquisition is unlikely (Tesla doesn’t need the legacy plants or brands with their baggage), but sometimes speculated – for instance, if a legacy OEM were to spin off its EV division, would Tesla merge? Probably not necessary for Tesla.
  - In the energy space, Tesla might acquire companies specializing in grid services or solar installation networks to boost Tesla Energy’s reach.
  
**Media Coverage Themes:** The media narrative around Tesla in the last year has centered on a few themes:
  - **“Price War in EVs”** – Many articles emphasize how Tesla’s price cuts forced a response from competitors and are reshaping industry pricing (squeezing other automakers’ EV profit ambitions) ([www.reuters.com](https://www.reuters.com/business/autos-transportation/byd-may-hand-back-top-ev-seller-title-tesla-after-q1-sales-decline-2024-04-02/#:~:text=33.7,year)). This is painted as Tesla using its strength to stave off competition, but at the cost of its own margins.
  - **“Tesla vs BYD / China vs U.S. EVs”** – With Tesla’s growth, media often compare it with BYD, noting how China’s EV market is both an opportunity and threat for Tesla. Elon Musk’s trip to China in mid-2023 (meeting government officials) got coverage as Tesla navigates U.S.-China tensions.
  - **“Technology Leadership and AI”** – Tesla’s AI Day presentations and bold claims (like building a humanoid **Optimus robot** prototype and its Dojo supercomputer for vision training) have media and analysts wondering if Tesla is more than an auto company, perhaps an AI company in disguise. There’s speculation if Tesla might license its self-driving tech to others or if it becomes a data/AI powerhouse thanks to its fleet’s driving data.
  - **“Environmental and ESG contradictions”** – Some press has been mixed: praising Tesla for its role in reducing emissions globally but criticizing it for labor issues or Musk’s statements that conflict with certain ESG values. Tesla was even removed from an S&P ESG index in 2022 over such concerns, which Musk loudly criticized.
  - **“Stock and Market dynamics”** – As Tesla’s stock swung, financial press covered how it could be considered overvalued or a bubble during peaks, and conversely whether sell-offs were buying opportunities. Tesla’s enormous market cap makes it influential on indices; for example, by mid-2023 Tesla comprised a sizable weight in the Nasdaq 100 and S&P 500, so its movements affect broader market indices.

**Summary of Recent News:** Tesla’s recent year has been marked by record production, bold pricing strategy, and strategic wins in charging infrastructure, counterbalanced by margin pressure, leadership transitions, and ever-growing competition. The company remains at the forefront of the EV revolution, but the conversation has shifted from “can EVs be profitable?” (Tesla answered yes) to “how will Tesla defend its lead?” in the face of oncoming rivals and an evolving market. Key watchpoints emerging from recent developments include Tesla’s pace of innovation (new models, FSD progress), its ability to stimulate demand without eroding margins, and external factors like regulations or Musk’s external ventures which can quickly change the risk outlook.

## 8. Overall Assessment  
**Strategic Position:** Tesla’s strategic position in the industry is strong – it is the pioneer and market leader of the EV transition, with a significant lead in production scale, technology integration, and brand recognition. The company’s transformation from a niche startup to the most valuable automaker in the world is a testament to its execution and first-mover advantages. Tesla’s current strategy focuses on leveraging its scale and cost advantage to **capture volume** (even at the expense of near-term margins) in order to solidify its market share and prepare for an autonomous future. The company is vertically integrated and sits at the crossroads of several industries: automotive, energy, and technology. This unique positioning gives it multiple avenues for growth (vehicles, energy storage, software services) but also means it faces diverse sets of competitors in each arena.

**Strengths:** Tesla’s key strengths include:
- **Technology and Innovation:** It has a technological edge in EV drive systems, battery management, and vehicle software. Tesla vehicles typically have class-leading ranges and a software user experience that is a benchmark for others. The continuous improvement model (via OTA updates) keeps Tesla ahead in feature offerings.
- **Brand and Customer Loyalty:** Tesla’s brand is synonymous with electric cars; it enjoys an Apple-like cachet in autos. Owners often become repeat customers and brand evangelists. This strong brand has been achieved with minimal marketing spend, a significant intangible asset.
- **Scale and Manufacturing Efficiency:** Tesla’s production scale (1M+ vehicles/year) and vertically integrated approach have led to cost efficiencies and better ability to weather supply issues. It has constructed and ramped large factories relatively quickly (e.g., Shanghai in under a year), showcasing operational prowess.
- **Financial Health:** With high margins (until recently, industry-leading) and a robust balance sheet (net cash position, solid cash flows), Tesla has financial flexibility. It can fund its growth internally and has liquidity to handle downturns or invest opportunistically.
- **Ecosystem and Infrastructure:** The Supercharger network, Tesla’s proprietary NACS charging standard, and its integrated Energy products form an ecosystem moat. New Tesla buyers get not just a car but access to a charging network and energy solutions, enhancing the value proposition relative to competitors.
- **Leadership Vision:** Elon Musk’s visionary leadership, while controversial at times, has been a driving force behind Tesla’s innovation and risk-taking culture. The company has dared to do things differently (selling direct, open-sourcing patents early on, etc.), which often gave it a step up vs. cautious incumbents.

**Weaknesses:** Despite its strengths, Tesla has some notable weaknesses and challenges:
- **Limited Product Portfolio (so far):** The current vehicle lineup, though successful, leaves segments unaddressed (no compact \$25k car yet, no vans, etc.). This makes Tesla vulnerable in segments where competitors have EVs (for instance, Chevy Bolt in the affordable hatch segment or various European city cars). The delay in launching a cheaper model means Tesla isn’t competing in the fastest-growing lower-priced EV segment.
- **Premium Pricing & Margin Dependence:** Tesla’s profitability has been partly propped up by selling higher-end models at premium prices and by regulatory credits. As it cuts prices to chase volume, it relies on cost reductions to preserve margin. If cost reductions (through new tech like 4680 cells or manufacturing simplifications) don’t keep pace with price cuts, profitability could erode further.
- **Service & Quality:** Rapid growth sometimes came at the expense of quality control. While improving, issues with build quality and after-sales service capacity are still cited by customers. The service network expansion has lagged car sales in some regions, which could hurt customer satisfaction in the long run.
- **Overreliance on Musk:** Tesla’s identity and success are highly tied to Elon Musk. This is a double-edged sword; his vision drives the company, but it also means Tesla’s perception is vulnerable to Musk’s personal actions or missteps. There’s also a key-man risk – no other executive at Tesla has the same clout or public profile, which is a governance concern.
- **Communication and PR**: Tesla’s unconventional communication strategy (minimal PR, often reactive tweets by Musk) can lead to confusion or negative narratives taking hold. For a large cap company, the lack of structured communication could be seen as a weakness in managing investor and public relations, especially during crises or recalls.
- **Geographic Risk Concentration:** Tesla’s heavy reliance on the Chinese market (both for production and sales) is a vulnerability. Geopolitical tensions or nationalist consumer shifts in China could impact Tesla more than some competitors who are more diversified or domestic in China.

**Opportunities:** Tesla has significant growth opportunities ahead:
- **Expansion of EV Market:** As global EV adoption accelerates (with many countries targeting 50% or more of new car sales to be electric by 2030), Tesla can tap huge new customer bases, especially in emerging markets. Tesla’s recent interest in markets like India or Southeast Asia demonstrate the potential to expand geographically.
- **New Models & Segments:** The upcoming Cybertruck opens the pickup segment; a future compact car could open the high-volume budget segment. Also, Tesla has hinted at a robo-van or people mover which could target transit and commercial markets. Each new segment Tesla enters is an opportunity to replicate its success.
- **Autonomous Driving & Robotaxi:** If Tesla can achieve full self-driving capability, it could launch a **robotaxi service** where Tesla owners or Tesla itself operate fleets of autonomous taxis. This could transform the business model from one-off car sales to continuous revenue from transportation services, akin to a rideshare company but without drivers. Even partial autonomy could be offered as subscriptions to generate recurring revenue.
- **Tesla Energy Growth:** The push for renewable energy and grid stabilization is a tailwind for the storage business. Megapacks (large battery installations) are selling rapidly to utilities and corporations as they need to buffer solar/wind energy and for grid services. This market is growing and Tesla has a competitive product. Similarly, solar roofs and Powerwalls could see broader adoption as home energy independence becomes attractive (especially with incentives for home batteries in places like California). Tesla’s energy division could become a much larger portion of revenue, diversifying the company.
- **Technology Monetization:** Tesla’s advancements in artificial intelligence (for driving) and possibly its Dojo supercomputer could be offered as services beyond its own cars. Musk has hinted at possibly licensing FSD software to other OEMs once it’s proven. If Tesla’s AI can be leveraged in other domains (robotics, etc.), that’s another revenue stream. The **Optimus humanoid robot**, while in its early stages, represents another futuristic opportunity – a successful multi-purpose robot for labor could open an entirely new market (though this remains speculative and longer-term).
- **Cost Leadership and Industry Partnerships:** Tesla’s continuous cost improvements (e.g., using more LFP batteries for cost savings, innovating in manufacturing) can allow it to undercut competitors and still expand margin in the long run. Also, by partnering (such as the Supercharger agreements), Tesla can generate revenue from competitors and possibly integrate itself as a backbone of EV infrastructure, which could come with government support and favorable positioning in infrastructure plans.

**Threats:** Tesla also faces serious threats:
- **Intense Competition:** This is arguably the top threat – nearly every automaker is launching EVs, often inspired by Tesla’s success. Competitors with deep pockets can afford to sell EVs at a loss for a while to gain share (e.g., some Chinese EVs are very price-competitive, and legacy automakers use profits from gas vehicles to subsidize EV development). There’s a risk that Tesla’s market share leadership erodes, especially in places like Europe and China where local brands might appeal more to local consumers or regulators.
- **Margin Pressure & Price War:** The EV price war that Tesla started could continue, forcing further price cuts. If a competitor like BYD can leverage even lower costs (given lower labor costs in China, etc.) to keep undercutting Tesla’s prices in key markets, Tesla might face sustained margin pressure. Already Tesla’s vehicle gross margin is off its peak ([www.reuters.com](https://www.reuters.com/business/autos-transportation/tesla-revenue-sees-surprise-rise-second-quarter-revenue-2024-07-23/#:~:text=Automotive%20gross%20margin%20excluding%20regulatory,unveiling%20delayed%20to%20October%2010)); if it were to converge to typical auto industry single-digit margins, the valuation narrative would be threatened.
- **Regulatory and Legal:** A major adverse regulatory action (e.g., a government banning use of Autopilot because of safety concerns, or a large fine/penalty for an issue like misleading FSD marketing) could hurt financially and reputationally. Additionally, if governments change EV incentive structures unpredictably, it could whiplash demand – e.g., if in a key market subsidies expire and Tesla’s cars become relatively more expensive again vs competitors who localize manufacturing to get incentives.
- **Supply Chain Disruptions:** The reliance on a few key suppliers (like battery producers) means if any one has an issue (technical or even political, like export controls), Tesla’s production could be hampered. Also, rising raw material costs could force vehicle price increases which might dampen demand or squeeze margins if Tesla holds prices.
- **Macroeconomic Slowdown:** A global recession or even regional recessions (esp. U.S. or China) could slow auto sales significantly. As a premium brand, Tesla might be more exposed to consumers pulling back on big purchases (though it could also benefit from consumers looking for lower operating cost vehicles).
- **Currency and Trade Risks:** Tesla earns significant revenue in China and Europe; currency fluctuations (a strong dollar in 2022 hurt Tesla’s reported earnings from overseas). Trade policies (like tariffs between U.S. and China) could increase costs of importing/exporting vehicles or components. Tesla’s plan to export cars (e.g., from China to other Asian or European markets) can be disrupted by trade tensions.

**Bull Case vs Bear Case:**  
- **Bull Case:** In a bullish scenario, Tesla’s execution remains stellar: it continues growing ~40% annually for the next few years, driven by Model Y and new models like Cybertruck and a future affordable car. Profit margins stabilize or even improve as cost reductions offset lower pricing. Tesla successfully launches full self-driving as a commercial service by 2025, creating a high-margin robotaxi revenue stream and justifying software-like valuations. Its energy business scales up, doubling or tripling revenue and contributing steady profits as battery costs drop. Under this scenario, Tesla solidifies its position not just as a car company but as a leader in clean energy and autonomy – a combination that could justify a trillion-plus market cap (some bulls even talk about Tesla being valued like Apple or Saudi Aramco on an energy-equivalent basis). Essentially, the bull case sees Tesla becoming one of the predominant companies of our time across multiple sectors, with long-term dominance akin to Apple in consumer tech – meaning years of growth ahead, high customer lock-in, and expanding competitive moats (like the charging network being a national standard, FSD being industry-leading, etc.). In this case, investors would be rewarded by both earnings growth and potential multiple expansion as confidence in Tesla’s future increases.

- **Bear Case:** In a bearish scenario, Tesla’s shine fades as it becomes “one of many” automakers in a crowded EV market. Growth would slow significantly – perhaps due to competition or market saturation – with annual growth dropping into the teens or single digits by late 2020s. Price wars could further cut margins, making Tesla’s profitability approach that of traditional auto companies. At the same time, hoped-for ventures might disappoint: FSD could continue to be “2 years away” indefinitely, never achieving true autonomy, meaning Tesla doesn’t unlock new revenue streams and even faces potential liability or stricter regulation on its driver assist systems. Economic factors (like higher interest rates or commodity inflation) could persist, making EVs more expensive and dampening demand. In this bear case, Tesla essentially transitions from a hyper-growth disruptor to a more mature manufacturing firm with earnings that, when valued at auto-industry multiples (e.g., 10x earnings), would imply a stock price well below current levels. Additionally, the bear case could involve some external shock – for example, an economic crisis in China cuts demand, or a major recall or scandal (safety issue, etc.) damages Tesla’s brand. If Elon Musk were to step away or sell a lot more stock, bears worry leadership vacuum or shareholder dilution could also weigh on the stock. In summary, the bear case envisions Tesla’s competitive advantages eroding and its valuation dramatically compressing to be more in line with industrial peers, which would be painful for a stock priced for growth.

**Risk/Reward and Investment Profile:** Tesla’s stock offers potentially high reward but with high risk. It is considered a high-beta, high-volatility stock – suitable for investors with higher risk tolerance who believe in the long-term EV/autonomy revolution and Tesla’s central role in it. The risk comes from the numerous factors discussed: execution risk, competitive risk, and valuation risk. In terms of **risk level**, it’s above average for a large-cap: unlike a stable dividend-paying blue-chip, Tesla can swing greatly based on future expectations. However, its inclusion in indices and its large market cap mean it’s somewhat less risky than in the past from a financial viability standpoint (bankruptcy risk now is low given cash and cash flow). It’s more about valuation risk now – paying a premium for growth that might not materialize as projected.

**Key Watch Points:** Investors and analysts should monitor several “critical watch points” for Tesla going forward:
  - **Demand vs. Pricing:** Are Tesla’s price cuts stimulating enough demand? Watch quarterly delivery numbers and order backlogs. If deliveries falter even with price cuts, that’s a red flag. Conversely, if Tesla can raise prices or hold them steady while maintaining volume, that’s positive.
  - **Margin Trajectory:** Keep an eye on automotive gross margin (particularly excluding credits) and operating margin. This will indicate if cost efficiencies (factory ramp-ups, cheaper batteries) are offsetting lost pricing. Also watch the role of regulatory credit revenue – reliance on it decreasing would mean Tesla is standing on its own profitability.
  - **Product Launches & Execution:** The success of the Cybertruck launch is one near-term indicator – can Tesla ramp it smoothly and with strong demand beyond the initial enthusiasts? Similarly, progress on the low-cost “next-gen” model will be crucial for long-term growth; any official announcement or timeline on that will be a major catalyst.
  - **FSD and Technology milestones:** Is Tesla truly getting closer to full autonomy? Updates on the capability of FSD Beta, and any metrics (like millions of miles per intervention) would show if Tesla’s approach is leading or lagging. Also, any hints at licensing FSD or opening it to other car brands could imply confidence in its tech. On the energy side, watch for Tesla’s battery production progress (4680 ramp) and Megapack deployments and margins for Tesla Energy – this could reveal if Tesla Energy is becoming a significant profit center.
  - **Competition and EV market developments:** Monitor competitor launches (e.g., how does GM’s upcoming EV lineup fare? Does Ford continue aggressive EV expansion or pull back? Are Chinese EV brands making inroads in Europe/Americas?). Also policy developments like EU emissions rules or US fuel economy standards can indirectly help or hurt Tesla (usually help EV adoption, but competition affects who benefits).
  - **China and Other Geographies:** Since China is vital, keep track of Tesla’s market share and pricing moves there. Any geopolitical news – e.g., U.S.-China trade friction that could hamper Tesla, or domestic sentiment in China – can impact Tesla’s fortunes. Additionally, new factory announcements (there have been talks of a potential Gigafactory in another country, maybe Canada or elsewhere) would signal Tesla’s growth plans.
  - **Leadership and Governance:** Elon Musk’s actions (share sales, focus between his companies) should be watched. Also, whether Tesla bolsters its management team (for instance, bringing in a COO or similar) could affect execution. Any insider selling beyond Musk, or large movements by key shareholders, could also signal sentiment shifts.
  - **Macro indicators:** Since Tesla is sensitive to economic conditions, things such as interest rate trends (affecting auto loan rates), oil prices (affecting EV attractiveness), and overall consumer confidence are important to gauge the environment Tesla is operating in.

**Investment Risk/Reward Profile:** Summing up, Tesla presents a compelling growth story with significant long-term upside if it executes well on its ambitious goals (EV dominance, full autonomy, energy revolution). The **bull case** envisions Tesla as a winner-take-most in a sustainable transport and energy future, which would make the current valuation look justified or even cheap. The **bear case** sees Tesla facing growing pains and heavy competition that normalize its growth and margins, implying the stock is overvalued. In the middle, Tesla is likely to continue growing, but perhaps at a moderate pace with competitive pressure – the stock could then trade more on fundamental earnings multiples which are still evolving. 

At this juncture, an equity research analyst would likely conclude that Tesla offers high growth potential but with elevated volatility; thus, position sizing and risk management are key for investors. **Ongoing due diligence** is warranted on all the watch points mentioned, as Tesla’s story is dynamic. The next few years will be critical to see whether Tesla can maintain its leadership and justify the sky-high expectations baked into its market cap, or whether the convergence of competition and execution challenges will bring it back to earth. For now, Tesla remains a unique entity – part automaker, part Silicon Valley disruptor – with a risk/reward profile that is as extraordinary as its journey so far. 

